In [1]:
# ==================================================================================================
# PROJECT 16 — CELL 1 / STEP 0
# FRESH-RUNTIME BOOTSTRAP AND RUNTIME-PRIORITIZED CANDIDATE DISCOVERY
#
# RUN THIS IN A NEW NOTEBOOK:
#   Thesis_project_16.ipynb
#
# SAFETY:
# - reads but never modifies the completion registry;
# - writes only Project 16 bootstrap/selection files;
# - never reads or modifies any prior-project condition-output files;
# - does not inject noise, reconstruct REC features, fit models, or start an experiment;
# - prepares the remaining projects for runtime-prioritized selection in Step 1A.
# ==================================================================================================

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil
import tarfile

import pandas as pd


print("=" * 136)
print("=== PROJECT 16 CELL 1 / STEP 0: FRESH-RUNTIME BOOTSTRAP AND CANDIDATE DISCOVERY ===")
print("=" * 136)


PROJECT_NUMBER = 16

STEP0_STATUS = (
    "PASS_PROJECT_16_FRESH_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_REGISTERED_PROJECTS = 15
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_UNREGISTERED_CANDIDATES = 10

REQUIRED_PROJECT_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
}

RUNTIME_PRIORITY_POLICY = {
    "purpose":
        "processing order only; protocol eligibility and final project set are unchanged",
    "primary":
        "ModelTrainingRows ascending",
    "secondary":
        "ModelEvaluationRows ascending",
    "tertiary":
        "RawExecutionRows ascending",
    "final_tie_break":
        "Project ascending",
    "scientific_effect":
        "none when all protocol-eligible projects are completed",
}


drive.mount(
    "/content/drive",
    force_remount=False,
)

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_EXTRACTION_ROOT = Path(
    "/content/datasets"
)

LOCAL_DATASET_ROOT = (
    LOCAL_EXTRACTION_ROOT
    / "datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_16_selection"
)

BOOTSTRAP_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_16_bootstrap_candidate_inventory.csv"
)

BOOTSTRAP_REPORT_PATH = (
    SELECTION_ROOT
    / "project_16_step0_report.json"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step0_status.json"
)


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def resolve_column(
    columns,
    *candidates,
):
    normalized = {
        str(column).strip().lower():
            column
        for column in columns
    }

    for candidate in candidates:
        key = str(
            candidate
        ).strip().lower()

        if key in normalized:
            return normalized[
                key
            ]

    raise RuntimeError(
        "Could not resolve any of these columns: "
        + ", ".join(
            candidates
        )
    )


def atomic_write_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_json(
    path,
    payload,
):
    atomic_write_text(
        path,
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
    )


def atomic_write_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def extract_archive_safely(
    archive_path,
    extraction_root,
):
    extraction_root = Path(
        extraction_root
    )

    extraction_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolved_root = extraction_root.resolve()
    extracted_files = 0

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace(
                    "\\",
                    "/",
                )
                .lstrip(
                    "/"
                )
            )

            target_path = (
                extraction_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target
                != resolved_root
                and resolved_root
                not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open(
                        "wb"
                    ) as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    return extracted_files


required_drive_paths = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
]

missing_drive_paths = [
    str(
        path
    )
    for path in required_drive_paths
    if not path.is_file()
]

if missing_drive_paths:
    raise FileNotFoundError(
        "Required Project 16 bootstrap inputs are missing:\n"
        + "\n".join(
            missing_drive_paths
        )
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Frozen TCP-CI archive SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs from the frozen Projects 1–15 state.\n"
        "Do not continue Project 16 until the unexpected registry change is investigated.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "Project Number",
    "Project_Number",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly frozen Projects 1–15."
    )


if not registry[
    registry_status_column
].astype(
    str
).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Not every registered predecessor is COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(
        str
    )
)


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                registry_project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


def local_dataset_looks_complete():
    if not LOCAL_DATASET_ROOT.is_dir():
        return False

    project_directories = [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ]

    return bool(
        len(
            project_directories
        )
        == 25
    )


if local_dataset_looks_complete():
    extraction_performed = False
    extracted_files = 0

    print(
        "\nA complete-looking local TCP-CI dataset is already present."
    )

else:
    extraction_performed = True

    print(
        "\nRestoring the frozen TCP-CI archive into the Project 16 runtime."
    )

    if LOCAL_EXTRACTION_ROOT.exists():
        shutil.rmtree(
            LOCAL_EXTRACTION_ROOT
        )

    extracted_files = extract_archive_safely(
        ARCHIVE_PATH,
        LOCAL_EXTRACTION_ROOT,
    )


if not LOCAL_DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Archive extraction did not create the expected dataset root:\n"
        f"{LOCAL_DATASET_ROOT}"
    )


all_project_directories = sorted(
    [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path:
        path.name,
)


if len(all_project_directories) != 25:
    raise RuntimeError(
        "Unexpected number of TCP-CI project directories.\n"
        f"Expected: 25\n"
        f"Actual:   {len(all_project_directories)}"
    )


reserved_projects = set()

candidate_rows = []

for source_directory in all_project_directories:
    project = source_directory.name

    source_files = {
        path.name
        for path in source_directory.iterdir()
        if path.is_file()
    }

    missing_required_files = sorted(
        REQUIRED_PROJECT_FILES
        - source_files
    )

    excluded_registered = (
        project in registered_projects
    )

    excluded_reserved = (
        project in reserved_projects
    )

    candidate_eligible_for_scan = (
        not excluded_registered
        and not excluded_reserved
        and not missing_required_files
    )

    candidate_rows.append({
        "Project":
            project,
        "ProjectSlug":
            project.replace(
                "@",
                "__",
            ),
        "SourceDirectory":
            str(
                source_directory
            ),
        "ExcludedRegistered":
            bool(
                excluded_registered
            ),
        "ExcludedReserved":
            bool(
                excluded_reserved
            ),
        "MissingRequiredFiles":
            "; ".join(
                missing_required_files
            ),
        "CandidateForProject16Scan":
            bool(
                candidate_eligible_for_scan
            ),
    })


inventory = pd.DataFrame(
    candidate_rows
)


project_16_candidates = (
    inventory.loc[
        inventory[
            "CandidateForProject16Scan"
        ]
    ]
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    project_16_candidates
) != EXPECTED_UNREGISTERED_CANDIDATES:
    raise RuntimeError(
        "Unexpected number of Project 16 candidates after excluding registered Projects 1–15.\n"
        f"Expected: {EXPECTED_UNREGISTERED_CANDIDATES}\n"
        f"Actual:   {len(project_16_candidates)}"
    )


if (
    project_16_candidates[
        "Project"
    ].isin(
        registered_projects
        | reserved_projects
    ).any()
):
    raise RuntimeError(
        "Registered identities leaked into the Project 16 candidate set."
    )


SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    BOOTSTRAP_INVENTORY_PATH,
    project_16_candidates,
)


created_at_utc = datetime.now(
    timezone.utc
).isoformat()


report = {
    "ProjectNumber":
        PROJECT_NUMBER,
    "Status":
        STEP0_STATUS,
    "CreatedAtUTC":
        created_at_utc,
    "ArchivePath":
        str(
            ARCHIVE_PATH
        ),
    "ArchiveSHA256":
        archive_sha256,
    "RegistryPath":
        str(
            REGISTRY_PATH
        ),
    "RegistrySHA256":
        registry_sha256_before,
    "RegisteredProjects":
        EXPECTED_REGISTERED_PROJECTS,
    "RegisteredStatuses":
        sorted(
            registry[
                registry_status_column
            ].astype(
                str
            ).unique().tolist()
        ),
    "DatasetRoot":
        str(
            LOCAL_DATASET_ROOT
        ),
    "SourceProjectDirectories":
        len(
            all_project_directories
        ),
    "Project16CandidateCount":
        len(
            project_16_candidates
        ),
    "CandidateInventory":
        str(
            BOOTSTRAP_INVENTORY_PATH
        ),
    "RuntimePriorityPolicy":
        RUNTIME_PRIORITY_POLICY,
    "ExtractionPerformed":
        bool(
            extraction_performed
        ),
    "ArchiveFilesExtracted":
        int(
            extracted_files
        ),
    "RegistryModified":
        False,
    "PriorProjectConditionOutputsAccessed":
        False,
    "PriorProjectConditionOutputsModified":
        False,
    "NoiseInjected":
        False,
    "ModelsFitted":
        False,
}


atomic_write_json(
    BOOTSTRAP_REPORT_PATH,
    report,
)

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,
        "Status":
            STEP0_STATUS,
        "CreatedAtUTC":
            created_at_utc,
        "Report":
            str(
                BOOTSTRAP_REPORT_PATH
            ),
    },
)


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during the Project 16 bootstrap."
    )


print("\nProject 16 candidates after excluding registered identities:")
print(
    project_16_candidates[
        [
            "Project",
            "ProjectSlug",
            "SourceDirectory",
        ]
    ].to_string(
        index=False
    )
)


print("\n")
print("=" * 136)
print("=== PROJECT 16 CELL 1 / STEP 0 RESULT ===")
print("=" * 136)

print(
    "Registered and frozen projects:",
    EXPECTED_REGISTERED_PROJECTS,
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "TCP-CI source directories:",
    len(
        all_project_directories
    ),
)

print(
    "Project 16 candidates:",
    len(
        project_16_candidates
    ),
)

print(
    "Runtime-priority policy:",
    RUNTIME_PRIORITY_POLICY,
)

print(
    "Candidate inventory:",
    BOOTSTRAP_INVENTORY_PATH,
)

print(
    "Completion registry modified:",
    False,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "\nSTATUS:",
    STEP0_STATUS,
)

print("=" * 136)


=== PROJECT 16 CELL 1 / STEP 0: FRESH-RUNTIME BOOTSTRAP AND CANDIDATE DISCOVERY ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring the frozen TCP-CI archive into the Project 16 runtime.

Project 16 candidates after excluding registered identities:
                      Project                    ProjectSlug                                          SourceDirectory
         EMResearch@EvoMaster          EMResearch__EvoMaster          /content/datasets/datasets/EMResearch@EvoMaster
     Graylog2@graylog2-server      Graylog2__graylog2-server      /content/datasets/datasets/Graylog2@graylog2-server
        SonarSource@sonarqube         SonarSource__sonarqube         /content/datasets/datasets/SonarSource@sonarqube
               apache@curator                apache__curator                /content/datasets/datasets/apache@curator
        apache@logging-log4j2         apache__logging-log4j2         

In [2]:
# ==================================================================================================
# PROJECT 16 — CELL 2 / STEP 1A
# ROBUST CANDIDATE DISCOVERY, PROTOCOL ELIGIBILITY, RUNTIME-PRIORITIZED RANKING,
# AND PROVISIONAL PROJECT 16 SELECTION
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# THIS CELL:
# - inspects all 11 candidates frozen by Project 16 Step 0;
# - validates the chronological 75/25 split and raw/model cohort viability;
# - deterministically ranks eligible candidates by estimated experiment cost (smallest first);
# - changes processing order only, not protocol eligibility or the intended final project set;
# - freezes only a provisional Project 16 selection for Step 1B;
# - does not run experiment conditions or fit models;
# - does not modify the completion registry or prior-project outputs.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 16 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_16_FRESH_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_16_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 15
EXPECTED_CANDIDATES = 10

RESERVED_ACTIVE_PROJECTS = set()

RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

# These three files are sufficient for deterministic selection.
# id_map.csv and entity_change_history.csv are checked and frozen later in Step 1B/2A.
REQUIRED_SELECTION_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_SOURCE_ROOT = Path(
    "/content/datasets/datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_16_selection"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step0_status.json"
)

BOOTSTRAP_CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_16_bootstrap_candidate_inventory.csv"
)

SCAN_PROGRESS_PATH = (
    SELECTION_ROOT
    / "project_16_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_16_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_16_candidate_inventory.csv"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_16_eligible_candidates_ranked.csv"
)

INELIGIBLE_PATH = (
    SELECTION_ROOT
    / "project_16_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    SELECTION_ROOT
    / "project_16_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_16_provisional_selection.json"
)

STEP1A_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_16_step1a_validation.csv"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_16_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
        1,
    )


def count_partitioned_rows(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0
    verdict_values = set()

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk_build = parse_integer_series(
            chunk[build_column],
            f"{label}.{build_column}",
        )

        chunk_verdict = parse_integer_series(
            chunk[verdict_column],
            f"{label}.{verdict_column}",
        )

        training_mask = chunk_build.isin(
            training_build_ids
        )

        evaluation_mask = chunk_build.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = chunk_verdict.ne(0)

        total_rows += len(chunk)

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            chunk_build.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            chunk_build.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (~linked_mask).sum()
        )

        verdict_values.update(
            int(value)
            for value in chunk_verdict.unique().tolist()
        )

    return {
        "Rows":
            int(total_rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(len(failing_training_builds)),

        "FailingEvaluationBuilds":
            int(len(failing_evaluation_builds)),

        "UnlinkedRows":
            int(unlinked_rows),

        "VerdictValuesJSON":
            json.dumps(
                sorted(verdict_values)
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


def reusable_scan_row_is_valid(
    row,
):
    required_fields = [
        "Project",
        "ProjectSlug",
        "SourceDirectory",
        "InspectionStatus",
        "InspectionError",
        "BuildIDColumn",
        "StartedAtColumn",
        "ExecutionBuildColumn",
        "ExecutionVerdictColumn",
        "DatasetBuildColumn",
        "DatasetVerdictColumn",
        "Builds",
        "TrainingBuilds",
        "EvaluationBuilds",
        "RawExecutionRows",
        "RawTrainingRows",
        "RawEvaluationRows",
        "RawTrainFailures",
        "RawEvaluationFailures",
        "RawFailingTrainingBuilds",
        "RawFailingEvaluationBuilds",
        "RawUnlinkedRows",
        "ModelReadyRows",
        "ModelTrainingRows",
        "ModelEvaluationRows",
        "ModelTrainFailures",
        "ModelEvaluationFailures",
        "ModelFailingTrainingBuilds",
        "ModelFailingEvaluationBuilds",
        "ModelUnlinkedRows",
    ]

    if any(
        field not in row
        for field in required_fields
    ):
        return False

    status = str(
        row.get(
            "InspectionStatus",
            "",
        )
    ).strip()

    error = str(
        row.get(
            "InspectionError",
            "",
        )
    ).strip().lower()

    return (
        status in {
            "ELIGIBLE",
            "INELIGIBLE",
        }
        and error in {
            "",
            "nan",
            "none",
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 0, REGISTRY, ARCHIVE, AND LOCAL SOURCE
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 16 Step 1A V2 inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not LOCAL_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        "The local Project 16 dataset source is missing:\n"
        f"{LOCAL_SOURCE_ROOT}"
    )


bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

if bootstrap_status.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 16 Step 0 is not in the expected PASS state."
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 16))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–15."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(str).tolist()
)


if registered_projects & RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "An active reservation unexpectedly remains after Project 15 registration."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


bootstrap_candidates = pd.read_csv(
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)


candidate_project_column = resolve_column(
    bootstrap_candidates.columns,
    "Project",
    "bootstrap candidate Project",
)

candidate_source_column = resolve_column(
    bootstrap_candidates.columns,
    "SourceDirectory",
    "bootstrap candidate SourceDirectory",
)


candidate_records = (
    bootstrap_candidates[
        [
            candidate_project_column,
            candidate_source_column,
        ]
    ]
    .rename(
        columns={
            candidate_project_column:
                "Project",

            candidate_source_column:
                "SourceDirectory",
        }
    )
    .copy()
)


candidate_records[
    "Project"
] = candidate_records[
    "Project"
].astype(str)


candidate_records[
    "SourceDirectory"
] = candidate_records[
    "SourceDirectory"
].astype(str)


if len(candidate_records) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 16 candidate count.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(candidate_records)}"
    )


if candidate_records[
    "Project"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Project 16 bootstrap candidate inventory contains duplicates."
    )


forbidden_candidates = (
    set(
        candidate_records[
            "Project"
        ]
    )
    & (
        registered_projects
        | RESERVED_ACTIVE_PROJECTS
    )
)


if forbidden_candidates:
    raise RuntimeError(
        "Project 16 inventory contains registered projects:\n"
        + "\n".join(
            sorted(forbidden_candidates)
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REUSE ANY VALID COMPLETED PROJECT 16 SCANS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


reusable_rows = {}


if SCAN_PROGRESS_PATH.is_file():
    try:
        previous_progress = pd.read_csv(
            SCAN_PROGRESS_PATH,
            low_memory=False,
        )

        valid_candidate_names = set(
            candidate_records[
                "Project"
            ]
        )

        for row in previous_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            if (
                project in valid_candidate_names
                and reusable_scan_row_is_valid(
                    row
                )
            ):
                reusable_rows[
                    project
                ] = row

        print(
            "\nReusable completed candidate scans:",
            len(reusable_rows),
        )

    except Exception as error:
        print(
            "\nPrevious scan progress was ignored:",
            type(error).__name__,
            str(error),
        )


# --------------------------------------------------------------------------------------------------
# 6. INSPECT ALL 11 CANDIDATES
# --------------------------------------------------------------------------------------------------

scan_rows = []


for candidate_index, candidate in enumerate(
    candidate_records.itertuples(
        index=False
    ),
    start=1,
):
    project = str(
        candidate.Project
    )

    source_directory = Path(
        candidate.SourceDirectory
    )

    print("-" * 132)
    print(
        f"[{candidate_index:02d}/{EXPECTED_CANDIDATES:02d}] "
        f"Inspecting: {project}"
    )


    if project in reusable_rows:
        row = dict(
            reusable_rows[
                project
            ]
        )

        row[
            "CandidateInspectionOrder"
        ] = candidate_index

        row[
            "ProtocolEligible"
        ] = (
            str(
                row[
                    "InspectionStatus"
                ]
            )
            == "ELIGIBLE"
        )

        row[
            "InspectionError"
        ] = ""

        scan_rows.append(
            row
        )

        print(
            "    Reused:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            int(
                row[
                    "Builds"
                ]
            ),
            "| Model eval failures:",
            int(
                row[
                    "ModelEvaluationFailures"
                ]
            ),
        )

        continue


    started = time.perf_counter()

    row = {
        "CandidateInspectionOrder":
            candidate_index,

        "Project":
            project,

        "ProjectSlug":
            project_slug(
                project
            ),

        "SourceDirectory":
            str(
                source_directory
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",
    }


    try:
        missing_files = [
            filename
            for filename in REQUIRED_SELECTION_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            raise FileNotFoundError(
                "Missing selection files: "
                + ", ".join(
                    missing_files
                )
            )


        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )


        build_columns = pd.read_csv(
            builds_path,
            nrows=0,
        ).columns.tolist()

        exe_columns = pd.read_csv(
            exe_path,
            nrows=0,
        ).columns.tolist()

        dataset_columns = pd.read_csv(
            dataset_path,
            nrows=0,
        ).columns.tolist()


        build_id_column = resolve_column(
            build_columns,
            "id",
            f"{project} builds.csv ID",
        )

        started_at_column = resolve_column(
            build_columns,
            "started_at",
            f"{project} builds.csv started_at",
        )

        execution_build_column = resolve_column(
            exe_columns,
            "build",
            f"{project} exe.csv build",
        )

        execution_verdict_column = resolve_column(
            exe_columns,
            "verdict",
            f"{project} exe.csv verdict",
        )

        dataset_build_column = resolve_column(
            dataset_columns,
            "Build",
            f"{project} dataset.csv Build",
        )

        dataset_verdict_column = resolve_column(
            dataset_columns,
            "Verdict",
            f"{project} dataset.csv Verdict",
        )


        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )


        builds[
            build_id_column
        ] = parse_integer_series(
            builds[
                build_id_column
            ],
            f"{project}.builds.id",
        )


        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )


        invalid_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )


        duplicate_build_id_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )


        if invalid_timestamps != 0:
            raise RuntimeError(
                f"Invalid build timestamps: {invalid_timestamps}"
            )


        if duplicate_build_id_rows != 0:
            raise RuntimeError(
                f"Duplicate build-ID rows: {duplicate_build_id_rows}"
            )


        ordered_builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        number_of_builds = len(
            ordered_builds
        )


        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )


        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )


        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "Chronological 75/25 split has an empty partition."
            )


        training_build_ids = set(
            ordered_builds.iloc[
                :training_build_count
            ][
                build_id_column
            ].astype(int).tolist()
        )


        evaluation_build_ids = set(
            ordered_builds.iloc[
                training_build_count:
            ][
                build_id_column
            ].astype(int).tolist()
        )


        if training_build_ids & evaluation_build_ids:
            raise RuntimeError(
                "Training/evaluation build partitions overlap."
            )


        raw_profile = count_partitioned_rows(
            csv_path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.exe",
            chunksize=500_000,
        )


        model_profile = count_partitioned_rows(
            csv_path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.dataset",
            chunksize=250_000,
        )


        eligibility_reasons = []


        eligibility_tests = [
            (
                raw_profile[
                    "TrainingRows"
                ] > 0,
                "No raw training rows",
            ),

            (
                raw_profile[
                    "EvaluationRows"
                ] > 0,
                "No raw evaluation rows",
            ),

            (
                raw_profile[
                    "TrainingFailures"
                ] > 0,
                "No raw training failures",
            ),

            (
                raw_profile[
                    "EvaluationFailures"
                ] > 0,
                "No raw evaluation failures",
            ),

            (
                model_profile[
                    "TrainingRows"
                ] > 0,
                "No model training rows",
            ),

            (
                model_profile[
                    "EvaluationRows"
                ] > 0,
                "No model evaluation rows",
            ),

            (
                model_profile[
                    "TrainingFailures"
                ] > 0,
                "No model training failures",
            ),

            (
                model_profile[
                    "EvaluationFailures"
                ] > 0,
                "No model evaluation failures",
            ),

            (
                raw_profile[
                    "UnlinkedRows"
                ] == 0,
                "Raw rows reference unknown builds",
            ),

            (
                model_profile[
                    "UnlinkedRows"
                ] == 0,
                "Model rows reference unknown builds",
            ),
        ]


        for passed, failure_reason in eligibility_tests:
            if not passed:
                eligibility_reasons.append(
                    failure_reason
                )


        protocol_eligible = (
            len(
                eligibility_reasons
            )
            == 0
        )


        row.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "RawExecutionRows":
                raw_profile[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_profile[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_profile[
                    "EvaluationRows"
                ],

            "RawTrainFailures":
                raw_profile[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_profile[
                    "EvaluationFailures"
                ],

            "RawFailingTrainingBuilds":
                raw_profile[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_profile[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_profile[
                    "UnlinkedRows"
                ],

            "RawVerdictValuesJSON":
                raw_profile[
                    "VerdictValuesJSON"
                ],

            "ModelReadyRows":
                model_profile[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_profile[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_profile[
                    "EvaluationRows"
                ],

            "ModelTrainFailures":
                model_profile[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_profile[
                    "EvaluationFailures"
                ],

            "ModelFailingTrainingBuilds":
                model_profile[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_profile[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_profile[
                    "UnlinkedRows"
                ],

            "ModelVerdictValuesJSON":
                model_profile[
                    "VerdictValuesJSON"
                ],

            "ProtocolEligible":
                protocol_eligible,

            "EligibilityReason":
                (
                    ""
                    if protocol_eligible
                    else "; ".join(
                        eligibility_reasons
                    )
                ),

            "InspectionStatus":
                (
                    "ELIGIBLE"
                    if protocol_eligible
                    else "INELIGIBLE"
                ),

            "InspectionError":
                "",
        })


        print(
            "    Status:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            number_of_builds,
            "| Model rows:",
            model_profile[
                "Rows"
            ],
            "| Model eval failures:",
            model_profile[
                "EvaluationFailures"
            ],
        )


    except Exception as error:
        row.update({
            "ProtocolEligible":
                False,

            "EligibilityReason":
                "Inspection error",

            "InspectionStatus":
                "ERROR",

            "InspectionError":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })

        print(
            "    ERROR:",
            row[
                "InspectionError"
            ],
        )


    row[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )


    scan_rows.append(
        row
    )


    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        pd.DataFrame(
            scan_rows
        ).sort_values(
            "CandidateInspectionOrder",
            kind="mergesort",
        ),
    )


scan_progress = (
    pd.DataFrame(
        scan_rows
    )
    .sort_values(
        "CandidateInspectionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DETERMINISTIC RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ERROR"
    )
].copy()


eligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ELIGIBLE"
    )
].copy()


ineligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "INELIGIBLE"
    )
].copy()


if not inspection_errors.empty:
    print(
        "\nCandidate inspection errors:"
    )

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 16 candidates could not be inspected. "
        "No provisional selection was frozen."
    )


if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 16 candidate was found."
    )


eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelTrainingRows",
            "ModelEvaluationRows",
            "RawExecutionRows",
            "Project",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        )
        + 1,
        dtype=np.int64,
    ),
)


top_candidate = eligible_candidates.iloc[
    0
]


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Projects 1–15 COMPLETE_AND_FROZEN",
    EXPECTED_REGISTERED_PROJECTS,
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ) == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 16 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


add_check(
    validation_records,
    "Project 11 frozen identity",
    "apache@shardingsphere",
    str(
        registry.loc[
            registry_project_numbers.eq(
                11
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                11
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "apache@shardingsphere",
)

add_check(
    validation_records,
    "Project 12 frozen identity",
    "zolyfarkas@spf4j",
    str(
        registry.loc[
            registry_project_numbers.eq(
                12
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                12
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "zolyfarkas@spf4j",
)

add_check(
    validation_records,
    "Project 13 frozen identity",
    "jcabi@jcabi-github",
    str(
        registry.loc[
            registry_project_numbers.eq(
                13
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                13
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "jcabi@jcabi-github",
)

add_check(
    validation_records,
    "Project 14 frozen identity",
    "JMRI@JMRI",
    str(
        registry.loc[
            registry_project_numbers.eq(
                14
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                14
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "JMRI@JMRI",
)

add_check(
    validation_records,
    "Project 15 frozen identity",
    "eclipse@steady",
    str(
        registry.loc[
            registry_project_numbers.eq(
                15
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                15
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "eclipse@steady",
)

add_check(
    validation_records,
    "Candidates inspected",
    EXPECTED_CANDIDATES,
    len(scan_progress),
    len(scan_progress)
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Unique candidate identities",
    EXPECTED_CANDIDATES,
    int(
        scan_progress[
            "Project"
        ].nunique()
    ),
    int(
        scan_progress[
            "Project"
        ].nunique()
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Registered/reserved candidates",
    0,
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ),
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Inspection errors",
    0,
    len(inspection_errors),
    len(inspection_errors)
    == 0,
)

add_check(
    validation_records,
    "Candidate accounting",
    EXPECTED_CANDIDATES,
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ),
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "At least one eligible candidate",
    "> 0",
    len(eligible_candidates),
    len(eligible_candidates)
    > 0,
)

add_check(
    validation_records,
    "Candidate ranks unique",
    len(eligible_candidates),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ) == len(
        eligible_candidates
    ),
)

add_check(
    validation_records,
    "Top rank",
    1,
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
    int(
        top_candidate[
            "CandidateRank"
        ]
    ) == 1,
)

add_check(
    validation_records,
    "Top candidate eligible",
    True,
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
)

add_check(
    validation_records,
    "Top candidate raw unlinked rows",
    0,
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model unlinked rows",
    0,
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model training failures",
    "> 0",
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ) > 0,
)

add_check(
    validation_records,
    "Top candidate model evaluation failures",
    "> 0",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ) > 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 16 Step 1A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed validation checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 16 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE AUTHORITATIVE STEP 1A OUTPUTS
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]


source_schema_audit = scan_progress[
    schema_columns
].copy()


atomic_write_csv(
    SCAN_PROGRESS_PATH,
    scan_progress,
)

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    scan_progress,
)

atomic_write_csv(
    ELIGIBLE_RANKED_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_PATH,
    ineligible_candidates,
)

atomic_write_csv(
    INSPECTION_ERRORS_PATH,
    inspection_errors,
)

atomic_write_csv(
    STEP1A_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


dimension_fields = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]


provisional_selection_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionState":
        "PROVISIONAL_PENDING_STEP_1B_FREEZE",

    "CandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "SourceDirectory":
        str(
            top_candidate[
                "SourceDirectory"
            ]
        ),

    "BuildIDColumn":
        str(
            top_candidate[
                "BuildIDColumn"
            ]
        ),

    "StartedAtColumn":
        str(
            top_candidate[
                "StartedAtColumn"
            ]
        ),

    "ExecutionBuildColumn":
        str(
            top_candidate[
                "ExecutionBuildColumn"
            ]
        ),

    "ExecutionVerdictColumn":
        str(
            top_candidate[
                "ExecutionVerdictColumn"
            ]
        ),

    "DatasetBuildColumn":
        str(
            top_candidate[
                "DatasetBuildColumn"
            ]
        ),

    "DatasetVerdictColumn":
        str(
            top_candidate[
                "DatasetVerdictColumn"
            ]
        ),

    "Dimensions": {
        field:
            int(
                top_candidate[
                    field
                ]
            )
        for field in dimension_fields
    },

    "RankingRule":
        RUNTIME_PRIORITY_RULE,

    "RankingPurpose":
        "Runtime-prioritized processing order only; protocol eligibility and final project set are unchanged",

    "EligibleCandidateCount":
        len(
            eligible_candidates
        ),

    "IneligibleCandidateCount":
        len(
            ineligible_candidates
        ),

    "ReservedActiveProjectsExcluded":
        sorted(
            RESERVED_ACTIVE_PROJECTS
        ),

    "CompletedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_selection_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_16_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalSelection":
        provisional_selection_payload,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project16ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_16_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project16ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL ISOLATION CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 16 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print(
    "\nRanked eligible Project 16 candidates:"
)

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


print(
    "\nProtocol-ineligible candidates:"
)

if ineligible_candidates.empty:
    print(
        "None"
    )

else:
    display(
        ineligible_candidates[
            [
                "Project",
                "Builds",
                "RawTrainFailures",
                "RawEvaluationFailures",
                "ModelTrainFailures",
                "ModelEvaluationFailures",
                "EligibilityReason",
            ]
        ]
    )


print("\n")
print("=" * 132)
print("=== PROJECT 16 CELL 2 / STEP 1A RESULT ===")
print("=" * 132)


print(
    "Registered projects:",
    len(
        registry
    ),
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Candidates inspected:",
    len(
        scan_progress
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)

print(
    "Runtime-priority ranking rule:",
    RUNTIME_PRIORITY_RULE,
)


print(
    "\nProvisional Project 16 candidate:"
)

print(
    "Candidate rank:",
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    str(
        top_candidate[
            "Project"
        ]
    ),
)

print(
    "Project slug:",
    str(
        top_candidate[
            "ProjectSlug"
        ]
    ),
)

print(
    "Source directory:",
    str(
        top_candidate[
            "SourceDirectory"
        ]
    ),
)


print(
    "\nCandidate dimensions:"
)

for field in dimension_fields:
    print(
        f"{field}:",
        int(
            top_candidate[
                field
            ]
        ),
    )


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–13 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 16 experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 16 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===
------------------------------------------------------------------------------------------------------------------------------------
[01/10] Inspecting: EMResearch@EvoMaster
    Status: ELIGIBLE | Builds: 583 | Model rows: 14460 | Model eval failures: 68
------------------------------------------------------------------------------------------------------------------------------------
[02/10] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE | Builds: 3668 | Model rows: 4822 | Model eval failures: 0
------------------------------------------------------------------------------------------------------------------------------------
[03/10] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE | Builds: 4286 | Model rows: 224550 | Model eval failures: 20
------------------------------------------------------------------------------------------------------------------------------------
[04/1

,Check,Expected,Actual,Pass
0,Completion registry rows,15,15,True
1,Projects 1–15 COMPLETE_AND_FROZEN,15,15,True
2,Project 16 registry rows,0,0,True
3,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
4,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True
5,Project 13 frozen identity,jcabi@jcabi-github,jcabi@jcabi-github,True
6,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
7,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
8,Candidates inspected,10,10,True
9,Unique candidate identities,10,10,True



Ranked eligible Project 16 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,apache@rocketmq,apache__rocketmq,536,402,134,97734,106,9,8,6548,4907,1641,105,9,8,0,0
1,2,yamcs@Yamcs,yamcs__Yamcs,504,378,126,58101,147,45,10,7533,6452,1081,145,45,10,0,0
2,3,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,450,337,113,67108,144,31,12,10580,8232,2348,142,31,12,0,0
3,4,EMResearch@EvoMaster,EMResearch__EvoMaster,583,437,146,59155,286,68,41,14460,9907,4553,284,68,41,0,0
4,5,apache@curator,apache__curator,517,387,130,59697,124,2,2,10509,10403,106,123,2,2,0,0
5,6,facebook@buck,facebook__buck,846,634,212,561294,1120,8,7,80898,75643,5255,1119,8,7,0,0
6,7,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0
7,8,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
8,9,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,5635027,1778,20,17,224550,205696,18854,1777,20,17,0,0



Protocol-ineligible candidates:


,Project,Builds,RawTrainFailures,RawEvaluationFailures,ModelTrainFailures,ModelEvaluationFailures,EligibilityReason
1,Graylog2@graylog2-server,3668,280,0,279,0,No raw evaluation failures; No model evaluatio...




=== PROJECT 16 CELL 2 / STEP 1A RESULT ===
Registered projects: 15
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Candidates inspected: 10
Protocol-eligible candidates: 9
Protocol-ineligible candidates: 1
Inspection errors: 0
Runtime-priority ranking rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Provisional Project 16 candidate:
Candidate rank: 1
Project: apache@rocketmq
Project slug: apache__rocketmq
Source directory: /content/datasets/datasets/apache@rocketmq

Candidate dimensions:
Builds: 536
TrainingBuilds: 402
EvaluationBuilds: 134
RawExecutionRows: 97734
RawTrainingRows: 71052
RawEvaluationRows: 26682
RawTrainFailures: 106
RawEvaluationFailures: 9
RawFailingTrainingBuilds: 49
RawFailingEvaluationBuilds: 8
RawUnlinkedRows: 0
ModelReadyRows: 6548
ModelTrainingRows: 

In [3]:
# ==================================================================================================
# PROJECT 16 — CELL 3 / STEP 1B
# FINAL SELECTION, CHRONOLOGY FREEZE, SOURCE MANIFEST, AND CHECKPOINT
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# SAFETY:
# - freezes the Project 16 identity selected by Step 1A;
# - freezes the complete source manifest and source-root SHA-256;
# - freezes the chronological 75/25 build split;
# - validates the exact raw/model dimensions discovered in Step 1A;
# - writes no completion-registry changes;
# - does not access or modify prior-project condition outputs;
# - does not start the Project 16 experiment.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 16 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
CANDIDATE_RANK = 1

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_16_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 15

EXPECTED_RESERVED_ACTIVE_PROJECTS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_DIMENSIONS = {
    "Builds": 536,
    "TrainingBuilds": 402,
    "EvaluationBuilds": 134,

    "RawExecutionRows": 97_734,
    "RawTrainingRows": 71_052,
    "RawEvaluationRows": 26_682,
    "RawTrainFailures": 106,
    "RawEvaluationFailures": 9,
    "RawFailingTrainingBuilds": 49,
    "RawFailingEvaluationBuilds": 8,
    "RawUnlinkedRows": 0,

    "ModelReadyRows": 6_548,
    "ModelTrainingRows": 4_907,
    "ModelEvaluationRows": 1_641,
    "ModelTrainFailures": 105,
    "ModelEvaluationFailures": 9,
    "ModelFailingTrainingBuilds": 48,
    "ModelFailingEvaluationBuilds": 8,
    "ModelUnlinkedRows": 0,
}

REQUIRED_SOURCE_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_16_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_16_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_16_provisional_selection.json"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_16_eligible_candidates_ranked.csv"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_16_fixed_chronological_builds.csv"
)

SOURCE_SCHEMA_SNAPSHOT_PATH = (
    SELECTION_ROOT
    / "project_16_selected_source_schema_snapshot.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_16_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def canonical_root_hash(
    manifest,
):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }

    missing_columns = (
        required_columns
        - set(manifest.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Source manifest is missing columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )

    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def profile_partition(
    frame,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
):
    training_mask = frame[
        build_column
    ].isin(
        training_build_ids
    )

    evaluation_mask = frame[
        build_column
    ].isin(
        evaluation_build_ids
    )

    linked_mask = (
        training_mask
        | evaluation_mask
    )

    failure_mask = frame[
        verdict_column
    ].ne(0)

    return {
        "Rows":
            int(len(frame)),

        "TrainingRows":
            int(training_mask.sum()),

        "EvaluationRows":
            int(evaluation_mask.sum()),

        "TrainingFailures":
            int(
                (
                    training_mask
                    & failure_mask
                ).sum()
            ),

        "EvaluationFailures":
            int(
                (
                    evaluation_mask
                    & failure_mask
                ).sum()
            ),

        "FailingTrainingBuilds":
            int(
                frame.loc[
                    training_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "FailingEvaluationBuilds":
            int(
                frame.loc[
                    evaluation_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "UnlinkedRows":
            int(
                (
                    ~linked_mask
                ).sum()
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_RANKED_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 16 Step 1B inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Selected Project 16 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


source_file_names = {
    path.name
    for path in SOURCE_DIRECTORY.iterdir()
    if path.is_file()
}


missing_required_source_files = sorted(
    REQUIRED_SOURCE_FILES
    - source_file_names
)


if missing_required_source_files:
    raise FileNotFoundError(
        "Selected Project 16 source is missing required files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1A, ARCHIVE, AND REGISTRY
# --------------------------------------------------------------------------------------------------

step1a_status_sha256 = sha256_file(
    STEP1A_STATUS_PATH
)

step1a_report_sha256 = sha256_file(
    STEP1A_REPORT_PATH
)

provisional_selection_sha256 = sha256_file(
    PROVISIONAL_SELECTION_PATH
)

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)


if step1a_status.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 16 Step 1A status is not PASS."
    )


if step1a_report.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 16 Step 1A report is not PASS."
    )


if provisional_selection.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 16 provisional selection state differs."
    )


if provisional_selection.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "Project 16 provisional project differs.\n"
        f"Expected: {PROJECT_NAME}\n"
        f"Actual:   {provisional_selection.get('Project')}"
    )


if provisional_selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Project 16 provisional slug differs."
    )


if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != CANDIDATE_RANK:
    raise RuntimeError(
        "Project 16 provisional candidate rank differs."
    )


if provisional_selection.get(
    "RankingRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 16 runtime-priority ranking rule differs."
    )


reserved_in_step1a = sorted(
    provisional_selection.get(
        "ReservedActiveProjectsExcluded",
        [],
    )
)


if reserved_in_step1a != EXPECTED_RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "Project 16 Step 1A unexpectedly excluded an active reservation."
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 16))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–15."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


if registry[
    registry_project_column
].eq(
    PROJECT_NAME
).any():
    raise RuntimeError(
        "The selected Project 16 identity is already registered."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


ranked_candidates = pd.read_csv(
    ELIGIBLE_RANKED_PATH,
    low_memory=False,
)


rank_one_rows = ranked_candidates.loc[
    pd.to_numeric(
        ranked_candidates[
            "CandidateRank"
        ],
        errors="coerce",
    ).eq(
        CANDIDATE_RANK
    )
]


if len(rank_one_rows) != 1:
    raise RuntimeError(
        "Step 1A ranked candidates do not contain exactly one rank-1 row."
    )


rank_one = rank_one_rows.iloc[0]


if (
    str(
        rank_one[
            "Project"
        ]
    ) != PROJECT_NAME
    or str(
        rank_one[
            "ProjectSlug"
        ]
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 1A rank-1 identity differs."
    )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE COMPLETE SOURCE MANIFEST
# --------------------------------------------------------------------------------------------------

source_files = sorted(
    [
        path
        for path in SOURCE_DIRECTORY.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            SOURCE_DIRECTORY
        ).as_posix(),
)


if not source_files:
    raise RuntimeError(
        "Selected Project 16 source directory contains no files."
    )


source_manifest_records = []


for source_path in source_files:
    relative_path = source_path.relative_to(
        SOURCE_DIRECTORY
    ).as_posix()

    source_manifest_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


source_manifest = pd.DataFrame(
    source_manifest_records
)


source_root_sha256 = canonical_root_hash(
    source_manifest
)

source_file_count = len(
    source_manifest
)

source_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 7. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

source_paths = {
    "builds.csv":
        SOURCE_DIRECTORY
        / "builds.csv",

    "exe.csv":
        SOURCE_DIRECTORY
        / "exe.csv",

    "dataset.csv":
        SOURCE_DIRECTORY
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIRECTORY
        / "entity_change_history.csv",

    "id_map.csv":
        SOURCE_DIRECTORY
        / "id_map.csv",
}


if (
    SOURCE_DIRECTORY
    / "contributors.csv"
).is_file():
    source_paths[
        "contributors.csv"
    ] = (
        SOURCE_DIRECTORY
        / "contributors.csv"
    )


schema_snapshot_records = []


for filename, file_path in source_paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    schema_snapshot_records.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(columns),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_snapshot = pd.DataFrame(
    schema_snapshot_records
)


build_columns = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    nrows=0,
).columns.tolist()


build_id_column = resolve_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_timestamp_column = resolve_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)

exe_build_column = resolve_column(
    exe_columns,
    "build",
    "exe.csv build",
)

exe_verdict_column = resolve_column(
    exe_columns,
    "verdict",
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset_columns,
    "Build",
    "dataset.csv Build",
)

dataset_verdict_column = resolve_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. FREEZE CHRONOLOGY AND 75/25 SPLIT
# --------------------------------------------------------------------------------------------------

builds = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_series(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


invalid_timestamp_rows = int(
    builds[
        build_timestamp_column
    ].isna().sum()
)


duplicate_build_id_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)


if invalid_timestamp_rows != 0:
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


if duplicate_build_id_rows != 0:
    raise RuntimeError(
        "builds.csv contains duplicate build IDs."
    )


timestamp_group_sizes = builds.groupby(
    build_timestamp_column
).size()


timestamp_tie_groups = int(
    timestamp_group_sizes.gt(1).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(1)
    ].sum()
)


ordered_builds = (
    builds.sort_values(
        [
            build_timestamp_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


number_of_builds = len(
    ordered_builds
)


training_build_count = int(
    math.floor(
        0.75
        * number_of_builds
    )
)


evaluation_build_count = int(
    number_of_builds
    - training_build_count
)


ordered_builds[
    "ChronologyOrder"
] = np.arange(
    1,
    number_of_builds
    + 1,
    dtype=np.int64,
)


ordered_builds[
    "Partition"
] = np.where(
    ordered_builds[
        "ChronologyOrder"
    ].le(
        training_build_count
    ),
    "TRAIN",
    "EVALUATION",
)


ordered_builds[
    "PartitionOrder"
] = (
    ordered_builds.groupby(
        "Partition",
        sort=False,
    ).cumcount()
    + 1
)


fixed_chronology = ordered_builds[
    [
        "ChronologyOrder",
        build_id_column,
        build_timestamp_column,
        "Partition",
        "PartitionOrder",
    ]
].rename(
    columns={
        build_id_column:
            "BuildID",

        build_timestamp_column:
            "StartedAtUTC",
    }
)


training_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(int).tolist()
)


evaluation_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(int).tolist()
)


partition_overlap = len(
    training_build_ids
    & evaluation_build_ids
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE RAW AND MODEL DIMENSIONS
# --------------------------------------------------------------------------------------------------

exe = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=[
        exe_build_column,
        exe_verdict_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_series(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_verdict_column
] = parse_integer_series(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


dataset = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=[
        dataset_build_column,
        dataset_verdict_column,
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_series(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_verdict_column
] = parse_integer_series(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


raw_profile = profile_partition(
    frame=exe,
    build_column=exe_build_column,
    verdict_column=exe_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


model_profile = profile_partition(
    frame=dataset,
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


actual_dimensions = {
    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "RawExecutionRows":
        raw_profile[
            "Rows"
        ],

    "RawTrainingRows":
        raw_profile[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_profile[
            "EvaluationRows"
        ],

    "RawTrainFailures":
        raw_profile[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_profile[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_profile[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_profile[
            "FailingEvaluationBuilds"
        ],

    "RawUnlinkedRows":
        raw_profile[
            "UnlinkedRows"
        ],

    "ModelReadyRows":
        model_profile[
            "Rows"
        ],

    "ModelTrainingRows":
        model_profile[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_profile[
            "EvaluationRows"
        ],

    "ModelTrainFailures":
        model_profile[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_profile[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_profile[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_profile[
            "FailingEvaluationBuilds"
        ],

    "ModelUnlinkedRows":
        model_profile[
            "UnlinkedRows"
        ],
}


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1A status",
    STEP1A_PASS_STATUS,
    step1a_status.get(
        "Status"
    ),
    step1a_status.get(
        "Status"
    ) == STEP1A_PASS_STATUS,
)

add_check(
    validation_records,
    "Candidate rank",
    CANDIDATE_RANK,
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ),
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ) == CANDIDATE_RANK,
)

add_check(
    validation_records,
    "Selected project",
    PROJECT_NAME,
    provisional_selection[
        "Project"
    ],
    provisional_selection[
        "Project"
    ] == PROJECT_NAME,
)

add_check(
    validation_records,
    "Selected project slug",
    PROJECT_SLUG,
    provisional_selection[
        "ProjectSlug"
    ],
    provisional_selection[
        "ProjectSlug"
    ] == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Archive SHA-256",
    EXPECTED_ARCHIVE_SHA256,
    archive_sha256,
    archive_sha256
    == EXPECTED_ARCHIVE_SHA256,
)

add_check(
    validation_records,
    "Registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha256_before,
    registry_sha256_before
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 16 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


add_check(
    validation_records,
    "Project 11 frozen identity",
    "apache@shardingsphere",
    str(
        registry.loc[
            registry_project_numbers.eq(
                11
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                11
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "apache@shardingsphere",
)

add_check(
    validation_records,
    "Project 12 frozen identity",
    "zolyfarkas@spf4j",
    str(
        registry.loc[
            registry_project_numbers.eq(
                12
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                12
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "zolyfarkas@spf4j",
)


add_check(
    validation_records,
    "Project 13 frozen identity",
    "jcabi@jcabi-github",
    str(
        registry.loc[
            registry_project_numbers.eq(
                13
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                13
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "jcabi@jcabi-github",
)

add_check(
    validation_records,
    "Project 14 frozen identity",
    "JMRI@JMRI",
    str(
        registry.loc[
            registry_project_numbers.eq(
                14
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                14
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "JMRI@JMRI",
)

add_check(
    validation_records,
    "Project 15 frozen identity",
    "eclipse@steady",
    str(
        registry.loc[
            registry_project_numbers.eq(
                15
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    ),
    str(
        registry.loc[
            registry_project_numbers.eq(
                15
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )
    == "eclipse@steady",
)

add_check(
    validation_records,
    "Active reservations excluded",
    EXPECTED_RESERVED_ACTIVE_PROJECTS,
    reserved_in_step1a,
    reserved_in_step1a
    == EXPECTED_RESERVED_ACTIVE_PROJECTS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    provisional_selection.get(
        "RankingRule"
    ),
    provisional_selection.get(
        "RankingRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Source files",
    "> 0",
    source_file_count,
    source_file_count > 0,
)

add_check(
    validation_records,
    "Source bytes",
    "> 0",
    source_bytes,
    source_bytes > 0,
)

add_check(
    validation_records,
    "Invalid timestamp rows",
    0,
    invalid_timestamp_rows,
    invalid_timestamp_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build-ID rows",
    0,
    duplicate_build_id_rows,
    duplicate_build_id_rows == 0,
)

add_check(
    validation_records,
    "Partition overlap",
    0,
    partition_overlap,
    partition_overlap == 0,
)


for metric, expected_value in EXPECTED_DIMENSIONS.items():
    actual_value = int(
        actual_dimensions[
            metric
        ]
    )

    add_check(
        validation_records,
        metric,
        expected_value,
        actual_value,
        actual_value
        == expected_value,
    )


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 16 Step 1B validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Project 16 Step 1B checks:")

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 16 STEP 1B VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE FROZEN OUTPUTS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    fixed_chronology,
)

atomic_write_csv(
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    source_schema_snapshot,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RuntimePriorityPurpose":
        (
            "Processing order only; protocol eligibility "
            "and final project set are unchanged"
        ),

    "ActiveReservations":
        [],

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "Step1AStatusSHA256":
        step1a_status_sha256,

    "Step1AReportSHA256":
        step1a_report_sha256,

    "ProvisionalSelectionSHA256":
        provisional_selection_sha256,

    "ChronologyRule":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "Dimensions":
        actual_dimensions,

    "FrozenSourceManifest":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronology":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "SourceSchemaSnapshot":
        str(
            SOURCE_SCHEMA_SNAPSHOT_PATH
        ),

    "Validation":
        str(
            STEP1B_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "Project14RegistryIdentity":
        required_registered_identities[14],

    "Project15RegistryIdentity":
        required_registered_identities[15],

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project16ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "SelectionCheckpoint":
        True,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceManifest":
        True,

    "DoNotChangeChronology":
        True,

    "DoNotChangeBuildPartitions":
        True,
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project16ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK AND IMMUTABILITY CHECKS
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 16 Step 1B."
    )


final_manifest_records = []


for row in source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise RuntimeError(
            "A frozen Project 16 source file disappeared:\n"
            f"{source_path}"
        )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_manifest_records
)


final_source_root_sha256 = canonical_root_hash(
    final_source_manifest
)


if final_source_root_sha256 != source_root_sha256:
    raise RuntimeError(
        "Project 16 source changed during Step 1B."
    )


checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 16 selection checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 16 Step 1B status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\nFrozen Project 16 source manifest:")

display(
    source_manifest
)


print("\nFixed Project 16 chronology sample:")

display(
    pd.concat(
        [
            fixed_chronology.head(10),
            fixed_chronology.tail(10),
        ],
        ignore_index=True,
    )
)


print("\n")
print("=" * 132)
print("=== PROJECT 16 CELL 3 / STEP 1B RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    CANDIDATE_RANK,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)


print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Active reservations:",
    EXPECTED_RESERVED_ACTIVE_PROJECTS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)


print("\nFrozen source:")

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nChronology:")

print(
    "Rule: started_at ascending; "
    "Build ID descending for timestamp ties"
)

print(
    "Builds:",
    number_of_builds,
)

print(
    "Training / evaluation builds:",
    len(
        training_build_ids
    ),
    "/",
    len(
        evaluation_build_ids
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Partition overlap:",
    partition_overlap,
)


print("\nRaw and model dimensions:")

for metric in EXPECTED_DIMENSIONS:
    print(
        f"{metric}:",
        actual_dimensions[
            metric
        ],
    )


print("\nIsolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–13 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 16 experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    selection_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 16 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 16 Step 1B validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_16_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_16_CANDIDATE_DISCOVERY_COMPLETE,True
1,Candidate rank,1,1,True
2,Selected project,apache@rocketmq,apache@rocketmq,True
3,Selected project slug,apache__rocketmq,apache__rocketmq,True
4,Archive SHA-256,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,True
5,Registry SHA-256,2c8803de6d0a448f1c505576829f110ae2144ca7f16e97...,2c8803de6d0a448f1c505576829f110ae2144ca7f16e97...,True
6,Registry rows,15,15,True
7,Project 16 registry rows,0,0,True
8,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
9,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True



Frozen Project 16 source manifest:


,RelativePath,SizeBytes,SHA256
0,builds.csv,43836,e4642221a292c3796de8e3adbe5fa00376d50b6144eee0...
1,contributors.csv,21715,2471a230298c83af181bd26ed1cf0c9fc6bf6cae219a86...
2,dataset.csv,5048628,86d68a4e4849ce102d43eefdfb5f5fbc96f3fb15fea864...
3,entity_change_history.csv,1735743,a0db989f6da64028fa7668eaa195bf2b9efbd3abfb4b4a...
4,exe.csv,3040921,10130c9b11d67c67ba18c8b3b455e507a9a379f0a3d769...
5,id_map.csv,272972,432bbfe25e8f374885226abc0bb9a2bd59232d04542eba...



Fixed Project 16 chronology sample:


,ChronologyOrder,BuildID,StartedAtUTC,Partition,PartitionOrder
0,1,666167895,2020-03-24 03:37:01+00:00,TRAIN,1
1,2,666175070,2020-03-24 04:18:25+00:00,TRAIN,2
2,3,666740558,2020-03-25 11:39:27+00:00,TRAIN,3
3,4,666747180,2020-03-25 12:02:32+00:00,TRAIN,4
4,5,666855963,2020-03-25 16:03:57+00:00,TRAIN,5
5,6,667647907,2020-03-27 11:17:40+00:00,TRAIN,6
6,7,667963337,2020-03-28 03:36:21+00:00,TRAIN,7
7,8,668646205,2020-03-30 09:04:35+00:00,TRAIN,8
8,9,668671709,2020-03-30 10:17:48+00:00,TRAIN,9
9,10,668725029,2020-03-30 12:36:10+00:00,TRAIN,10




=== PROJECT 16 CELL 3 / STEP 1B RESULT ===

Project identity:
Project number: 16
Project: apache@rocketmq
Project slug: apache__rocketmq
Candidate rank: 1
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Selection state: FINAL_AND_FROZEN

Frozen source:
Source directory: /content/datasets/datasets/apache@rocketmq
Source files: 6
Source bytes: 10163815
Source root SHA-256: e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e

Chronology:
Rule: started_at ascending; Build ID descending for timestamp ties
Builds: 536
Training / evaluation builds: 402 / 134
Timestamp tie groups: 0
Partition overlap: 0

Raw and model dimensions:
Builds: 536
TrainingBuilds: 402
EvaluationBuilds: 13

In [4]:
# ==================================================================================================
# PROJECT 16 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# PURPOSE:
# - validate the frozen Project 16 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commits to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 16 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 16 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_16_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "b3cf69922236c45ae62599e141d0d29f0c3737a334370daffcbfa709abff250a"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 10_163_815

EXPECTED_BUILDS = 536
EXPECTED_TRAIN_BUILDS = 402
EXPECTED_EVAL_BUILDS = 134

EXPECTED_RAW_ROWS = 97_734
EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_RAW_TRAIN_FAILURES = 106
EXPECTED_RAW_EVAL_FAILURES = 9

EXPECTED_MODEL_ROWS = 6_548
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_16_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_16_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )

            result.append(
                token
            )

    return result


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(
        path
    ).is_file()
]

missing_inputs.extend(
    str(
        SOURCE_DIR
        / filename
    )
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIR
        / filename
    ).is_file()
)


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 16 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN SELECTION, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 16 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 16 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 16 identity differs."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 16 runtime-priority rule differs."
    )


active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 16 active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {active_reservations}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 15
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            16,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–15."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


frozen_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []


for row in frozen_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 16 source file is missing:\n"
            f"{source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)

current_source_root = source_root_hash(
    current_manifest
)

current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 16 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


build_order = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN VALIDATION
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        [
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        [
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(
        0
    ).sum()
)


if (
    raw_duplicate_pairs
    or model_duplicate_pairs
):
    raise RuntimeError(
        "Duplicate Build-Test pairs were found.\n"
        f"Raw duplicate rows: {raw_duplicate_pairs}\n"
        f"Model duplicate rows: {model_duplicate_pairs}"
    )


raw_pairs = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pairs = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


joined = model_pairs.merge(
    raw_pairs,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


verdict_mismatches = int(
    joined[
        "ModelVerdict"
    ].ne(
        joined[
            "RawVerdict"
        ]
    ).sum()
)


raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)


raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)


raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)


model_training_mask = joined[
    "Build"
].isin(
    training_builds
)


model_evaluation_mask = joined[
    "Build"
].isin(
    evaluation_builds
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


join_audit[
    "Pass"
] = (
    join_audit[
        "Expected"
    ].astype(
        str
    )
    == join_audit[
        "Actual"
    ].astype(
        str
    )
)


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,

        "PresentInDataset":
            feature
            in dataset.columns,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

token_rows = []
builds_without_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    tokens = extract_commit_tokens(
        raw_commits
    )

    if not tokens:
        builds_without_tokens += 1

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        token_rows.append({
            "BuildID":
                int(
                    build_id
                ),

            "ChronologyOrder":
                int(
                    build_order[
                        int(
                            build_id
                        )
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    token_rows
)


if build_tokens.empty:
    raise RuntimeError(
        "No build commit tokens could be extracted."
    )


build_tokens = (
    build_tokens.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ENTITY HISTORY AND ALIAS-AWARE ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


entity_history = (
    entity_history.loc[
        entity_history[
            "NormalisedCommit"
        ].ne(
            ""
        ),
        [
            entity_id_column,
            "NormalisedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    )
)


history_commits = sorted(
    entity_history[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commits
)


id_raw = pd.read_csv(
    paths[
        "id_map.csv"
    ],
    usecols=[
        id_key_column,
        id_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


orientation_rows = []


for column in [
    id_key_column,
    id_value_column,
]:
    numeric = pd.to_numeric(
        id_raw[
            column
        ],
        errors="coerce",
    )

    numeric_filled = numeric.fillna(
        0
    )

    valid_integral = (
        numeric.notna()
        & np.isclose(
            numeric_filled,
            np.floor(
                numeric_filled
            ),
            rtol=0,
            atol=0,
        )
    )

    parsed_ids = set(
        numeric.loc[
            valid_integral
        ].astype(
            "int64"
        )
    )

    overlap = len(
        parsed_ids
        & history_entity_ids
    )

    orientation_rows.append({
        "Column":
            column,

        "Rows":
            len(
                id_raw
            ),

        "IntegralNumericRows":
            int(
                valid_integral.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~valid_integral
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(
                parsed_ids
            ),

        "MatchingHistoryEntityIDs":
            overlap,

        "HistoryEntityCoveragePercent":
            (
                100.0
                * overlap
                / len(
                    history_entity_ids
                )
                if history_entity_ids
                else 0.0
            ),
    })


id_orientation = pd.DataFrame(
    orientation_rows
)


best_orientation = (
    id_orientation.sort_values(
        [
            "MatchingHistoryEntityIDs",
            "IntegralNumericRows",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    best_orientation
) < 2:
    raise RuntimeError(
        "id_map orientation audit is incomplete."
    )


if (
    int(
        best_orientation.loc[
            0,
            "MatchingHistoryEntityIDs",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "MatchingHistoryEntityIDs",
        ]
    )
    and int(
        best_orientation.loc[
            0,
            "IntegralNumericRows",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "IntegralNumericRows",
        ]
    )
):
    raise RuntimeError(
        "Could not uniquely resolve the EntityId column in id_map.csv."
    )


resolved_id_column = str(
    best_orientation.loc[
        0,
        "Column",
    ]
)


resolved_path_column = (
    id_value_column
    if resolved_id_column
    == id_key_column
    else id_key_column
)


resolved_numeric = pd.to_numeric(
    id_raw[
        resolved_id_column
    ],
    errors="coerce",
)


resolved_numeric_filled = resolved_numeric.fillna(
    0
)


valid_resolved = (
    resolved_numeric.notna()
    & np.isclose(
        resolved_numeric_filled,
        np.floor(
            resolved_numeric_filled
        ),
        rtol=0,
        atol=0,
    )
)


invalid_resolved_rows = int(
    (
        ~valid_resolved
    ).sum()
)


if invalid_resolved_rows:
    raise RuntimeError(
        "Resolved id_map EntityId column contains "
        f"{invalid_resolved_rows} invalid rows."
    )


resolved_id_map = pd.DataFrame({
    "EntityPath":
        id_raw[
            resolved_path_column
        ].astype(
            str
        ).str.strip(),

    "EntityId":
        resolved_numeric.astype(
            "int64"
        ),
})


empty_path_rows = int(
    resolved_id_map[
        "EntityPath"
    ].eq(
        ""
    ).sum()
)


exact_duplicate_rows = int(
    len(
        resolved_id_map
    )
    - len(
        resolved_id_map.drop_duplicates(
            [
                "EntityPath",
                "EntityId",
            ]
        )
    )
)


resolved_id_map = (
    resolved_id_map.drop_duplicates(
        [
            "EntityPath",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "EntityId",
            "EntityPath",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


duplicate_entity_id_rows = int(
    resolved_id_map.duplicated(
        "EntityId",
        keep=False,
    ).sum()
)


entity_ids_with_multiple_paths = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().gt(
        1
    ).sum()
)


paths_with_multiple_ids = int(
    resolved_id_map.groupby(
        "EntityPath"
    )[
        "EntityId"
    ].nunique().gt(
        1
    ).sum()
)


maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().max()
)


id_map_entity_ids = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# 12. COMMIT MATCHING AND BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

match_started = time.perf_counter()

match_rows = []


for row in build_tokens.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    matched_commit = None


    if token in history_commit_set:
        match_type = "EXACT"
        matched_commit = token
        candidate_count = 1

    else:
        candidates = [
            commit
            for commit in history_commits
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]

        if len(
            candidates
        ) == 1:
            match_type = (
                "UNIQUE_PREFIX"
            )

            matched_commit = candidates[
                0
            ]

            candidate_count = 1

        elif len(
            candidates
        ) == 0:
            match_type = (
                "UNMATCHED"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX"
            )

            candidate_count = len(
                candidates
            )


    match_rows.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_audit = (
    pd.DataFrame(
        match_rows
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


commit_matching_seconds = float(
    time.perf_counter()
    - match_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)


prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX"
    ).sum()
)


unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)


ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX"
    ).sum()
)


matched_token_rows = int(
    exact_matches
    + prefix_matches
)


commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        commit_audit
    )
)


matched_build_commits = (
    commit_audit.loc[
        commit_audit[
            "MatchedCommit"
        ].notna(),
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


entity_for_join = (
    entity_history.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity = (
    matched_build_commits.merge(
        entity_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity[
    "EntityId"
] = build_entity[
    "EntityId"
].astype(
    "int64"
)


build_entity = (
    build_entity[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


unmatched_commit_builds = sorted(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        ),
        "BuildID",
    ].astype(
        int
    ).unique().tolist()
)


mapping_incomplete_builds = sorted(
    set(
        builds_without_entities
    )
    | set(
        unmatched_commit_builds
    )
)


mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    )
)


mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


alias_summary = (
    resolved_id_map.groupby(
        "EntityId",
        as_index=False,
    )
    .agg(
        EntityPathAliasCount=(
            "EntityPath",
            "nunique",
        ),

        CanonicalEntityPath=(
            "EntityPath",
            "min",
        ),
    )
)


entity_id_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(
                mapped_entity_ids
            )
    })
    .merge(
        alias_summary,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


entity_id_audit[
    "PresentInIDMap"
] = entity_id_audit[
    "EntityPathAliasCount"
].notna()


entity_id_audit[
    "EntityPathAliasCount"
] = entity_id_audit[
    "EntityPathAliasCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            mapping_incomplete_builds
        ),
        [
            "BuildID",
            "ChronologyOrder",
            "Partition",
        ],
    ]
    .copy()
)


unmatched_counts = (
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        )
    ]
    .groupby(
        "BuildID"
    )
    .size()
    .rename(
        "UnmatchedCommitTokens"
    )
)


mapped_entity_counts = (
    build_entity.groupby(
        "BuildID"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "MappedEntityCount"
    )
)


mapping_incomplete_frame = (
    mapping_incomplete_frame.merge(
        unmatched_counts,
        on="BuildID",
        how="left",
    )
    .merge(
        mapped_entity_counts,
        on="BuildID",
        how="left",
    )
)


mapping_incomplete_frame[
    "UnmatchedCommitTokens"
] = mapping_incomplete_frame[
    "UnmatchedCommitTokens"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "MappedEntityCount"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "HasMappedEntities"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].gt(
    0
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    checks,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    checks,
    "Dataset key columns",
    3,
    3,
    (
        dataset_build_column
        in dataset.columns
        and dataset_test_column
        in dataset.columns
        and dataset_verdict_column
        in dataset.columns
    ),
)

add_check(
    checks,
    "Predictor count consistency",
    len(
        dataset.columns
    )
    - 3,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == (
        len(
            dataset.columns
        )
        - 3
    ),
)

add_check(
    checks,
    "REC features",
    19,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == 19
        and not missing_rec_features
    ),
)

for row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            row.Metric
        ),
        row.Expected,
        row.Actual,
        bool(
            row.Pass
        ),
    )

add_check(
    checks,
    "Invalid resolved id_map IDs",
    0,
    invalid_resolved_rows,
    invalid_resolved_rows
    == 0,
)

add_check(
    checks,
    "Empty id_map paths",
    0,
    empty_path_rows,
    empty_path_rows
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped entity IDs missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    15,
    len(
        registry
    ),
    len(
        registry
    ) == 15,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    checks,
    "Project 16 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 16 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 16 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 16 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


report_payload = {
    **summary_payload,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "ActiveReservations":
        active_reservations,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 16 Step 2A."
    )


final_manifest_records = []


for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 16 source changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),
            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 132
)

print(
    "=== PROJECT 16 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 132
)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

for predecessor_number in sorted(
    required_registered_identities
):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )


print(
    "Active reservations:",
    active_reservations,
)


print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)


print(
    "\nBuild-Test joins:"
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print(
    "\nid_map.csv resolution:"
)

print(
    "Resolved EntityId column:",
    resolved_id_column,
)

print(
    "Resolved path column:",
    resolved_path_column,
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)


print(
    "\nCommit and entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact commit matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Step 2B mapping audit required:",
    bool(
        unmatched_tokens
        or builds_without_entities
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–15 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 132
)


=== PROJECT 16 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===

Project 16 Step 2A validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,b3cf69922236c45ae62599e141d0d29f0c3737a334370d...,b3cf69922236c45ae62599e141d0d29f0c3737a334370d...,True
1,Source root SHA-256,e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65dd...,e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65dd...,True
2,Source files,6,6,True
3,Source bytes,10163815,10163815,True
4,Builds,536,536,True
5,Training builds,402,402,True
6,Evaluation builds,134,134,True
7,Raw rows,97734,97734,True
8,Model rows,6548,6548,True
9,Dataset key columns,3,3,True



Build-Test join audit:


,Metric,Expected,Actual,Pass
0,RawRows,97734,97734,True
1,RawTrainingRows,71052,71052,True
2,RawEvaluationRows,26682,26682,True
3,RawTrainingFailures,106,106,True
4,RawEvaluationFailures,9,9,True
5,ModelRows,6548,6548,True
6,ModelTrainingRows,4907,4907,True
7,ModelEvaluationRows,1641,1641,True
8,ModelTrainingFailures,105,105,True
9,ModelEvaluationFailures,9,9,True



id_map orientation audit:


...


Mapping-incomplete builds:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities
0,712125832,88,TRAIN,1,0,False
1,747593450,175,TRAIN,1,0,False
2,748265487,176,TRAIN,1,0,False
3,771761066,348,TRAIN,1,0,False
4,771763616,349,TRAIN,1,0,False
5,771767101,350,TRAIN,1,0,False
6,771873583,365,TRAIN,1,0,False
7,773450607,399,TRAIN,1,0,False



Build-entity sample:


,BuildID,ChronologyOrder,MatchedCommit,EntityId
0,666167895,1,3974677f04815609951c17059d85d3795eb51247,1
1,666167895,1,3974677f04815609951c17059d85d3795eb51247,36
2,666167895,1,3974677f04815609951c17059d85d3795eb51247,92
3,666167895,1,3974677f04815609951c17059d85d3795eb51247,190
4,666167895,1,3974677f04815609951c17059d85d3795eb51247,376
5,666167895,1,3974677f04815609951c17059d85d3795eb51247,415
6,666167895,1,3974677f04815609951c17059d85d3795eb51247,424
7,666167895,1,3974677f04815609951c17059d85d3795eb51247,474
8,666167895,1,3974677f04815609951c17059d85d3795eb51247,476
9,666167895,1,3974677f04815609951c17059d85d3795eb51247,527



=== PROJECT 16 CELL 4 / STEP 2A RESULT ===
Project: apache@rocketmq
Project slug: apache__rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Builds: 536
Training / evaluation builds: 402 / 134
Raw execution rows: 97734
Model-ready rows: 6548
Dataset columns: 154
Predictor columns: 151
REC features: 19

Build-Test joins:
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links: 0
Model/raw verdict mismatches: 0
Non-finite duration rows: 0
Negative duration rows: 0

id_map.csv resolution:
Resolved EntityId column: value
Resolved path column: key
Duplicate EntityId rows accepted as aliases: 1627
EntityIds with multiple paths: 790
Paths with multip

In [5]:
# ==================================================================================================
# PROJECT 16 — CELL 5 / STEP 2B
# NO-TIMESTAMP-TIE VECTORIZED CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   apache@rocketmq
#
# WHY THIS IMPLEMENTATION IS SAFE:
# - Project 16 has zero build-timestamp tie groups.
# - Each raw Build-Test pair is unique.
# - Therefore each test's history order is uniquely determined by frozen build chronology.
# - REC_Age uses the global first-appearance build order from the clean raw execution history.
# - The 16 non-file history features are reconstructed with vectorized cumulative calculations.
# - The two file-history features are reconstructed from the Step 2A build-entity map.
# - Clean anchor offsets preserve any accepted source-level file-mapping residuals exactly.
#
# RUN THIS AS A NEW CELL IN Thesis_project_16.ipynb.
# DO NOT RERUN PROJECT 16 STEPS 0–2A.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from collections import defaultdict

import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 136)
print("=== PROJECT 16 CELL 5 / STEP 2B: VECTORIZED CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_16_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_16_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

IMPLEMENTATION_VERSION = (
    "PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WITH_RAW_ONLY_TEST_HANDLING"
)

EXPECTED_SELECTION_SHA256 = (
    "b3cf69922236c45ae62599e141d0d29f0c3737a334370daffcbfa709abff250a"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 15

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 10_163_815

EXPECTED_BUILDS = 536
EXPECTED_TRAIN_BUILDS = 402
EXPECTED_EVAL_BUILDS = 134
EXPECTED_TIMESTAMP_TIE_GROUPS = 0

EXPECTED_RAW_ROWS = 97_734
EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_RAW_TRAIN_FAILURES = 106
EXPECTED_RAW_EVAL_FAILURES = 9

EXPECTED_MODEL_ROWS = 6_548
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 598
EXPECTED_EXACT_COMMIT_MATCHES = 590
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 8
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 528
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 8
EXPECTED_BUILD_ENTITY_ROWS = 5_058

EXPECTED_MAPPING_INCOMPLETE_BUILDS = {
    712125832,
    747593450,
    748265487,
    771761066,
    771763616,
    771767101,
    771873583,
    773450607,
}
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = {
    "TRAIN",
}
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 0

RECENT_WINDOW = 6

SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2

DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9

ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_16_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_16_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

TIMESTAMP_TIE_GROUPS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
)

TEST_ORDER_SEARCH_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_test_order_search_audit.csv"
)

INFERRED_EXECUTION_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

GLOBAL_AGE_ORDER_SEARCH_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_global_age_order_search.csv"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def prefix_sum(
    values,
):
    values = np.asarray(
        values
    )

    dtype = (
        np.float64
        if values.dtype.kind == "f"
        else np.int64
    )

    result = np.empty(
        len(values) + 1,
        dtype=dtype,
    )

    result[0] = 0

    np.cumsum(
        values,
        out=result[1:],
    )

    return result


def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.full(
        len(denominator),
        -1.0,
        dtype=float,
    )

    valid = denominator > 0

    result[
        valid
    ] = (
        numerator[
            valid
        ]
        / denominator[
            valid
        ]
    )

    return result


def calculate_file_rate(
    target_builds,
    current_changed_entities,
    entity_changed_builds,
):
    if not target_builds:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(
                entity_id
            )
        )

        if not changed_builds:
            continue

        overlap_count = len(
            target_builds.intersection(
                changed_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_requested_group_features(
    builds,
    verdicts,
    durations,
    global_positions,
    requested_positions,
    changed_entities_by_build,
    entity_changed_builds,
):
    builds = np.asarray(
        builds,
        dtype=np.int64,
    )

    verdicts = np.asarray(
        verdicts,
        dtype=np.int64,
    )

    durations = np.asarray(
        durations,
        dtype=np.float64,
    )

    global_positions = np.asarray(
        global_positions,
        dtype=np.int64,
    )

    requested_positions = np.asarray(
        requested_positions,
        dtype=np.int64,
    )

    n = len(
        builds
    )

    all_positions = np.arange(
        n,
        dtype=np.int64,
    )

    failure = (
        verdicts
        != SUCCESS_VERDICT_CODE
    ).astype(
        np.int64
    )

    assertion = (
        verdicts
        == ASSERTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    exception = (
        verdicts
        == EXCEPTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    transition = np.zeros(
        n,
        dtype=np.int64,
    )

    if n > 1:
        transition[
            1:
        ] = (
            verdicts[
                1:
            ]
            != verdicts[
                :-1
            ]
        ).astype(
            np.int64
        )

    duration_prefix = prefix_sum(
        durations
    )

    failure_prefix = prefix_sum(
        failure
    )

    assertion_prefix = prefix_sum(
        assertion
    )

    exception_prefix = prefix_sum(
        exception
    )

    transition_prefix = prefix_sum(
        transition
    )

    positions = requested_positions

    history_length = positions.astype(
        float
    )

    recent_start = np.maximum(
        0,
        positions - RECENT_WINDOW,
    )

    recent_length = (
        positions
        - recent_start
    ).astype(
        float
    )

    last_failure_inclusive = np.maximum.accumulate(
        np.where(
            failure > 0,
            all_positions,
            -1,
        )
    )

    last_transition_inclusive = np.maximum.accumulate(
        np.where(
            transition > 0,
            all_positions,
            -1,
        )
    )

    prior_failure_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    prior_transition_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    positive_history = positions > 0

    prior_failure_position[
        positive_history
    ] = last_failure_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    prior_transition_position[
        positive_history
    ] = last_transition_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    recent_max = np.full(
        n,
        np.nan,
        dtype=float,
    )

    for offset in range(
        1,
        RECENT_WINDOW + 1,
    ):
        if n <= offset:
            continue

        recent_max[
            offset:
        ] = np.fmax(
            recent_max[
                offset:
            ],
            durations[
                :-offset
            ],
        )

    total_max_inclusive = np.maximum.accumulate(
        durations
    )

    previous_indices = np.maximum(
        positions - 1,
        0,
    )

    reconstructed = {
        "REC_Age":
            (
                global_positions[
                    positions
                ]
                - global_positions[
                    0
                ]
            ).astype(
                float
            ),

        "REC_LastFailureAge":
            np.where(
                prior_failure_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_failure_position
                ).astype(
                    float
                ),
            ),

        "REC_LastTransitionAge":
            np.where(
                prior_transition_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_transition_position
                ).astype(
                    float
                ),
            ),

        "REC_RecentAvgExeTime":
            safe_divide(
                (
                    duration_prefix[
                        positions
                    ]
                    - duration_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentMaxExeTime":
            np.where(
                positive_history,
                recent_max[
                    positions
                ],
                -1.0,
            ),

        "REC_RecentFailRate":
            safe_divide(
                (
                    failure_prefix[
                        positions
                    ]
                    - failure_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentAssertRate":
            safe_divide(
                (
                    assertion_prefix[
                        positions
                    ]
                    - assertion_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentExcRate":
            safe_divide(
                (
                    exception_prefix[
                        positions
                    ]
                    - exception_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentTransitionRate":
            safe_divide(
                (
                    transition_prefix[
                        positions
                    ]
                    - transition_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_TotalAvgExeTime":
            safe_divide(
                duration_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalMaxExeTime":
            np.where(
                positive_history,
                total_max_inclusive[
                    previous_indices
                ],
                -1.0,
            ),

        "REC_TotalFailRate":
            safe_divide(
                failure_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalAssertRate":
            safe_divide(
                assertion_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalExcRate":
            safe_divide(
                exception_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalTransitionRate":
            safe_divide(
                transition_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_LastVerdict":
            np.where(
                positive_history,
                verdicts[
                    previous_indices
                ],
                -1,
            ).astype(
                float
            ),

        "REC_LastExeTime":
            np.where(
                positive_history,
                durations[
                    previous_indices
                ],
                -1.0,
            ),
    }

    file_failure_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    file_transition_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    failure_event_positions = np.flatnonzero(
        failure > 0
    )

    transition_event_positions = np.flatnonzero(
        transition > 0
    )

    failure_pointer = 0
    transition_pointer = 0

    prior_failure_builds = set()
    prior_transition_builds = set()

    requested_order = np.argsort(
        positions,
        kind="mergesort",
    )

    for requested_index in requested_order:
        current_position = int(
            positions[
                requested_index
            ]
        )

        while (
            failure_pointer
            < len(
                failure_event_positions
            )
            and int(
                failure_event_positions[
                    failure_pointer
                ]
            )
            < current_position
        ):
            prior_failure_builds.add(
                int(
                    builds[
                        failure_event_positions[
                            failure_pointer
                        ]
                    ]
                )
            )

            failure_pointer += 1

        while (
            transition_pointer
            < len(
                transition_event_positions
            )
            and int(
                transition_event_positions[
                    transition_pointer
                ]
            )
            < current_position
        ):
            prior_transition_builds.add(
                int(
                    builds[
                        transition_event_positions[
                            transition_pointer
                        ]
                    ]
                )
            )

            transition_pointer += 1

        current_build = int(
            builds[
                current_position
            ]
        )

        current_entities = changed_entities_by_build.get(
            current_build,
            frozenset(),
        )

        file_failure_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_failure_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

        file_transition_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_transition_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

    reconstructed[
        "REC_MaxTestFileFailRate"
    ] = file_failure_rate

    reconstructed[
        "REC_MaxTestFileTransitionRate"
    ] = file_transition_rate

    return (
        reconstructed,
        transition,
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 16 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2a_status = load_json(
    STEP2A_STATUS_PATH
)

step2a_report = load_json(
    STEP2A_REPORT_PATH
)

entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 16 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    selection.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 16 Step 1B is not frozen successfully."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or step2a_report.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or entity_mapping_summary.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):
    raise RuntimeError(
        "Project 16 Step 2A outputs are not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–15."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


if selection.get(
    "Project"
) != PROJECT_NAME or selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Frozen Project 16 identity differs."
    )


active_reservations = selection.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 16 active-reservation state differs."
    )


if selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 16 runtime-priority rule differs."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 16 source file is missing:\n"
            f"{source_path}"
        )

    current_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_manifest_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The local Project 16 source does not match "
        "the frozen source manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


chronology[
    "ChronologyOrder"
] = parse_int(
    chronology[
        "ChronologyOrder"
    ],
    "chronology.ChronologyOrder",
)


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "The frozen chronology contains invalid timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


timestamp_group_sizes = (
    chronology.groupby(
        "StartedAtUTC"
    )
    .size()
)


timestamp_tie_groups_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


if timestamp_tie_groups_count != 0:
    raise RuntimeError(
        "Project 16 unexpectedly contains timestamp ties. "
        "This no-tie reconstruction cell must not continue."
    )


timestamp_tie_groups_frame = pd.DataFrame(
    columns=[
        "TieGroup",
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsJSON",
        "PermutationCount",
    ]
)


build_chronology_map = chronology.set_index(
    "BuildID"
)[
    "ChronologyOrder"
].astype(
    int
).to_dict()


build_timestamp_map = chronology.set_index(
    "BuildID"
)[
    "StartedAtUTC"
].to_dict()


dataset_header = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    nrows=0,
).columns.tolist()


model_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

model_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

model_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    usecols=[
        model_build_column,
        model_test_column,
        model_verdict_column,
    ] + REC_FEATURES,
    low_memory=False,
)


dataset[
    model_build_column
] = parse_int(
    dataset[
        model_build_column
    ],
    "dataset.Build",
)


dataset[
    model_test_column
] = parse_int(
    dataset[
        model_test_column
    ],
    "dataset.Test",
)


dataset[
    model_verdict_column
] = parse_int(
    dataset[
        model_verdict_column
    ],
    "dataset.Verdict",
)


dataset = dataset.rename(
    columns={
        model_build_column:
            "Build",

        model_test_column:
            "Test",

        model_verdict_column:
            "Verdict",
    }
).reset_index(
    drop=True
)


dataset[
    "_ModelRow"
] = np.arange(
    len(
        dataset
    ),
    dtype=np.int64,
)


print(
    "Loading the 97,734-row clean execution history."
)


exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.verdict",
)


exe[
    exe_job_column
] = pd.to_numeric(
    exe[
        exe_job_column
    ],
    errors="coerce",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


if exe[
    exe_job_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing/non-numeric job values."
    )


if not np.isfinite(
    exe[
        exe_duration_column
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )


if exe[
    exe_duration_column
].lt(
    0
).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )


observed_verdict_codes = sorted(
    int(
        value
    )
    for value in exe[
        exe_verdict_column
    ].unique().tolist()
)


if not set(
    observed_verdict_codes
).issubset({
    0,
    1,
    2,
    3,
}):
    raise RuntimeError(
        "exe.csv contains an unsupported verdict code.\n"
        f"Observed codes: {observed_verdict_codes}"
    )


exe = exe.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "ChronologyOrder"
] = exe[
    "Build"
].map(
    build_chronology_map
)


if exe[
    "ChronologyOrder"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to frozen chronology."
    )


exe[
    "ChronologyOrder"
] = exe[
    "ChronologyOrder"
].astype(
    np.int64
)


raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_pairs != 0
    or model_duplicate_pairs != 0
):
    raise RuntimeError(
        "Duplicate Build-Test pairs prevent exact REC reconstruction."
    )


raw_build_ids = set(
    exe[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)


global_build_sequence = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            raw_build_ids
        )
    ]
    .sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )[
        "BuildID"
    ]
    .astype(
        int
    )
    .tolist()
)


global_build_position = {
    int(
        build_id
    ):
        position
    for position, build_id in enumerate(
        global_build_sequence
    )
}


exe[
    "GlobalBuildPosition"
] = exe[
    "Build"
].map(
    global_build_position
)


if exe[
    "GlobalBuildPosition"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to global first-appearance order."
    )


exe[
    "GlobalBuildPosition"
] = exe[
    "GlobalBuildPosition"
].astype(
    np.int64
)


print(
    "Sorting raw execution history by Test and frozen chronology."
)


sort_started = time.perf_counter()


exe = (
    exe.sort_values(
        [
            "Test",
            "ChronologyOrder",
            "Build",
            "Job",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exe[
    "InferredTestOrder"
] = (
    exe.groupby(
        "Test",
        sort=False,
    )
    .cumcount()
    .astype(
        np.int64
    )
)


sort_seconds = float(
    time.perf_counter()
    - sort_started
)


commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)


build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)


mapping_incomplete_source = pd.read_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    low_memory=False,
)


commit_audit[
    "BuildID"
] = parse_int(
    commit_audit[
        "BuildID"
    ],
    "commit_audit.BuildID",
)


build_entity[
    "BuildID"
] = parse_int(
    build_entity[
        "BuildID"
    ],
    "build_entity.BuildID",
)


build_entity[
    "EntityId"
] = parse_int(
    build_entity[
        "EntityId"
    ],
    "build_entity.EntityId",
)


mapping_incomplete_source[
    "BuildID"
] = parse_int(
    mapping_incomplete_source[
        "BuildID"
    ],
    "mapping_incomplete.BuildID",
)


mapping_incomplete_source_partitions = sorted(
    set(
        mapping_incomplete_source[
            "Partition"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )
)


mapping_incomplete_source_rows_with_entities = int(
    mapping_incomplete_source[
        "HasMappedEntities"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
    .sum()
)


normalised_match_type = (
    commit_audit[
        "MatchType"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


exact_match_mask = normalised_match_type.eq(
    "EXACT"
)

prefix_match_mask = normalised_match_type.eq(
    "UNIQUE_PREFIX"
)

unmatched_mask = normalised_match_type.eq(
    "UNMATCHED"
)

ambiguous_mask = normalised_match_type.eq(
    "AMBIGUOUS_PREFIX"
)


unknown_match_type_rows = int(
    (
        ~(
            exact_match_mask
            | prefix_match_mask
            | unmatched_mask
            | ambiguous_mask
        )
    ).sum()
)


if unknown_match_type_rows != 0:
    raise RuntimeError(
        "Commit audit contains unknown MatchType rows."
    )


exact_matches = int(
    exact_match_mask.sum()
)

prefix_matches = int(
    prefix_match_mask.sum()
)

unmatched_tokens = int(
    unmatched_mask.sum()
)

ambiguous_tokens = int(
    ambiguous_mask.sum()
)


unmatched_token_builds = sorted(
    commit_audit.loc[
        unmatched_mask,
        "BuildID",
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


mapping_incomplete_builds = sorted(
    set(
        unmatched_token_builds
    )
    | set(
        builds_without_entities
    )
)


changed_entities_by_build = {
    int(
        build_id
    ):
        frozenset(
            int(
                entity_id
            )
            for entity_id in values
        )
    for build_id, values in build_entity.groupby(
        "BuildID",
        sort=False,
    )[
        "EntityId"
    ]
}


entity_changed_builds_accumulator = defaultdict(
    set
)


for row in build_entity[
    [
        "BuildID",
        "EntityId",
    ]
].itertuples(
    index=False
):
    entity_changed_builds_accumulator[
        int(
            row.EntityId
        )
    ].add(
        int(
            row.BuildID
        )
    )


entity_changed_builds = {
    entity_id:
        frozenset(
            build_ids
        )
    for entity_id, build_ids in entity_changed_builds_accumulator.items()
}


del entity_changed_builds_accumulator
gc.collect()


raw_build_counts = exe.groupby(
    "Build",
    sort=False,
).size()


raw_build_failures = (
    exe[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        exe[
            "Build"
        ]
    )
    .sum()
)


model_build_counts = dataset.groupby(
    "Build",
    sort=False,
).size()


model_build_failures = (
    dataset[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        dataset[
            "Build"
        ]
    )
    .sum()
)


unmatched_mapping_audit = (
    mapping_incomplete_source.copy()
    .sort_values(
        "BuildID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


unmatched_mapping_audit[
    "RawExecutionRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "RawFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_failures
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelReadyRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_failures
).fillna(
    0
).astype(
    int
)


print(
    "\nTimestamp tie groups:"
)

display(
    timestamp_tie_groups_frame
)


print(
    "\nUnmatched mapping audit:"
)

display(
    unmatched_mapping_audit
)


# --------------------------------------------------------------------------------------------------
# 6. VECTORIZED PER-TEST RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

reconstruction_started = time.perf_counter()


model_group_indices = dataset.groupby(
    "Test",
    sort=False,
).indices


model_build_array = dataset[
    "Build"
].to_numpy(
    dtype=np.int64
)


result_arrays = {
    feature:
        np.full(
            len(
                dataset
            ),
            np.nan,
            dtype=np.float64,
        )
    for feature in REC_FEATURES
}


filled_model_rows = np.zeros(
    len(
        dataset
    ),
    dtype=bool,
)


raw_test_array = exe[
    "Test"
].to_numpy(
    dtype=np.int64
)


test_starts = np.concatenate([
    np.array(
        [
            0,
        ],
        dtype=np.int64,
    ),

    (
        np.flatnonzero(
            raw_test_array[
                1:
            ]
            != raw_test_array[
                :-1
            ]
        )
        + 1
    ).astype(
        np.int64
    ),

    np.array(
        [
            len(
                exe
            ),
        ],
        dtype=np.int64,
    ),
])


test_order_search_records = []


raw_build_array = exe[
    "Build"
].to_numpy(
    dtype=np.int64
)

raw_verdict_array = exe[
    "Verdict"
].to_numpy(
    dtype=np.int64
)

raw_duration_array = exe[
    "Duration"
].to_numpy(
    dtype=np.float64
)

raw_global_position_array = exe[
    "GlobalBuildPosition"
].to_numpy(
    dtype=np.int64
)


total_tests = len(
    test_starts
) - 1


for test_number in range(
    total_tests
):
    start = int(
        test_starts[
            test_number
        ]
    )

    end = int(
        test_starts[
            test_number
            + 1
        ]
    )

    test_id = int(
        raw_test_array[
            start
        ]
    )

    raw_rows_for_test = end - start

    model_rows = model_group_indices.get(
        test_id
    )

    if model_rows is None:
        test_order_search_records.append({
            "Test":
                test_id,

            "RawExecutionRows":
                raw_rows_for_test,

            "ModelReadyRows":
                0,

            "TimestampTieGroupsForTest":
                0,

            "CandidateOrderCombinations":
                1,

            "MinimumMismatchValues":
                0,

            "ZeroMismatchCandidates":
                1,

            "BestMismatchCountsJSON":
                json.dumps(
                    {},
                    sort_keys=True,
                ),

            "SearchMode":
                "RAW_ONLY_TEST_NO_TIE_DIRECT_ORDER",
        })

        continue

    model_rows = np.asarray(
        model_rows,
        dtype=np.int64,
    )

    requested_builds = model_build_array[
        model_rows
    ]

    group_builds = raw_build_array[
        start:end
    ]

    position_by_build = {
        int(
            build_id
        ):
            position
        for position, build_id in enumerate(
            group_builds
        )
    }

    missing_requested_builds = [
        int(
            build_id
        )
        for build_id in requested_builds
        if int(
            build_id
        )
        not in position_by_build
    ]

    if missing_requested_builds:
        raise RuntimeError(
            "A model-ready test contains Build rows missing from raw history.\n"
            f"Test={test_id}; "
            f"missing sample={missing_requested_builds[:20]}"
        )

    requested_positions = np.array([
        position_by_build[
            int(
                build_id
            )
        ]
        for build_id in requested_builds
    ], dtype=np.int64)

    (
        reconstructed_group,
        transition_group,
    ) = reconstruct_requested_group_features(
        builds=group_builds,
        verdicts=raw_verdict_array[
            start:end
        ],
        durations=raw_duration_array[
            start:end
        ],
        global_positions=raw_global_position_array[
            start:end
        ],
        requested_positions=requested_positions,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )

    for feature in REC_FEATURES:
        result_arrays[
            feature
        ][
            model_rows
        ] = reconstructed_group[
            feature
        ]

    filled_model_rows[
        model_rows
    ] = True

    test_order_search_records.append({
        "Test":
            test_id,

        "RawExecutionRows":
            raw_rows_for_test,

        "ModelReadyRows":
            len(
                model_rows
            ),

        "TimestampTieGroupsForTest":
            0,

        "CandidateOrderCombinations":
            1,

        "MinimumMismatchValues":
            0,

        "ZeroMismatchCandidates":
            1,

        "BestMismatchCountsJSON":
            json.dumps(
                {},
                sort_keys=True,
            ),

        "SearchMode":
            "MODEL_READY_TEST_NO_TIE_DIRECT_ORDER",
    })

    if (
        (
            test_number
            + 1
        )
        % 500
        == 0
        or (
            test_number
            + 1
        )
        == total_tests
    ):
        print(
            "Vectorized REC reconstruction progress:",
            test_number + 1,
            "/",
            total_tests,
            "tests | reconstructed rows:",
            int(
                filled_model_rows.sum()
            ),
        )


reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)


if not filled_model_rows.all():
    missing_model_rows = np.flatnonzero(
        ~filled_model_rows
    )

    raise RuntimeError(
        "Clean REC reconstruction did not fill every model-ready row.\n"
        f"Missing rows: {len(missing_model_rows)}; "
        f"sample={missing_model_rows[:20].tolist()}"
    )


clean_reconstructed = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


for feature in REC_FEATURES:
    clean_reconstructed[
        feature
    ] = result_arrays[
        feature
    ]


test_order_search_audit = pd.DataFrame(
    test_order_search_records
)


model_ready_tests = int(
    test_order_search_audit[
        "ModelReadyRows"
    ].gt(
        0
    ).sum()
)


raw_only_tests = int(
    test_order_search_audit[
        "ModelReadyRows"
    ].eq(
        0
    ).sum()
)


tests_with_timestamp_ties = int(
    test_order_search_audit[
        "TimestampTieGroupsForTest"
    ].gt(
        0
    ).sum()
)


tests_with_nonzero_order_mismatches = int(
    test_order_search_audit[
        "MinimumMismatchValues"
    ].gt(
        0
    ).sum()
)


tests_with_ambiguous_zero_orders = int(
    test_order_search_audit[
        "ZeroMismatchCandidates"
    ].gt(
        1
    ).sum()
)


total_test_order_mismatch_values = int(
    test_order_search_audit[
        "MinimumMismatchValues"
    ].sum()
)


print(
    "\nVectorized reconstruction summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Tests",

            "Value":
                total_tests,
        },

        {
            "Metric":
                "Model-ready tests",

            "Value":
                model_ready_tests,
        },

        {
            "Metric":
                "Raw-only tests",

            "Value":
                raw_only_tests,
        },

        {
            "Metric":
                "Tests touching timestamp ties",

            "Value":
                tests_with_timestamp_ties,
        },

        {
            "Metric":
                "Reconstructed model rows",

            "Value":
                int(
                    filled_model_rows.sum()
                ),
        },

        {
            "Metric":
                "Raw sort seconds",

            "Value":
                sort_seconds,
        },

        {
            "Metric":
                "REC reconstruction seconds",

            "Value":
                reconstruction_seconds,
        },
    ])
)


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL BUILD ORDER AND DIRECT CLEAN COMPARISON
# --------------------------------------------------------------------------------------------------

frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder":
        np.arange(
            1,
            len(
                global_build_sequence
            )
            + 1,
            dtype=np.int64,
        ),

    "BuildID":
        global_build_sequence,
})


frozen_global_build_order[
    "StartedAtUTC"
] = frozen_global_build_order[
    "BuildID"
].map(
    build_timestamp_map
)


reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


missing_reconstructed_rows = int(
    clean_reconstructed[
        REC_FEATURES
    ].isna().any(
        axis=1
    ).sum()
)


if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )


if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction contains missing values."
    )


comparison_records = []
mismatch_examples = []


anchor_offsets = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)


rows_at_mapping_incomplete_build = dataset[
    "Build"
].isin(
    mapping_incomplete_build_set
).to_numpy()


for feature in REC_FEATURES:
    original_values = dataset[
        feature
    ].to_numpy(
        dtype=float
    )

    reconstructed_values = clean_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(
            original_values
        ).all()
        or not np.isfinite(
            reconstructed_values
        ).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = (
        ~direct_match_mask
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[
        feature
    ] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in FILE_HISTORY_REC,

        "Rows":
            len(
                dataset
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            int(
                direct_mismatch_mask.sum()
            ),

        "DirectMismatchesAtMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & rows_at_mapping_incomplete_build
                ).sum()
            ),

        "DirectMismatchesOutsideMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & (
                        ~rows_at_mapping_incomplete_build
                    )
                ).sum()
            ),

        "NonZeroAnchorOffsets":
            int(
                (
                    direct_difference
                    != 0
                ).sum()
            ),

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_match_mask
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MaximumAbsoluteAnchoredDifference":
            float(
                np.max(
                    np.abs(
                        original_values
                        - anchored_values
                    )
                )
            ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[
        :20
    ]

    for mismatch_index in mismatch_indices:
        mismatch_examples.append({
            "Build":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Build"
                    ]
                ),

            "Test":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Test"
                    ]
                ),

            "Feature":
                feature,

            "Original":
                float(
                    original_values[
                        mismatch_index
                    ]
                ),

            "Reconstructed":
                float(
                    reconstructed_values[
                        mismatch_index
                    ]
                ),

            "Difference":
                float(
                    direct_difference[
                        mismatch_index
                    ]
                ),

            "MappingIncompleteBuild":
                bool(
                    rows_at_mapping_incomplete_build[
                        mismatch_index
                    ]
                ),
        })


comparison_summary = pd.DataFrame(
    comparison_records
)


mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "Original",
        "Reconstructed",
        "Difference",
        "MappingIncompleteBuild",
    ],
)


anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows":
            "MatchingRows",

        "AnchoredMismatchingRows":
            "MismatchingRows",
    }
)


anchor_validation[
    "Pass"
] = anchor_validation[
    "MismatchingRows"
].eq(
    0
)


direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)


verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_DEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_INDEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


anchored_mismatch_values = int(
    anchor_validation[
        "MismatchingRows"
    ].sum()
)


nonzero_anchor_offset_values = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).sum()
)


rows_with_any_nonzero_anchor_offset = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).any(
        axis=1
    ).sum()
)


unmatched_mapping_effect_is_confined = bool(
    non_file_direct_mismatches == 0
    and file_mismatches_outside_mapping_incomplete_build == 0
)


zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)


age_mismatch_rows = int(
    comparison_summary.loc[
        comparison_summary[
            "Feature"
        ].eq(
            "REC_Age"
        ),
        "DirectMismatchingRows",
    ].iloc[
        0
    ]
)


global_age_order_search = pd.DataFrame([
    {
        "Candidate":
            1,

        "AgeMismatchRows":
            age_mismatch_rows,

        "BuildOrderSHA256":
            hashlib.sha256(
                ",".join(
                    str(
                        build_id
                    )
                    for build_id in global_build_sequence
                ).encode(
                    "utf-8"
                )
            ).hexdigest(),

        "TieOrdersJSON":
            "[]",
    },
])


global_age_combination_count = 1
zero_age_candidates = int(
    age_mismatch_rows == 0
)
best_age_mismatches = age_mismatch_rows


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

raw_train_mask = exe[
    "Build"
].isin(
    training_builds
)


raw_eval_mask = exe[
    "Build"
].isin(
    evaluation_builds
)


model_train_mask = dataset[
    "Build"
].isin(
    training_builds
)


model_eval_mask = dataset[
    "Build"
].isin(
    evaluation_builds
)


validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)


add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get(
        "Status"
    ),
    step2a_status.get(
        "Status"
    )
    == EXPECTED_STEP2A_STATUS,
)


add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)


add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)


add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_groups_count,
    timestamp_tie_groups_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)


add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    int(
        raw_train_mask.sum()
    ),
    int(
        raw_train_mask.sum()
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    int(
        raw_eval_mask.sum()
    ),
    int(
        raw_eval_mask.sum()
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    int(
        model_train_mask.sum()
    ),
    int(
        model_train_mask.sum()
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    int(
        model_eval_mask.sum()
    ),
    int(
        model_eval_mask.sum()
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    )
    == EXPECTED_DATASET_COLUMNS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_pairs,
    raw_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_pairs,
    model_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Official assertion verdict code",
    2,
    ASSERTION_VERDICT_CODE,
    ASSERTION_VERDICT_CODE
    == 2,
)


add_check(
    validation_records,
    "Official exception verdict code",
    1,
    EXCEPTION_VERDICT_CODE,
    EXCEPTION_VERDICT_CODE
    == 1,
)


add_check(
    validation_records,
    "Per-test no-tie accounting",
    total_tests,
    model_ready_tests
    + raw_only_tests,
    (
        model_ready_tests
        + raw_only_tests
    )
    == total_tests,
)


add_check(
    validation_records,
    "Tests touching timestamp ties",
    0,
    tests_with_timestamp_ties,
    tests_with_timestamp_ties
    == 0,
)


add_check(
    validation_records,
    "Tests with non-zero order mismatches",
    0,
    tests_with_nonzero_order_mismatches,
    tests_with_nonzero_order_mismatches
    == 0,
)


add_check(
    validation_records,
    "Total order mismatch values",
    0,
    total_test_order_mismatch_values,
    total_test_order_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age mismatch rows",
    0,
    best_age_mismatches,
    best_age_mismatches
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age zero-match candidates",
    1,
    zero_age_candidates,
    zero_age_candidates
    == 1,
)


add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(
        commit_audit
    ),
    len(
        commit_audit
    )
    == EXPECTED_COMMIT_TOKEN_ROWS,
)


add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unique-prefix matches",
    EXPECTED_PREFIX_COMMIT_MATCHES,
    prefix_matches,
    prefix_matches
    == EXPECTED_PREFIX_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(
        builds_with_entities
    ),
    len(
        builds_with_entities
    )
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(
        builds_without_entities
    ),
    len(
        builds_without_entities
    )
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Mapping-incomplete build identities",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    mapping_incomplete_builds,
    set(
        mapping_incomplete_builds
    )
    == EXPECTED_MAPPING_INCOMPLETE_BUILDS,
)


add_check(
    validation_records,
    "Mapping-incomplete source rows",
    len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    len(
        mapping_incomplete_source
    ),
    len(
        mapping_incomplete_source
    )
    == len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
)


add_check(
    validation_records,
    "Mapping-incomplete partitions",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    ),
    mapping_incomplete_source_partitions,
    set(
        mapping_incomplete_source_partitions
    )
    == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS,
)


add_check(
    validation_records,
    "Mapping-incomplete rows with mapped entities",
    EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
    mapping_incomplete_source_rows_with_entities,
    mapping_incomplete_source_rows_with_entities
    == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
)


add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(
        build_entity
    ),
    len(
        build_entity
    )
    == EXPECTED_BUILD_ENTITY_ROWS,
)


add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(
        clean_reconstructed
    ),
    len(
        clean_reconstructed
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows
    == 0,
)


add_check(
    validation_records,
    "Missing reconstructed values",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows
    == 0,
)


add_check(
    validation_records,
    "Non-file direct mismatch values",
    0,
    non_file_direct_mismatches,
    non_file_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build
    == 0,
)


add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)


add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features
    == 0,
)


add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection.get(
        "RuntimePriorityRule"
    ),
    selection.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    validation_records,
    "Project 16 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 16 Step 2B validation:"
)

display(
    validation
)


print(
    "\nClean REC comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor validation:"
)

display(
    anchor_validation
)


if not failed_validation.empty:
    print(
        "\nFailed Step 2B checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 2B PASS checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 16 STEP 2B VALIDATION FAILED. "
        "DO NOT START THE EXPERIMENT."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


exe[
    "StartedAtUTC"
] = exe[
    "Build"
].map(
    build_timestamp_map
)


inferred_execution_order_for_storage = (
    exe[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "StartedAtUTC",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)


atomic_csv(
    TIMESTAMP_TIE_GROUPS_PATH,
    timestamp_tie_groups_frame,
)


atomic_csv(
    TEST_ORDER_SEARCH_AUDIT_PATH,
    test_order_search_audit,
)


print(
    "\nWriting the frozen 97,734-row execution-order parquet."
)


atomic_parquet(
    INFERRED_EXECUTION_ORDER_PATH,
    inferred_execution_order_for_storage,
)


atomic_csv(
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    global_age_order_search,
)


atomic_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    frozen_global_build_order,
)


atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)


atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)


atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

execution_order_metadata = pq.ParquetFile(
    INFERRED_EXECUTION_ORDER_PATH
)


execution_order_readback_rows = int(
    execution_order_metadata.metadata.num_rows
)


execution_order_readback_columns = set(
    execution_order_metadata.schema.names
)


required_execution_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if (
    execution_order_readback_rows
    != EXPECTED_RAW_ROWS
    or not required_execution_order_columns.issubset(
        execution_order_readback_columns
    )
):
    raise RuntimeError(
        "Frozen execution-order parquet metadata readback failed."
    )


reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if len(
    reconstructed_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )


readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


readback_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original_values = readback_join[
        feature
    ].to_numpy(
        dtype=float
    )

    readback_mismatch_values += int(
        (
            ~np.isclose(
                reproduced_values,
                original_values,
                rtol=ANCHOR_RTOL,
                atol=ANCHOR_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH,
    TIMESTAMP_TIE_GROUPS_PATH,
    TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "OfficialVerdictSemantics": {
        "Success":
            SUCCESS_VERDICT_CODE,

        "Exception":
            EXCEPTION_VERDICT_CODE,

        "Assertion":
            ASSERTION_VERDICT_CODE,
    },

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "ModelReadyTests":
        model_ready_tests,

    "RawOnlyTests":
        raw_only_tests,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "TestsWithMultipleZeroMismatchOrders":
        tests_with_ambiguous_zero_orders,

    "GlobalAgeOrderCombinations":
        global_age_combination_count,

    "GlobalAgeZeroMismatchCandidates":
        zero_age_candidates,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "RawSortSeconds":
        sort_seconds,

    "RECReconstructionSeconds":
        reconstruction_seconds,

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "RawExecutionRows":
        len(
            inferred_execution_order_for_storage
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "ReconstructedRows":
        len(
            clean_reconstructed
        ),

    "GlobalBuildOrderRows":
        len(
            frozen_global_build_order
        ),

    "CommitTokenRows":
        len(
            commit_audit
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        mapping_incomplete_builds,

    "MappingIncompletePartitions":
        mapping_incomplete_source_partitions,

    "MappingIncompleteRowsWithMappedEntities":
        mapping_incomplete_source_rows_with_entities,

    "DirectMismatchValues":
        direct_mismatch_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_anchor_offset,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_offset_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ReadbackMismatchValues":
        readback_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        active_reservations,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_16_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        rec_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 16 Step 2B."
    )


final_source_manifest_records = []

for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_manifest_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 16 source changed during Step 2B."
    )


checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP2B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP2B_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP2B_STATUS
):
    raise RuntimeError(
        "Project 16 Step 2B checkpoint/status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 16 CELL 5 / STEP 2B RESULT ===")
print("=" * 136)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Active reservations:",
    active_reservations,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)


print(
    "\nOfficial verdict semantics:"
)

print(
    "Success:",
    SUCCESS_VERDICT_CODE,
)

print(
    "Exception:",
    EXCEPTION_VERDICT_CODE,
)

print(
    "Assertion:",
    ASSERTION_VERDICT_CODE,
)


print(
    "\nNo-tie execution-order freeze:"
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups_count,
)

print(
    "Raw execution-order rows:",
    execution_order_readback_rows,
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Tests:",
    total_tests,
)

print(
    "Model-ready tests:",
    model_ready_tests,
)

print(
    "Raw-only tests:",
    raw_only_tests,
)

print(
    "Tests with non-zero order mismatches:",
    tests_with_nonzero_order_mismatches,
)

print(
    "Global REC_Age mismatch rows:",
    best_age_mismatches,
)


print(
    "\nClean REC reconstruction:"
)

print(
    "Raw history rows:",
    len(
        inferred_execution_order_for_storage
    ),
)

print(
    "Model rows requested/reconstructed:",
    len(
        dataset
    ),
    "/",
    len(
        clean_reconstructed
    ),
)

print(
    "Direct mismatch values:",
    direct_mismatch_values,
)

print(
    "Non-file direct mismatch values:",
    non_file_direct_mismatches,
)

print(
    "File-history direct mismatch values:",
    file_history_direct_mismatches,
)

print(
    "File-history mismatches outside mapping-incomplete builds:",
    file_mismatches_outside_mapping_incomplete_build,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_anchor_offset,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_offset_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatch values:",
    anchored_mismatch_values,
)

print(
    "Readback mismatch values:",
    readback_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    zero_percent_clean_reproduced_exactly,
)


print(
    "\nMapping audit:"
)

print(
    "Commit-token rows:",
    len(
        commit_audit
    ),
)

print(
    "Exact / prefix / unmatched / ambiguous:",
    exact_matches,
    "/",
    prefix_matches,
    "/",
    unmatched_tokens,
    "/",
    ambiguous_tokens,
)

print(
    "Builds with / without mapped entities:",
    len(
        builds_with_entities
    ),
    "/",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Mapping-incomplete partitions:",
    mapping_incomplete_source_partitions,
)

print(
    "Mapping-incomplete rows with mapped entities:",
    mapping_incomplete_source_rows_with_entities,
)

print(
    "Unmatched mapping effect confined:",
    unmatched_mapping_effect_is_confined,
)


print(
    "\nRuntime:"
)

print(
    "Raw sort seconds:",
    round(
        sort_seconds,
        2,
    ),
)

print(
    "REC reconstruction seconds:",
    round(
        reconstruction_seconds,
        2,
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–15 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nREC reconstruction checkpoint:"
)

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    rec_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP2B_STATUS,
)

print("=" * 136)


=== PROJECT 16 CELL 5 / STEP 2B: VECTORIZED CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===
Loading the 97,734-row clean execution history.
Sorting raw execution history by Test and frozen chronology.

Timestamp tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount



Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows
0,712125832,88,TRAIN,1,0,False,195,0,0,0
1,747593450,175,TRAIN,1,0,False,198,0,0,0
2,748265487,176,TRAIN,1,0,False,198,0,0,0
3,771761066,348,TRAIN,1,0,False,176,0,0,0
4,771763616,349,TRAIN,1,0,False,203,0,0,0
5,771767101,350,TRAIN,1,0,False,203,0,0,0
6,771873583,365,TRAIN,1,0,False,203,0,0,0
7,773450607,399,TRAIN,1,0,False,204,0,0,0



Vectorized reconstruction summary:


...


Clean REC comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,DirectMismatchesAtMappingIncompleteBuild,DirectMismatchesOutsideMappingIncompleteBuild,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference,MaximumAbsoluteAnchoredDifference
0,REC_Age,VERDICT_INDEPENDENT,False,6548,6548,0,0,0,0,6548,0,0.000000e+00,0.000000e+00,0.0
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,6548,6548,0,0,0,0,6548,0,0.000000e+00,0.000000e+00,0.0
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,6548,6548,0,0,0,0,6548,0,0.000000e+00,0.000000e+00,0.0
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,6548,6548,0,0,0,1081,6548,0,2.910383e-11,1.423996e-13,0.0
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,6548,6548,0,0,0,0,6548,0,0.000000e+00,0.000000e+00,0.0
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,6548,6548,0,0,0,37,6548,0,5.551115e-17,3.136702e-19,0.0
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,6548,6548,0,0,0,12,6548,0,5.551115e-17,1.017309e-19,0.0
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,6548,6548,0,0,0,25,6548,0,5.551115e-17,2.119393e-19,0.0
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,6548,6548,0,0,0,16,6548,0,5.551115e-17,1.356412e-19,0.0
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,6548,6548,0,0,0,933,6548,0,1.455192e-11,1.067023e-13,0.0



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,6548,6548,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,6548,6548,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,6548,6548,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,6548,6548,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,6548,6548,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,6548,6548,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,6548,6548,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,6548,6548,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,6548,6548,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,6548,6548,0,0.0,True



Writing the frozen 97,734-row execution-order parquet.


=== PROJECT 16 CELL 5 / STEP 2B RESULT ===
Project: apache@rocketmq
Project slug: apache__rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Source root SHA-256: e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e

Official verdict semantics:
Success: 0
Exception: 1
Assertion: 2

No-tie execution-order freeze:
Timestamp tie groups: 0
Raw execution-order rows: 97734
Global build-order rows: 536
Tests: 219
Model-ready tests: 216
Raw-only tests: 3
Tests with non-zero order mismatches: 0
Global REC_Age mismatch rows: 0

Clean REC reconstruction:
Raw history rows: 97734
Model rows requested/reconstructed: 6548 / 6548


In [6]:
# ==================================================================================================
# PROJECT 16 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN AND COHORT FREEZE
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# PURPOSE:
# - verify the frozen Project 16 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the Project 16 failure-subtype distribution;
# - generate deterministic project/seed random streams for label-noise injection;
# - prove nested masks across all noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the evaluation partition clean and immutable;
# - perform no model fitting and no registry write.
#
# SAFETY:
# - Projects 1–15 must remain COMPLETE_AND_FROZEN and unchanged;
# - Project 16 must remain absent from the completion registry;
# - no prior-project condition output is accessed or modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 16 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_16_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_16_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "b3cf69922236c45ae62599e141d0d29f0c3737a334370daffcbfa709abff250a"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "71a43f55bc825d93abfdf6e4285174b41eed5c3dcee48f818a15a6193cb33db0"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 15

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 10_163_815

EXPECTED_BUILDS = 536
EXPECTED_TRAIN_BUILDS = 402
EXPECTED_EVAL_BUILDS = 134

EXPECTED_RAW_ROWS = 97_734
EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_RAW_TRAIN_FAILURES = 106
EXPECTED_RAW_EVAL_FAILURES = 9

EXPECTED_MODEL_ROWS = 6_548
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 8

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_16_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_16_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_16_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target_path = str(
        Path(
            path
        )
    )

    matches = [
        entry
        for entry in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            entry.get(
                "Path",
                "",
            )
        ) == target_path
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "The Project 16 REC checkpoint does not contain exactly "
            f"one manifest entry for {target_path}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR / "dataset.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 16 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 16 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 16 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 16 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 16 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


expected_rec_freeze_flags = {
    # The implementation label is frozen exactly as written by Project 16 Step 2B.
    "ImplementationVersion":
        "PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WITH_RAW_ONLY_TEST_HANDLING",

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_16_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    if rec_checkpoint.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "The Project 16 REC checkpoint does not match the frozen "
            f"Step 2B contract: {flag_name}={expected_value!r}."
        )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 16 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–15."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


selection_active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    None,
)


if selection_active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 16 selection checkpoint active reservations differ."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 16 selection checkpoint runtime-priority rule differs."
    )


if rec_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 16 REC checkpoint active reservations differ."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 16 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


expected_inferred_execution_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    INFERRED_EXECUTION_ORDER_PATH,
)

expected_global_build_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
)

expected_clean_reconstructed_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_RECONSTRUCTED_PATH,
)

expected_clean_anchor_offsets_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_ANCHOR_OFFSETS_PATH,
)

actual_inferred_execution_order_sha256 = sha256_file(
    INFERRED_EXECUTION_ORDER_PATH
)

actual_global_build_order_sha256 = sha256_file(
    FROZEN_GLOBAL_BUILD_ORDER_PATH
)

actual_clean_reconstructed_sha256 = sha256_file(
    CLEAN_RECONSTRUCTED_PATH
)

actual_clean_anchor_offsets_sha256 = sha256_file(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    actual_inferred_execution_order_sha256
    != expected_inferred_execution_order_sha256
    or actual_global_build_order_sha256
    != expected_global_build_order_sha256
    or actual_clean_reconstructed_sha256
    != expected_clean_reconstructed_sha256
    or actual_clean_anchor_offsets_sha256
    != expected_clean_anchor_offsets_sha256
):
    raise RuntimeError(
        "One or more frozen Project 16 Step 2B artifacts "
        "do not match the REC checkpoint manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, MODEL DATA, AND THE FROZEN V6 RAW EXECUTION ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


# V6 froze the exact per-test execution order required to reproduce all 19 REC features.
# This is the canonical raw-history cohort for every Project 16 noise condition.
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


missing_inferred_columns = (
    required_inferred_columns
    - set(
        inferred_execution_order.columns
    )
)


if missing_inferred_columns:
    raise RuntimeError(
        "The frozen V6 inferred execution-order file is missing columns:\n"
        + "\n".join(
            sorted(
                missing_inferred_columns
            )
        )
    )


inferred_execution_order[
    "Build"
] = parse_int(
    inferred_execution_order[
        "Build"
    ],
    "inferred_execution_order.Build",
)

inferred_execution_order[
    "Test"
] = parse_int(
    inferred_execution_order[
        "Test"
    ],
    "inferred_execution_order.Test",
)

inferred_execution_order[
    "Verdict"
] = parse_int(
    inferred_execution_order[
        "Verdict"
    ],
    "inferred_execution_order.Verdict",
)

inferred_execution_order[
    "InferredTestOrder"
] = parse_int(
    inferred_execution_order[
        "InferredTestOrder"
    ],
    "inferred_execution_order.InferredTestOrder",
)

inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)

inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "The frozen raw execution order contains missing/non-numeric "
        "job or duration values."
    )


if not np.isfinite(
    inferred_execution_order[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "The frozen raw execution order contains non-finite durations."
    )


if inferred_execution_order[
    "Duration"
].lt(
    0
).any():
    raise RuntimeError(
        "The frozen raw execution order contains negative durations."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_build_test_rows
    or raw_duplicate_test_order_rows
):
    raise RuntimeError(
        "The frozen V6 execution order contains duplicate keys."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "The frozen V6 global build-order file is missing required columns."
    )


frozen_global_build_order[
    "GlobalBuildOrder"
] = parse_int(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order[
    "BuildID"
] = parse_int(
    frozen_global_build_order[
        "BuildID"
    ],
    "frozen_global_build_order.BuildID",
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            1,
            EXPECTED_BUILDS
            + 1,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "The frozen V6 global build order is invalid."
    )


# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_valid = bool(
    failure_subtypes.astype(
        int
    ).tolist()
    == [
        1,
        2,
    ]
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 16 clean training failures do not use exactly "
        "the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 16 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "fixed chronological split: started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "RawHistoryOrder":
        (
            "Project 16 Step 2B frozen per-test execution order; "
            "timestamp-tie order inferred from exact clean REC reproduction"
        ),

    "GlobalRECAgeBuildOrder":
        (
            "Project 16 Step 2B frozen global build first-appearance order"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint implementation",
    "PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WITH_RAW_ONLY_TEST_HANDLING",
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == "PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WITH_RAW_ONLY_TEST_HANDLING",
)

add_check(
    validation_records,
    "REC checkpoint schema version",
    1,
    rec_checkpoint.get(
        "CheckpointVersion"
    ),
    rec_checkpoint.get(
        "CheckpointVersion"
    )
    == 1,
)

add_check(
    validation_records,
    "Frozen inferred execution-order SHA-256",
    expected_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256
    == expected_inferred_execution_order_sha256,
)

add_check(
    validation_records,
    "Frozen global build-order SHA-256",
    expected_global_build_order_sha256,
    actual_global_build_order_sha256,
    actual_global_build_order_sha256
    == expected_global_build_order_sha256,
)

add_check(
    validation_records,
    "Frozen clean reconstruction SHA-256",
    expected_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256
    == expected_clean_reconstructed_sha256,
)

add_check(
    validation_records,
    "Frozen clean anchor-offset SHA-256",
    expected_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256
    == expected_clean_anchor_offsets_sha256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Failure subtype values",
    [
        1,
        2,
    ],
    failure_subtypes.astype(
        int
    ).tolist(),
    failure_subtype_values_valid,
)

add_check(
    validation_records,
    "Failure subtype profile sum",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    ) == EXPECTED_REGISTERED_PROJECTS,
)

for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project == required_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_active_reservations,
    selection_active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Project 16 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 16 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 16 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        "Project 16 Step 2B exact per-test inferred order",

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 16 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 16 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 16 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 16 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 16 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print("\nFrozen clean-history order:")

print(
    "Inferred execution-order rows:",
    len(
        exe
    ),
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Raw duplicate Test-order rows:",
    raw_duplicate_test_order_rows,
)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 16 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–15 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)


=== PROJECT 16 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 16 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_16_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_16_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,b3cf69922236c45ae62599e141d0d29f0c3737a334370d...,b3cf69922236c45ae62599e141d0d29f0c3737a334370d...,True
3,REC checkpoint SHA-256,71a43f55bc825d93abfdf6e4285174b41eed5c3dcee48f...,71a43f55bc825d93abfdf6e4285174b41eed5c3dcee48f...,True
4,REC checkpoint implementation,PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WIT...,PROJECT_16_V1_NO_TIMESTAMP_TIES_VECTORIZED_WIT...,True
...,...,...,...,...
66,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
67,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
68,Active reservations,[],[],True
69,Project 16 registry rows,0,0,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,69,0.650943
1,2,37,0.349057



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,3194741083,3744228384,71052,0,...,0,106,106,4907,0,105,105,62527d52b815d39b23d8a7aeaad105e6f64c32fef01d62...,0922787f6597840f586614f4f695d24302075a195f910c...,3cc0b5417b4ccdaaa4ec71ce19456eb7a9bd3381b720f3...
1,2,noise_05__seed_01,1,2,5,1,3194741083,3744228384,71052,3581,...,4,106,3679,4907,241,105,338,fd72ed651f7bbdd56e27d2a427e0f7e47ab5e0fa95a5e4...,52832127a81e17f5879edb3182c4848e822f77544291f4...,e1a2dcbc9bddf03b7089736aa6b62f8555c5edd3ff8591...
2,3,noise_10__seed_01,1,3,10,1,3194741083,3744228384,71052,7153,...,8,106,7243,4907,494,105,583,42a8570cfa8c8392dd75c8d6197aabc03c28d16cf98fcc...,8980b167efb349cf8544c235fb118c851c675c48c2f4d4...,a5eb8208ad23989bcd583dc93a141c97e13dc789e2ae60...
3,4,noise_15__seed_01,1,4,15,1,3194741083,3744228384,71052,10579,...,12,106,10661,4907,750,105,833,1d3f5f9a3d725c0abc49311214bf242bc0e8bafae0b300...,7f40e652d8233cf7fa8bfd419f3c6f406ed76c08734008...,f39e1b271ec50b40eef5dc80b43ced6141aa3f9822c396...
4,5,noise_20__seed_01,1,5,20,1,3194741083,3744228384,71052,14005,...,13,106,14085,4907,978,105,1059,fa8af91501f1157f02221a2c2f4b1082c49be221f0f7ff...,da6f1ad474ef309366760646bd8997b5fe224c9e1e3001...,8b064ec845fa7173e3437e34e96f57d4655c31f9d67018...
5,6,noise_25__seed_01,1,6,25,1,3194741083,3744228384,71052,17623,...,18,106,17693,4907,1221,105,1292,7799fed5d48d92b2ffc6edb9fdbc8e5ec15d04d46a965a...,6a1061bb9d9bd5bcbc4f43d807c3826837c0a4bc169e75...,f79a92262f9e091dc49562c080a7ab77ce21a62ac2b8ce...
6,7,noise_30__seed_01,1,7,30,1,3194741083,3744228384,71052,21187,...,20,106,21253,4907,1439,105,1506,b5ad76a702c598d7104f7dfa43c6c69692251f0f8d15ad...,ed29652b458a0d6a21066fc252f654086a9c15b642a4b8...,631a2cef658d150039fb32c1dc28b33924507185f132a0...
7,8,noise_40__seed_01,1,8,40,1,3194741083,3744228384,71052,28382,...,29,106,28430,4907,1963,105,2012,3114596a8223f858c0cefecd97e2c6a0c94b94e1c10ab8...,5ae8192b730d179845bac76d819e32077c7de2923ed118...,9cd63b660cacd722c9c16b69842f4c5fc5805a08e2f41f...
8,9,noise_50__seed_01,1,9,50,1,3194741083,3744228384,71052,35579,...,45,106,35595,4907,2472,105,2489,8bf757c7b7a0e4c61721eaa7126c0170c92f57ecac8124...,b6a7ab16678c449ef4cdb75a032626a7ef5c2f0c1093e0...,87ce2d5db889a1b2e9a2e7cbdaf079209cd194704c4efe...
9,262,noise_00__seed_30,30,1,0,30,2714151920,3769187290,71052,0,...,0,106,106,4907,0,105,105,62527d52b815d39b23d8a7aeaad105e6f64c32fef01d62...,0922787f6597840f586614f4f695d24302075a195f910c...,3cc0b5417b4ccdaaa4ec71ce19456eb7a9bd3381b720f3...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 16 CELL 6 / STEP 3A RESULT ===

Project:
apache@rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen clean-history order:
Inferred execution-order rows: 97734
Global build-order rows: 536
Raw duplicate Build-Test rows: 0
Raw duplicate Test-order rows: 0

Fixed cohorts:
Raw training rows: 71052
Raw evaluation rows: 26682
Raw training failures: 106
Raw evaluation failures: 9
Model training rows: 4907
Model evaluation rows: 1641
Model training failures: 105
Model evaluation failures: 9
Model failing evaluation builds: 8

Noise plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
RNG-manifest rows: 2131560
Failure subtypes: [1, 

In [7]:
# ==================================================================================================
# PROJECT 16 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# PURPOSE:
# - verify the frozen Project 16 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 16 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 16 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_16_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_16_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "996cde153bbbd71087a4de3670054eaf5d7e466d5aa37c92ccaa6b59d45da6e6"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 8

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_16_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 16 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 16 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 16 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 16 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 16 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != 15
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            16,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–15."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 16 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 16 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 16 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    15,
    len(
        registry
    ),
    len(
        registry
    )
    == 15,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 16 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 16 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 16 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 16 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To15Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project16ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project16ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 16 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 16 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 16 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 16 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 16 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 16 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–15 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 16 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 16 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===


/tmp/ipykernel_3320/1748369127.py:1370: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training_numeric[
/tmp/ipykernel_3320/1748369127.py:1374: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  evaluation_numeric[
/tmp/ipykernel_3320/1748369127.py:1370: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train


Project 16 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_16_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_16_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,996cde153bbbd71087a4de3670054eaf5d7e466d5aa37c...,996cde153bbbd71087a4de3670054eaf5d7e466d5aa37c...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65dd...,e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65dd...,True
4,Raw training rows,71052,71052,True
5,Raw evaluation rows,26682,26682,True
6,Model training rows,4907,4907,True
7,Model evaluation rows,1641,1641,True
8,Model training failures,105,105,True
9,Model evaluation failures,9,9,True



Runtime versions:


...


Metric self-tests:


,Check,Expected,Actual,Pass
0,Manual APFD,0.8,0.8,True
1,APFDc rewards quick failing test first,True,True,True
2,All-pass APFD is NaN,True,True,True
3,All-pass APFDc is NaN,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,412078,412078,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,148773,148773,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,584685,584685,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,277570,277570,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,42443,42443,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,15401,15401,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,93,93,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5246,5246,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,30369303,30369303,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,84518,84518,True




=== PROJECT 16 CELL 7 / STEP 4A RESULT ===

Project:
apache@rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen experiment contract:
Predictors: 151
REC features: 19
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD
Positive class: failure = 1
Median imputation: condition-training medians
Ranking tie-break: Test ascending
Rolling retraining: False

Fixed cohorts:
Model training rows: 4907
Model evaluation rows: 1641
Training failures: 105
Evaluation failures: 9
Failing evaluation builds: 8

Runtime validation:
Step 3A output-manifest failures: 0
Runtime-version failur

In [8]:
# ==================================================================================================
# PROJECT 16 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# SMOKE CONDITIONS:
# - 0% noise, repetition seed 1
# - 50% noise, repetition seed 1
#
# THIS CELL:
# - verifies the frozen Step 4A runtime/model contract;
# - reconstructs condition-specific dependent REC features;
# - preserves all six verdict-independent REC features;
# - applies the frozen clean-anchor offsets;
# - trains all four ML techniques once per smoke condition;
# - evaluates ML plus Random, LatestFail, and QTF-Avg;
# - validates APFDc/APFD outputs and baseline invariance;
# - writes only Project 16 smoke-test outputs and checkpoint/status files;
# - does not modify the registry or full 270-condition raw-result root.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 16 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_16_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_16_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_16_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

STEP4B_STATUS = (
    "PASS_PROJECT_16_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)

CONDITION_STATUS = (
    "PASS_PROJECT_16_SMOKE_CONDITION"
)

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "c635bb42072c980c7d561d680b69a9efb1f5c669a7fc042cc59288566846bf05"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "996cde153bbbd71087a4de3670054eaf5d7e466d5aa37c92ccaa6b59d45da6e6"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "71a43f55bc825d93abfdf6e4285174b41eed5c3dcee48f818a15a6193cb33db0"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "b3cf69922236c45ae62599e141d0d29f0c3737a334370daffcbfa709abff250a"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_BUILDS = 536
EXPECTED_RAW_ROWS = 97_734
EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_ROWS = 6_548
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9
EXPECTED_FAILING_EVAL_BUILDS = 8
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 134
EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS
EXPECTED_RNG_MANIFEST_ROWS = 2_131_560
EXPECTED_REGISTERED_PROJECTS = 15

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

SMOKE_CONDITION_IDS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

SMOKE_REPETITION_SEED = 1
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@rocketmq"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_16_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_16_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_16_fixed_chronological_builds.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_selection_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

BUILD_ENTITY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_smoke_test"
)

SMOKE_CONDITION_INVENTORY_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_condition_inventory.csv"
)

SMOKE_COMBINED_CONDITION_AUDIT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
)

SMOKE_COMBINED_PROJECT_RUNS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"
)

SMOKE_COMBINED_BUILD_METRICS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
)

SMOKE_COMBINED_MODEL_FITS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_report.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4b_status.json"
)

SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_16_smoke_test_checkpoint.json"
)

# The smoke test must never write to this future full-run root.
FULL_RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                "inferred_test_order",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)


def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def audit_checkpoint_manifest(
    payload,
    manifest_key,
    label,
):
    manifest = payload.get(
        manifest_key,
        [],
    )

    if not isinstance(
        manifest,
        list,
    ) or not manifest:
        raise RuntimeError(
            f"{label} contains no {manifest_key}."
        )

    records = []

    for item in manifest:
        path = Path(
            item[
                "Path"
            ]
        )

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha256 = str(
            item[
                "SHA256"
            ]
        ).lower()

        exists = path.is_file()

        actual_bytes = (
            int(
                path.stat().st_size
            )
            if exists
            else -1
        )

        actual_sha256 = (
            sha256_file(
                path
            )
            if exists
            else "MISSING"
        )

        records.append({
            "Checkpoint":
                label,

            "Path":
                str(
                    path
                ),

            "ExpectedBytes":
                expected_bytes,

            "ActualBytes":
                actual_bytes,

            "ExpectedSHA256":
                expected_sha256,

            "ActualSHA256":
                actual_sha256,

            "Pass":
                bool(
                    exists
                    and actual_bytes
                    == expected_bytes
                    and actual_sha256
                    == expected_sha256
                ),
        })

    audit = pd.DataFrame(
        records
    )

    failures = int(
        (
            ~audit[
                "Pass"
            ]
        ).sum()
    )

    return (
        audit,
        failures,
    )


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 16 Step 4B inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 runtime-contract checkpoint SHA-256 differs."
    )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)

if selection_checkpoint.get(
    "Status"
) != "PASS_PROJECT_16_SELECTION_AND_SOURCE_FROZEN":
    raise RuntimeError(
        "Selection checkpoint does not contain the frozen Step 1B PASS status."
    )

if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise RuntimeError(
        "REC checkpoint does not contain the frozen Step 2B PASS status."
    )

if noise_plan_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise RuntimeError(
        "Noise-plan checkpoint does not contain the frozen Step 3A PASS status."
    )

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active-reservation state differs."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 4B."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–15."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 16 is already present in the completion registry."
    )

active_reservations = []

if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 16 freeze."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 16 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 16 source root differs before Step 4B."
    )

rec_manifest_audit, rec_manifest_failures = (
    audit_checkpoint_manifest(
        rec_checkpoint,
        "OutputManifest",
        "REC checkpoint",
    )
)

noise_manifest_audit, noise_manifest_failures = (
    audit_checkpoint_manifest(
        noise_plan_checkpoint,
        "OutputManifest",
        "Noise-plan checkpoint",
    )
)

if rec_manifest_failures != 0:
    print(
        "\nFailed REC output-manifest checks:"
    )

    display(
        rec_manifest_audit.loc[
            ~rec_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen REC outputs changed."
    )

if noise_manifest_failures != 0:
    print(
        "\nFailed noise-plan output-manifest checks:"
    )

    display(
        noise_manifest_audit.loc[
            ~noise_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen noise-plan outputs changed."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 16 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)
frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")
if len(inferred_execution_order) != EXPECTED_RAW_ROWS:
    raise RuntimeError("Frozen inferred execution-order row count differs.")
if len(frozen_global_build_order) != EXPECTED_BUILDS:
    raise RuntimeError("Frozen global build-order row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

smoke_plan = (
    condition_plan.loc[
        condition_plan["ConditionID"].isin(
            SMOKE_CONDITION_IDS
        )
    ]
    .copy()
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(smoke_plan) != EXPECTED_SMOKE_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain exactly the two smoke conditions."
    )

if smoke_plan["ConditionID"].tolist() != SMOKE_CONDITION_IDS:
    raise RuntimeError(
        "Smoke-condition order differs from the frozen contract."
    )

if smoke_plan["NoisePercent"].astype(int).tolist() != SMOKE_NOISE_LEVELS:
    raise RuntimeError(
        "Smoke noise levels differ from the frozen contract."
    )

if not smoke_plan["RepetitionSeed"].astype(int).eq(
    SMOKE_REPETITION_SEED
).all():
    raise RuntimeError(
        "Smoke repetition seed differs from the frozen contract."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_MANIFEST_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

rng_seed = pd.read_parquet(
    RNG_MANIFEST_PATH,
    filters=[
        (
            "RepetitionSeed",
            "==",
            SMOKE_REPETITION_SEED,
        ),
    ],
)

rng_seed[raw_order_column] = parse_int(
    rng_seed[raw_order_column],
    "rng_seed.RawTrainingRowOrder",
)

rng_seed = (
    rng_seed.sort_values(
        raw_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(rng_seed) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Seed-1 RNG stream has the wrong row count."
    )

if not np.array_equal(
    rng_seed[raw_order_column].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_RAW_TRAIN_ROWS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Seed-1 RNG stream row order differs."
    )

flip_uniform = rng_seed[
    "FlipUniform"
].to_numpy(dtype=np.float64)
sampled_failure_subtype = rng_seed[
    "SampledFailureSubtype"
].to_numpy(dtype=np.int16)

if not np.isfinite(flip_uniform).all():
    raise RuntimeError(
        "Seed-1 flip-uniform stream contains non-finite values."
    )

if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
    raise RuntimeError(
        "Seed-1 flip-uniform values are outside [0,1)."
    )

if sorted(np.unique(sampled_failure_subtype).tolist()) != [1, 2]:
    raise RuntimeError(
        "Seed-1 failure-subtype stream differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

required_inferred_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "InferredTestOrder",
}

missing_inferred_order_columns = (
    required_inferred_order_columns
    - set(inferred_execution_order.columns)
)

if missing_inferred_order_columns:
    raise RuntimeError(
        "Frozen inferred execution order is missing columns: "
        f"{sorted(missing_inferred_order_columns)}"
    )

for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[column] = parse_int(
        inferred_execution_order[column],
        f"inferred_execution_order.{column}",
    )

inferred_execution_order["Job"] = pd.to_numeric(
    inferred_execution_order["Job"],
    errors="coerce",
)

inferred_execution_order["Duration"] = pd.to_numeric(
    inferred_execution_order["Duration"],
    errors="coerce",
)

if not np.isfinite(
    inferred_execution_order["Job"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite jobs."
    )

if not np.isfinite(
    inferred_execution_order["Duration"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite durations."
    )

if inferred_execution_order["Duration"].lt(0).any():
    raise RuntimeError(
        "Frozen inferred execution order contains negative durations."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Build",
        "Test",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate Build-Test rows."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Test",
        "InferredTestOrder",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate per-test order rows."
    )

required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}

missing_global_order_columns = (
    required_global_order_columns
    - set(frozen_global_build_order.columns)
)

if missing_global_order_columns:
    raise RuntimeError(
        "Frozen global build order is missing columns: "
        f"{sorted(missing_global_order_columns)}"
    )

frozen_global_build_order["GlobalBuildOrder"] = parse_int(
    frozen_global_build_order["GlobalBuildOrder"],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order["BuildID"] = parse_int(
    frozen_global_build_order["BuildID"],
    "frozen_global_build_order.BuildID",
)

frozen_global_build_order = (
    frozen_global_build_order.sort_values(
        "GlobalBuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not np.array_equal(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_BUILDS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen global build-order sequence is not canonical."
    )

if frozen_global_build_order["BuildID"].nunique() != EXPECTED_BUILDS:
    raise RuntimeError(
        "Frozen global build order contains duplicate build IDs."
    )

ordered_builds = (
    frozen_global_build_order[
        "BuildID"
    ]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing InferredTestOrder."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

combined_raw_order = (
    pd.concat(
        [
            raw_training[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
            raw_evaluation[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

inferred_order_reference = (
    inferred_execution_order[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

raw_order_key_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            combined_raw_order[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
            != inferred_order_reference[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
        ).sum()
    )
)

raw_order_numeric_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            ~np.isclose(
                combined_raw_order[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                inferred_order_reference[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                rtol=0,
                atol=0,
                equal_nan=False,
            )
        ).sum()
    )
)

if (
    raw_order_key_mismatches != 0
    or raw_order_numeric_mismatches != 0
):
    raise RuntimeError(
        "The fixed raw cohorts no longer reproduce the frozen V6 "
        "inferred execution order."
    )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

random_scores = np.empty(
    EXPECTED_MODEL_EVAL_ROWS,
    dtype=np.float64,
)

for build_id in sorted(
    evaluation_meta_base["Build"].unique()
):
    build_indices = np.flatnonzero(
        evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        == int(build_id)
    )

    random_scores[build_indices] = np.random.default_rng(
        deterministic_random_build_seed(
            SMOKE_REPETITION_SEED,
            int(build_id),
        )
    ).random(len(build_indices))

if not np.isfinite(random_scores).all():
    raise RuntimeError(
        "Random baseline produced non-finite scores."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN THE TWO END-TO-END SMOKE CONDITIONS
# --------------------------------------------------------------------------------------------------

# Remove only incomplete/previous Project 16 smoke-test outputs.
# Frozen Steps 0–4A and the future full-result root are untouched.
if SMOKE_ROOT.exists():
    shutil.rmtree(
        SMOKE_ROOT
    )

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

raw_training_hash_before = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_before = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_before = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_before = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)

condition_inventory_records = []
all_condition_audits = []
all_project_runs = []
all_build_metrics = []
all_model_fits = []
all_rankings_for_invariance = []

smoke_execution_started = time.perf_counter()

for smoke_index, plan_row in enumerate(
    smoke_plan.itertuples(index=False),
    start=1,
):
    condition_started = time.perf_counter()
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)

    print("\n" + "-" * 136)
    print(
        f"[{smoke_index}/{EXPECTED_SMOKE_CONDITIONS}] "
        f"Running {condition_key}"
    )
    print("-" * 136)

    condition_dir = (
        SMOKE_ROOT
        / condition_key
    )
    condition_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ranking_path = (
        condition_dir
        / "rankings.csv.gz"
    )
    build_metrics_path = (
        condition_dir
        / "build_metrics.csv"
    )
    project_runs_path = (
        condition_dir
        / "project_runs.csv"
    )
    model_fits_path = (
        condition_dir
        / "model_fits.csv"
    )
    training_medians_path = (
        condition_dir
        / "training_medians.csv"
    )
    condition_audit_path = (
        condition_dir
        / "condition_audit.csv"
    )
    condition_summary_path = (
        condition_dir
        / "condition_summary.json"
    )
    completion_marker_path = (
        condition_dir
        / "COMPLETE.json"
    )

    flip_mask = (
        flip_uniform
        < (noise_percent / 100.0)
    )

    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = (
        flip_mask
        & (clean_raw_training_verdict == 0)
    )
    failure_to_pass_mask = (
        flip_mask
        & (clean_raw_training_verdict != 0)
    )

    noisy_raw_training_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtype[
        pass_to_failure_mask
    ]
    noisy_raw_training_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(
        plan_row.FlipMaskSHA256
    )
    expected_noisy_raw_sha256 = str(
        plan_row.NoisyRawVerdictSHA256
    )
    expected_noisy_model_sha256 = str(
        plan_row.NoisyModelVerdictSHA256
    )

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )

    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )

    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (
            noisy_model_training_verdict
            != clean_model_training_verdict
        ).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(
        noisy_model_training_binary.sum()
    )

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    print(
        "  Reconstructing REC features from the condition-specific history."
    )

    train_history = pd.DataFrame({
        "build": raw_training["Build"].to_numpy(dtype=np.int64),
        "test": raw_training["Test"].to_numpy(dtype=np.int64),
        "job": raw_training["Job"].to_numpy(),
        "verdict": noisy_raw_training_verdict.astype(np.int16),
        "duration": raw_training["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_training[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    evaluation_history = pd.DataFrame({
        "build": raw_evaluation["Build"].to_numpy(dtype=np.int64),
        "test": raw_evaluation["Test"].to_numpy(dtype=np.int64),
        "job": raw_evaluation["Job"].to_numpy(),
        "verdict": clean_raw_evaluation_verdict.astype(np.int16),
        "duration": raw_evaluation["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_evaluation[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    execution_history = pd.concat(
        [
            train_history,
            evaluation_history,
        ],
        ignore_index=True,
    )

    if execution_history.duplicated(
        subset=[
            "build",
            "test",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate Build-Test rows."
        )

    if execution_history.duplicated(
        subset=[
            "test",
            "inferred_test_order",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate per-test order rows."
        )

    execution_history = (
        execution_history.sort_values(
            [
                "test",
                "inferred_test_order",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    rec_started = time.perf_counter()

    reconstructed = reconstruct_rec_features(
        execution_history=execution_history,
        requested_rows=model_all[["Build", "Test"]],
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
        recent_window=RECENT_WINDOW,
    )

    rec_seconds = time.perf_counter() - rec_started

    if len(reconstructed) != EXPECTED_MODEL_ROWS:
        raise RuntimeError(
            f"{condition_key}: reconstructed REC row count differs."
        )

    if reconstructed.duplicated(
        subset=["Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: reconstructed REC contains duplicate keys."
        )

    reconstructed["Build"] = parse_int(
        reconstructed["Build"],
        f"{condition_key}.reconstructed.Build",
    )
    reconstructed["Test"] = parse_int(
        reconstructed["Test"],
        f"{condition_key}.reconstructed.Test",
    )

    reconstructed_indexed = reconstructed.set_index(
        [
            "Build",
            "Test",
        ]
    )

    missing_reconstructed_keys = model_key_index.difference(
        reconstructed_indexed.index
    )

    if len(missing_reconstructed_keys) != 0:
        raise RuntimeError(
            f"{condition_key}: reconstruction does not cover all model rows."
        )

    reconstructed_values_all = reconstructed_indexed.loc[
        model_key_index,
        REC_FEATURES,
    ].to_numpy(dtype=np.float64)

    anchored_values_all = (
        reconstructed_values_all
        + anchor_values_all
    )

    independent_reconstruction_mismatches = int(
        (~np.isclose(
            anchored_values_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            clean_original_rec_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            rtol=0,
            atol=1e-12,
        )).sum()
    )

    if independent_reconstruction_mismatches != 0:
        raise RuntimeError(
            f"{condition_key}: verdict-independent REC reconstruction changed."
        )

    condition_numeric_all = all_base_numeric.copy()

    dependent_rec_values_all = anchored_values_all[:, [
        REC_FEATURES.index(feature)
        for feature in VERDICT_DEPENDENT_REC
    ]]

    condition_numeric_all[:, dependent_predictor_indices] = (
        dependent_rec_values_all
    )

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    condition_original_dependent_all = all_base_numeric[
        :, dependent_predictor_indices
    ]

    dependent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, dependent_predictor_indices],
            condition_original_dependent_all,
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        )).sum()
    )

    independent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, independent_predictor_indices],
            all_base_numeric[:, independent_predictor_indices],
            rtol=0,
            atol=0,
            equal_nan=True,
        )).sum()
    )

    if independent_rec_changes != 0:
        raise RuntimeError(
            f"{condition_key}: preserved independent REC predictors changed."
        )

    if noise_percent == 0:
        zero_rec_mismatches = int(
            (~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )).sum()
        )

        if zero_rec_mismatches != 0:
            raise RuntimeError(
                "0% smoke condition did not reproduce the clean REC cohort."
            )

        if number_flipped != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly flipped raw labels."
            )

        if model_label_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed model labels."
            )

        if dependent_rec_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed dependent REC values."
            )

    if noise_percent > 0:
        if number_flipped <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no dependent REC values."
            )

    medians = np.nanmedian(
        condition_training_numeric,
        axis=0,
    )

    nonfinite_median_indices = np.flatnonzero(
        ~np.isfinite(medians)
    )

    if len(nonfinite_median_indices) != 0:
        bad_features = [
            predictor_columns[index]
            for index in nonfinite_median_indices
        ]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(
        condition_training_numeric
    )
    evaluation_missing_mask = ~np.isfinite(
        condition_evaluation_numeric
    )

    if training_missing_mask.any():
        row_indices, column_indices = np.where(
            training_missing_mask
        )
        condition_training_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(
            evaluation_missing_mask
        )
        condition_evaluation_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite after imputation."
        )

    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite after imputation."
        )

    # Keep float64 throughout the smoke test. This matches the frozen Step 4A
    # numeric/imputation contract and avoids changing ranking/model behaviour
    # through an unapproved dtype conversion.

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()
        fit_status = "PASS_MODEL_FIT"
        fit_error = ""

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )

            fit_seconds = time.perf_counter() - fit_started
            scores = positive_probability(
                model,
                condition_evaluation_numeric,
            )

            technique_scores[technique] = scores

        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)

            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })

            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps(
                [
                    int(value)
                    for value in np.asarray(model.classes_).tolist()
                ]
            ),
            "Status": fit_status,
            "Error": fit_error,
        })

        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )

    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :,
        predictor_index["REC_LastFailureAge"],
    ].astype(float)

    condition_eval_qtf = condition_evaluation_numeric[
        :,
        predictor_index["REC_TotalAvgExeTime"],
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = (
        -condition_eval_last_failure_age
    )
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []

    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )

    ranking_techniques = sorted(
        rankings["Technique"].unique().tolist()
    )

    if ranking_techniques != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )

    ranking_rows_per_technique = rankings.groupby(
        "Technique"
    ).size()

    if not ranking_rows_per_technique.eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )

    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_ranking_rows != 0:
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(
        rankings
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )

    if sorted(project_runs["Technique"].tolist()) != sorted(
        ALL_TECHNIQUES
    ):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    metric_columns = [
        "APFDc",
        "APFD",
    ]

    build_metric_values = build_metrics[
        metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )

    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    project_metric_values = project_runs[
        project_metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )

    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(len(reconstructed)),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    atomic_csv(
        ranking_path,
        rankings,
        compression="gzip",
    )
    atomic_csv(
        build_metrics_path,
        build_metrics,
    )
    atomic_csv(
        project_runs_path,
        project_runs,
    )
    atomic_csv(
        model_fits_path,
        model_fits,
    )
    atomic_csv(
        training_medians_path,
        training_medians,
    )
    atomic_csv(
        condition_audit_path,
        condition_audit,
    )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]

    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "OutputManifest": condition_output_manifest,
    }

    atomic_json(
        condition_summary_path,
        condition_summary,
    )

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json(
        completion_marker_path,
        completion_marker,
    )

    condition_manifest = directory_manifest(
        condition_dir
    )
    condition_root_sha256 = directory_root_hash(
        condition_manifest
    )

    condition_inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": smoke_index,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": condition_root_sha256,
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "ModelFitRows": int(len(model_fits)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
    })

    all_condition_audits.append(condition_audit)
    all_project_runs.append(project_runs)
    all_build_metrics.append(build_metrics)
    all_model_fits.append(model_fits)
    all_rankings_for_invariance.append(
        rankings.loc[
            rankings["Technique"].isin(
                [
                    "Random",
                    "QTF-Avg",
                ]
            )
        ].copy()
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del train_history
    del evaluation_history
    del execution_history
    del reconstructed
    del reconstructed_indexed
    del reconstructed_values_all
    del anchored_values_all
    del condition_numeric_all
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. COMBINE SMOKE OUTPUTS AND VERIFY BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

condition_inventory = pd.DataFrame(
    condition_inventory_records
)
combined_condition_audit = pd.concat(
    all_condition_audits,
    ignore_index=True,
)
combined_project_runs = pd.concat(
    all_project_runs,
    ignore_index=True,
)
combined_build_metrics = pd.concat(
    all_build_metrics,
    ignore_index=True,
)
combined_model_fits = pd.concat(
    all_model_fits,
    ignore_index=True,
)
combined_invariance_rankings = pd.concat(
    all_rankings_for_invariance,
    ignore_index=True,
)

baseline_invariance_records = []

for technique in [
    "Random",
    "QTF-Avg",
]:
    zero_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(0)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    noisy_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(50)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    same_keys = bool(
        zero_rows[["Build", "Test"]].equals(
            noisy_rows[["Build", "Test"]]
        )
    )

    score_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (~np.isclose(
                zero_rows["Score"].to_numpy(dtype=float),
                noisy_rows["Score"].to_numpy(dtype=float),
                rtol=0,
                atol=0,
            )).sum()
        )
    )

    rank_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (
                zero_rows["Rank"].to_numpy(dtype=np.int64)
                != noisy_rows["Rank"].to_numpy(dtype=np.int64)
            ).sum()
        )
    )

    baseline_invariance_records.append({
        "Technique": technique,
        "Rows": int(len(zero_rows)),
        "SameBuildTestKeys": same_keys,
        "ScoreMismatches": score_mismatches,
        "RankMismatches": rank_mismatches,
        "Pass": (
            len(zero_rows) == EXPECTED_MODEL_EVAL_ROWS
            and len(noisy_rows) == EXPECTED_MODEL_EVAL_ROWS
            and same_keys
            and score_mismatches == 0
            and rank_mismatches == 0
        ),
    })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)
baseline_invariance_failures = int(
    (~baseline_invariance["Pass"]).sum()
)

if baseline_invariance_failures != 0:
    print("\nBaseline invariance failures:")
    display(
        baseline_invariance.loc[
            ~baseline_invariance["Pass"]
        ]
    )
    raise RuntimeError(
        "Random or QTF-Avg changed across the two smoke noise levels."
    )

atomic_csv(
    SMOKE_CONDITION_INVENTORY_PATH,
    condition_inventory,
)
atomic_csv(
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)
atomic_csv(
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)
atomic_csv(
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)
atomic_csv(
    SMOKE_COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)
atomic_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_hash_after = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_after = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_after = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_after = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)
registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

current_source_rows_after = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

source_root_sha256_after = source_root_hash(
    pd.DataFrame(current_source_rows_after)
)

full_raw_result_root_exists_after = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_after = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_after = directory_root_hash(
    full_raw_result_manifest_after
)

full_raw_result_unchanged = bool(
    full_raw_result_root_existed_before
    == full_raw_result_root_exists_after
    and full_raw_result_root_hash_before
    == full_raw_result_root_hash_after
    and len(full_raw_result_manifest_before)
    == len(full_raw_result_manifest_after)
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
].iloc[0]

positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(50)
].iloc[0]

validation_records = []

add_check(
    validation_records,
    "Step 4A status",
    EXPECTED_STEP4A_STATUS,
    step4a_status.get("Status"),
    step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
)
add_check(
    validation_records,
    "Runtime checkpoint SHA-256",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    runtime_checkpoint_sha256,
    runtime_checkpoint_sha256 == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_plan_checkpoint_sha256,
    noise_plan_checkpoint_sha256 == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256 == EXPECTED_REC_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256 == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC output-manifest failures",
    0,
    rec_manifest_failures,
    rec_manifest_failures == 0,
)
add_check(
    validation_records,
    "Noise-plan output-manifest failures",
    0,
    noise_manifest_failures,
    noise_manifest_failures == 0,
)
add_check(
    validation_records,
    "Step 4A output-manifest failures",
    0,
    runtime_manifest_failures,
    runtime_manifest_failures == 0,
)
add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_sha256_after,
    source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256,
)
add_check(
    validation_records,
    "Smoke conditions",
    EXPECTED_SMOKE_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Smoke condition keys",
    SMOKE_CONDITION_IDS,
    condition_inventory["ConditionKey"].tolist(),
    condition_inventory["ConditionKey"].tolist() == SMOKE_CONDITION_IDS,
)
add_check(
    validation_records,
    "Condition statuses",
    CONDITION_STATUS,
    sorted(condition_inventory["Status"].unique().tolist()),
    condition_inventory["Status"].eq(CONDITION_STATUS).all(),
)
add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_SMOKE_CONDITIONS,
    len(combined_condition_audit),
    len(combined_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Total ML fits",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
)
add_check(
    validation_records,
    "Model-fit failures",
    0,
    int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()),
    combined_model_fits["Status"].eq("PASS_MODEL_FIT").all(),
)
add_check(
    validation_records,
    "Total project-run rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Total build-metric rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Technique set",
    sorted(ALL_TECHNIQUES),
    sorted(combined_project_runs["Technique"].unique().tolist()),
    sorted(combined_project_runs["Technique"].unique().tolist())
    == sorted(ALL_TECHNIQUES),
)
add_check(
    validation_records,
    "Project-run rows per condition violations",
    0,
    int((
        combined_project_runs.groupby("ConditionKey").size()
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_project_runs.groupby("ConditionKey").size()
        == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Build-metric rows per condition violations",
    0,
    int((
        combined_build_metrics.groupby("ConditionKey").size()
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_build_metrics.groupby("ConditionKey").size()
        == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Zero-noise raw flips",
    0,
    int(zero_audit["NumberFlipped"]),
    int(zero_audit["NumberFlipped"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise model-label changes",
    0,
    int(zero_audit["ModelLabelChanges"]),
    int(zero_audit["ModelLabelChanges"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise dependent REC changes",
    0,
    int(zero_audit["DependentRECChanges"]),
    int(zero_audit["DependentRECChanges"]) == 0,
)
add_check(
    validation_records,
    "Positive-noise raw flips",
    "> 0",
    int(positive_audit["NumberFlipped"]),
    int(positive_audit["NumberFlipped"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise model-label changes",
    "> 0",
    int(positive_audit["ModelLabelChanges"]),
    int(positive_audit["ModelLabelChanges"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise dependent REC changes",
    "> 0",
    int(positive_audit["DependentRECChanges"]),
    int(positive_audit["DependentRECChanges"]) > 0,
)
add_check(
    validation_records,
    "Independent REC changes",
    0,
    int(combined_condition_audit["IndependentRECChanges"].sum()),
    int(combined_condition_audit["IndependentRECChanges"].sum()) == 0,
)
add_check(
    validation_records,
    "Frozen raw-order key mismatches",
    0,
    raw_order_key_mismatches,
    raw_order_key_mismatches == 0,
)
add_check(
    validation_records,
    "Frozen raw-order numeric mismatches",
    0,
    raw_order_numeric_mismatches,
    raw_order_numeric_mismatches == 0,
)
add_check(
    validation_records,
    "Independent REC reconstruction mismatches",
    0,
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()),
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()) == 0,
)
add_check(
    validation_records,
    "Noise-plan hash mismatches",
    0,
    int((
        combined_condition_audit["ExpectedFlipMaskSHA256"]
        != combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
        != combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
        != combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()),
    bool(
        (
            combined_condition_audit["ExpectedFlipMaskSHA256"]
            == combined_condition_audit["ActualFlipMaskSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
            == combined_condition_audit["ActualNoisyRawVerdictSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
            == combined_condition_audit["ActualNoisyModelVerdictSHA256"]
        ).all()
    ),
)
add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)
add_check(
    validation_records,
    "Project metrics non-finite",
    0,
    int((~np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    )).sum()),
    bool(np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    ).all()),
)
add_check(
    validation_records,
    "Project metrics outside [0,1]",
    0,
    int((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            < 0
        )
        | (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            > 1
        )
    ).sum()),
    bool((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            >= 0
        )
        & (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            <= 1
        )
    ).all()),
)
add_check(
    validation_records,
    "Raw training cohort unchanged",
    raw_training_hash_before,
    raw_training_hash_after,
    raw_training_hash_after == raw_training_hash_before,
)
add_check(
    validation_records,
    "Raw evaluation cohort unchanged",
    raw_evaluation_hash_before,
    raw_evaluation_hash_after,
    raw_evaluation_hash_after == raw_evaluation_hash_before,
)
add_check(
    validation_records,
    "Model training cohort unchanged",
    model_training_hash_before,
    model_training_hash_after,
    model_training_hash_after == model_training_hash_before,
)
add_check(
    validation_records,
    "Model evaluation cohort unchanged",
    model_evaluation_hash_before,
    model_evaluation_hash_after,
    model_evaluation_hash_after == model_evaluation_hash_before,
)
add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after,
    registry_sha256_after == registry_sha256_before,
)
add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Registry Project 16 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)
add_check(
    validation_records,
    "Full raw-result root unchanged",
    True,
    full_raw_result_unchanged,
    full_raw_result_unchanged,
)

validation = pd.DataFrame(
    validation_records
)
failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 16 Step 4B validation:")
display(validation)

print("\nBaseline invariance audit:")
display(baseline_invariance)

print("\nSmoke project-run results:")
display(
    combined_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

if not failed_validation.empty:
    print("\nFailed Project 16 Step 4B checks:")
    display(failed_validation)
    print("\nNo Step 4B PASS status or checkpoint was written.")
    raise RuntimeError(
        "PROJECT 16 STEP 4B VALIDATION FAILED. DO NOT START THE FULL EXPERIMENT."
    )

atomic_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, STATUS, AND FINAL READBACK
# --------------------------------------------------------------------------------------------------

smoke_execution_seconds = (
    time.perf_counter()
    - smoke_execution_started
)
completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

smoke_output_paths = [
    SMOKE_CONDITION_INVENTORY_PATH,
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    SMOKE_COMBINED_MODEL_FITS_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]

for condition_key in SMOKE_CONDITION_IDS:
    condition_dir = SMOKE_ROOT / condition_key
    smoke_output_paths.extend([
        path
        for path in condition_dir.rglob("*")
        if path.is_file()
    ])

smoke_output_paths = sorted(
    set(smoke_output_paths),
    key=lambda path: str(path),
)

smoke_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in smoke_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "SmokeConditionKeys": SMOKE_CONDITION_IDS,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": int(len(combined_build_metrics)),
    "ProjectRunRows": int(len(combined_project_runs)),
    "TrainingMedianRows": int(
        condition_inventory["TrainingMedianRows"].sum()
    ),
    "ZeroNoiseRawFlips": int(zero_audit["NumberFlipped"]),
    "ZeroNoiseModelLabelChanges": int(
        zero_audit["ModelLabelChanges"]
    ),
    "ZeroNoiseDependentRECChanges": int(
        zero_audit["DependentRECChanges"]
    ),
    "PositiveNoiseRawFlips": int(
        positive_audit["NumberFlipped"]
    ),
    "PositiveNoiseModelLabelChanges": int(
        positive_audit["ModelLabelChanges"]
    ),
    "PositiveNoiseDependentRECChanges": int(
        positive_audit["DependentRECChanges"]
    ),
    "IndependentRECChanges": int(
        combined_condition_audit["IndependentRECChanges"].sum()
    ),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": int(len(validation)),
    "FailedValidationChecks": int(len(failed_validation)),
    "SmokeExecutionSeconds": float(smoke_execution_seconds),
    "OutputManifest": smoke_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To15Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "FullRawResultRootModified": False,
    "FullExperimentStarted": False,
}

atomic_json(
    SMOKE_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "SmokeTestPassed": True,
    "RuntimeContractFrozen": True,
    "NoisePlanFrozen": True,
    "EvaluationCohortImmutable": True,
    "ReadyForFull270ConditionExperiment": True,
}

atomic_json(
    SMOKE_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "FailedValidationChecks": int(len(failed_validation)),
    "Checkpoint": str(SMOKE_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(SMOKE_CHECKPOINT_PATH),
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "FullExperimentStarted": False,
}

atomic_json(
    STEP4B_STATUS_PATH,
    status_payload,
)

checkpoint_readback = load_json(
    SMOKE_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP4B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B checkpoint readback failed."
    )

if status_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B status readback failed."
    )

if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Step 4B finalisation."
    )

final_source_rows = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    final_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

if source_root_hash(pd.DataFrame(final_source_rows)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 16 source changed during Step 4B finalisation."
    )

print("\n" + "=" * 136)
print("=== PROJECT 16 CELL 8 / STEP 4B RESULT ===")
print("=" * 136)
print()
print("Project:")
print(PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)
print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Two-condition end-to-end smoke test:")
print("Conditions:", SMOKE_CONDITION_IDS)
print("Conditions passed:", len(condition_inventory), "/", EXPECTED_SMOKE_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Noise and REC audit:")
print("0% raw flips:", int(zero_audit["NumberFlipped"]))
print("0% model-label changes:", int(zero_audit["ModelLabelChanges"]))
print("0% dependent REC changes:", int(zero_audit["DependentRECChanges"]))
print("50% raw flips:", int(positive_audit["NumberFlipped"]))
print("50% model-label changes:", int(positive_audit["ModelLabelChanges"]))
print("50% dependent REC changes:", int(positive_audit["DependentRECChanges"]))
print("Independent REC changes:", int(combined_condition_audit["IndependentRECChanges"].sum()))
print()
print("Baselines and metrics:")
print("Random/QTF-Avg invariance failures:", baseline_invariance_failures)
print("Techniques:", ALL_TECHNIQUES)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 16 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–15 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print("Full experiment raw-result root modified:", False)
print("Full 270-condition experiment started:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Smoke-test checkpoint:")
print(SMOKE_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(SMOKE_CHECKPOINT_PATH))
print()
print("Runtime seconds:", round(smoke_execution_seconds, 2))
print()
print("STATUS:", STEP4B_STATUS)
print("=" * 136)


=== PROJECT 16 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 16 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.

----------------------------------------------------------------------------------------------------------------------------------------
[1/2] Running noise_00__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 219 tests | reconstructed rows: 2509
    REC reconstruction progress: 200 / 219 tests | reconstructed rows: 6326
    REC reconstruction progress: 219 / 219 tests | reconstructed rows: 6548
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 105 | condition seconds: 55.87

----------------------------------------------------------------------------------------------------------------------------------------
[2/2] Running noise_50__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 219 tests | reconstructed rows: 2509
    REC reconstruction progress: 200 / 219 tests | reconstructed rows: 6326
    REC reconstruction progress: 219 / 219 tests | reconstructed rows: 6548
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 35579 | model-label changes: 2472 | dependent REC changes: 71949
  Training failures: 2489 | condition seconds: 34.12

Project 16 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,1641,True,0,0,True
1,QTF-Avg,1641,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,134,8,1641,9,0.299580,0.309140,0.103191,0.120098
1,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LightGBM,134,8,1641,9,0.952212,0.962415,0.981944,0.986520
2,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,NaiveBayes,134,8,1641,9,0.476445,0.479743,0.856686,0.891611
3,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,QTF-Avg,134,8,1641,9,0.539643,0.432478,0.107454,0.017242
4,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,Random,134,8,1641,9,0.345334,0.275528,0.372112,0.393382
5,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,RandomForest,134,8,1641,9,0.858692,0.947116,0.972183,0.977941
6,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,XGBoost,134,8,1641,9,0.964272,0.967551,0.993573,0.992647
7,16,apache@rocketmq,apache__rocketmq,noise_50__seed_01,50,1,LatestFail,134,8,1641,9,0.920061,0.968889,0.927330,0.992647
8,16,apache@rocketmq,apache__rocketmq,noise_50__seed_01,50,1,LightGBM,134,8,1641,9,0.664256,0.664489,0.636571,0.623467
9,16,apache@rocketmq,apache__rocketmq,noise_50__seed_01,50,1,NaiveBayes,134,8,1641,9,0.698909,0.700549,0.950813,0.990160



=== PROJECT 16 CELL 8 / STEP 4B RESULT ===

Project:
apache@rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 22974
Build-metric rows: 112
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 35579
50% model-label changes: 2472
50% dependent REC changes: 71949
Independent REC changes: 0

Baselines and metrics:
Random/QTF-Avg invariance failures: 0
Techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', '

In [9]:
# ==================================================================================================
# PROJECT 16 — CELL 9 / STEP 5A V2 CHECKPOINT-SCHEMA-COMPATIBLE ACCELERATED
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT WITH VECTORIZED REC ENGINE
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# PURPOSE:
# - validate the frozen Project 16 Step 4B smoke-test checkpoint and every upstream contract;
# - validate the accelerated REC engine against the exact frozen Step 4B smoke outputs;
# - execute all 270 noise/seed conditions with scientifically identical inputs and outputs;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze the complete Project 16 raw-result root for independent Step 5B revalidation.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–15;
# - no prior-project condition-output access;
# - clean evaluation data remain immutable;
# - incomplete condition outputs are preserved in quarantine before rerun.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)

print("=" * 136)
print("=== PROJECT 16 CELL 9 / STEP 5A V2: CHECKPOINT-SCHEMA-COMPATIBLE FULL 270-CONDITION EXPERIMENT ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_16_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_16_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "c635bb42072c980c7d561d680b69a9efb1f5c669a7fc042cc59288566846bf05"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "b2921d447ed5ac57badbdae55dc79830e45d18ea541c21d849976b4fb9954c4c"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "996cde153bbbd71087a4de3670054eaf5d7e466d5aa37c92ccaa6b59d45da6e6"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "71a43f55bc825d93abfdf6e4285174b41eed5c3dcee48f818a15a6193cb33db0"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "b3cf69922236c45ae62599e141d0d29f0c3737a334370daffcbfa709abff250a"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
)
EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_REGISTERED_PROJECTS = 15

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 71_052
EXPECTED_RAW_EVAL_ROWS = 26_682
EXPECTED_MODEL_TRAIN_ROWS = 4_907
EXPECTED_MODEL_EVAL_ROWS = 1_641
EXPECTED_MODEL_ROWS = 6_548
EXPECTED_MODEL_TRAIN_FAILURES = 105
EXPECTED_MODEL_EVAL_FAILURES = 9
EXPECTED_FAILING_EVAL_BUILDS = 8
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 134
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 2_131_560
ACCELERATED_ENGINE_VERSION = "PROJECT_16_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/apache@rocketmq")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_16_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_16_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_16_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_16_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_16_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_16_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_16_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_16_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
INCOMPLETE_BACKUP_ROOT = FULL_EXPERIMENT_ROOT / "incomplete_condition_backups"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_16_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)

def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)

def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)

def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()

def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")

def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })

def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )

def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )

def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )

def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )

def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)

def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the exact Step 2B V6-frozen InferredTestOrder is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result

def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))

def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    audit_columns = [
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "RawTrainingRows",
        "NumberFlipped",
        "ExpectedNumberFlipped",
        "PassToFailure",
        "FailureToPass",
        "ModelTrainingRows",
        "ModelLabelChanges",
        "ExpectedModelLabelChanges",
        "TrainingFailures",
        "ExpectedTrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ReconstructedRows",
        "Predictors",
        "MLFits",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ]
    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)

def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]

def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs

DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]

def directory_manifest(root):
    root = Path(root)
    rows = []

    if root.exists():
        for path in sorted(
            [
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ],
            key=lambda candidate: candidate.relative_to(root).as_posix(),
        ):
            rows.append({
                "RelativePath": path.relative_to(root).as_posix(),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            })

    return pd.DataFrame(
        rows,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )

def directory_root_hash(manifest):
    if manifest is None:
        raise TypeError(
            "Directory manifest cannot be None."
        )

    missing_columns = [
        column
        for column in DIRECTORY_MANIFEST_COLUMNS
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Directory manifest is missing required columns: "
            + ", ".join(missing_columns)
        )

    digest = hashlib.sha256()

    if manifest.empty:
        return digest.hexdigest()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()

# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 16 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 16 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 16 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 16 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

if smoke_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Smoke-test checkpoint active reservations differ."
    )

# Step 4B validates the runtime-priority rule against the frozen Step 4A
# runtime checkpoint, but its checkpoint schema does not duplicate that field.
# Therefore, validate the frozen linkage instead of requiring an absent key.
if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke-test checkpoint does not link to the frozen runtime contract."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )

# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–15."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–15 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 16 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 16 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 16 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)

# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 16 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )["BuildID"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing the V6-frozen InferredTestOrder column."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "InferredTestOrder": np.concatenate((
        raw_training["InferredTestOrder"].to_numpy(dtype=np.int64),
        raw_evaluation["InferredTestOrder"].to_numpy(dtype=np.int64),
    )),
})

if combined_history.duplicated(
    subset=["Test", "InferredTestOrder"],
    keep=False,
).any():
    raise RuntimeError(
        "Accelerated REC history contains duplicate V6 per-test order keys."
    )

# Project 16 must use the exact per-test execution order frozen by Step 2B V6.
# The fixed chronological Build-ID tie-break is not the REC history order for this project.
combined_history = (
    combined_history.sort_values(
        ["Test", "InferredTestOrder"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "BuildID",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)

# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND FULL CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}

def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }

def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination

print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending conditions:", EXPECTED_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The frozen 0% and 50% seed-1 smoke conditions are executed/validated first.
equivalence_records_by_key = {}
for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
    equivalence_row = condition_plan.loc[
        condition_plan["ConditionID"].eq(equivalence_key)
    ]
    if len(equivalence_row) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
        )
    equivalence_plan_row = next(equivalence_row.itertuples(index=False))
    equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
    if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
        equivalence_record = compare_full_condition_to_smoke(
            equivalence_key,
            equivalence_dir,
        )
        if not equivalence_record["Pass"]:
            raise RuntimeError(
                f"Existing accelerated condition {equivalence_key} differs from Step 4B."
            )
        equivalence_records_by_key[equivalence_key] = equivalence_record

smoke_first_plan = condition_plan.loc[
    condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].copy()
smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
    key: index
    for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
})
smoke_first_plan = smoke_first_plan.sort_values(
    "__SmokeOrder",
    kind="mergesort",
).drop(columns="__SmokeOrder")
remaining_plan = condition_plan.loc[
    ~condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].sort_values("ConditionOrder", kind="mergesort")
execution_plan = pd.concat(
    [smoke_first_plan, remaining_plan],
    ignore_index=True,
)

if len(execution_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError("Accelerated execution plan does not contain 270 conditions.")

if equivalence_records_by_key:
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for plan_row in execution_plan.itertuples(index=False):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[{condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = pd.read_parquet(
            RNG_MANIFEST_PATH,
            filters=[("RepetitionSeed", "==", repetition_seed)],
        )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(eval_build_array == int(build_id))
            random_scores[build_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(build_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[{condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical 833,541-row reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        atomic_csv(
            ACCELERATED_EQUIVALENCE_PATH,
            pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                "ConditionKey",
                kind="mergesort",
            ),
        )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": "PROJECT_11_FULL_EXPERIMENT_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "PriorProjectConditionOutputsAccessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()

# --------------------------------------------------------------------------------------------------
# 9. FINAL 270-CONDITION REVALIDATION
# --------------------------------------------------------------------------------------------------

print("\nValidating all 270 completed conditions.")

inventory_records = []
condition_audits = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for plan_row in condition_plan.itertuples(index=False):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Final validation failed for {plan_row.ConditionID}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    project_run_frames.append(pd.read_csv(condition_dir / "project_runs.csv"))
    build_metric_frames.append(pd.read_csv(condition_dir / "build_metrics.csv"))
    model_fit_frames.append(pd.read_csv(condition_dir / "model_fits.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

condition_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)

baseline_invariance_records = []
for repetition_seed in REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~baseline_invariance["Pass"]).sum())
qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

raw_training_hash_after = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_after = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_after = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_after = sha256_file(MODEL_EVALUATION_COHORT_PATH)
registry_sha256_after = sha256_file(REGISTRY_PATH)

current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]

noise_plan_hash_mismatches = int(
    combined_condition_audit[
        "ExpectedFlipMaskSHA256"
    ].ne(combined_condition_audit["ActualFlipMaskSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyRawVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyRawVerdictSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyModelVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyModelVerdictSHA256"]).sum()
)

project_metric_columns = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
project_metric_values = combined_project_runs[
    project_metric_columns
].to_numpy(dtype=float)

if not ACCELERATED_EQUIVALENCE_PATH.is_file():
    raise FileNotFoundError(
        "Accelerated-engine equivalence audit is missing."
    )
accelerated_equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)
accelerated_equivalence_passes = int(
    accelerated_equivalence["Pass"].astype(bool).sum()
)

validation_records = []
add_check(validation_records, "Step 4B passed", EXPECTED_STEP4B_STATUS, smoke_checkpoint.get("Status"), smoke_checkpoint.get("Status") == EXPECTED_STEP4B_STATUS)
add_check(validation_records, "Smoke checkpoint SHA-256", EXPECTED_SMOKE_CHECKPOINT_SHA256, smoke_checkpoint_sha256, smoke_checkpoint_sha256 == EXPECTED_SMOKE_CHECKPOINT_SHA256)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Accelerated smoke-equivalence rows", 2, len(accelerated_equivalence), len(accelerated_equivalence) == 2)
add_check(validation_records, "Accelerated smoke-equivalence keys", sorted(SMOKE_EQUIVALENCE_KEYS), sorted(accelerated_equivalence["ConditionKey"].tolist()), sorted(accelerated_equivalence["ConditionKey"].tolist()) == sorted(SMOKE_EQUIVALENCE_KEYS))
add_check(validation_records, "Accelerated smoke-equivalence failures", 0, int((~accelerated_equivalence["Pass"].astype(bool)).sum()), accelerated_equivalence["Pass"].astype(bool).all())
add_check(validation_records, "Completed conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation_records, "Noise levels", NOISE_LEVELS, sorted(condition_inventory["NoisePercent"].unique().tolist()), sorted(condition_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Repetition seeds", REPETITION_SEEDS, sorted(condition_inventory["RepetitionSeed"].unique().tolist()), sorted(condition_inventory["RepetitionSeed"].unique().tolist()) == REPETITION_SEEDS)
add_check(validation_records, "Duplicate condition keys", 0, int(condition_inventory["ConditionKey"].duplicated(keep=False).sum()), not condition_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Duplicate condition coordinates", 0, int(condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).sum()), not condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).any())
add_check(validation_records, "Condition-order sequence", list(range(1, EXPECTED_CONDITIONS + 1)), condition_inventory["ConditionOrder"].tolist(), condition_inventory["ConditionOrder"].tolist() == list(range(1, EXPECTED_CONDITIONS + 1)))
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(condition_inventory["Files"].unique().tolist()), condition_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation_records, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation_records, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation_records, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation_records, "Model fits", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation_records, "Condition-audit rows", EXPECTED_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_CONDITIONS)
add_check(validation_records, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Active predictors", EXPECTED_PREDICTORS, sorted(combined_condition_audit["Predictors"].unique().tolist()), combined_condition_audit["Predictors"].eq(EXPECTED_PREDICTORS).all())
add_check(validation_records, "Condition statuses", [CONDITION_STATUS], sorted(combined_condition_audit["Status"].unique().tolist()), combined_condition_audit["Status"].eq(CONDITION_STATUS).all())
add_check(validation_records, "Model-fit failures", 0, int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()), combined_model_fits["Status"].eq("PASS_MODEL_FIT").all())
add_check(validation_records, "Project-run technique set", sorted(ALL_TECHNIQUES), sorted(combined_project_runs["Technique"].unique().tolist()), sorted(combined_project_runs["Technique"].unique().tolist()) == sorted(ALL_TECHNIQUES))
add_check(validation_records, "Model-fit technique set", sorted(ML_TECHNIQUES), sorted(combined_model_fits["Technique"].unique().tolist()), sorted(combined_model_fits["Technique"].unique().tolist()) == sorted(ML_TECHNIQUES))
add_check(validation_records, "Zero-noise conditions", len(REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "QTF global score variants", 1, qtf_global_score_variants, qtf_global_score_variants == 1)
add_check(validation_records, "QTF global rank variants", 1, qtf_global_rank_variants, qtf_global_rank_variants == 1)
add_check(validation_records, "Project metrics non-finite", 0, int((~np.isfinite(project_metric_values)).sum()), np.isfinite(project_metric_values).all())
add_check(validation_records, "Project metrics outside [0,1]", 0, int(((project_metric_values < 0) | (project_metric_values > 1)).sum()), bool(((project_metric_values >= 0) & (project_metric_values <= 1)).all()))
add_check(validation_records, "Raw training cohort unchanged", raw_training_hash_before, raw_training_hash_after, raw_training_hash_after == raw_training_hash_before)
add_check(validation_records, "Raw evaluation cohort unchanged", raw_evaluation_hash_before, raw_evaluation_hash_after, raw_evaluation_hash_after == raw_evaluation_hash_before)
add_check(validation_records, "Model training cohort unchanged", model_training_hash_before, model_training_hash_after, model_training_hash_after == model_training_hash_before)
add_check(validation_records, "Model evaluation cohort unchanged", model_evaluation_hash_before, model_evaluation_hash_after, model_evaluation_hash_after == model_evaluation_hash_before)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Registry rows", EXPECTED_REGISTERED_PROJECTS, len(registry), len(registry) == EXPECTED_REGISTERED_PROJECTS)
for predecessor_number, predecessor_project in required_registered_identities.items():
    predecessor_actual = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            project_column,
        ].iloc[0]
    )
    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        predecessor_actual,
        predecessor_actual == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    runtime_checkpoint.get("ActiveReservations"),
    runtime_checkpoint.get("ActiveReservations") == EXPECTED_ACTIVE_RESERVATIONS,
)
add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)
add_check(
    validation_records,
    "Smoke checkpoint runtime-contract linkage",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    ),
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    )
    == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(validation_records, "Registry Project 16 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

validation = pd.DataFrame(validation_records)
failed_validation = validation.loc[~validation["Pass"]]

print("\nStep 5A validation:")
display(validation)

if not failed_validation.empty:
    print("\nFailed Step 5A checks:")
    display(failed_validation)
    print("\nCompleted condition checkpoints remain resume-safe.")
    raise RuntimeError(
        "PROJECT 16 STEP 5A FINAL VALIDATION FAILED."
    )

# --------------------------------------------------------------------------------------------------
# 10. FREEZE FULL RAW ROOT AND STEP 5A CHECKPOINT
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(STEP5A_VALIDATION_PATH, validation)

full_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    STEP5A_VALIDATION_PATH,
]
aggregate_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in aggregate_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "AcceleratedSmokeEquivalenceRows": len(accelerated_equivalence),
    "AcceleratedSmokeEquivalenceFailures": int((~accelerated_equivalence["Pass"].astype(bool)).sum()),
    "AcceleratedEquivalenceAudit": str(ACCELERATED_EQUIVALENCE_PATH),
    "CompletedAtUTC": completed_at_utc,
    "SmokeCheckpointSHA256": smoke_checkpoint_sha256,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "RawRoot": str(FULL_RAW_RESULT_ROOT),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "RawManifest": str(RAW_MANIFEST_PATH),
    "RawManifestSHA256": sha256_file(RAW_MANIFEST_PATH),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisRun": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "FullExecutionSecondsThisInvocation": float(full_execution_seconds),
    "AggregateOutputManifest": aggregate_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To15Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "EvaluationCohortImmutable": True,
    "ResumeSafe": True,
}
atomic_json(STEP5A_REPORT_PATH, report_payload)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint_payload)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "Conditions": len(condition_inventory),
    "MLFits": len(combined_model_fits),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "FailedValidationChecks": len(failed_validation),
    "RegistryModified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5A_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_CONDITIONS,
        "ExpectedConditions": EXPECTED_CONDITIONS,
        "RawRootSHA256": raw_root_sha256,
        "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "RegistryModified": False,
        "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

        "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,
    },
)

checkpoint_readback = load_json(STEP5A_CHECKPOINT_PATH)
status_readback = load_json(STEP5A_STATUS_PATH)
if checkpoint_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A checkpoint readback failed.")
if status_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A status readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during Step 5A finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 16 CELL 9 / STEP 5A ACCELERATED RESULT ===")
print("=" * 136)
print()
print("Project:", PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[14],
)
print(
    "Project 15 identity:",
    required_registered_identities[15],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Accelerated engine:")
print("Engine version:", ACCELERATED_ENGINE_VERSION)
print("Clean REC mismatches:", accelerated_clean_mismatch_values)
print("Frozen smoke-equivalence failures:", int((~accelerated_equivalence["Pass"].astype(bool)).sum()))
print()
print("Full experiment:")
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Raw result freeze:")
print("Raw files:", raw_files)
print("Raw bytes:", raw_bytes)
print("Raw root SHA-256:", raw_root_sha256)
print()
print("Checkpoint/resume:")
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Resume safe:", True)
print()
print("Baselines and metrics:")
print("Baseline invariance failures:", baseline_invariance_failures)
print("QTF global score variants:", qtf_global_score_variants)
print("QTF global rank variants:", qtf_global_rank_variants)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 16 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–15 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Step 5A checkpoint:")
print(STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print()
print("Runtime seconds this invocation:", round(full_execution_seconds, 2))
print()
print("STATUS:", STEP5A_STATUS)
print("=" * 136)


=== PROJECT 16 CELL 9 / STEP 5A V2: CHECKPOINT-SCHEMA-COMPATIBLE FULL 270-CONDITION EXPERIMENT ===

Loading frozen Project 16 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 219 / 6548 / 781

Scanning existing condition checkpoints...
Valid completed conditions: 0
Incomplete/invalid condition directories: 0
Pending conditions: 270

Loading deterministic RNG stream for seed 1.

--------------------------------------------------------------------------------------------------------------
[1/270] Running noise_00__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_00__seed_01
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 105 | condition seconds: 27.72

--------------------------------------------------------------------------------------------------------------
[9/270] Running noise_50__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_50__seed_01
  Completed: noise_50__seed_01
  Raw flips: 35579 | model-label changes: 2472 | dependent REC changes: 71949
  Training failures: 2489 | condition seconds: 32.81

--------------------------------------------------------------------------------------------------------------
[2/270] Running noise_05__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_30
  Raw flips: 28327 | model-label changes: 2012 | dependent REC changes: 70134
  Training failures: 2037 | condition seconds: 10.08

--------------------------------------------------------------------------------------------------------------
[270/270] Running noise_50__seed_30
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_30
  Raw flips: 35414 | model-label changes: 2476 | dependent REC changes: 71761
  Training failures: 2491 | condition seconds: 6.58

Validating all 270 completed conditions.

Step 5A validation:


,Check,Expected,Actual,Pass
0,Step 4B passed,PASS_PROJECT_16_TWO_CONDITION_END_TO_END_SMOKE...,PASS_PROJECT_16_TWO_CONDITION_END_TO_END_SMOKE...,True
1,Smoke checkpoint SHA-256,b2921d447ed5ac57badbdae55dc79830e45d18ea541c21...,b2921d447ed5ac57badbdae55dc79830e45d18ea541c21...,True
2,Accelerated clean REC mismatches,0,0,True
3,Accelerated smoke-equivalence rows,2,2,True
4,Accelerated smoke-equivalence keys,"[noise_00__seed_01, noise_50__seed_01]","[noise_00__seed_01, noise_50__seed_01]",True
5,Accelerated smoke-equivalence failures,0,0,True
6,Completed conditions,270,270,True
7,Noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
8,Repetition seeds,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
9,Duplicate condition keys,0,0,True



=== PROJECT 16 CELL 9 / STEP 5A ACCELERATED RESULT ===

Project: apache@rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Accelerated engine:
Engine version: PROJECT_16_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER
Clean REC mismatches: 0
Frozen smoke-equivalence failures: 0

Full experiment:
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 3101490
Build-metric rows: 15120
Project-run rows: 1890
Condition-audit rows: 270
Training-median rows: 40770

Raw result freeze:
Raw files: 2160
Raw bytes: 53410207
Raw root SHA-256: 04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085

Checkpoint/resume:
Completed this invocation: 270
Skipped validat

In [10]:
# ==================================================================================================
# PROJECT 16 — CELL 10 / STEP 5B
# CORRECTED PROJECT-SPECIFIC COUNT CONTRACT, RAW REVALIDATION, AND COMPACT AGGREGATION
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# CURRENT REGISTRY CONTRACT:
# - Projects 1–15 must be present exactly once and COMPLETE_AND_FROZEN.
# - Project 11 must be apache@shardingsphere.
# - Project 12 must be zolyfarkas@spf4j.
# - Project 13 must be jcabi@jcabi-github.
# - Project 14 must be JMRI@JMRI.
# - Project 15 must be eclipse@steady.
# - Project 16 must still be absent.
#
# THIS CELL:
# - independently hashes all 2,160 Project 16 raw files;
# - validates every condition checkpoint and compact output;
# - recounts all 3,101,490 compressed ranking rows;
# - independently validates noise hashes, REC invariance, metrics, and baselines;
# - creates analysis-ready aggregates across all 30 seeds;
# - writes the Project 16 Step 5B checkpoint;
# - does not rerun conditions or fit models;
# - does not access or modify prior-project condition outputs;
# - does not register Project 16.
#
# BASELINE-INVARIANCE CONTRACT:
# - Random and QTF-Avg are compared independently within each metric;
# - APFDc and APFD are never compared against one another.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gzip, hashlib, json, os, time
import numpy as np
import pandas as pd

print('=' * 136)
print('=== PROJECT 16 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===')
print('=' * 136)

PROJECT_NUMBER = 16
PROJECT_NAME = 'apache@rocketmq'
PROJECT_SLUG = 'apache__rocketmq'
PROJECT_SHORT = 'ROCKETMQ'
STEP5A_STATUS = 'PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_COMPLETE'
CONDITION_STATUS = 'PASS_FULL_CONDITION'
STEP5B_STATUS = 'PASS_PROJECT_16_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN'

EXPECTED_STEP5A_SHA = 'bdb17f099601882d4d5f6e9e0818501341a5f74de26a6ed1146f6e6ab5f2649d'
EXPECTED_RAW_ROOT_SHA = '04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085'
EXPECTED_REGISTRY_SHA = '2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085'
EXPECTED_SOURCE_ROOT_SHA = 'e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e'

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
ML_TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
INVARIANT_BASELINES = ['Random', 'QTF-Avg']
PROJECT_METRICS = ['MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']
BUILD_METRICS = ['APFDc', 'APFD']

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 53_410_207
EXPECTED_RANKING_ROWS_PER_CONDITION = 1_641 * 7
EXPECTED_BUILD_ROWS_PER_CONDITION = 8 * 7
EXPECTED_PROJECT_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_TOTAL_RANKING_ROWS = 3_101_490
EXPECTED_TOTAL_BUILD_ROWS = 15_120
EXPECTED_TOTAL_PROJECT_ROWS = 1_890
EXPECTED_TOTAL_FIT_ROWS = 1_080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40_770

# Project 16 fixed evaluation counts.
# These are validated in Step 1A/1B, Step 2A, Step 4A, Step 4B, and Step 5A.
EXPECTED_SCORED_FAILING_BUILDS = 8
EXPECTED_EVALUATION_BUILDS = 134
EXPECTED_EVALUATION_FAILURES = 9

EXPECTED_CONDITION_FILES = {
    'rankings.csv.gz', 'build_metrics.csv', 'project_runs.csv', 'model_fits.csv',
    'training_medians.csv', 'condition_audit.csv', 'condition_summary.json', 'COMPLETE.json'
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {'condition_summary.json', 'COMPLETE.json'}

# Mount only Drive. No source re-extraction is needed for Step 5B.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Thesis_Experiment')
NOTES = ROOT / 'Notes'
RESULTS = ROOT / 'Results'
REGISTRY = NOTES / 'completed_project_registry.csv'
PROJECT_ROOT = RESULTS / 'Aggregated' / PROJECT_SLUG
RAW_ROOT = RESULTS / 'Raw' / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f'{PROJECT_SHORT}_full_experiment'
PLAN = PROJECT_ROOT / f'{PROJECT_SHORT}_noise_plan' / f'{PROJECT_SHORT}_condition_plan.csv'
STEP5A_CHECKPOINT = NOTES / 'project_16_step5a_checkpoint.json'
STEP5A_STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5a_status.json'
STEP5A_REPORT = FULL_ROOT / f'{PROJECT_SHORT}_step5a_report.json'
STEP5A_RAW_MANIFEST = FULL_ROOT / f'{PROJECT_SHORT}_raw_manifest.csv'
STEP5A_BASELINE = FULL_ROOT / f'{PROJECT_SHORT}_baseline_invariance.csv'

OUT = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b'
CURRENT_MANIFEST = OUT / f'{PROJECT_SHORT}_independent_raw_manifest.csv'
CONDITION_INVENTORY = OUT / f'{PROJECT_SHORT}_independent_condition_inventory.csv'
REVALIDATED_PROJECT_RUNS = OUT / f'{PROJECT_SHORT}_revalidated_project_runs.csv'
REVALIDATED_BUILD_METRICS = OUT / f'{PROJECT_SHORT}_revalidated_build_metrics.csv'
REVALIDATED_MODEL_FITS = OUT / f'{PROJECT_SHORT}_revalidated_model_fits.csv'
REVALIDATED_CONDITION_AUDIT = OUT / f'{PROJECT_SHORT}_revalidated_condition_audit.csv'
REVALIDATED_MEDIANS = OUT / f'{PROJECT_SHORT}_revalidated_training_medians.csv'
NOISE_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_technique_summary.csv'
SEED_DELTAS = OUT / f'{PROJECT_SHORT}_seed_level_noise_deltas.csv'
DELTA_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_delta_summary.csv'
VALIDATION_PATH = OUT / f'{PROJECT_SHORT}_step5b_validation.csv'
REPORT_PATH = OUT / f'{PROJECT_SHORT}_step5b_report.json'
STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b_status.json'
CHECKPOINT_PATH = NOTES / 'project_16_step5b_checkpoint.json'


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write('\n')
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    df.to_csv(tmp, index=False, lineterminator='\n')
    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise RuntimeError(f'Could not resolve {label}; columns={list(columns)}')


def normalize_manifest(df, label):
    p = resolve_col(df.columns, ['RelativePath'], f'{label} path')
    b = resolve_col(df.columns, ['Bytes', 'SizeBytes'], f'{label} bytes')
    s = resolve_col(df.columns, ['SHA256'], f'{label} sha')
    out = df[[p, b, s]].copy(); out.columns = ['RelativePath', 'Bytes', 'SHA256']
    out['RelativePath'] = out['RelativePath'].astype(str).str.replace('\\', '/', regex=False)
    out['Bytes'] = pd.to_numeric(out['Bytes'], errors='raise').astype('int64')
    out['SHA256'] = out['SHA256'].astype(str).str.lower()
    return out.sort_values('RelativePath', kind='mergesort').reset_index(drop=True)


def root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values('RelativePath', kind='mergesort').itertuples(index=False):
        h.update(f'{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n'.encode('utf-8'))
    return h.hexdigest()


def gzip_rows(path):
    n = 0
    with gzip.open(path, 'rb') as f:
        for _ in f:
            n += 1
    return max(0, n - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Pass': bool(passed)})


def metric_nonfinite(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int((~np.isfinite(arr)).sum())


def metric_outside(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int(((arr < 0) | (arr > 1)).sum())


# Required Drive inputs.
required = [REGISTRY, PLAN, STEP5A_CHECKPOINT, STEP5A_STATUS_PATH, STEP5A_REPORT,
            STEP5A_RAW_MANIFEST, STEP5A_BASELINE, RAW_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Project 16 Step 5B inputs:\n' + '\n'.join(missing))

# Frozen Step 5A and registry.
step5a_sha = sha256_file(STEP5A_CHECKPOINT)
step5a = load_json(STEP5A_CHECKPOINT)
step5a_status = load_json(STEP5A_STATUS_PATH)
step5a_report = load_json(STEP5A_REPORT)
if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(f'Step 5A checkpoint SHA differs. Expected={EXPECTED_STEP5A_SHA}; actual={step5a_sha}')
for label, payload in [('checkpoint', step5a), ('status', step5a_status), ('report', step5a_report)]:
    if payload.get('Status') != STEP5A_STATUS:
        raise RuntimeError(f'Step 5A {label} is not in PASS state.')
if step5a.get('SourceRootSHA256') != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError('Step 5A source-root SHA differs.')
if step5a.get('RawRootSHA256') != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError('Step 5A frozen raw-root SHA differs.')

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f'Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA}; actual={registry_sha_before}')
registry = pd.read_csv(REGISTRY, dtype=str).fillna('')
pn_col = resolve_col(registry.columns, ['ProjectNumber'], 'registry ProjectNumber')
project_col = resolve_col(registry.columns, ['Project'], 'registry Project')
st_col = resolve_col(registry.columns, ['Status'], 'registry Status')
pnums = pd.to_numeric(registry[pn_col], errors='raise').astype(int)

if len(registry) != 15 or sorted(pnums.tolist()) != list(range(1, 16)):
    raise RuntimeError(
        'Registry must contain exactly Projects 1–15 before Project 16 Step 5B.'
    )

if not registry[st_col].eq('COMPLETE_AND_FROZEN').all():
    raise RuntimeError(
        'Projects 1–15 are not all COMPLETE_AND_FROZEN.'
    )

required_registered_identities = {
    11: 'apache@shardingsphere',
    12: 'zolyfarkas@spf4j',
    13: 'jcabi@jcabi-github',
    14: 'JMRI@JMRI',
    15: 'eclipse@steady',
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(required_number)
    ]

    if (
        len(matching_rows) != 1
        or matching_rows.iloc[0][project_col] != required_project
    ):
        raise RuntimeError(
            'A required frozen predecessor has a different registry identity.\n'
            f'Project number: {required_number}\n'
            f'Expected project: {required_project}'
        )

if pnums.eq(16).any() or registry[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError(
        'Project 16 is unexpectedly already present in the completion registry.'
    )

# Validate Step 5A aggregate-output manifest.
agg_manifest = step5a.get('AggregateOutputManifest', [])
if not isinstance(agg_manifest, list) or not agg_manifest:
    raise RuntimeError('Step 5A checkpoint has no AggregateOutputManifest.')
agg_failures = 0
for item in agg_manifest:
    p = Path(item['Path'])
    ok = p.is_file() and p.stat().st_size == int(item['Bytes']) and sha256_file(p) == str(item['SHA256'])
    agg_failures += int(not ok)
if agg_failures:
    raise RuntimeError(f'{agg_failures} Step 5A aggregate outputs changed.')

# Independently hash all raw files.
print('\nIndependently hashing all 2,160 raw files.')
hash_start = time.perf_counter()
paths = sorted([p for p in RAW_ROOT.rglob('*') if p.is_file()], key=lambda p: p.relative_to(RAW_ROOT).as_posix())
manifest_rows = []
for i, p in enumerate(paths, 1):
    manifest_rows.append({'RelativePath': p.relative_to(RAW_ROOT).as_posix(),
                          'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)})
    if i % 200 == 0 or i == len(paths):
        print(f'  Raw hashing progress: {i} / {len(paths)} files')
current_manifest = normalize_manifest(pd.DataFrame(manifest_rows), 'current manifest')
hash_seconds = time.perf_counter() - hash_start
frozen_manifest = normalize_manifest(pd.read_csv(STEP5A_RAW_MANIFEST), 'frozen manifest')
current_raw_sha = root_hash(current_manifest)
current_raw_bytes = int(current_manifest['Bytes'].sum())
merged_manifest = frozen_manifest.merge(current_manifest, on='RelativePath', how='outer',
                                        suffixes=('_frozen', '_current'), indicator=True)
missing_raw = int(merged_manifest['_merge'].eq('left_only').sum())
unexpected_raw = int(merged_manifest['_merge'].eq('right_only').sum())
size_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['Bytes_frozen'].ne(merged_manifest['Bytes_current'])).sum())
hash_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['SHA256_frozen'].ne(merged_manifest['SHA256_current'])).sum())

# Condition-by-condition independent validation and compact reload.
plan = pd.read_csv(PLAN, low_memory=False)
id_col = resolve_col(plan.columns, ['ConditionID', 'ConditionKey'], 'condition identifier')
order_col = resolve_col(plan.columns, ['ConditionOrder'], 'condition order')
noise_col = resolve_col(plan.columns, ['NoisePercent'], 'noise percent')
seed_col = resolve_col(plan.columns, ['RepetitionSeed'], 'repetition seed')
for col in [order_col, noise_col, seed_col]:
    plan[col] = pd.to_numeric(plan[col], errors='raise').astype(int)
plan = plan.sort_values(order_col, kind='mergesort').reset_index(drop=True)

inventory_rows, project_frames, build_frames, fit_frames, audit_frames, median_frames = [], [], [], [], [], []
marker_fail = summary_fail = file_set_fail = embedded_fail = ranking_count_fail = 0
print('\nRevalidating all 270 condition directories.')
condition_start = time.perf_counter()
for i, row in enumerate(plan.itertuples(index=False), 1):
    key = str(getattr(row, id_col)); order = int(getattr(row, order_col))
    noise = int(getattr(row, noise_col)); seed = int(getattr(row, seed_col))
    d = RAW_ROOT / key
    if not d.is_dir():
        raise FileNotFoundError(f'Missing condition directory: {d}')
    actual_files = {p.name for p in d.iterdir() if p.is_file()}
    file_ok = actual_files == EXPECTED_CONDITION_FILES
    file_set_fail += int(not file_ok)
    complete_path, summary_path = d / 'COMPLETE.json', d / 'condition_summary.json'
    complete, summary = load_json(complete_path), load_json(summary_path)
    complete_ok = (complete.get('Status') == CONDITION_STATUS and complete.get('ConditionKey') == key and
                   str(complete.get('ConditionSummaryPath')) == str(summary_path) and
                   str(complete.get('ConditionSummarySHA256')).lower() == sha256_file(summary_path))
    summary_ok = (summary.get('Status') == CONDITION_STATUS and summary.get('ConditionKey') == key and
                  int(summary.get('NoisePercent', -1)) == noise and int(summary.get('RepetitionSeed', -1)) == seed)
    marker_fail += int(not complete_ok); summary_fail += int(not summary_ok)
    output_manifest = summary.get('OutputManifest', [])
    local_embedded_fail = 0
    names = set()
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        local_embedded_fail += 1
    else:
        for item in output_manifest:
            p = Path(item.get('Path', '')); names.add(p.name)
            ok = (p.parent == d and p.is_file() and p.stat().st_size == int(item.get('Bytes', -1)) and
                  sha256_file(p) == str(item.get('SHA256', '')).lower())
            local_embedded_fail += int(not ok)
        local_embedded_fail += int(names != CONDITION_OUTPUT_FILES)
    embedded_fail += local_embedded_fail
    ranking_rows = gzip_rows(d / 'rankings.csv.gz')
    ranking_count_fail += int(ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION)
    build = pd.read_csv(d / 'build_metrics.csv', low_memory=False)
    project = pd.read_csv(d / 'project_runs.csv', low_memory=False)
    fits = pd.read_csv(d / 'model_fits.csv', low_memory=False)
    audit = pd.read_csv(d / 'condition_audit.csv', low_memory=False)
    medians = pd.read_csv(d / 'training_medians.csv', low_memory=False)
    expected_counts = [EXPECTED_BUILD_ROWS_PER_CONDITION, EXPECTED_PROJECT_ROWS_PER_CONDITION,
                       EXPECTED_FIT_ROWS_PER_CONDITION, 1, EXPECTED_MEDIAN_ROWS_PER_CONDITION]
    actual_counts = [len(build), len(project), len(fits), len(audit), len(medians)]
    if actual_counts != expected_counts:
        raise RuntimeError(f'{key}: compact output counts differ. expected={expected_counts}; actual={actual_counts}')
    for field, count in [('RankingRows', ranking_rows), ('BuildMetricRows', len(build)),
                         ('ProjectRunRows', len(project)), ('MLFits', len(fits)),
                         ('TrainingMedianRows', len(medians))]:
        if int(summary.get(field, -1)) != count:
            raise RuntimeError(f'{key}: condition_summary {field} differs.')
    project_frames.append(project); build_frames.append(build); fit_frames.append(fits)
    audit_frames.append(audit); median_frames.append(medians)
    inventory_rows.append({
        'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
        'ConditionOrder': order, 'ConditionKey': key, 'NoisePercent': noise, 'RepetitionSeed': seed,
        'ConditionDirectory': str(d), 'CompletionStatus': complete.get('Status'),
        'SummaryStatus': summary.get('Status'), 'Files': len(actual_files),
        'ConditionBytes': int(sum(p.stat().st_size for p in d.iterdir() if p.is_file())),
        'RankingRows': ranking_rows, 'BuildMetricRows': len(build), 'ProjectRunRows': len(project),
        'ModelFits': len(fits), 'ConditionAuditRows': len(audit), 'TrainingMedianRows': len(medians),
        'FileSetPass': file_ok, 'CompletionMarkerPass': complete_ok, 'ConditionSummaryPass': summary_ok,
        'EmbeddedManifestFailures': local_embedded_fail,
        'CompletionMarkerSHA256': sha256_file(complete_path), 'ConditionSummarySHA256': sha256_file(summary_path),
    })
    if i % 30 == 0 or i == len(plan):
        print(f'  Condition revalidation progress: {i} / {len(plan)}')
condition_seconds = time.perf_counter() - condition_start

inventory = pd.DataFrame(inventory_rows).sort_values('ConditionOrder', kind='mergesort').reset_index(drop=True)
project_runs = pd.concat(project_frames, ignore_index=True)
build_metrics = pd.concat(build_frames, ignore_index=True)
model_fits = pd.concat(fit_frames, ignore_index=True)
condition_audit = pd.concat(audit_frames, ignore_index=True)
training_medians = pd.concat(median_frames, ignore_index=True)

# Contract audits.
coordinate_count = len(inventory[['NoisePercent', 'RepetitionSeed']].drop_duplicates())
dup_keys = int(inventory.duplicated(['ConditionKey'], keep=False).sum())
dup_coords = int(inventory.duplicated(['NoisePercent', 'RepetitionSeed'], keep=False).sum())
order_viol = int((inventory['ConditionOrder'].to_numpy(int) != np.arange(1, 271)).sum())
files_viol = int(inventory['Files'].ne(8).sum())
ranking_viol = int(inventory['RankingRows'].ne(EXPECTED_RANKING_ROWS_PER_CONDITION).sum())
small_per_condition_viol = int(inventory['BuildMetricRows'].ne(EXPECTED_BUILD_ROWS_PER_CONDITION).sum() +
                               inventory['ProjectRunRows'].ne(7).sum() + inventory['ModelFits'].ne(4).sum() +
                               inventory['ConditionAuditRows'].ne(1).sum() + inventory['TrainingMedianRows'].ne(151).sum())
project_techniques = sorted(project_runs['Technique'].astype(str).unique().tolist())
fit_techniques = sorted(model_fits['Technique'].astype(str).unique().tolist())
dup_project = int(project_runs.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_build = int(build_metrics.duplicated(['ConditionKey', 'Technique', 'Build'], keep=False).sum())
dup_fit = int(model_fits.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_audit = int(condition_audit.duplicated(['ConditionKey'], keep=False).sum())
dup_median = int(training_medians.duplicated(['ConditionKey', 'PredictorOrder'], keep=False).sum())
fit_fail = int((~model_fits['Status'].astype(str).eq('PASS_MODEL_FIT')).sum())
fit_errors = int(model_fits['Error'].fillna('').astype(str).str.len().gt(0).sum())
project_nonfinite = metric_nonfinite(project_runs, PROJECT_METRICS)
project_outside = metric_outside(project_runs, PROJECT_METRICS)
build_nonfinite = metric_nonfinite(build_metrics, BUILD_METRICS)
build_outside = metric_outside(build_metrics, BUILD_METRICS)
median_nonfinite = int((~np.isfinite(pd.to_numeric(training_medians['TrainingMedian'], errors='coerce').to_numpy(float))).sum())
median_predictor_viol = int(training_medians.groupby('ConditionKey')['Predictor'].nunique().ne(151).sum())
scored_build_viol = int(
    project_runs[
        'ScoredFailingBuilds'
    ].ne(
        EXPECTED_SCORED_FAILING_BUILDS
    ).sum()
)

eval_build_viol = int(
    project_runs[
        'EvaluationBuilds'
    ].ne(
        EXPECTED_EVALUATION_BUILDS
    ).sum()
)

eval_failure_viol = int(
    project_runs[
        'EvaluationFailures'
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)
zero = condition_audit[condition_audit['NoisePercent'].eq(0)]
positive = condition_audit[condition_audit['NoisePercent'].gt(0)]
zero_flip_viol = int(zero['NumberFlipped'].ne(0).sum())
zero_model_viol = int(zero['ModelLabelChanges'].ne(0).sum())
zero_rec_viol = int(zero['DependentRECChanges'].ne(0).sum())
pos_raw_viol = int(positive['NumberFlipped'].le(0).sum())
pos_model_viol = int(positive['ModelLabelChanges'].le(0).sum())
pos_rec_viol = int(positive['DependentRECChanges'].le(0).sum())
independent_viol = int(condition_audit['IndependentRECChanges'].ne(0).sum())
independent_recon_viol = int(condition_audit['IndependentReconstructionMismatches'].ne(0).sum())
noise_hash_mismatch = 0
for e, a in [('ExpectedFlipMaskSHA256', 'ActualFlipMaskSHA256'),
             ('ExpectedNoisyRawVerdictSHA256', 'ActualNoisyRawVerdictSHA256'),
             ('ExpectedNoisyModelVerdictSHA256', 'ActualNoisyModelVerdictSHA256')]:
    noise_hash_mismatch += int((condition_audit[e].astype(str) != condition_audit[a].astype(str)).sum())

baseline = pd.read_csv(STEP5A_BASELINE, low_memory=False)
if 'Pass' in baseline.columns:
    bpass = baseline['Pass'].astype(str).str.strip().str.lower().isin({'true', '1'})
    ranking_baseline_fail = int((~bpass).sum())
else:
    mismatch_cols = [c for c in baseline.columns if 'mismatch' in c.lower()]
    ranking_baseline_fail = int(baseline[mismatch_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(float).sum())
# Project-metric invariance must be evaluated independently for each metric.
# The previous V1 expression compared the maximum of one metric with the
# minimum of another metric. Because APFDc and APFD naturally have different
# values, that incorrectly marked all 60 seed/baseline groups as failures even
# though each individual metric was invariant across noise.
metric_baseline_fail = 0
metric_baseline_max_range = 0.0

for _, g in project_runs[
    project_runs['Technique'].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        'RepetitionSeed',
        'Technique',
    ],
    sort=False,
):
    arr = g[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            arr,
            axis=0,
        )
        - np.min(
            arr,
            axis=0,
        )
    )

    metric_baseline_max_range = max(
        metric_baseline_max_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    metric_baseline_fail += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )

# Compact aggregates.
agg_start = time.perf_counter()
noise_summary = project_runs.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Runs=('ConditionKey', 'count'), Seeds=('RepetitionSeed', 'nunique'),
    Mean_MeanAPFDc=('MeanAPFDc', 'mean'), SD_MeanAPFDc=('MeanAPFDc', 'std'), Median_MeanAPFDc=('MeanAPFDc', 'median'),
    Mean_MedianAPFDc=('MedianAPFDc', 'mean'), SD_MedianAPFDc=('MedianAPFDc', 'std'), Median_MedianAPFDc=('MedianAPFDc', 'median'),
    Mean_MeanAPFD=('MeanAPFD', 'mean'), SD_MeanAPFD=('MeanAPFD', 'std'), Median_MeanAPFD=('MeanAPFD', 'median'),
    Mean_MedianAPFD=('MedianAPFD', 'mean'), SD_MedianAPFD=('MedianAPFD', 'std'), Median_MedianAPFD=('MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
clean = project_runs[project_runs['NoisePercent'].eq(0)][['RepetitionSeed', 'Technique'] + PROJECT_METRICS].rename(
    columns={c: f'Clean_{c}' for c in PROJECT_METRICS})
seed_deltas = project_runs.merge(clean, on=['RepetitionSeed', 'Technique'], how='left', validate='many_to_one')
for c in PROJECT_METRICS:
    seed_deltas[f'Delta_{c}'] = seed_deltas[c] - seed_deltas[f'Clean_{c}']
delta_cols = [f'Delta_{c}' for c in PROJECT_METRICS]
seed_deltas = seed_deltas[['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent',
                           'RepetitionSeed', 'Technique'] + PROJECT_METRICS +
                          [f'Clean_{c}' for c in PROJECT_METRICS] + delta_cols].sort_values(
                              ['NoisePercent', 'Technique', 'RepetitionSeed'], kind='mergesort').reset_index(drop=True)
delta_summary = seed_deltas.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Seeds=('RepetitionSeed', 'nunique'),
    Mean_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'mean'), SD_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'std'), Median_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'median'),
    Mean_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'mean'), SD_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'std'), Median_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'median'),
    Mean_Delta_MeanAPFD=('Delta_MeanAPFD', 'mean'), SD_Delta_MeanAPFD=('Delta_MeanAPFD', 'std'), Median_Delta_MeanAPFD=('Delta_MeanAPFD', 'median'),
    Mean_Delta_MedianAPFD=('Delta_MedianAPFD', 'mean'), SD_Delta_MedianAPFD=('Delta_MedianAPFD', 'std'), Median_Delta_MedianAPFD=('Delta_MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
agg_seconds = time.perf_counter() - agg_start
summary_nonfinite = metric_nonfinite(noise_summary, [c for c in noise_summary.columns if c not in {'NoisePercent', 'Technique'}])
delta_nonfinite = metric_nonfinite(delta_summary, [c for c in delta_summary.columns if c not in {'NoisePercent', 'Technique'}])
clean_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['NoisePercent'].eq(0)][delta_cols].to_numpy(float)) > 1e-15).sum())
baseline_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['Technique'].isin(INVARIANT_BASELINES)][delta_cols].to_numpy(float)) > 1e-15).sum())

# Validation table.
checks = []
add_check(checks, 'Step 5A status', STEP5A_STATUS, step5a.get('Status'), step5a.get('Status') == STEP5A_STATUS)
add_check(checks, 'Step 5A checkpoint SHA-256', EXPECTED_STEP5A_SHA, step5a_sha, step5a_sha == EXPECTED_STEP5A_SHA)
add_check(checks, 'Frozen raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, step5a.get('RawRootSHA256'), step5a.get('RawRootSHA256') == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Independent current raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, current_raw_sha, current_raw_sha == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Step 5A aggregate-manifest failures', 0, agg_failures, agg_failures == 0)
add_check(checks, 'Condition marker failures', 0, marker_fail, marker_fail == 0)
add_check(checks, 'Condition summary failures', 0, summary_fail, summary_fail == 0)
add_check(checks, 'Condition file-set failures', 0, file_set_fail, file_set_fail == 0)
add_check(checks, 'Embedded output-manifest failures', 0, embedded_fail, embedded_fail == 0)
add_check(checks, 'Conditions', 270, len(inventory), len(inventory) == 270)
add_check(checks, 'Condition coordinates', 270, coordinate_count, coordinate_count == 270)
add_check(checks, 'Duplicate condition keys', 0, dup_keys, dup_keys == 0)
add_check(checks, 'Duplicate condition coordinates', 0, dup_coords, dup_coords == 0)
add_check(checks, 'Condition-order violations', 0, order_viol, order_viol == 0)
add_check(checks, 'Files-per-condition violations', 0, files_viol, files_viol == 0)
add_check(checks, 'Raw files', EXPECTED_RAW_FILES, len(current_manifest), len(current_manifest) == EXPECTED_RAW_FILES)
add_check(checks, 'Raw bytes', EXPECTED_RAW_BYTES, current_raw_bytes, current_raw_bytes == EXPECTED_RAW_BYTES)
add_check(checks, 'Missing raw files', 0, missing_raw, missing_raw == 0)
add_check(checks, 'Unexpected raw files', 0, unexpected_raw, unexpected_raw == 0)
add_check(checks, 'Raw size mismatches', 0, size_mismatch, size_mismatch == 0)
add_check(checks, 'Raw SHA-256 mismatches', 0, hash_mismatch, hash_mismatch == 0)
add_check(checks, 'Ranking rows', EXPECTED_TOTAL_RANKING_ROWS, int(inventory['RankingRows'].sum()), int(inventory['RankingRows'].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(checks, 'Ranking row-count failures', 0, ranking_count_fail + ranking_viol, ranking_count_fail + ranking_viol == 0)
add_check(checks, 'Project-run rows', EXPECTED_TOTAL_PROJECT_ROWS, len(project_runs), len(project_runs) == EXPECTED_TOTAL_PROJECT_ROWS)
add_check(checks, 'Build-metric rows', EXPECTED_TOTAL_BUILD_ROWS, len(build_metrics), len(build_metrics) == EXPECTED_TOTAL_BUILD_ROWS)
add_check(checks, 'Model-fit rows', EXPECTED_TOTAL_FIT_ROWS, len(model_fits), len(model_fits) == EXPECTED_TOTAL_FIT_ROWS)
add_check(checks, 'Condition-audit rows', EXPECTED_TOTAL_AUDIT_ROWS, len(condition_audit), len(condition_audit) == EXPECTED_TOTAL_AUDIT_ROWS)
add_check(checks, 'Training-median rows', EXPECTED_TOTAL_MEDIAN_ROWS, len(training_medians), len(training_medians) == EXPECTED_TOTAL_MEDIAN_ROWS)
add_check(checks, 'Small rows-per-condition violations', 0, small_per_condition_viol, small_per_condition_viol == 0)
add_check(checks, 'Project-run technique set', sorted(TECHNIQUES), project_techniques, project_techniques == sorted(TECHNIQUES))
add_check(checks, 'Model-fit technique set', sorted(ML_TECHNIQUES), fit_techniques, fit_techniques == sorted(ML_TECHNIQUES))
add_check(checks, 'Duplicate project/build/fit/audit/median rows', 0, dup_project + dup_build + dup_fit + dup_audit + dup_median, dup_project + dup_build + dup_fit + dup_audit + dup_median == 0)
add_check(checks, 'Model-fit failures', 0, fit_fail + fit_errors, fit_fail + fit_errors == 0)
add_check(
    checks,
    'Scored-failing-build count violations',
    0,
    scored_build_viol,
    scored_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-build count violations',
    0,
    eval_build_viol,
    eval_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-failure count violations',
    0,
    eval_failure_viol,
    eval_failure_viol == 0,
)

add_check(
    checks,
    'Combined scored/evaluated/failure count violations',
    0,
    scored_build_viol + eval_build_viol + eval_failure_viol,
    scored_build_viol + eval_build_viol + eval_failure_viol == 0,
)
add_check(checks, 'Project metric invalid values', 0, project_nonfinite + project_outside, project_nonfinite + project_outside == 0)
add_check(checks, 'Build metric invalid values', 0, build_nonfinite + build_outside, build_nonfinite + build_outside == 0)
add_check(checks, 'Training-median invalid values', 0, median_nonfinite + median_predictor_viol, median_nonfinite + median_predictor_viol == 0)
add_check(checks, 'Zero-noise conditions', 30, len(zero), len(zero) == 30)
add_check(checks, 'Zero-noise violations', 0, zero_flip_viol + zero_model_viol + zero_rec_viol, zero_flip_viol + zero_model_viol + zero_rec_viol == 0)
add_check(checks, 'Positive-noise violations', 0, pos_raw_viol + pos_model_viol + pos_rec_viol, pos_raw_viol + pos_model_viol + pos_rec_viol == 0)
add_check(checks, 'Independent REC violations', 0, independent_viol + independent_recon_viol, independent_viol + independent_recon_viol == 0)
add_check(checks, 'Noise-plan hash mismatches', 0, noise_hash_mismatch, noise_hash_mismatch == 0)
add_check(checks, 'Ranking-level baseline-invariance failures', 0, ranking_baseline_fail, ranking_baseline_fail == 0)
add_check(checks, 'Project-metric baseline-invariance failures', 0, metric_baseline_fail, metric_baseline_fail == 0)
add_check(checks, 'Noise-technique summary rows', 63, len(noise_summary), len(noise_summary) == 63)
add_check(checks, 'Noise-technique summary count/nonfinite violations', 0, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite == 0)
add_check(checks, 'Seed-level delta rows', 1890, len(seed_deltas), len(seed_deltas) == 1890)
add_check(checks, 'Clean delta non-zero values', 0, clean_delta_nonzero, clean_delta_nonzero == 0)
add_check(checks, 'Invariant-baseline delta non-zero values', 0, baseline_delta_nonzero, baseline_delta_nonzero == 0)
add_check(checks, 'Noise-delta summary rows', 63, len(delta_summary), len(delta_summary) == 63)
add_check(checks, 'Noise-delta summary count/nonfinite violations', 0, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite == 0)
add_check(checks, 'Registry rows', 15, len(registry), len(registry) == 15)

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(
            required_number
        )
    ]

    add_check(
        checks,
        f'Registry Project {required_number} rows',
        1,
        len(
            matching_rows
        ),
        len(
            matching_rows
        )
        == 1,
    )

    add_check(
        checks,
        f'Project {required_number} frozen identity',
        required_project,
        (
            str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            if len(
                matching_rows
            )
            == 1
            else None
        ),
        (
            len(
                matching_rows
            )
            == 1
            and str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            == required_project
        ),
    )

add_check(
    checks,
    'Registry Project 16 rows',
    0,
    int(
        pnums.eq(
            16
        ).sum()
    ),
    int(
        pnums.eq(
            16
        ).sum()
    )
    == 0,
)

validation = pd.DataFrame(checks)
failed = validation[~validation['Pass']]
print('\nProject 16 Step 5B validation:')
display(validation)
if not failed.empty:
    print('\nFailed checks:'); display(failed)
    raise RuntimeError('PROJECT 16 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.')

# Freeze outputs.
OUT.mkdir(parents=True, exist_ok=True)
for path, frame in [
    (CURRENT_MANIFEST, current_manifest), (CONDITION_INVENTORY, inventory),
    (REVALIDATED_PROJECT_RUNS, project_runs), (REVALIDATED_BUILD_METRICS, build_metrics),
    (REVALIDATED_MODEL_FITS, model_fits), (REVALIDATED_CONDITION_AUDIT, condition_audit),
    (REVALIDATED_MEDIANS, training_medians), (NOISE_SUMMARY, noise_summary),
    (SEED_DELTAS, seed_deltas), (DELTA_SUMMARY, delta_summary), (VALIDATION_PATH, validation),
]:
    atomic_csv(path, frame)
output_paths = [CURRENT_MANIFEST, CONDITION_INVENTORY, REVALIDATED_PROJECT_RUNS,
                REVALIDATED_BUILD_METRICS, REVALIDATED_MODEL_FITS, REVALIDATED_CONDITION_AUDIT,
                REVALIDATED_MEDIANS, NOISE_SUMMARY, SEED_DELTAS, DELTA_SUMMARY, VALIDATION_PATH]
output_manifest = [{'Path': str(p), 'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)} for p in output_paths]
completed = datetime.now(timezone.utc).isoformat()
report = {
    'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
    'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
    'FrozenRawRootSHA256': EXPECTED_RAW_ROOT_SHA, 'IndependentRawRootSHA256': current_raw_sha,
    'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes, 'Conditions': len(inventory),
    'ExpectedScoredFailingBuilds': EXPECTED_SCORED_FAILING_BUILDS,
    'ExpectedEvaluationBuilds': EXPECTED_EVALUATION_BUILDS,
    'ExpectedEvaluationFailures': EXPECTED_EVALUATION_FAILURES,
    'MLFits': len(model_fits), 'RankingRows': int(inventory['RankingRows'].sum()),
    'BuildMetricRows': len(build_metrics), 'ProjectRunRows': len(project_runs),
    'ConditionAuditRows': len(condition_audit), 'TrainingMedianRows': len(training_medians),
    'NoiseTechniqueSummaryRows': len(noise_summary), 'SeedLevelNoiseDeltaRows': len(seed_deltas),
    'NoiseDeltaSummaryRows': len(delta_summary),
    'StandardDeviationDefinition': 'Sample SD across 30 seeds; pandas std, ddof=1',
    'RankingLevelBaselineInvarianceFailures': int(ranking_baseline_fail),
    'ProjectMetricBaselineInvarianceFailures': int(metric_baseline_fail),
    'ProjectMetricBaselineMaximumWithinMetricRange': float(metric_baseline_max_range),
    'RawHashingSeconds': float(hash_seconds), 'ConditionRevalidationSeconds': float(condition_seconds),
    'AggregationSeconds': float(agg_seconds), 'OutputManifest': output_manifest,
    'ValidationChecks': len(validation), 'FailedValidationChecks': len(failed),
    'RegistrySHA256': registry_sha_before,
    'RegistryModified': False,
    'Projects1To15Modified': False,
    'Project15RegistryIdentity': required_registered_identities[15],
    'Project15ConditionOutputsAccessed': False,
    'Project15ConditionOutputsModified': False,
    'PriorProjectConditionOutputsAccessed': False,
    'PriorProjectWriteAttempted': False,
    'ModelsFitted': False,
    'ConditionsRerun': False,
}
atomic_json(REPORT_PATH, report)
checkpoint = {**report, 'CheckpointVersion': 1,
              'CheckpointType': 'PROJECT_16_RAW_REVALIDATION_AND_COMPACT_AGGREGATION',
              'RawResultsRevalidated': True, 'CompactAggregatesFrozen': True,
              'ReadyForFinalPackageAndRegistration': True}
atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
status = {'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
          'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
          'RawRootSHA256': current_raw_sha, 'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes,
          'Conditions': len(inventory), 'MLFits': len(model_fits), 'Checkpoint': str(CHECKPOINT_PATH),
          'CheckpointSHA256': checkpoint_sha,
          'ReadyForFinalPackageAndRegistration': True,
          'RegistryModified': False,
          'Projects1To15Modified': False,
          'Project15ConditionOutputsAccessed': False,
          'Project15ConditionOutputsModified': False,
          'PriorProjectConditionOutputsAccessed': False}
atomic_json(STATUS_PATH, status)

# Readback and immutability.
if load_json(CHECKPOINT_PATH).get('Status') != STEP5B_STATUS or load_json(STATUS_PATH).get('Status') != STEP5B_STATUS:
    raise RuntimeError('Step 5B checkpoint/status readback failed.')
for item in output_manifest:
    p = Path(item['Path'])
    if not p.is_file() or p.stat().st_size != item['Bytes'] or sha256_file(p) != item['SHA256']:
        raise RuntimeError(f'Step 5B output readback failed: {p}')
registry_sha_after = sha256_file(REGISTRY)
if registry_sha_after != registry_sha_before:
    raise RuntimeError('Registry changed during Project 16 Step 5B.')
if sha256_file(STEP5A_CHECKPOINT) != EXPECTED_STEP5A_SHA:
    raise RuntimeError('Step 5A checkpoint changed during Step 5B.')

print('\nNoise-technique summary:')
display(noise_summary)
print('\nNoise-delta summary:')
display(delta_summary)
print('\n' + '=' * 136)
print('=== PROJECT 16 CELL 10 / STEP 5B RESULT ===')
print('=' * 136)

print('Project:', PROJECT_NAME)
print('Project slug:', PROJECT_SLUG)
print('Step 5A checkpoint SHA-256:', step5a_sha)
print('Frozen raw-root SHA-256:', EXPECTED_RAW_ROOT_SHA)
print('Independent current raw-root SHA-256:', current_raw_sha)

print('\nRaw-output revalidation:')
print('Conditions:', len(inventory), '/', EXPECTED_CONDITIONS)
print('Raw files:', len(current_manifest), '/', EXPECTED_RAW_FILES)
print('Raw bytes:', current_raw_bytes, '/', EXPECTED_RAW_BYTES)
print(
    'Missing / unexpected / size / SHA mismatches:',
    missing_raw,
    '/',
    unexpected_raw,
    '/',
    size_mismatch,
    '/',
    hash_mismatch,
)
print('Embedded output-manifest failures:', embedded_manifest_fail)

print('\nExperiment totals:')
print('ML fits:', len(model_fits), '/', EXPECTED_TOTAL_FIT_ROWS)
print(
    'Ranking rows:',
    int(
        inventory[
            'RankingRows'
        ].sum()
    ),
    '/',
    EXPECTED_TOTAL_RANKING_ROWS,
)
print('Build-metric rows:', len(build_metrics), '/', EXPECTED_TOTAL_BUILD_ROWS)
print('Project-run rows:', len(project_runs), '/', EXPECTED_TOTAL_PROJECT_ROWS)
print('Condition-audit rows:', len(condition_audit), '/', EXPECTED_TOTAL_AUDIT_ROWS)
print('Training-median rows:', len(training_medians), '/', EXPECTED_TOTAL_MEDIAN_ROWS)

print('\nAnalysis-ready aggregates:')
print('Noise-technique summary rows:', len(noise_summary))
print('Seed-level noise-delta rows:', len(seed_deltas))
print('Noise-delta summary rows:', len(delta_summary))
print('Sample SD calculated with ddof=1:', True)
print('Ranking-level baseline-invariance failures:', ranking_baseline_fail)
print('Project-metric baseline-invariance failures:', metric_baseline_fail)
print('Maximum within-metric baseline range:', metric_baseline_max_range)

print('\nImmutability and isolation:')
print('Completion registry unchanged:', registry_sha_after == registry_sha_before)
print('Registry Project 11 rows:', int(pnums.eq(11).sum()))
print('Registry Project 12 rows:', int(pnums.eq(12).sum()))
print('Registry Project 13 rows:', int(pnums.eq(13).sum()))
print('Registry Project 14 rows:', int(pnums.eq(14).sum()))
print('Registry Project 15 rows:', int(pnums.eq(15).sum()))
print('Registry Project 16 rows:', int(pnums.eq(16).sum()))
print('Project 11 identity:', required_registered_identities[11])
print('Project 12 identity:', required_registered_identities[12])
print('Project 13 identity:', required_registered_identities[13])
print('Project 14 identity:', required_registered_identities[14])
print('Project 15 identity:', required_registered_identities[15])
print('Projects 1–15 modified:', 0)
print('Project 15 condition outputs accessed:', False)
print('Project 15 condition outputs modified:', False)
print('Prior project condition outputs accessed:', False)
print('Prior project write attempted:', False)
print('Conditions rerun:', False)
print('Models fitted:', False)

print('\nRuntime:')
print('Raw hashing seconds:', round(hash_seconds, 2))
print('Condition revalidation seconds:', round(condition_seconds, 2))
print('Compact aggregation seconds:', round(agg_seconds, 2))

print('\nValidation:')
print('Checks:', len(validation))
print('Failed checks:', len(failed))

print('\nProject 16 Step 5B checkpoint:')
print(CHECKPOINT_PATH)
print('Checkpoint SHA-256:', checkpoint_sha)

print('\nSTATUS:', STEP5B_STATUS)
print('=' * 136)


=== PROJECT 16 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Condition revalidatio

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,bdb17f099601882d4d5f6e9e0818501341a5f74de26a6e...,bdb17f099601882d4d5f6e9e0818501341a5f74de26a6e...,True
2,Frozen raw-root SHA-256,04d5b0e9f621e19141ba19c55914891cfea85b43c2374d...,04d5b0e9f621e19141ba19c55914891cfea85b43c2374d...,True
3,Independent current raw-root SHA-256,04d5b0e9f621e19141ba19c55914891cfea85b43c2374d...,04d5b0e9f621e19141ba19c55914891cfea85b43c2374d...,True
4,Step 5A aggregate-manifest failures,0,0,True
...,...,...,...,...
61,Registry Project 14 rows,1,1,True
62,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
63,Registry Project 15 rows,1,1,True
64,Project 15 frozen identity,eclipse@steady,eclipse@steady,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.299580,0.000000,0.299580,0.309140,0.000000,0.309140,0.103191,0.000000,0.103191,0.120098,0.000000,0.120098
1,0,LightGBM,30,30,0.952212,0.000000,0.952212,0.962415,0.000000,0.962415,0.981944,0.000000,0.981944,0.986520,0.000000,0.986520
2,0,NaiveBayes,30,30,0.476445,0.000000,0.476445,0.479743,0.000000,0.479743,0.856686,0.000000,0.856686,0.891611,0.000000,0.891611
3,0,QTF-Avg,30,30,0.539643,0.000000,0.539643,0.432478,0.000000,0.432478,0.107454,0.000000,0.107454,0.017242,0.000000,0.017242
4,0,Random,30,30,0.467455,0.098294,0.454867,0.451948,0.146717,0.456193,0.477180,0.093971,0.463233,0.453277,0.127221,0.439951
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.622152,0.221303,0.691536,0.616152,0.311446,0.682110,0.675844,0.318592,0.853016,0.680382,0.453034,0.985294
59,50,QTF-Avg,30,30,0.539643,0.000000,0.539643,0.432478,0.000000,0.432478,0.107454,0.000000,0.107454,0.017242,0.000000,0.017242
60,50,Random,30,30,0.467455,0.098294,0.454867,0.451948,0.146717,0.456193,0.477180,0.093971,0.463233,0.453277,0.127221,0.439951
61,50,RandomForest,30,30,0.441423,0.155045,0.440699,0.440729,0.176308,0.420281,0.403882,0.168873,0.386433,0.381810,0.199733,0.397672



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,0.145708,0.221303,0.215092,0.136409,0.311446,0.202367,-0.180843,0.318592,-0.003671,-0.211229,0.453034,0.093683
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.423992,0.148564,-0.407289,-0.450325,0.157189,-0.433522,-0.561600,0.170046,-0.579197,-0.588003,0.200052,-0.572917



=== PROJECT 16 CELL 10 / STEP 5B RESULT ===
Project: apache@rocketmq
Project slug: apache__rocketmq
Step 5A checkpoint SHA-256: bdb17f099601882d4d5f6e9e0818501341a5f74de26a6ed1146f6e6ab5f2649d
Frozen raw-root SHA-256: 04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085
Independent current raw-root SHA-256: 04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 53410207 / 53410207
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0


NameError: name 'embedded_manifest_fail' is not defined

In [11]:
# ==================================================================================================
# PROJECT 16 — CELL 10B / STEP 5B READBACK FINALIZER
# RECOVER THE COMPLETED STEP 5B AFTER THE FINAL DISPLAY-ONLY NameError
#
# RUN THIS AS A NEW CELL BELOW THE FAILED PROJECT 16 STEP 5B CELL.
#
# WHY THIS CELL EXISTS:
# - the expensive raw hashing, all 270 condition revalidations, compact aggregation,
#   validation, output writes, checkpoint write, status write, and readback checks
#   completed before the original cell reached its final console-print block;
# - the original cell then failed only because the final display used the undefined
#   variable name `embedded_manifest_fail` instead of the existing `embedded_fail`;
# - this cell independently validates the already-written Step 5B checkpoint, status,
#   report, validation table, and compact-output manifest;
# - it does not rehash the 2,160 raw files, rerun conditions, fit models, or rewrite
#   the frozen Step 5B checkpoint.
# ==================================================================================================

from google.colab import drive

from pathlib import Path

import hashlib
import json

import pandas as pd


print("=" * 136)
print("=== PROJECT 16 CELL 10B / STEP 5B READBACK FINALIZER ===")
print("=" * 136)


PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

STEP5A_STATUS = (
    "PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)

STEP5B_STATUS = (
    "PASS_PROJECT_16_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
)

EXPECTED_STEP5A_SHA256 = (
    "bdb17f099601882d4d5f6e9e0818501341a5f74de26a6ed1146f6e6ab5f2649d"
)

EXPECTED_RAW_ROOT_SHA256 = (
    "04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085"
)

EXPECTED_REGISTRY_SHA256 = (
    "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
)

EXPECTED_COUNTS = {
    "RawFiles": 2_160,
    "RawBytes": 53_410_207,
    "Conditions": 270,
    "MLFits": 1_080,
    "RankingRows": 3_101_490,
    "BuildMetricRows": 15_120,
    "ProjectRunRows": 1_890,
    "ConditionAuditRows": 270,
    "TrainingMedianRows": 40_770,
    "NoiseTechniqueSummaryRows": 63,
    "SeedLevelNoiseDeltaRows": 1_890,
    "NoiseDeltaSummaryRows": 63,
    "ValidationChecks": 66,
    "FailedValidationChecks": 0,
}


drive.mount(
    "/content/drive",
    force_remount=False,
)


ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES
    / "completed_project_registry.csv"
)

STEP5A_CHECKPOINT_PATH = (
    NOTES
    / "project_16_step5a_checkpoint.json"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES
    / "project_16_step5b_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
)

STEP5B_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
)

STEP5B_REPORT_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT}_step5b_report.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b_status.json"
)

STEP5B_VALIDATION_PATH = (
    STEP5B_ROOT
    / f"{PROJECT_SHORT}_step5b_validation.csv"
)


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def as_bool_series(
    series,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.fillna(
            False
        )

    normalized = (
        series
        .astype(
            str
        )
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true",
        "1",
        "yes",
    }

    rejected = {
        "false",
        "0",
        "no",
        "",
        "nan",
        "none",
    }

    unknown = sorted(
        set(
            normalized.unique().tolist()
        )
        - accepted
        - rejected
    )

    if unknown:
        raise RuntimeError(
            "The Step 5B validation table contains unrecognized Pass values:\n"
            + "\n".join(
                unknown
            )
        )

    return normalized.isin(
        accepted
    )


required_paths = [
    REGISTRY_PATH,
    STEP5A_CHECKPOINT_PATH,
    STEP5B_CHECKPOINT_PATH,
    STEP5B_REPORT_PATH,
    STEP5B_STATUS_PATH,
    STEP5B_VALIDATION_PATH,
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "The completed Step 5B artifacts required for recovery are missing:\n"
        + "\n".join(
            missing_paths
        )
        + "\nDo not rerun this finalizer until those artifacts exist."
    )


registry_sha256 = sha256_file(
    REGISTRY_PATH
)

step5a_sha256 = sha256_file(
    STEP5A_CHECKPOINT_PATH
)

step5b_sha256 = sha256_file(
    STEP5B_CHECKPOINT_PATH
)


if registry_sha256 != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "The completion registry changed after Project 15 registration.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256}"
    )


if step5a_sha256 != EXPECTED_STEP5A_SHA256:
    raise RuntimeError(
        "The frozen Project 16 Step 5A checkpoint changed.\n"
        f"Expected: {EXPECTED_STEP5A_SHA256}\n"
        f"Actual:   {step5a_sha256}"
    )


checkpoint = load_json(
    STEP5B_CHECKPOINT_PATH
)

report = load_json(
    STEP5B_REPORT_PATH
)

status = load_json(
    STEP5B_STATUS_PATH
)


for label, payload in [
    (
        "Step 5B checkpoint",
        checkpoint,
    ),
    (
        "Step 5B report",
        report,
    ),
    (
        "Step 5B status",
        status,
    ),
]:
    if int(
        payload.get(
            "ProjectNumber",
            -1,
        )
    ) != PROJECT_NUMBER:
        raise RuntimeError(
            f"{label} project number differs."
        )

    if str(
        payload.get(
            "Project",
            "",
        )
    ) != PROJECT_NAME:
        raise RuntimeError(
            f"{label} project identity differs."
        )

    if str(
        payload.get(
            "ProjectSlug",
            "",
        )
    ) != PROJECT_SLUG:
        raise RuntimeError(
            f"{label} project slug differs."
        )

    if str(
        payload.get(
            "Status",
            "",
        )
    ) != STEP5B_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 5B PASS status."
        )


if checkpoint.get(
    "CheckpointType"
) != "PROJECT_16_RAW_REVALIDATION_AND_COMPACT_AGGREGATION":
    raise RuntimeError(
        "The Step 5B checkpoint type differs."
    )


if not bool(
    checkpoint.get(
        "RawResultsRevalidated",
        False,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint is not marked RawResultsRevalidated."
    )


if not bool(
    checkpoint.get(
        "CompactAggregatesFrozen",
        False,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint is not marked CompactAggregatesFrozen."
    )


if not bool(
    checkpoint.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint is not ready for final packaging and registration."
    )


if str(
    checkpoint.get(
        "Step5ACheckpointSHA256",
        "",
    )
).lower() != EXPECTED_STEP5A_SHA256:
    raise RuntimeError(
        "The Step 5B checkpoint does not link to the frozen Step 5A checkpoint."
    )


for field in [
    "FrozenRawRootSHA256",
    "IndependentRawRootSHA256",
]:
    if str(
        checkpoint.get(
            field,
            "",
        )
    ).lower() != EXPECTED_RAW_ROOT_SHA256:
        raise RuntimeError(
            f"The Step 5B checkpoint {field} differs."
        )


for field, expected_value in EXPECTED_COUNTS.items():
    actual_value = checkpoint.get(
        field
    )

    if actual_value is None:
        raise RuntimeError(
            f"The Step 5B checkpoint is missing {field}."
        )

    if int(
        actual_value
    ) != int(
        expected_value
    ):
        raise RuntimeError(
            f"The Step 5B checkpoint {field} differs.\n"
            f"Expected: {expected_value}\n"
            f"Actual:   {actual_value}"
        )


if bool(
    checkpoint.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint incorrectly reports a registry modification."
    )


if bool(
    checkpoint.get(
        "Projects1To15Modified",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint incorrectly reports modification of Projects 1–15."
    )


if bool(
    checkpoint.get(
        "ModelsFitted",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint incorrectly reports model fitting."
    )


if bool(
    checkpoint.get(
        "ConditionsRerun",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint incorrectly reports rerun conditions."
    )


if bool(
    checkpoint.get(
        "Project15ConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint reports access to Project 15 condition outputs."
    )


if bool(
    checkpoint.get(
        "Project15ConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "The Step 5B checkpoint reports modification of Project 15 condition outputs."
    )


if str(
    status.get(
        "Checkpoint",
        "",
    )
) != str(
    STEP5B_CHECKPOINT_PATH
):
    raise RuntimeError(
        "The Step 5B status points to a different checkpoint path."
    )


if str(
    status.get(
        "CheckpointSHA256",
        "",
    )
).lower() != step5b_sha256:
    raise RuntimeError(
        "The Step 5B status checkpoint SHA-256 differs from the live checkpoint."
    )


if str(
    status.get(
        "RawRootSHA256",
        "",
    )
).lower() != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "The Step 5B status raw-root SHA-256 differs."
    )


validation = pd.read_csv(
    STEP5B_VALIDATION_PATH,
    low_memory=False,
)

required_validation_columns = {
    "Check",
    "Expected",
    "Actual",
    "Pass",
}

if not required_validation_columns.issubset(
    validation.columns
):
    raise RuntimeError(
        "The Step 5B validation table has an unexpected schema."
    )


validation_pass = as_bool_series(
    validation[
        "Pass"
    ]
)

failed_validation_rows = int(
    (
        ~validation_pass
    ).sum()
)

if len(
    validation
) != EXPECTED_COUNTS[
    "ValidationChecks"
]:
    raise RuntimeError(
        "The Step 5B validation-row count differs.\n"
        f"Expected: {EXPECTED_COUNTS['ValidationChecks']}\n"
        f"Actual:   {len(validation)}"
    )


if failed_validation_rows != 0:
    raise RuntimeError(
        "The frozen Step 5B validation table contains failed checks."
    )


output_manifest = checkpoint.get(
    "OutputManifest",
    [],
)

if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 5B checkpoint contains no compact-output manifest."
    )


missing_output_files = 0
size_mismatches = 0
sha256_mismatches = 0

for item in output_manifest:
    output_path = Path(
        item.get(
            "Path",
            "",
        )
    )

    if not output_path.is_file():
        missing_output_files += 1
        continue

    expected_bytes = int(
        item.get(
            "Bytes",
            -1,
        )
    )

    expected_sha256 = str(
        item.get(
            "SHA256",
            "",
        )
    ).lower()

    if int(
        output_path.stat().st_size
    ) != expected_bytes:
        size_mismatches += 1

    if sha256_file(
        output_path
    ) != expected_sha256:
        sha256_mismatches += 1


if (
    missing_output_files
    or size_mismatches
    or sha256_mismatches
):
    raise RuntimeError(
        "The frozen Step 5B compact outputs failed readback validation.\n"
        f"Missing files: {missing_output_files}\n"
        f"Size mismatches: {size_mismatches}\n"
        f"SHA-256 mismatches: {sha256_mismatches}"
    )


report_fields = [
    "RawFiles",
    "RawBytes",
    "Conditions",
    "MLFits",
    "RankingRows",
    "BuildMetricRows",
    "ProjectRunRows",
    "ConditionAuditRows",
    "TrainingMedianRows",
    "NoiseTechniqueSummaryRows",
    "SeedLevelNoiseDeltaRows",
    "NoiseDeltaSummaryRows",
    "ValidationChecks",
    "FailedValidationChecks",
]

for field in report_fields:
    if report.get(
        field
    ) != checkpoint.get(
        field
    ):
        raise RuntimeError(
            f"The Step 5B report and checkpoint differ for {field}."
        )


print("\nThe expensive Step 5B work had already completed before the display-only error.")
print("The undefined name was only in the final console summary:")
print("  erroneous name: embedded_manifest_fail")
print("  existing variable: embedded_fail")

print("\nReadback validation:")
print(
    "Step 5B checkpoint SHA-256:",
    step5b_sha256,
)

print(
    "Validation rows:",
    len(
        validation
    ),
)

print(
    "Failed validation rows:",
    failed_validation_rows,
)

print(
    "Compact-output manifest files:",
    len(
        output_manifest
    ),
)

print(
    "Missing / size / SHA mismatches:",
    missing_output_files,
    "/",
    size_mismatches,
    "/",
    sha256_mismatches,
)

print("\nRaw-output freeze:")
print(
    "Conditions:",
    checkpoint[
        "Conditions"
    ],
    "/",
    EXPECTED_COUNTS[
        "Conditions"
    ],
)

print(
    "Raw files:",
    checkpoint[
        "RawFiles"
    ],
    "/",
    EXPECTED_COUNTS[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    checkpoint[
        "RawBytes"
    ],
    "/",
    EXPECTED_COUNTS[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    checkpoint[
        "IndependentRawRootSHA256"
    ],
)

print("\nExperiment totals:")
print(
    "ML fits:",
    checkpoint[
        "MLFits"
    ],
    "/",
    EXPECTED_COUNTS[
        "MLFits"
    ],
)

print(
    "Ranking rows:",
    checkpoint[
        "RankingRows"
    ],
    "/",
    EXPECTED_COUNTS[
        "RankingRows"
    ],
)

print(
    "Build-metric rows:",
    checkpoint[
        "BuildMetricRows"
    ],
    "/",
    EXPECTED_COUNTS[
        "BuildMetricRows"
    ],
)

print(
    "Project-run rows:",
    checkpoint[
        "ProjectRunRows"
    ],
    "/",
    EXPECTED_COUNTS[
        "ProjectRunRows"
    ],
)

print("\nIsolation:")
print(
    "Completion registry unchanged:",
    registry_sha256
    == EXPECTED_REGISTRY_SHA256,
)

print(
    "Projects 1–15 modified:",
    0,
)

print(
    "Project 15 condition outputs accessed:",
    False,
)

print(
    "Project 15 condition outputs modified:",
    False,
)

print(
    "Conditions rerun by this finalizer:",
    False,
)

print(
    "Models fitted by this finalizer:",
    False,
)

print("\nProject 16 Step 5B checkpoint:")
print(
    STEP5B_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    step5b_sha256,
)

print(
    "\nSTATUS:",
    STEP5B_STATUS,
)

print("=" * 136)


=== PROJECT 16 CELL 10B / STEP 5B READBACK FINALIZER ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

The expensive Step 5B work had already completed before the display-only error.
The undefined name was only in the final console summary:
  erroneous name: embedded_manifest_fail
  existing variable: embedded_fail

Readback validation:
Step 5B checkpoint SHA-256: e294708adebef9a7a34d537688ab2193939ef09c7e0cabb5e874312c84c89213
Validation rows: 66
Failed validation rows: 0
Compact-output manifest files: 11
Missing / size / SHA mismatches: 0 / 0 / 0

Raw-output freeze:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 53410207 / 53410207
Raw root SHA-256: 04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 3101490 / 3101490
Build-metric rows: 15120 / 15120
Project-run rows: 1890 / 1890

Isolation:
Completion registry unchanged: Tru

In [12]:
# ==================================================================================================
# PROJECT 16 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   apache@rocketmq
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 16 Step 5A checkpoint SHA-256:
#   bdb17f099601882d4d5f6e9e0818501341a5f74de26a6ed1146f6e6ab5f2649d
# - Project 16 Step 5B checkpoint SHA-256:
#   e294708adebef9a7a34d537688ab2193939ef09c7e0cabb5e874312c84c89213
# - Project 16 raw-root SHA-256:
#   04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085
# - Registry before registration:
#   exactly Projects 1–15, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 16 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_16_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_16_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_16_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "2c8803de6d0a448f1c505576829f110ae2144ca7f16e976d548b2ab797597085"
STEP5A_SHA_EXPECTED = "bdb17f099601882d4d5f6e9e0818501341a5f74de26a6ed1146f6e6ab5f2649d"
STEP5B_SHA_EXPECTED = "e294708adebef9a7a34d537688ab2193939ef09c7e0cabb5e874312c84c89213"
SOURCE_ROOT_SHA = "e7241f9fb031f059afcda78ffc5f1dc6bb6109bf1d65ddeb6252641db8b54f4e"
RAW_ROOT_SHA = "04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 53410207, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 3101490, "BuildMetricRows": 15120, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 536,
    "TrainingBuilds": 402, "EvaluationBuilds": 134, "RawRows": 97734,
    "RawTrainingRows": 71052, "RawEvaluationRows": 26682,
    "RawTrainingFailures": 106, "RawEvaluationFailures": 9, "ModelRows": 6548,
    "ModelTrainingRows": 4907, "ModelEvaluationRows": 1641,
    "ModelTrainingFailures": 105, "ModelEvaluationFailures": 9,
    "ModelFailingEvaluationBuilds": 8, "Predictors": 151, "RECFeatures": 19,
}
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_16_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_16.csv"

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)
STEP5A_CP = NOTES / "project_16_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_16_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_16_selection_checkpoint.json",
    NOTES / "project_16_rec_reconstruction_checkpoint.json",
    NOTES / "project_16_noise_plan_checkpoint.json",
    NOTES / "project_16_runtime_contract_checkpoint.json",
    NOTES / "project_16_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 16 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "Project15ConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to Project 15 condition outputs."
    )

if bool(
    step5b.get(
        "Project15ConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of Project 15 condition outputs."
    )

step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must contain exactly completed Projects 1–15, with Project 16 absent.
registry_sha_before = sha(REGISTRY)

if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(
        "Registry SHA differs before Project 16 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna("")

pn_col = resolve(reg_before.columns, "ProjectNumber")
project_col = resolve(reg_before.columns, "Project")
status_col = resolve(reg_before.columns, "Status")

pnums = pd.to_numeric(
    reg_before[pn_col],
    errors="raise",
).astype(int)

if len(reg_before) != 15 or sorted(pnums.tolist()) != list(range(1, 16)):
    raise RuntimeError("Registry must contain exactly Projects 1–15.")

if not reg_before[status_col].eq(COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–15 are not all COMPLETE_AND_FROZEN.")

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = reg_before.loc[pnums.eq(required_number)]
    if len(matching_rows) != 1 or matching_rows.iloc[0][project_col] != required_project:
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if pnums.eq(PROJECT_NUMBER).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Project 16 is already present in the completion registry.")

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-16 registry backup does not match the live registry."
    )

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

# Use the frozen Step 5B completion timestamp so the package is deterministic
# across safe reruns.
created_at = str(
    step5b.get(
        "CompletedAtUTC",
        ""
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project16_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 16 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
        "Project15ConditionOutputsAccessed": False,
        "Project15ConditionOutputsModified": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 16 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 16 final package already exists and "
                "was not modified."
            )

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems. A direct
        # os.replace(tmp_root, FINAL_ROOT) therefore raises EXDEV. Publish in
        # two stages:
        #   1. copy the completed local package to a sibling staging directory
        #      on Google Drive;
        #   2. verify every staged payload file;
        #   3. rename the staging directory to FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root,
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + comparison.loc[
                    (
                        comparison["_merge"].ne(
                            "both"
                        )
                        | comparison[
                            "Bytes_candidate"
                        ].ne(
                            comparison[
                                "Bytes_staged"
                            ]
                        )
                        | comparison[
                            "SHA256_candidate"
                        ].ne(
                            comparison[
                                "SHA256_staged"
                            ]
                        )
                    )
                ].head(
                    20
                ).to_string(
                    index=False
                )
            )

        # This rename is within the Google Drive filesystem, so it does not
        # cross a device boundary.
        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 16 package failed readback validation.")

# Build a complete Project 16 registry row.
#
# The registry has evolved across Projects 1–15. Some columns are protocol
# descriptors, some are project-specific counts, and some are paths to frozen
# audit artefacts. V2 deliberately stopped because it did not map every
# variable column. V3 handles the complete observed schema explicitly.
#
# For protocol fields whose textual formatting has varied historically
# (Seeds, NoiseLevels, Techniques, DoNotRerun), use the exact frozen
# Project 15 representation. Project 16 uses the same protocol.
project_15_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        15
    )
]

if len(
    project_15_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 15 registry template row."
    )

project_15_template = project_15_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_15_template[
                registry_column
            ]
        ).strip()


protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}


for protocol_key, fallback_value in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        ""
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds": protocol_template_values["seeds"],
    "noiselevels": protocol_template_values["noiselevels"],
    "techniques": protocol_template_values["techniques"],
    "evaluationrows": COUNTS["ModelEvaluationRows"],
    "evaluationfailures": COUNTS["ModelEvaluationFailures"],
    "finaldirectory": str(FINAL_ROOT),
    "finalauditreport": str(REPORT_PATH),
    "donotrerun": protocol_template_values["donotrerun"],
    "freezerecord": str(CHECKPOINT_PATH),
    "rawresultsmanifest": str(STEP5B_RAW_MANIFEST_PATH),
    "finalpackagemanifest": str(MANIFEST_PATH),
    "rawresultsrootsha256": RAW_ROOT_SHA,
    "finalauditstatus": STEP5C_STATUS,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )


new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; no registry write "
        "was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [reg_before, pd.DataFrame([new_row])],
    ignore_index=True,
)
reg_candidate[pn_col] = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int).astype(str)
candidate_nums = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int)

project16_candidate = reg_candidate.loc[candidate_nums.eq(16)]

if len(reg_candidate) != 16 or sorted(candidate_nums.tolist()) != list(range(1, 17)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–16.")

if (
    not reg_candidate[status_col].eq(COMPLETE_STATUS).all()
    or len(project16_candidate) != 1
    or project16_candidate.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError("Candidate Project 16 registry row failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 15, len(reg_before), len(reg_before) == 15)
check(rows, "Registry rows candidate", 16, len(reg_candidate), len(reg_candidate) == 16)
check(rows, "Candidate Project 16 rows", 1, len(project16_candidate), len(project16_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}


registry_field_validation_failures = 0

for expected_column_name, expected_value in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project16_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )


check(
    rows,
    "Explicit Project 16 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(rows)
print("\nProject 16 Step 5C pre-write validation:"); display(pre)
print("\nProject 16 registry row candidate:"); display(project16_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 16 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project16_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if (
    len(tmp_read) != 16
    or sorted(tmp_nums.tolist()) != list(range(1, 17))
    or not tmp_read[status_col].eq(COMPLETE_STATUS).all()
    or int(tmp_nums.eq(16).sum()) != 1
):
    tmp_reg.unlink(missing_ok=True)
    raise RuntimeError("Temporary Project 16 registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
project12_after = reg_after.loc[after_nums.eq(12)]
project13_after = reg_after.loc[after_nums.eq(13)]
project14_after = reg_after.loc[after_nums.eq(14)]
project15_after = reg_after.loc[after_nums.eq(15)]
project16_after = reg_after.loc[after_nums.eq(16)]
registry_sha_after = sha(REGISTRY)

if (
    len(reg_after) != 16
    or sorted(after_nums.tolist()) != list(range(1, 17))
    or not reg_after[status_col].eq(COMPLETE_STATUS).all()
    or len(project11_after) != 1
    or project11_after.iloc[0][project_col] != "apache@shardingsphere"
    or len(project12_after) != 1
    or project12_after.iloc[0][project_col] != "zolyfarkas@spf4j"
    or len(project13_after) != 1
    or project13_after.iloc[0][project_col] != "jcabi@jcabi-github"
    or len(project14_after) != 1
    or project14_after.iloc[0][project_col] != "JMRI@JMRI"
    or len(project15_after) != 1
    or project15_after.iloc[0][project_col] != "eclipse@steady"
    or len(project16_after) != 1
    or project16_after.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed Project 16 post-write validation. "
        f"Backup: {BACKUP_PATH}"
    )

check(rows, "Registry rows after", 16, len(reg_after), len(reg_after) == 16)
check(rows, "COMPLETE_AND_FROZEN projects after", 16, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), int(reg_after[status_col].eq(COMPLETE_STATUS).sum()) == 16)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry Project 12 rows after", 1, len(project12_after), len(project12_after) == 1)
check(rows, "Registry Project 13 rows after", 1, len(project13_after), len(project13_after) == 1)
check(rows, "Registry Project 14 rows after", 1, len(project14_after), len(project14_after) == 1)
check(rows, "Registry Project 15 rows after", 1, len(project15_after), len(project15_after) == 1)
check(rows, "Registry Project 16 rows after", 1, len(project16_after), len(project16_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed)
    raise RuntimeError("PROJECT 16 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after),
    "Project12RegistryRowsAfter": len(project12_after),
    "Project13RegistryRowsAfter": len(project13_after),
    "Project14RegistryRowsAfter": len(project14_after),
    "Project15RegistryRowsAfter": len(project15_after),
    "Project16RegistryRowsAfter": len(project16_after),
    "Project15ConditionOutputsAccessed": False,
    "Project15ConditionOutputsModified": False,
    "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectWriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "CheckpointType": "PROJECT_16_FINAL_PACKAGE_AND_REGISTRY", "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha,
    "ProjectCompleteAndFrozen": True,
    "Project15ConditionOutputsAccessed": False,
    "Project15ConditionOutputsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 16 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Project 11 identity:", required_registered_identities[11])
print("Project 12 identity:", required_registered_identities[12])
print("Project 13 identity:", required_registered_identities[13])
print("Project 14 identity:", required_registered_identities[14])
print("Project 15 identity:", required_registered_identities[15])
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Project 12 registry rows:", len(project12_after))
print("Project 13 registry rows:", len(project13_after))
print("Project 14 registry rows:", len(project14_after))
print("Project 15 registry rows:", len(project15_after))
print("Project 16 registry rows:", len(project16_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 15 condition outputs accessed:", False)
print("Project 15 condition outputs modified:", False)
print("Prior project condition outputs accessed:", False)
print("Prior project write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 16 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print("Explicit registry-schema fields validated:", len(required_registry_field_expectations))
print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)


=== PROJECT 16 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project 16 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,bdb17f099601882d4d5f6e9e0818501341a5f74de26a6e...,bdb17f099601882d4d5f6e9e0818501341a5f74de26a6e...,True
1,Step 5B checkpoint SHA-256,e294708adebef9a7a34d537688ab2193939ef09c7e0cab...,e294708adebef9a7a34d537688ab2193939ef09c7e0cab...,True
2,Step 5A manifest failures,0,0,True
3,Step 5B manifest failures,0,0,True
4,Package missing files,0,0,True
5,Package unexpected files,0,0,True
6,Package size mismatches,0,0,True
7,Package SHA-256 mismatches,0,0,True
8,Registry rows before,15,15,True
9,Registry rows candidate,16,16,True



Project 16 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
15,16,apache@rocketmq,apache__rocketmq,COMPLETE_AND_FROZEN,270,30,9,7,134,1641,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,04d5b0e9f621e19141ba19c55914891cfea85b43c2374d...,2d4337589593a85a44d0e12124bbc03d9430d3792b1b3d...,1080,5358150.0,PASS_PROJECT_16_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 16 CELL 11 / STEP 5C RESULT ===
Project number: 16
Project: apache@rocketmq
Project slug: apache__rocketmq
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 53410207
Raw root SHA-256: 04d5b0e9f621e19141ba19c55914891cfea85b43c2374dde5b53c9f472d61085

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/apache__rocketmq
Package files: 32
Package bytes: 10467143
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: 2d4337589593a85a44d0e12124bbc03d9430d3792b1b3dbd9170e732d53521da

Completion registry:
Registry rows: 16
COMPLETE_AND_FROZEN projects: 16
Project 11 registry rows: 1
Project 12 registry rows: 1
Project 13 registry rows: 1
Project 14 r

In [13]:
# ==================================================================================================
# PROJECT 17 — CELL 1 / STEP 0
# SAME-NOTEBOOK POST-PROJECT-16 BOOTSTRAP AND CANDIDATE DISCOVERY
#
# RUN THIS AS THE NEXT NEW CELL IN THE EXISTING:
#   Thesis_project_16.ipynb
#
# PROJECT 16 IS COMPLETE_AND_FROZEN AND MUST NOT BE RERUN.
#
# SAFETY:
# - validates the frozen 16-project completion registry and Project 16 completion checkpoint;
# - reads but never modifies the completion registry;
# - writes only Project 17 bootstrap/selection files;
# - never reads or modifies any prior-project condition-output files;
# - does not inject noise, reconstruct REC features, fit models, or start an experiment;
# - prepares the nine remaining projects for runtime-prioritized selection in Step 1A.
# ==================================================================================================

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil
import tarfile

import pandas as pd


print("=" * 136)
print("=== PROJECT 17 CELL 1 / STEP 0: SAME-NOTEBOOK POST-PROJECT-16 BOOTSTRAP ===")
print("=" * 136)


PROJECT_NUMBER = 17

STEP0_STATUS = (
    "PASS_PROJECT_17_SAME_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_REGISTERED_PROJECTS = 16
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

ACTIVE_RESERVED_PROJECTS = {}

EXPECTED_CANDIDATES = 9

REQUIRED_PROJECT_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
}

RUNTIME_PRIORITY_POLICY = {
    "purpose":
        "processing order only; protocol eligibility and final project set are unchanged",
    "primary":
        "ModelTrainingRows ascending",
    "secondary":
        "ModelEvaluationRows ascending",
    "tertiary":
        "RawExecutionRows ascending",
    "final_tie_break":
        "Project ascending",
    "scientific_effect":
        "none when all protocol-eligible projects are completed",
}


drive.mount(
    "/content/drive",
    force_remount=False,
)

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

PROJECT_16_STEP5C_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_16_step5c_checkpoint.json"
)

EXPECTED_PROJECT_16_STEP5C_SHA256 = (
    "e346db468f71741a8c02d6e24cd68d90f6ae9f07b0404495672f44c73e14fb1a"
)

LOCAL_EXTRACTION_ROOT = Path(
    "/content/datasets"
)

LOCAL_DATASET_ROOT = (
    LOCAL_EXTRACTION_ROOT
    / "datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_17_selection"
)

BOOTSTRAP_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_17_bootstrap_candidate_inventory.csv"
)

BOOTSTRAP_REPORT_PATH = (
    SELECTION_ROOT
    / "project_17_step0_report.json"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step0_status.json"
)


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def resolve_column(
    columns,
    *candidates,
):
    normalized = {
        str(column).strip().lower():
            column
        for column in columns
    }

    for candidate in candidates:
        key = str(
            candidate
        ).strip().lower()

        if key in normalized:
            return normalized[
                key
            ]

    raise RuntimeError(
        "Could not resolve any of these columns: "
        + ", ".join(
            candidates
        )
    )


def atomic_write_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_json(
    path,
    payload,
):
    atomic_write_text(
        path,
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
    )


def atomic_write_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def extract_archive_safely(
    archive_path,
    extraction_root,
):
    extraction_root = Path(
        extraction_root
    )

    extraction_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolved_root = extraction_root.resolve()
    extracted_files = 0

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace(
                    "\\",
                    "/",
                )
                .lstrip(
                    "/"
                )
            )

            target_path = (
                extraction_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target
                != resolved_root
                and resolved_root
                not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open(
                        "wb"
                    ) as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    return extracted_files


required_drive_paths = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    PROJECT_16_STEP5C_CHECKPOINT_PATH,
]

missing_drive_paths = [
    str(
        path
    )
    for path in required_drive_paths
    if not path.is_file()
]

if missing_drive_paths:
    raise FileNotFoundError(
        "Required Project 17 bootstrap inputs are missing:\n"
        + "\n".join(
            missing_drive_paths
        )
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Frozen TCP-CI archive SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


project_16_step5c_sha256 = sha256_file(
    PROJECT_16_STEP5C_CHECKPOINT_PATH
)

if project_16_step5c_sha256 != EXPECTED_PROJECT_16_STEP5C_SHA256:
    raise RuntimeError(
        "Project 16 completion checkpoint SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_PROJECT_16_STEP5C_SHA256}\n"
        f"Actual:   {project_16_step5c_sha256}"
    )

project_16_step5c_checkpoint = json.loads(
    PROJECT_16_STEP5C_CHECKPOINT_PATH.read_text(encoding="utf-8")
)

if project_16_step5c_checkpoint.get("Status") != (
    "PASS_PROJECT_16_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
):
    raise RuntimeError(
        "Project 16 completion checkpoint is not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs from the frozen Projects 1–16 state.\n"
        "Do not continue Project 17 until the unexpected registry change is investigated.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "Project Number",
    "Project_Number",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly frozen Projects 1–16."
    )


if not registry[
    registry_status_column
].astype(
    str
).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Not every registered predecessor is COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(
        str
    )
)


if ACTIVE_RESERVED_PROJECTS:
    raise RuntimeError(
        "Project 17 bootstrap expects no active project reservations."
    )


EXPECTED_PREDECESSOR_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}

for predecessor_number, expected_project in EXPECTED_PREDECESSOR_IDENTITIES.items():
    matches = registry.loc[
        registry_project_numbers.eq(predecessor_number),
        registry_project_column,
    ].astype(str).tolist()

    if matches != [expected_project]:
        raise RuntimeError(
            f"Frozen Project {predecessor_number} identity mismatch.\n"
            f"Expected: {expected_project}\n"
            f"Actual:   {matches}"
        )


def local_dataset_looks_complete():
    if not LOCAL_DATASET_ROOT.is_dir():
        return False

    project_directories = [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ]

    return bool(
        len(
            project_directories
        )
        == 25
    )


if local_dataset_looks_complete():
    extraction_performed = False
    extracted_files = 0

    print(
        "\nA complete-looking local TCP-CI dataset is already present."
    )

else:
    extraction_performed = True

    print(
        "\nRestoring the frozen TCP-CI archive into the Project 17 runtime."
    )

    if LOCAL_EXTRACTION_ROOT.exists():
        shutil.rmtree(
            LOCAL_EXTRACTION_ROOT
        )

    extracted_files = extract_archive_safely(
        ARCHIVE_PATH,
        LOCAL_EXTRACTION_ROOT,
    )


if not LOCAL_DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Archive extraction did not create the expected dataset root:\n"
        f"{LOCAL_DATASET_ROOT}"
    )


all_project_directories = sorted(
    [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path:
        path.name,
)


if len(all_project_directories) != 25:
    raise RuntimeError(
        "Unexpected number of TCP-CI project directories.\n"
        f"Expected: 25\n"
        f"Actual:   {len(all_project_directories)}"
    )


reserved_projects = set(
    ACTIVE_RESERVED_PROJECTS.values()
)

candidate_rows = []

for source_directory in all_project_directories:
    project = source_directory.name

    source_files = {
        path.name
        for path in source_directory.iterdir()
        if path.is_file()
    }

    missing_required_files = sorted(
        REQUIRED_PROJECT_FILES
        - source_files
    )

    excluded_registered = (
        project in registered_projects
    )

    excluded_reserved = (
        project in reserved_projects
    )

    candidate_eligible_for_scan = (
        not excluded_registered
        and not excluded_reserved
        and not missing_required_files
    )

    candidate_rows.append({
        "Project":
            project,
        "ProjectSlug":
            project.replace(
                "@",
                "__",
            ),
        "SourceDirectory":
            str(
                source_directory
            ),
        "ExcludedRegistered":
            bool(
                excluded_registered
            ),
        "ExcludedReserved":
            bool(
                excluded_reserved
            ),
        "MissingRequiredFiles":
            "; ".join(
                missing_required_files
            ),
        "CandidateForProject17Scan":
            bool(
                candidate_eligible_for_scan
            ),
    })


inventory = pd.DataFrame(
    candidate_rows
)


project_17_candidates = (
    inventory.loc[
        inventory[
            "CandidateForProject17Scan"
        ]
    ]
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    project_17_candidates
) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected number of Project 17 candidates after excluding "
        "frozen Projects 1–16.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(project_17_candidates)}"
    )


if (
    project_17_candidates[
        "Project"
    ].isin(
        registered_projects
        | reserved_projects
    ).any()
):
    raise RuntimeError(
        "A registered identity leaked into the Project 17 candidate set."
    )


SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    BOOTSTRAP_INVENTORY_PATH,
    project_17_candidates,
)


created_at_utc = datetime.now(
    timezone.utc
).isoformat()


report = {
    "ProjectNumber":
        PROJECT_NUMBER,
    "Status":
        STEP0_STATUS,
    "CreatedAtUTC":
        created_at_utc,
    "ArchivePath":
        str(
            ARCHIVE_PATH
        ),
    "ArchiveSHA256":
        archive_sha256,
    "RegistryPath":
        str(
            REGISTRY_PATH
        ),
    "RegistrySHA256":
        registry_sha256_before,
    "Project16Step5CCheckpoint":
        str(
            PROJECT_16_STEP5C_CHECKPOINT_PATH
        ),
    "Project16Step5CCheckpointSHA256":
        project_16_step5c_sha256,
    "RegisteredProjects":
        EXPECTED_REGISTERED_PROJECTS,
    "RegisteredStatuses":
        sorted(
            registry[
                registry_status_column
            ].astype(
                str
            ).unique().tolist()
        ),
    "ActiveReservations":
        {
            str(
                key
            ):
                value
            for key, value in ACTIVE_RESERVED_PROJECTS.items()
        },
    "FrozenPredecessorIdentities":
        {
            str(key): value
            for key, value in EXPECTED_PREDECESSOR_IDENTITIES.items()
        },
    "DatasetRoot":
        str(
            LOCAL_DATASET_ROOT
        ),
    "SourceProjectDirectories":
        len(
            all_project_directories
        ),
    "Project17CandidateCount":
        len(
            project_17_candidates
        ),
    "CandidateInventory":
        str(
            BOOTSTRAP_INVENTORY_PATH
        ),
    "RuntimePriorityPolicy":
        RUNTIME_PRIORITY_POLICY,
    "ExtractionPerformed":
        bool(
            extraction_performed
        ),
    "ArchiveFilesExtracted":
        int(
            extracted_files
        ),
    "RegistryModified":
        False,
    "PriorProjectConditionOutputsAccessed":
        False,
    "PriorProjectConditionOutputsModified":
        False,
    "NoiseInjected":
        False,
    "ModelsFitted":
        False,
}


atomic_write_json(
    BOOTSTRAP_REPORT_PATH,
    report,
)

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,
        "Status":
            STEP0_STATUS,
        "CreatedAtUTC":
            created_at_utc,
        "Report":
            str(
                BOOTSTRAP_REPORT_PATH
            ),
    },
)


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during the Project 17 bootstrap."
    )


print("\nProject 17 candidates after excluding registered identities:")
print(
    project_17_candidates[
        [
            "Project",
            "ProjectSlug",
            "SourceDirectory",
        ]
    ].to_string(
        index=False
    )
)


print("\n")
print("=" * 136)
print("=== PROJECT 17 CELL 1 / STEP 0 RESULT ===")
print("=" * 136)

print(
    "Registered and frozen projects:",
    EXPECTED_REGISTERED_PROJECTS,
)

print(
    "Active reservations:",
    [],
)

print(
    "TCP-CI source directories:",
    len(
        all_project_directories
    ),
)

print(
    "Project 17 candidates:",
    len(
        project_17_candidates
    ),
)

print(
    "Runtime-priority policy:",
    RUNTIME_PRIORITY_POLICY,
)

print(
    "Candidate inventory:",
    BOOTSTRAP_INVENTORY_PATH,
)

print(
    "Completion registry modified:",
    False,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "\nSTATUS:",
    STEP0_STATUS,
)

print("=" * 136)


=== PROJECT 17 CELL 1 / STEP 0: SAME-NOTEBOOK POST-PROJECT-16 BOOTSTRAP ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

A complete-looking local TCP-CI dataset is already present.

Project 17 candidates after excluding registered identities:
                      Project                    ProjectSlug                                          SourceDirectory
         EMResearch@EvoMaster          EMResearch__EvoMaster          /content/datasets/datasets/EMResearch@EvoMaster
     Graylog2@graylog2-server      Graylog2__graylog2-server      /content/datasets/datasets/Graylog2@graylog2-server
        SonarSource@sonarqube         SonarSource__sonarqube         /content/datasets/datasets/SonarSource@sonarqube
               apache@curator                apache__curator                /content/datasets/datasets/apache@curator
        apache@logging-log4j2         apache__logging-log4j2         /content/data

In [14]:
# ==================================================================================================
# PROJECT 17 — CELL 2 / STEP 1A
# ROBUST CANDIDATE DISCOVERY, PROTOCOL ELIGIBILITY, RUNTIME-PRIORITIZED RANKING,
# AND PROVISIONAL PROJECT 17 SELECTION
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# THIS CELL:
# - inspects all 9 candidates frozen by Project 17 Step 0;
# - validates the chronological 75/25 split and raw/model cohort viability;
# - deterministically ranks eligible candidates by estimated experiment cost (smallest first);
# - changes processing order only, not protocol eligibility or the intended final project set;
# - freezes only a provisional Project 17 selection for Step 1B;
# - does not run experiment conditions or fit models;
# - does not modify the completion registry or Projects 1–16;
# - writes only Project 17 selection artifacts;
# - does not access prior-project condition outputs.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 17 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_17_SAME_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_17_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 16
EXPECTED_CANDIDATES = 9

RESERVED_ACTIVE_PROJECTS = set()

RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

# These three files are sufficient for deterministic selection.
# id_map.csv and entity_change_history.csv are checked and frozen later in Step 1B/2A.
REQUIRED_SELECTION_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_SOURCE_ROOT = Path(
    "/content/datasets/datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_17_selection"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step0_status.json"
)

BOOTSTRAP_CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_17_bootstrap_candidate_inventory.csv"
)

SCAN_PROGRESS_PATH = (
    SELECTION_ROOT
    / "project_17_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_17_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_17_candidate_inventory.csv"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_17_eligible_candidates_ranked.csv"
)

INELIGIBLE_PATH = (
    SELECTION_ROOT
    / "project_17_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    SELECTION_ROOT
    / "project_17_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_17_provisional_selection.json"
)

STEP1A_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_17_step1a_validation.csv"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_17_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
        1,
    )


def count_partitioned_rows(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0
    verdict_values = set()

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk_build = parse_integer_series(
            chunk[build_column],
            f"{label}.{build_column}",
        )

        chunk_verdict = parse_integer_series(
            chunk[verdict_column],
            f"{label}.{verdict_column}",
        )

        training_mask = chunk_build.isin(
            training_build_ids
        )

        evaluation_mask = chunk_build.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = chunk_verdict.ne(0)

        total_rows += len(chunk)

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            chunk_build.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            chunk_build.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (~linked_mask).sum()
        )

        verdict_values.update(
            int(value)
            for value in chunk_verdict.unique().tolist()
        )

    return {
        "Rows":
            int(total_rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(len(failing_training_builds)),

        "FailingEvaluationBuilds":
            int(len(failing_evaluation_builds)),

        "UnlinkedRows":
            int(unlinked_rows),

        "VerdictValuesJSON":
            json.dumps(
                sorted(verdict_values)
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


def reusable_scan_row_is_valid(
    row,
):
    required_fields = [
        "Project",
        "ProjectSlug",
        "SourceDirectory",
        "InspectionStatus",
        "InspectionError",
        "BuildIDColumn",
        "StartedAtColumn",
        "ExecutionBuildColumn",
        "ExecutionVerdictColumn",
        "DatasetBuildColumn",
        "DatasetVerdictColumn",
        "Builds",
        "TrainingBuilds",
        "EvaluationBuilds",
        "RawExecutionRows",
        "RawTrainingRows",
        "RawEvaluationRows",
        "RawTrainFailures",
        "RawEvaluationFailures",
        "RawFailingTrainingBuilds",
        "RawFailingEvaluationBuilds",
        "RawUnlinkedRows",
        "ModelReadyRows",
        "ModelTrainingRows",
        "ModelEvaluationRows",
        "ModelTrainFailures",
        "ModelEvaluationFailures",
        "ModelFailingTrainingBuilds",
        "ModelFailingEvaluationBuilds",
        "ModelUnlinkedRows",
    ]

    if any(
        field not in row
        for field in required_fields
    ):
        return False

    status = str(
        row.get(
            "InspectionStatus",
            "",
        )
    ).strip()

    error = str(
        row.get(
            "InspectionError",
            "",
        )
    ).strip().lower()

    return (
        status in {
            "ELIGIBLE",
            "INELIGIBLE",
        }
        and error in {
            "",
            "nan",
            "none",
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 0, REGISTRY, ARCHIVE, AND LOCAL SOURCE
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 17 Step 1A V2 inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not LOCAL_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        "The local Project 17 dataset source is missing:\n"
        f"{LOCAL_SOURCE_ROOT}"
    )


bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

if bootstrap_status.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 17 Step 0 is not in the expected PASS state."
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 17))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–16."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(str).tolist()
)


if registered_projects & RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "A reserved active-project identity is unexpectedly present in the completion registry."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


bootstrap_candidates = pd.read_csv(
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)


candidate_project_column = resolve_column(
    bootstrap_candidates.columns,
    "Project",
    "bootstrap candidate Project",
)

candidate_source_column = resolve_column(
    bootstrap_candidates.columns,
    "SourceDirectory",
    "bootstrap candidate SourceDirectory",
)


candidate_records = (
    bootstrap_candidates[
        [
            candidate_project_column,
            candidate_source_column,
        ]
    ]
    .rename(
        columns={
            candidate_project_column:
                "Project",

            candidate_source_column:
                "SourceDirectory",
        }
    )
    .copy()
)


candidate_records[
    "Project"
] = candidate_records[
    "Project"
].astype(str)


candidate_records[
    "SourceDirectory"
] = candidate_records[
    "SourceDirectory"
].astype(str)


if len(candidate_records) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 17 candidate count.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(candidate_records)}"
    )


if candidate_records[
    "Project"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Project 17 bootstrap candidate inventory contains duplicates."
    )


forbidden_candidates = (
    set(
        candidate_records[
            "Project"
        ]
    )
    & (
        registered_projects
        | RESERVED_ACTIVE_PROJECTS
    )
)


if forbidden_candidates:
    raise RuntimeError(
        "Project 17 inventory contains registered/reserved projects:\n"
        + "\n".join(
            sorted(forbidden_candidates)
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REUSE ANY VALID COMPLETED PROJECT 17 SCANS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


reusable_rows = {}


if SCAN_PROGRESS_PATH.is_file():
    try:
        previous_progress = pd.read_csv(
            SCAN_PROGRESS_PATH,
            low_memory=False,
        )

        valid_candidate_names = set(
            candidate_records[
                "Project"
            ]
        )

        for row in previous_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            if (
                project in valid_candidate_names
                and reusable_scan_row_is_valid(
                    row
                )
            ):
                reusable_rows[
                    project
                ] = row

        print(
            "\nReusable completed candidate scans:",
            len(reusable_rows),
        )

    except Exception as error:
        print(
            "\nPrevious scan progress was ignored:",
            type(error).__name__,
            str(error),
        )


# --------------------------------------------------------------------------------------------------
# 6. INSPECT ALL 11 CANDIDATES
# --------------------------------------------------------------------------------------------------

scan_rows = []


for candidate_index, candidate in enumerate(
    candidate_records.itertuples(
        index=False
    ),
    start=1,
):
    project = str(
        candidate.Project
    )

    source_directory = Path(
        candidate.SourceDirectory
    )

    print("-" * 132)
    print(
        f"[{candidate_index:02d}/{EXPECTED_CANDIDATES:02d}] "
        f"Inspecting: {project}"
    )


    if project in reusable_rows:
        row = dict(
            reusable_rows[
                project
            ]
        )

        row[
            "CandidateInspectionOrder"
        ] = candidate_index

        row[
            "ProtocolEligible"
        ] = (
            str(
                row[
                    "InspectionStatus"
                ]
            )
            == "ELIGIBLE"
        )

        row[
            "InspectionError"
        ] = ""

        scan_rows.append(
            row
        )

        print(
            "    Reused:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            int(
                row[
                    "Builds"
                ]
            ),
            "| Model eval failures:",
            int(
                row[
                    "ModelEvaluationFailures"
                ]
            ),
        )

        continue


    started = time.perf_counter()

    row = {
        "CandidateInspectionOrder":
            candidate_index,

        "Project":
            project,

        "ProjectSlug":
            project_slug(
                project
            ),

        "SourceDirectory":
            str(
                source_directory
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",
    }


    try:
        missing_files = [
            filename
            for filename in REQUIRED_SELECTION_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            raise FileNotFoundError(
                "Missing selection files: "
                + ", ".join(
                    missing_files
                )
            )


        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )


        build_columns = pd.read_csv(
            builds_path,
            nrows=0,
        ).columns.tolist()

        exe_columns = pd.read_csv(
            exe_path,
            nrows=0,
        ).columns.tolist()

        dataset_columns = pd.read_csv(
            dataset_path,
            nrows=0,
        ).columns.tolist()


        build_id_column = resolve_column(
            build_columns,
            "id",
            f"{project} builds.csv ID",
        )

        started_at_column = resolve_column(
            build_columns,
            "started_at",
            f"{project} builds.csv started_at",
        )

        execution_build_column = resolve_column(
            exe_columns,
            "build",
            f"{project} exe.csv build",
        )

        execution_verdict_column = resolve_column(
            exe_columns,
            "verdict",
            f"{project} exe.csv verdict",
        )

        dataset_build_column = resolve_column(
            dataset_columns,
            "Build",
            f"{project} dataset.csv Build",
        )

        dataset_verdict_column = resolve_column(
            dataset_columns,
            "Verdict",
            f"{project} dataset.csv Verdict",
        )


        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )


        builds[
            build_id_column
        ] = parse_integer_series(
            builds[
                build_id_column
            ],
            f"{project}.builds.id",
        )


        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )


        invalid_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )


        duplicate_build_id_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )


        if invalid_timestamps != 0:
            raise RuntimeError(
                f"Invalid build timestamps: {invalid_timestamps}"
            )


        if duplicate_build_id_rows != 0:
            raise RuntimeError(
                f"Duplicate build-ID rows: {duplicate_build_id_rows}"
            )


        ordered_builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        number_of_builds = len(
            ordered_builds
        )


        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )


        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )


        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "Chronological 75/25 split has an empty partition."
            )


        training_build_ids = set(
            ordered_builds.iloc[
                :training_build_count
            ][
                build_id_column
            ].astype(int).tolist()
        )


        evaluation_build_ids = set(
            ordered_builds.iloc[
                training_build_count:
            ][
                build_id_column
            ].astype(int).tolist()
        )


        if training_build_ids & evaluation_build_ids:
            raise RuntimeError(
                "Training/evaluation build partitions overlap."
            )


        raw_profile = count_partitioned_rows(
            csv_path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.exe",
            chunksize=500_000,
        )


        model_profile = count_partitioned_rows(
            csv_path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.dataset",
            chunksize=250_000,
        )


        eligibility_reasons = []


        eligibility_tests = [
            (
                raw_profile[
                    "TrainingRows"
                ] > 0,
                "No raw training rows",
            ),

            (
                raw_profile[
                    "EvaluationRows"
                ] > 0,
                "No raw evaluation rows",
            ),

            (
                raw_profile[
                    "TrainingFailures"
                ] > 0,
                "No raw training failures",
            ),

            (
                raw_profile[
                    "EvaluationFailures"
                ] > 0,
                "No raw evaluation failures",
            ),

            (
                model_profile[
                    "TrainingRows"
                ] > 0,
                "No model training rows",
            ),

            (
                model_profile[
                    "EvaluationRows"
                ] > 0,
                "No model evaluation rows",
            ),

            (
                model_profile[
                    "TrainingFailures"
                ] > 0,
                "No model training failures",
            ),

            (
                model_profile[
                    "EvaluationFailures"
                ] > 0,
                "No model evaluation failures",
            ),

            (
                raw_profile[
                    "UnlinkedRows"
                ] == 0,
                "Raw rows reference unknown builds",
            ),

            (
                model_profile[
                    "UnlinkedRows"
                ] == 0,
                "Model rows reference unknown builds",
            ),
        ]


        for passed, failure_reason in eligibility_tests:
            if not passed:
                eligibility_reasons.append(
                    failure_reason
                )


        protocol_eligible = (
            len(
                eligibility_reasons
            )
            == 0
        )


        row.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "RawExecutionRows":
                raw_profile[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_profile[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_profile[
                    "EvaluationRows"
                ],

            "RawTrainFailures":
                raw_profile[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_profile[
                    "EvaluationFailures"
                ],

            "RawFailingTrainingBuilds":
                raw_profile[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_profile[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_profile[
                    "UnlinkedRows"
                ],

            "RawVerdictValuesJSON":
                raw_profile[
                    "VerdictValuesJSON"
                ],

            "ModelReadyRows":
                model_profile[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_profile[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_profile[
                    "EvaluationRows"
                ],

            "ModelTrainFailures":
                model_profile[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_profile[
                    "EvaluationFailures"
                ],

            "ModelFailingTrainingBuilds":
                model_profile[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_profile[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_profile[
                    "UnlinkedRows"
                ],

            "ModelVerdictValuesJSON":
                model_profile[
                    "VerdictValuesJSON"
                ],

            "ProtocolEligible":
                protocol_eligible,

            "EligibilityReason":
                (
                    ""
                    if protocol_eligible
                    else "; ".join(
                        eligibility_reasons
                    )
                ),

            "InspectionStatus":
                (
                    "ELIGIBLE"
                    if protocol_eligible
                    else "INELIGIBLE"
                ),

            "InspectionError":
                "",
        })


        print(
            "    Status:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            number_of_builds,
            "| Model rows:",
            model_profile[
                "Rows"
            ],
            "| Model eval failures:",
            model_profile[
                "EvaluationFailures"
            ],
        )


    except Exception as error:
        row.update({
            "ProtocolEligible":
                False,

            "EligibilityReason":
                "Inspection error",

            "InspectionStatus":
                "ERROR",

            "InspectionError":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })

        print(
            "    ERROR:",
            row[
                "InspectionError"
            ],
        )


    row[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )


    scan_rows.append(
        row
    )


    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        pd.DataFrame(
            scan_rows
        ).sort_values(
            "CandidateInspectionOrder",
            kind="mergesort",
        ),
    )


scan_progress = (
    pd.DataFrame(
        scan_rows
    )
    .sort_values(
        "CandidateInspectionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DETERMINISTIC RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ERROR"
    )
].copy()


eligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ELIGIBLE"
    )
].copy()


ineligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "INELIGIBLE"
    )
].copy()


if not inspection_errors.empty:
    print(
        "\nCandidate inspection errors:"
    )

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 17 candidates could not be inspected. "
        "No provisional selection was frozen."
    )


if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 17 candidate was found."
    )


eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelTrainingRows",
            "ModelEvaluationRows",
            "RawExecutionRows",
            "Project",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        )
        + 1,
        dtype=np.int64,
    ),
)


top_candidate = eligible_candidates.iloc[
    0
]


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Projects 1–16 COMPLETE_AND_FROZEN",
    EXPECTED_REGISTERED_PROJECTS,
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ) == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 17 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Candidates inspected",
    EXPECTED_CANDIDATES,
    len(scan_progress),
    len(scan_progress)
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Unique candidate identities",
    EXPECTED_CANDIDATES,
    int(
        scan_progress[
            "Project"
        ].nunique()
    ),
    int(
        scan_progress[
            "Project"
        ].nunique()
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Registered/reserved candidates",
    0,
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ),
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Inspection errors",
    0,
    len(inspection_errors),
    len(inspection_errors)
    == 0,
)

add_check(
    validation_records,
    "Candidate accounting",
    EXPECTED_CANDIDATES,
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ),
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "At least one eligible candidate",
    "> 0",
    len(eligible_candidates),
    len(eligible_candidates)
    > 0,
)

add_check(
    validation_records,
    "Candidate ranks unique",
    len(eligible_candidates),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ) == len(
        eligible_candidates
    ),
)

add_check(
    validation_records,
    "Top rank",
    1,
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
    int(
        top_candidate[
            "CandidateRank"
        ]
    ) == 1,
)

add_check(
    validation_records,
    "Top candidate eligible",
    True,
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
)

add_check(
    validation_records,
    "Top candidate raw unlinked rows",
    0,
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model unlinked rows",
    0,
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model training failures",
    "> 0",
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ) > 0,
)

add_check(
    validation_records,
    "Top candidate model evaluation failures",
    "> 0",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ) > 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 17 Step 1A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed validation checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 17 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE AUTHORITATIVE STEP 1A OUTPUTS
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]


source_schema_audit = scan_progress[
    schema_columns
].copy()


atomic_write_csv(
    SCAN_PROGRESS_PATH,
    scan_progress,
)

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    scan_progress,
)

atomic_write_csv(
    ELIGIBLE_RANKED_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_PATH,
    ineligible_candidates,
)

atomic_write_csv(
    INSPECTION_ERRORS_PATH,
    inspection_errors,
)

atomic_write_csv(
    STEP1A_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


dimension_fields = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]


provisional_selection_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionState":
        "PROVISIONAL_PENDING_STEP_1B_FREEZE",

    "CandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "SourceDirectory":
        str(
            top_candidate[
                "SourceDirectory"
            ]
        ),

    "BuildIDColumn":
        str(
            top_candidate[
                "BuildIDColumn"
            ]
        ),

    "StartedAtColumn":
        str(
            top_candidate[
                "StartedAtColumn"
            ]
        ),

    "ExecutionBuildColumn":
        str(
            top_candidate[
                "ExecutionBuildColumn"
            ]
        ),

    "ExecutionVerdictColumn":
        str(
            top_candidate[
                "ExecutionVerdictColumn"
            ]
        ),

    "DatasetBuildColumn":
        str(
            top_candidate[
                "DatasetBuildColumn"
            ]
        ),

    "DatasetVerdictColumn":
        str(
            top_candidate[
                "DatasetVerdictColumn"
            ]
        ),

    "Dimensions": {
        field:
            int(
                top_candidate[
                    field
                ]
            )
        for field in dimension_fields
    },

    "RankingRule":
        RUNTIME_PRIORITY_RULE,

    "RankingPurpose":
        "Runtime-prioritized processing order only; protocol eligibility and final project set are unchanged",

    "EligibleCandidateCount":
        len(
            eligible_candidates
        ),

    "IneligibleCandidateCount":
        len(
            ineligible_candidates
        ),

    "ReservedActiveProjectsExcluded":
        sorted(
            RESERVED_ACTIVE_PROJECTS
        ),

    "CompletedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_selection_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_17_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalSelection":
        provisional_selection_payload,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project17ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_17_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project17ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL ISOLATION CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 17 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print(
    "\nRanked eligible Project 17 candidates:"
)

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


print(
    "\nProtocol-ineligible candidates:"
)

if ineligible_candidates.empty:
    print(
        "None"
    )

else:
    display(
        ineligible_candidates[
            [
                "Project",
                "Builds",
                "RawTrainFailures",
                "RawEvaluationFailures",
                "ModelTrainFailures",
                "ModelEvaluationFailures",
                "EligibilityReason",
            ]
        ]
    )


print("\n")
print("=" * 132)
print("=== PROJECT 17 CELL 2 / STEP 1A RESULT ===")
print("=" * 132)


print(
    "Registered projects:",
    len(
        registry
    ),
)

for required_number in sorted(
    required_registered_identities
):
    print(
        f"Project {required_number} identity:",
        required_registered_identities[
            required_number
        ],
    )


print(
    "Candidates inspected:",
    len(
        scan_progress
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)

print(
    "Runtime-priority ranking rule:",
    RUNTIME_PRIORITY_RULE,
)


print(
    "\nProvisional Project 17 candidate:"
)

print(
    "Candidate rank:",
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    str(
        top_candidate[
            "Project"
        ]
    ),
)

print(
    "Project slug:",
    str(
        top_candidate[
            "ProjectSlug"
        ]
    ),
)

print(
    "Source directory:",
    str(
        top_candidate[
            "SourceDirectory"
        ]
    ),
)


print(
    "\nCandidate dimensions:"
)

for field in dimension_fields:
    print(
        f"{field}:",
        int(
            top_candidate[
                field
            ]
        ),
    )


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 17 experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 17 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===
------------------------------------------------------------------------------------------------------------------------------------
[01/09] Inspecting: EMResearch@EvoMaster
    Status: ELIGIBLE | Builds: 583 | Model rows: 14460 | Model eval failures: 68
------------------------------------------------------------------------------------------------------------------------------------
[02/09] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE | Builds: 3668 | Model rows: 4822 | Model eval failures: 0
------------------------------------------------------------------------------------------------------------------------------------
[03/09] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE | Builds: 4286 | Model rows: 224550 | Model eval failures: 20
------------------------------------------------------------------------------------------------------------------------------------
[04/0

,Check,Expected,Actual,Pass
0,Completion registry rows,16,16,True
1,Projects 1–16 COMPLETE_AND_FROZEN,16,16,True
2,Project 17 registry rows,0,0,True
3,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
4,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True
5,Project 13 frozen identity,jcabi@jcabi-github,jcabi@jcabi-github,True
6,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
7,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
8,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
9,Candidates inspected,9,9,True



Ranked eligible Project 17 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,yamcs@Yamcs,yamcs__Yamcs,504,378,126,58101,147,45,10,7533,6452,1081,145,45,10,0,0
1,2,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,450,337,113,67108,144,31,12,10580,8232,2348,142,31,12,0,0
2,3,EMResearch@EvoMaster,EMResearch__EvoMaster,583,437,146,59155,286,68,41,14460,9907,4553,284,68,41,0,0
3,4,apache@curator,apache__curator,517,387,130,59697,124,2,2,10509,10403,106,123,2,2,0,0
4,5,facebook@buck,facebook__buck,846,634,212,561294,1120,8,7,80898,75643,5255,1119,8,7,0,0
5,6,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0
6,7,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
7,8,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,5635027,1778,20,17,224550,205696,18854,1777,20,17,0,0



Protocol-ineligible candidates:


,Project,Builds,RawTrainFailures,RawEvaluationFailures,ModelTrainFailures,ModelEvaluationFailures,EligibilityReason
1,Graylog2@graylog2-server,3668,280,0,279,0,No raw evaluation failures; No model evaluatio...




=== PROJECT 17 CELL 2 / STEP 1A RESULT ===
Registered projects: 16
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Candidates inspected: 9
Protocol-eligible candidates: 8
Protocol-ineligible candidates: 1
Inspection errors: 0
Runtime-priority ranking rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Provisional Project 17 candidate:
Candidate rank: 1
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Source directory: /content/datasets/datasets/yamcs@Yamcs

Candidate dimensions:
Builds: 504
TrainingBuilds: 378
EvaluationBuilds: 126
RawExecutionRows: 58101
RawTrainingRows: 42277
RawEvaluationRows: 15824
RawTrainFailures: 147
RawEvaluationFailures: 45
RawFailingTrainingBuilds: 63
RawFailingEvaluationBuilds: 10
RawUnlinkedRows: 0
ModelReadyRows

In [15]:
# ==================================================================================================
# PROJECT 17 — CELL 3 / STEP 1B
# FINAL SELECTION, CHRONOLOGY FREEZE, SOURCE MANIFEST, AND CHECKPOINT
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# SAFETY:
# - freezes the Project 17 identity selected by Step 1A;
# - freezes the complete source manifest and source-root SHA-256;
# - freezes the chronological 75/25 build split;
# - validates the exact raw/model dimensions discovered in Step 1A;
# - writes no completion-registry changes;
# - does not access or modify prior-project condition outputs;
# - does not start the Project 17 experiment.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 17 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
CANDIDATE_RANK = 1

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_17_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 16

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_DIMENSIONS = {
    "Builds": 504,
    "TrainingBuilds": 378,
    "EvaluationBuilds": 126,

    "RawExecutionRows": 58_101,
    "RawTrainingRows": 42_277,
    "RawEvaluationRows": 15_824,
    "RawTrainFailures": 147,
    "RawEvaluationFailures": 45,
    "RawFailingTrainingBuilds": 63,
    "RawFailingEvaluationBuilds": 10,
    "RawUnlinkedRows": 0,

    "ModelReadyRows": 7_533,
    "ModelTrainingRows": 6_452,
    "ModelEvaluationRows": 1_081,
    "ModelTrainFailures": 145,
    "ModelEvaluationFailures": 45,
    "ModelFailingTrainingBuilds": 62,
    "ModelFailingEvaluationBuilds": 10,
    "ModelUnlinkedRows": 0,
}

REQUIRED_SOURCE_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_17_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_17_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_17_provisional_selection.json"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_17_eligible_candidates_ranked.csv"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_17_fixed_chronological_builds.csv"
)

SOURCE_SCHEMA_SNAPSHOT_PATH = (
    SELECTION_ROOT
    / "project_17_selected_source_schema_snapshot.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_17_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def canonical_root_hash(
    manifest,
):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }

    missing_columns = (
        required_columns
        - set(manifest.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Source manifest is missing columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )

    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def profile_partition(
    frame,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
):
    training_mask = frame[
        build_column
    ].isin(
        training_build_ids
    )

    evaluation_mask = frame[
        build_column
    ].isin(
        evaluation_build_ids
    )

    linked_mask = (
        training_mask
        | evaluation_mask
    )

    failure_mask = frame[
        verdict_column
    ].ne(0)

    return {
        "Rows":
            int(len(frame)),

        "TrainingRows":
            int(training_mask.sum()),

        "EvaluationRows":
            int(evaluation_mask.sum()),

        "TrainingFailures":
            int(
                (
                    training_mask
                    & failure_mask
                ).sum()
            ),

        "EvaluationFailures":
            int(
                (
                    evaluation_mask
                    & failure_mask
                ).sum()
            ),

        "FailingTrainingBuilds":
            int(
                frame.loc[
                    training_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "FailingEvaluationBuilds":
            int(
                frame.loc[
                    evaluation_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "UnlinkedRows":
            int(
                (
                    ~linked_mask
                ).sum()
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_RANKED_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 17 Step 1B inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Selected Project 17 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


source_file_names = {
    path.name
    for path in SOURCE_DIRECTORY.iterdir()
    if path.is_file()
}


missing_required_source_files = sorted(
    REQUIRED_SOURCE_FILES
    - source_file_names
)


if missing_required_source_files:
    raise FileNotFoundError(
        "Selected Project 17 source is missing required files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1A, ARCHIVE, AND REGISTRY
# --------------------------------------------------------------------------------------------------

step1a_status_sha256 = sha256_file(
    STEP1A_STATUS_PATH
)

step1a_report_sha256 = sha256_file(
    STEP1A_REPORT_PATH
)

provisional_selection_sha256 = sha256_file(
    PROVISIONAL_SELECTION_PATH
)

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)


if step1a_status.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 17 Step 1A status is not PASS."
    )


if step1a_report.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 17 Step 1A report is not PASS."
    )


if provisional_selection.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 17 provisional selection state differs."
    )


if provisional_selection.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "Project 17 provisional project differs.\n"
        f"Expected: {PROJECT_NAME}\n"
        f"Actual:   {provisional_selection.get('Project')}"
    )


if provisional_selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Project 17 provisional slug differs."
    )


if Path(
    provisional_selection.get(
        "SourceDirectory",
        "",
    )
) != SOURCE_DIRECTORY:
    raise RuntimeError(
        "Project 17 provisional source directory differs."
    )


if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != CANDIDATE_RANK:
    raise RuntimeError(
        "Project 17 provisional candidate rank differs."
    )


if provisional_selection.get(
    "RankingRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 17 runtime-priority ranking rule differs."
    )


step1a_dimensions = {
    key: int(value)
    for key, value in provisional_selection.get(
        "Dimensions",
        {},
    ).items()
}

if step1a_dimensions != EXPECTED_DIMENSIONS:
    raise RuntimeError(
        "Project 17 Step 1A dimensions differ from the frozen Step 1B contract.\n"
        f"Expected: {EXPECTED_DIMENSIONS}\n"
        f"Actual:   {step1a_dimensions}"
    )


reserved_in_step1a = sorted(
    provisional_selection.get(
        "ReservedActiveProjectsExcluded",
        [],
    )
)

if reserved_in_step1a != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 17 Step 1A active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {reserved_in_step1a}"
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 17))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–16."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


if registry[
    registry_project_column
].eq(
    PROJECT_NAME
).any():
    raise RuntimeError(
        "The selected Project 17 identity is already registered."
    )




required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


ranked_candidates = pd.read_csv(
    ELIGIBLE_RANKED_PATH,
    low_memory=False,
)


rank_one_rows = ranked_candidates.loc[
    pd.to_numeric(
        ranked_candidates[
            "CandidateRank"
        ],
        errors="coerce",
    ).eq(
        CANDIDATE_RANK
    )
]


if len(rank_one_rows) != 1:
    raise RuntimeError(
        "Step 1A ranked candidates do not contain exactly one rank-1 row."
    )


rank_one = rank_one_rows.iloc[0]


if (
    str(
        rank_one[
            "Project"
        ]
    ) != PROJECT_NAME
    or str(
        rank_one[
            "ProjectSlug"
        ]
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 1A rank-1 identity differs."
    )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE COMPLETE SOURCE MANIFEST
# --------------------------------------------------------------------------------------------------

source_files = sorted(
    [
        path
        for path in SOURCE_DIRECTORY.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            SOURCE_DIRECTORY
        ).as_posix(),
)


if not source_files:
    raise RuntimeError(
        "Selected Project 17 source directory contains no files."
    )


source_manifest_records = []


for source_path in source_files:
    relative_path = source_path.relative_to(
        SOURCE_DIRECTORY
    ).as_posix()

    source_manifest_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


source_manifest = pd.DataFrame(
    source_manifest_records
)


source_root_sha256 = canonical_root_hash(
    source_manifest
)

source_file_count = len(
    source_manifest
)

source_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 7. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

source_paths = {
    "builds.csv":
        SOURCE_DIRECTORY
        / "builds.csv",

    "exe.csv":
        SOURCE_DIRECTORY
        / "exe.csv",

    "dataset.csv":
        SOURCE_DIRECTORY
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIRECTORY
        / "entity_change_history.csv",

    "id_map.csv":
        SOURCE_DIRECTORY
        / "id_map.csv",
}


if (
    SOURCE_DIRECTORY
    / "contributors.csv"
).is_file():
    source_paths[
        "contributors.csv"
    ] = (
        SOURCE_DIRECTORY
        / "contributors.csv"
    )


schema_snapshot_records = []


for filename, file_path in source_paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    schema_snapshot_records.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(columns),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_snapshot = pd.DataFrame(
    schema_snapshot_records
)


build_columns = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    nrows=0,
).columns.tolist()


build_id_column = resolve_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_timestamp_column = resolve_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)

exe_build_column = resolve_column(
    exe_columns,
    "build",
    "exe.csv build",
)

exe_verdict_column = resolve_column(
    exe_columns,
    "verdict",
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset_columns,
    "Build",
    "dataset.csv Build",
)

dataset_verdict_column = resolve_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. FREEZE CHRONOLOGY AND 75/25 SPLIT
# --------------------------------------------------------------------------------------------------

builds = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_series(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


invalid_timestamp_rows = int(
    builds[
        build_timestamp_column
    ].isna().sum()
)


duplicate_build_id_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)


if invalid_timestamp_rows != 0:
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


if duplicate_build_id_rows != 0:
    raise RuntimeError(
        "builds.csv contains duplicate build IDs."
    )


timestamp_group_sizes = builds.groupby(
    build_timestamp_column
).size()


timestamp_tie_groups = int(
    timestamp_group_sizes.gt(1).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(1)
    ].sum()
)


ordered_builds = (
    builds.sort_values(
        [
            build_timestamp_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


number_of_builds = len(
    ordered_builds
)


training_build_count = int(
    math.floor(
        0.75
        * number_of_builds
    )
)


evaluation_build_count = int(
    number_of_builds
    - training_build_count
)


ordered_builds[
    "ChronologyOrder"
] = np.arange(
    1,
    number_of_builds
    + 1,
    dtype=np.int64,
)


ordered_builds[
    "Partition"
] = np.where(
    ordered_builds[
        "ChronologyOrder"
    ].le(
        training_build_count
    ),
    "TRAIN",
    "EVALUATION",
)


ordered_builds[
    "PartitionOrder"
] = (
    ordered_builds.groupby(
        "Partition",
        sort=False,
    ).cumcount()
    + 1
)


fixed_chronology = ordered_builds[
    [
        "ChronologyOrder",
        build_id_column,
        build_timestamp_column,
        "Partition",
        "PartitionOrder",
    ]
].rename(
    columns={
        build_id_column:
            "BuildID",

        build_timestamp_column:
            "StartedAtUTC",
    }
)


training_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(int).tolist()
)


evaluation_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(int).tolist()
)


partition_overlap = len(
    training_build_ids
    & evaluation_build_ids
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE RAW AND MODEL DIMENSIONS
# --------------------------------------------------------------------------------------------------

exe = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=[
        exe_build_column,
        exe_verdict_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_series(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_verdict_column
] = parse_integer_series(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


dataset = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=[
        dataset_build_column,
        dataset_verdict_column,
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_series(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_verdict_column
] = parse_integer_series(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


raw_profile = profile_partition(
    frame=exe,
    build_column=exe_build_column,
    verdict_column=exe_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


model_profile = profile_partition(
    frame=dataset,
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


actual_dimensions = {
    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "RawExecutionRows":
        raw_profile[
            "Rows"
        ],

    "RawTrainingRows":
        raw_profile[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_profile[
            "EvaluationRows"
        ],

    "RawTrainFailures":
        raw_profile[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_profile[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_profile[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_profile[
            "FailingEvaluationBuilds"
        ],

    "RawUnlinkedRows":
        raw_profile[
            "UnlinkedRows"
        ],

    "ModelReadyRows":
        model_profile[
            "Rows"
        ],

    "ModelTrainingRows":
        model_profile[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_profile[
            "EvaluationRows"
        ],

    "ModelTrainFailures":
        model_profile[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_profile[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_profile[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_profile[
            "FailingEvaluationBuilds"
        ],

    "ModelUnlinkedRows":
        model_profile[
            "UnlinkedRows"
        ],
}


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1A status",
    STEP1A_PASS_STATUS,
    step1a_status.get(
        "Status"
    ),
    step1a_status.get(
        "Status"
    ) == STEP1A_PASS_STATUS,
)

add_check(
    validation_records,
    "Candidate rank",
    CANDIDATE_RANK,
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ),
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ) == CANDIDATE_RANK,
)

add_check(
    validation_records,
    "Selected project",
    PROJECT_NAME,
    provisional_selection[
        "Project"
    ],
    provisional_selection[
        "Project"
    ] == PROJECT_NAME,
)

add_check(
    validation_records,
    "Selected project slug",
    PROJECT_SLUG,
    provisional_selection[
        "ProjectSlug"
    ],
    provisional_selection[
        "ProjectSlug"
    ] == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Archive SHA-256",
    EXPECTED_ARCHIVE_SHA256,
    archive_sha256,
    archive_sha256
    == EXPECTED_ARCHIVE_SHA256,
)

add_check(
    validation_records,
    "Registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha256_before,
    registry_sha256_before
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 17 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_predecessor = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            registry_project_column,
        ].iloc[0]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_predecessor,
        actual_predecessor == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    reserved_in_step1a,
    reserved_in_step1a == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    provisional_selection.get(
        "RankingRule"
    ),
    provisional_selection.get(
        "RankingRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Source files",
    "> 0",
    source_file_count,
    source_file_count > 0,
)

add_check(
    validation_records,
    "Source bytes",
    "> 0",
    source_bytes,
    source_bytes > 0,
)

add_check(
    validation_records,
    "Invalid timestamp rows",
    0,
    invalid_timestamp_rows,
    invalid_timestamp_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build-ID rows",
    0,
    duplicate_build_id_rows,
    duplicate_build_id_rows == 0,
)

add_check(
    validation_records,
    "Partition overlap",
    0,
    partition_overlap,
    partition_overlap == 0,
)


for metric, expected_value in EXPECTED_DIMENSIONS.items():
    actual_value = int(
        actual_dimensions[
            metric
        ]
    )

    add_check(
        validation_records,
        metric,
        expected_value,
        actual_value,
        actual_value
        == expected_value,
    )


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 17 Step 1B validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Project 17 Step 1B checks:")

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 17 STEP 1B VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE FROZEN OUTPUTS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    fixed_chronology,
)

atomic_write_csv(
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    source_schema_snapshot,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RuntimePriorityPurpose":
        (
            "Processing order only; protocol eligibility "
            "and final project set are unchanged"
        ),

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "Step1AStatusSHA256":
        step1a_status_sha256,

    "Step1AReportSHA256":
        step1a_report_sha256,

    "ProvisionalSelectionSHA256":
        provisional_selection_sha256,

    "ChronologyRule":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "Dimensions":
        actual_dimensions,

    "FrozenSourceManifest":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronology":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "SourceSchemaSnapshot":
        str(
            SOURCE_SCHEMA_SNAPSHOT_PATH
        ),

    "Validation":
        str(
            STEP1B_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project17ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "SelectionCheckpoint":
        True,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceManifest":
        True,

    "DoNotChangeChronology":
        True,

    "DoNotChangeBuildPartitions":
        True,
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project17ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK AND IMMUTABILITY CHECKS
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 17 Step 1B."
    )


final_manifest_records = []


for row in source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise RuntimeError(
            "A frozen Project 17 source file disappeared:\n"
            f"{source_path}"
        )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_manifest_records
)


final_source_root_sha256 = canonical_root_hash(
    final_source_manifest
)


if final_source_root_sha256 != source_root_sha256:
    raise RuntimeError(
        "Project 17 source changed during Step 1B."
    )


checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 17 selection checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 17 Step 1B status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\nFrozen Project 17 source manifest:")

display(
    source_manifest
)


print("\nFixed Project 17 chronology sample:")

display(
    pd.concat(
        [
            fixed_chronology.head(10),
            fixed_chronology.tail(10),
        ],
        ignore_index=True,
    )
)


print("\n")
print("=" * 132)
print("=== PROJECT 17 CELL 3 / STEP 1B RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    CANDIDATE_RANK,
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[predecessor_number],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)


print("\nFrozen source:")

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nChronology:")

print(
    "Rule: started_at ascending; "
    "Build ID descending for timestamp ties"
)

print(
    "Builds:",
    number_of_builds,
)

print(
    "Training / evaluation builds:",
    len(
        training_build_ids
    ),
    "/",
    len(
        evaluation_build_ids
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Partition overlap:",
    partition_overlap,
)


print("\nRaw and model dimensions:")

for metric in EXPECTED_DIMENSIONS:
    print(
        f"{metric}:",
        actual_dimensions[
            metric
        ],
    )


print("\nIsolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 17 experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    selection_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 17 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 17 Step 1B validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_17_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_17_CANDIDATE_DISCOVERY_COMPLETE,True
1,Candidate rank,1,1,True
2,Selected project,yamcs@Yamcs,yamcs@Yamcs,True
3,Selected project slug,yamcs__Yamcs,yamcs__Yamcs,True
4,Archive SHA-256,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,True
5,Registry SHA-256,d0d3c32bc2374014357113905ed42282bc22f1d63888eb...,d0d3c32bc2374014357113905ed42282bc22f1d63888eb...,True
6,Registry rows,16,16,True
7,Project 17 registry rows,0,0,True
8,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
9,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True



Frozen Project 17 source manifest:


,RelativePath,SizeBytes,SHA256
0,builds.csv,79461,87a94dfd107db9584aa10f4408fb8fbee2effd846daaec...
1,contributors.csv,2958,b8f56b449742792ad0ab2029758a41a3d783d8ffaed9c5...
2,dataset.csv,6522700,19f4ee8120b993cc79c569d7541f948de9bc2dd7800851...
3,entity_change_history.csv,5985537,7df1e31e4a8de5acf83f32df2f80b112e61741c4542d39...
4,exe.csv,1815807,8c9e0b78b9ee60c54be7f4d62efc7712420a49baf00a5c...
5,id_map.csv,700031,e7578e20f45d198f57aedf92a6b74b22fab83c11b45366...



Fixed Project 17 chronology sample:


,ChronologyOrder,BuildID,StartedAtUTC,Partition,PartitionOrder
0,1,312474785,2017-12-06 15:22:36+00:00,TRAIN,1
1,2,340954297,2018-02-13 13:50:28+00:00,TRAIN,2
2,3,341031428,2018-02-13 16:41:58+00:00,TRAIN,3
3,4,341039027,2018-02-13 16:58:47+00:00,TRAIN,4
4,5,341355991,2018-02-14 09:54:10+00:00,TRAIN,5
5,6,341359252,2018-02-14 10:04:13+00:00,TRAIN,6
6,7,341404440,2018-02-14 13:23:12+00:00,TRAIN,7
7,8,343299488,2018-02-19 10:20:34+00:00,TRAIN,8
8,9,343306063,2018-02-19 10:40:29+00:00,TRAIN,9
9,10,343314691,2018-02-19 11:16:24+00:00,TRAIN,10




=== PROJECT 17 CELL 3 / STEP 1B RESULT ===

Project identity:
Project number: 17
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Candidate rank: 1
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Selection state: FINAL_AND_FROZEN

Frozen source:
Source directory: /content/datasets/datasets/yamcs@Yamcs
Source files: 6
Source bytes: 15106494
Source root SHA-256: 64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2

Chronology:
Rule: started_at ascending; Build ID descending for timestamp ties
Builds: 504
Training / evaluation builds: 378 / 126
Timestamp tie groups: 1
Partition overlap: 0

Raw and model dimensions:
Builds: 504
TrainingBuilds:

In [16]:
# ==================================================================================================
# PROJECT 17 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 17 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commits to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 17 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 17 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

SOURCE_DIR = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_17_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d69f27517f91bfac416"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 15_106_494

EXPECTED_BUILDS = 504
EXPECTED_TRAIN_BUILDS = 378
EXPECTED_EVAL_BUILDS = 126

EXPECTED_RAW_ROWS = 58_101
EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_RAW_TRAIN_FAILURES = 147
EXPECTED_RAW_EVAL_FAILURES = 45

EXPECTED_MODEL_ROWS = 7_533
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_17_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_17_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )

            result.append(
                token
            )

    return result


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(
        path
    ).is_file()
]

missing_inputs.extend(
    str(
        SOURCE_DIR
        / filename
    )
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIR
        / filename
    ).is_file()
)


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 17 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN SELECTION, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 17 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 17 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 17 identity differs."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 17 runtime-priority rule differs."
    )


active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 17 active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {active_reservations}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 16
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            17,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–16."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


frozen_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []


for row in frozen_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 17 source file is missing:\n"
            f"{source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)

current_source_root = source_root_hash(
    current_manifest
)

current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 17 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


build_order = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN VALIDATION
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        [
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        [
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(
        0
    ).sum()
)


if (
    raw_duplicate_pairs
    or model_duplicate_pairs
):
    raise RuntimeError(
        "Duplicate Build-Test pairs were found.\n"
        f"Raw duplicate rows: {raw_duplicate_pairs}\n"
        f"Model duplicate rows: {model_duplicate_pairs}"
    )


raw_pairs = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pairs = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


joined = model_pairs.merge(
    raw_pairs,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


verdict_mismatches = int(
    joined[
        "ModelVerdict"
    ].ne(
        joined[
            "RawVerdict"
        ]
    ).sum()
)


raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)


raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)


raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)


model_training_mask = joined[
    "Build"
].isin(
    training_builds
)


model_evaluation_mask = joined[
    "Build"
].isin(
    evaluation_builds
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


join_audit[
    "Pass"
] = (
    join_audit[
        "Expected"
    ].astype(
        str
    )
    == join_audit[
        "Actual"
    ].astype(
        str
    )
)


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,

        "PresentInDataset":
            feature
            in dataset.columns,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

token_rows = []
builds_without_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    tokens = extract_commit_tokens(
        raw_commits
    )

    if not tokens:
        builds_without_tokens += 1

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        token_rows.append({
            "BuildID":
                int(
                    build_id
                ),

            "ChronologyOrder":
                int(
                    build_order[
                        int(
                            build_id
                        )
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    token_rows
)


if build_tokens.empty:
    raise RuntimeError(
        "No build commit tokens could be extracted."
    )


build_tokens = (
    build_tokens.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ENTITY HISTORY AND ALIAS-AWARE ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


entity_history = (
    entity_history.loc[
        entity_history[
            "NormalisedCommit"
        ].ne(
            ""
        ),
        [
            entity_id_column,
            "NormalisedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    )
)


history_commits = sorted(
    entity_history[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commits
)


id_raw = pd.read_csv(
    paths[
        "id_map.csv"
    ],
    usecols=[
        id_key_column,
        id_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


orientation_rows = []


for column in [
    id_key_column,
    id_value_column,
]:
    numeric = pd.to_numeric(
        id_raw[
            column
        ],
        errors="coerce",
    )

    numeric_filled = numeric.fillna(
        0
    )

    valid_integral = (
        numeric.notna()
        & np.isclose(
            numeric_filled,
            np.floor(
                numeric_filled
            ),
            rtol=0,
            atol=0,
        )
    )

    parsed_ids = set(
        numeric.loc[
            valid_integral
        ].astype(
            "int64"
        )
    )

    overlap = len(
        parsed_ids
        & history_entity_ids
    )

    orientation_rows.append({
        "Column":
            column,

        "Rows":
            len(
                id_raw
            ),

        "IntegralNumericRows":
            int(
                valid_integral.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~valid_integral
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(
                parsed_ids
            ),

        "MatchingHistoryEntityIDs":
            overlap,

        "HistoryEntityCoveragePercent":
            (
                100.0
                * overlap
                / len(
                    history_entity_ids
                )
                if history_entity_ids
                else 0.0
            ),
    })


id_orientation = pd.DataFrame(
    orientation_rows
)


best_orientation = (
    id_orientation.sort_values(
        [
            "MatchingHistoryEntityIDs",
            "IntegralNumericRows",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    best_orientation
) < 2:
    raise RuntimeError(
        "id_map orientation audit is incomplete."
    )


if (
    int(
        best_orientation.loc[
            0,
            "MatchingHistoryEntityIDs",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "MatchingHistoryEntityIDs",
        ]
    )
    and int(
        best_orientation.loc[
            0,
            "IntegralNumericRows",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "IntegralNumericRows",
        ]
    )
):
    raise RuntimeError(
        "Could not uniquely resolve the EntityId column in id_map.csv."
    )


resolved_id_column = str(
    best_orientation.loc[
        0,
        "Column",
    ]
)


resolved_path_column = (
    id_value_column
    if resolved_id_column
    == id_key_column
    else id_key_column
)


resolved_numeric = pd.to_numeric(
    id_raw[
        resolved_id_column
    ],
    errors="coerce",
)


resolved_numeric_filled = resolved_numeric.fillna(
    0
)


valid_resolved = (
    resolved_numeric.notna()
    & np.isclose(
        resolved_numeric_filled,
        np.floor(
            resolved_numeric_filled
        ),
        rtol=0,
        atol=0,
    )
)


invalid_resolved_rows = int(
    (
        ~valid_resolved
    ).sum()
)


if invalid_resolved_rows:
    raise RuntimeError(
        "Resolved id_map EntityId column contains "
        f"{invalid_resolved_rows} invalid rows."
    )


resolved_id_map = pd.DataFrame({
    "EntityPath":
        id_raw[
            resolved_path_column
        ].astype(
            str
        ).str.strip(),

    "EntityId":
        resolved_numeric.astype(
            "int64"
        ),
})


empty_path_rows = int(
    resolved_id_map[
        "EntityPath"
    ].eq(
        ""
    ).sum()
)


exact_duplicate_rows = int(
    len(
        resolved_id_map
    )
    - len(
        resolved_id_map.drop_duplicates(
            [
                "EntityPath",
                "EntityId",
            ]
        )
    )
)


resolved_id_map = (
    resolved_id_map.drop_duplicates(
        [
            "EntityPath",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "EntityId",
            "EntityPath",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


duplicate_entity_id_rows = int(
    resolved_id_map.duplicated(
        "EntityId",
        keep=False,
    ).sum()
)


entity_ids_with_multiple_paths = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().gt(
        1
    ).sum()
)


paths_with_multiple_ids = int(
    resolved_id_map.groupby(
        "EntityPath"
    )[
        "EntityId"
    ].nunique().gt(
        1
    ).sum()
)


maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().max()
)


id_map_entity_ids = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# 12. COMMIT MATCHING AND BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

match_started = time.perf_counter()

match_rows = []


for row in build_tokens.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    matched_commit = None


    if token in history_commit_set:
        match_type = "EXACT"
        matched_commit = token
        candidate_count = 1

    else:
        candidates = [
            commit
            for commit in history_commits
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]

        if len(
            candidates
        ) == 1:
            match_type = (
                "UNIQUE_PREFIX"
            )

            matched_commit = candidates[
                0
            ]

            candidate_count = 1

        elif len(
            candidates
        ) == 0:
            match_type = (
                "UNMATCHED"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX"
            )

            candidate_count = len(
                candidates
            )


    match_rows.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_audit = (
    pd.DataFrame(
        match_rows
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


commit_matching_seconds = float(
    time.perf_counter()
    - match_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)


prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX"
    ).sum()
)


unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)


ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX"
    ).sum()
)


matched_token_rows = int(
    exact_matches
    + prefix_matches
)


commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        commit_audit
    )
)


matched_build_commits = (
    commit_audit.loc[
        commit_audit[
            "MatchedCommit"
        ].notna(),
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


entity_for_join = (
    entity_history.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity = (
    matched_build_commits.merge(
        entity_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity[
    "EntityId"
] = build_entity[
    "EntityId"
].astype(
    "int64"
)


build_entity = (
    build_entity[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


unmatched_commit_builds = sorted(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        ),
        "BuildID",
    ].astype(
        int
    ).unique().tolist()
)


mapping_incomplete_builds = sorted(
    set(
        builds_without_entities
    )
    | set(
        unmatched_commit_builds
    )
)


mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    )
)


mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


alias_summary = (
    resolved_id_map.groupby(
        "EntityId",
        as_index=False,
    )
    .agg(
        EntityPathAliasCount=(
            "EntityPath",
            "nunique",
        ),

        CanonicalEntityPath=(
            "EntityPath",
            "min",
        ),
    )
)


entity_id_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(
                mapped_entity_ids
            )
    })
    .merge(
        alias_summary,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


entity_id_audit[
    "PresentInIDMap"
] = entity_id_audit[
    "EntityPathAliasCount"
].notna()


entity_id_audit[
    "EntityPathAliasCount"
] = entity_id_audit[
    "EntityPathAliasCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            mapping_incomplete_builds
        ),
        [
            "BuildID",
            "ChronologyOrder",
            "Partition",
        ],
    ]
    .copy()
)


unmatched_counts = (
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        )
    ]
    .groupby(
        "BuildID"
    )
    .size()
    .rename(
        "UnmatchedCommitTokens"
    )
)


mapped_entity_counts = (
    build_entity.groupby(
        "BuildID"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "MappedEntityCount"
    )
)


mapping_incomplete_frame = (
    mapping_incomplete_frame.merge(
        unmatched_counts,
        on="BuildID",
        how="left",
    )
    .merge(
        mapped_entity_counts,
        on="BuildID",
        how="left",
    )
)


mapping_incomplete_frame[
    "UnmatchedCommitTokens"
] = mapping_incomplete_frame[
    "UnmatchedCommitTokens"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "MappedEntityCount"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "HasMappedEntities"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].gt(
    0
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    checks,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    checks,
    "Dataset key columns",
    3,
    3,
    (
        dataset_build_column
        in dataset.columns
        and dataset_test_column
        in dataset.columns
        and dataset_verdict_column
        in dataset.columns
    ),
)

add_check(
    checks,
    "Predictor count consistency",
    len(
        dataset.columns
    )
    - 3,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == (
        len(
            dataset.columns
        )
        - 3
    ),
)

add_check(
    checks,
    "REC features",
    19,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == 19
        and not missing_rec_features
    ),
)

for row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            row.Metric
        ),
        row.Expected,
        row.Actual,
        bool(
            row.Pass
        ),
    )

add_check(
    checks,
    "Invalid resolved id_map IDs",
    0,
    invalid_resolved_rows,
    invalid_resolved_rows
    == 0,
)

add_check(
    checks,
    "Empty id_map paths",
    0,
    empty_path_rows,
    empty_path_rows
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped entity IDs missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    16,
    len(
        registry
    ),
    len(
        registry
    ) == 16,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    checks,
    "Project 17 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 17 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 17 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 17 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


report_payload = {
    **summary_payload,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "ActiveReservations":
        active_reservations,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 17 Step 2A."
    )


final_manifest_records = []


for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 17 source changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),
            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 132
)

print(
    "=== PROJECT 17 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 132
)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

for predecessor_number in sorted(
    required_registered_identities
):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )


print(
    "Active reservations:",
    active_reservations,
)


print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)


print(
    "\nBuild-Test joins:"
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print(
    "\nid_map.csv resolution:"
)

print(
    "Resolved EntityId column:",
    resolved_id_column,
)

print(
    "Resolved path column:",
    resolved_path_column,
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)


print(
    "\nCommit and entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact commit matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Step 2B mapping audit required:",
    bool(
        unmatched_tokens
        or builds_without_entities
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 132
)


=== PROJECT 17 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===

Project 17 Step 2A validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d...,7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d...,True
1,Source root SHA-256,64cf358897e6363470ffb08f3239dd46044867f510a094...,64cf358897e6363470ffb08f3239dd46044867f510a094...,True
2,Source files,6,6,True
3,Source bytes,15106494,15106494,True
4,Builds,504,504,True
5,Training builds,378,378,True
6,Evaluation builds,126,126,True
7,Raw rows,58101,58101,True
8,Model rows,7533,7533,True
9,Dataset key columns,3,3,True



Build-Test join audit:


,Metric,Expected,Actual,Pass
0,RawRows,58101,58101,True
1,RawTrainingRows,42277,42277,True
2,RawEvaluationRows,15824,15824,True
3,RawTrainingFailures,147,147,True
4,RawEvaluationFailures,45,45,True
5,ModelRows,7533,7533,True
6,ModelTrainingRows,6452,6452,True
7,ModelEvaluationRows,1081,1081,True
8,ModelTrainingFailures,145,145,True
9,ModelEvaluationFailures,45,45,True



id_map orientation audit:


...


Mapping-incomplete builds:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities
0,349740857,41,TRAIN,1,0,False
1,368944017,101,TRAIN,1,19,True



Build-entity sample:


,BuildID,ChronologyOrder,MatchedCommit,EntityId
0,312474785,1,54066586844cd46ab0781ffd0206bc30abac801d,1
1,312474785,1,679ad863a9d6b6d7d451ec61ec23cab0494fa097,1
2,312474785,1,0105861bfb6d75a5633ac7875113737dca33fb40,5
3,312474785,1,0e89f7f2a244ee08df74e6e8ee8d0cdb5521e37c,5
4,312474785,1,0e983a18b19e2572b3e0a977c99b4d3d0046ea54,5
5,312474785,1,126b2ccc8c29453b40f8f6d5f359b11c79423580,5
6,312474785,1,17a87cf646e79de2f0f88c14adda54366cb692d9,5
7,312474785,1,2764f9e2fa49c108e4952c6c062019eee9a6a93b,5
8,312474785,1,2ebe58d0776255f5000a7000e0a4932fc18534fe,5
9,312474785,1,32d56d34fe7c033937abfe6be8017df8c9092073,5



=== PROJECT 17 CELL 4 / STEP 2A RESULT ===
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Builds: 504
Training / evaluation builds: 378 / 126
Raw execution rows: 58101
Model-ready rows: 7533
Dataset columns: 154
Predictor columns: 151
REC features: 19

Build-Test joins:
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links: 0
Model/raw verdict mismatches: 0
Non-finite duration rows: 0
Negative duration rows: 0

id_map.csv resolution:
Resolved EntityId column: value
Resolved path column: key
Duplicate EntityId rows accepted as aliases: 6272
EntityIds with multiple

In [17]:
# ==================================================================================================
# PROJECT 17 — CELL 5 / STEP 2B
# EXACT TIE-AWARE CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   yamcs@Yamcs
#
# WHY THIS IMPLEMENTATION IS SAFE:
# - Project 17 has one frozen timestamp-tie group.
# - Each raw Build-Test pair is unique.
# - Every affected test is evaluated under every feasible tied-build order.
# - The 16 non-file history features infer per-test order independently of file mapping.
# - REC_Age independently selects a compatible global build order.
# - The 16 non-file history features are reconstructed with vectorized cumulative calculations.
# - The two file-history features are reconstructed from the Step 2A build-entity map.
# - Clean anchor offsets preserve any accepted source-level file-mapping residuals exactly.
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
# DO NOT RERUN PROJECT 16 OR PROJECT 17 STEPS 0–2A.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from collections import defaultdict
from itertools import permutations, product
import math

import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 136)
print("=== PROJECT 17 CELL 5 / STEP 2B: EXACT TIE-AWARE CLEAN REC RECONSTRUCTION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

SOURCE_DIR = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_17_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_17_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

IMPLEMENTATION_VERSION = (
    "PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING"
)

EXPECTED_SELECTION_SHA256 = (
    "7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d69f27517f91bfac416"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 16

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 15_106_494

EXPECTED_BUILDS = 504
EXPECTED_TRAIN_BUILDS = 378
EXPECTED_EVAL_BUILDS = 126
EXPECTED_TIMESTAMP_TIE_GROUPS = 1

EXPECTED_RAW_ROWS = 58_101
EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_RAW_TRAIN_FAILURES = 147
EXPECTED_RAW_EVAL_FAILURES = 45

EXPECTED_MODEL_ROWS = 7_533
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 1_495
EXPECTED_EXACT_COMMIT_MATCHES = 1_493
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 2
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 503
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 1
EXPECTED_BUILD_ENTITY_ROWS = 23_763

EXPECTED_MAPPING_INCOMPLETE_BUILDS = {
    349740857,
    368944017,
}
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = {
    "TRAIN",
}
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 1

RECENT_WINDOW = 6

SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2

DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9

ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

TIE_INFERENCE_FEATURES = [
    feature
    for feature in REC_FEATURES
    if feature != "REC_Age"
    and feature not in FILE_HISTORY_REC
]

MAX_TIE_ORDER_COMBINATIONS = 1_024

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_17_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_17_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

TIMESTAMP_TIE_GROUPS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
)

TEST_ORDER_SEARCH_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_test_order_search_audit.csv"
)

INFERRED_EXECUTION_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

GLOBAL_AGE_ORDER_SEARCH_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_global_age_order_search.csv"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def prefix_sum(
    values,
):
    values = np.asarray(
        values
    )

    dtype = (
        np.float64
        if values.dtype.kind == "f"
        else np.int64
    )

    result = np.empty(
        len(values) + 1,
        dtype=dtype,
    )

    result[0] = 0

    np.cumsum(
        values,
        out=result[1:],
    )

    return result


def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.full(
        len(denominator),
        -1.0,
        dtype=float,
    )

    valid = denominator > 0

    result[
        valid
    ] = (
        numerator[
            valid
        ]
        / denominator[
            valid
        ]
    )

    return result


def calculate_file_rate(
    target_builds,
    current_changed_entities,
    entity_changed_builds,
):
    if not target_builds:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(
                entity_id
            )
        )

        if not changed_builds:
            continue

        overlap_count = len(
            target_builds.intersection(
                changed_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_requested_group_features(
    builds,
    verdicts,
    durations,
    global_positions,
    requested_positions,
    changed_entities_by_build,
    entity_changed_builds,
):
    builds = np.asarray(
        builds,
        dtype=np.int64,
    )

    verdicts = np.asarray(
        verdicts,
        dtype=np.int64,
    )

    durations = np.asarray(
        durations,
        dtype=np.float64,
    )

    global_positions = np.asarray(
        global_positions,
        dtype=np.int64,
    )

    requested_positions = np.asarray(
        requested_positions,
        dtype=np.int64,
    )

    n = len(
        builds
    )

    all_positions = np.arange(
        n,
        dtype=np.int64,
    )

    failure = (
        verdicts
        != SUCCESS_VERDICT_CODE
    ).astype(
        np.int64
    )

    assertion = (
        verdicts
        == ASSERTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    exception = (
        verdicts
        == EXCEPTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    transition = np.zeros(
        n,
        dtype=np.int64,
    )

    if n > 1:
        transition[
            1:
        ] = (
            verdicts[
                1:
            ]
            != verdicts[
                :-1
            ]
        ).astype(
            np.int64
        )

    duration_prefix = prefix_sum(
        durations
    )

    failure_prefix = prefix_sum(
        failure
    )

    assertion_prefix = prefix_sum(
        assertion
    )

    exception_prefix = prefix_sum(
        exception
    )

    transition_prefix = prefix_sum(
        transition
    )

    positions = requested_positions

    history_length = positions.astype(
        float
    )

    recent_start = np.maximum(
        0,
        positions - RECENT_WINDOW,
    )

    recent_length = (
        positions
        - recent_start
    ).astype(
        float
    )

    last_failure_inclusive = np.maximum.accumulate(
        np.where(
            failure > 0,
            all_positions,
            -1,
        )
    )

    last_transition_inclusive = np.maximum.accumulate(
        np.where(
            transition > 0,
            all_positions,
            -1,
        )
    )

    prior_failure_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    prior_transition_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    positive_history = positions > 0

    prior_failure_position[
        positive_history
    ] = last_failure_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    prior_transition_position[
        positive_history
    ] = last_transition_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    recent_max = np.full(
        n,
        np.nan,
        dtype=float,
    )

    for offset in range(
        1,
        RECENT_WINDOW + 1,
    ):
        if n <= offset:
            continue

        recent_max[
            offset:
        ] = np.fmax(
            recent_max[
                offset:
            ],
            durations[
                :-offset
            ],
        )

    total_max_inclusive = np.maximum.accumulate(
        durations
    )

    previous_indices = np.maximum(
        positions - 1,
        0,
    )

    reconstructed = {
        "REC_Age":
            (
                global_positions[
                    positions
                ]
                - global_positions[
                    0
                ]
            ).astype(
                float
            ),

        "REC_LastFailureAge":
            np.where(
                prior_failure_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_failure_position
                ).astype(
                    float
                ),
            ),

        "REC_LastTransitionAge":
            np.where(
                prior_transition_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_transition_position
                ).astype(
                    float
                ),
            ),

        "REC_RecentAvgExeTime":
            safe_divide(
                (
                    duration_prefix[
                        positions
                    ]
                    - duration_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentMaxExeTime":
            np.where(
                positive_history,
                recent_max[
                    positions
                ],
                -1.0,
            ),

        "REC_RecentFailRate":
            safe_divide(
                (
                    failure_prefix[
                        positions
                    ]
                    - failure_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentAssertRate":
            safe_divide(
                (
                    assertion_prefix[
                        positions
                    ]
                    - assertion_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentExcRate":
            safe_divide(
                (
                    exception_prefix[
                        positions
                    ]
                    - exception_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentTransitionRate":
            safe_divide(
                (
                    transition_prefix[
                        positions
                    ]
                    - transition_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_TotalAvgExeTime":
            safe_divide(
                duration_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalMaxExeTime":
            np.where(
                positive_history,
                total_max_inclusive[
                    previous_indices
                ],
                -1.0,
            ),

        "REC_TotalFailRate":
            safe_divide(
                failure_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalAssertRate":
            safe_divide(
                assertion_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalExcRate":
            safe_divide(
                exception_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalTransitionRate":
            safe_divide(
                transition_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_LastVerdict":
            np.where(
                positive_history,
                verdicts[
                    previous_indices
                ],
                -1,
            ).astype(
                float
            ),

        "REC_LastExeTime":
            np.where(
                positive_history,
                durations[
                    previous_indices
                ],
                -1.0,
            ),
    }

    file_failure_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    file_transition_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    failure_event_positions = np.flatnonzero(
        failure > 0
    )

    transition_event_positions = np.flatnonzero(
        transition > 0
    )

    failure_pointer = 0
    transition_pointer = 0

    prior_failure_builds = set()
    prior_transition_builds = set()

    requested_order = np.argsort(
        positions,
        kind="mergesort",
    )

    for requested_index in requested_order:
        current_position = int(
            positions[
                requested_index
            ]
        )

        while (
            failure_pointer
            < len(
                failure_event_positions
            )
            and int(
                failure_event_positions[
                    failure_pointer
                ]
            )
            < current_position
        ):
            prior_failure_builds.add(
                int(
                    builds[
                        failure_event_positions[
                            failure_pointer
                        ]
                    ]
                )
            )

            failure_pointer += 1

        while (
            transition_pointer
            < len(
                transition_event_positions
            )
            and int(
                transition_event_positions[
                    transition_pointer
                ]
            )
            < current_position
        ):
            prior_transition_builds.add(
                int(
                    builds[
                        transition_event_positions[
                            transition_pointer
                        ]
                    ]
                )
            )

            transition_pointer += 1

        current_build = int(
            builds[
                current_position
            ]
        )

        current_entities = changed_entities_by_build.get(
            current_build,
            frozenset(),
        )

        file_failure_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_failure_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

        file_transition_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_transition_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

    reconstructed[
        "REC_MaxTestFileFailRate"
    ] = file_failure_rate

    reconstructed[
        "REC_MaxTestFileTransitionRate"
    ] = file_transition_rate

    return (
        reconstructed,
        transition,
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 17 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2a_status = load_json(
    STEP2A_STATUS_PATH
)

step2a_report = load_json(
    STEP2A_REPORT_PATH
)

entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 17 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    selection.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 17 Step 1B is not frozen successfully."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or step2a_report.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or entity_mapping_summary.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):
    raise RuntimeError(
        "Project 17 Step 2A outputs are not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–16."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


if selection.get(
    "Project"
) != PROJECT_NAME or selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Frozen Project 17 identity differs."
    )


active_reservations = selection.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 17 active-reservation state differs."
    )


if selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 17 runtime-priority rule differs."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 17 source file is missing:\n"
            f"{source_path}"
        )

    current_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_manifest_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The local Project 17 source does not match "
        "the frozen source manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


chronology[
    "ChronologyOrder"
] = parse_int(
    chronology[
        "ChronologyOrder"
    ],
    "chronology.ChronologyOrder",
)


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "The frozen chronology contains invalid timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


timestamp_group_sizes = (
    chronology.groupby(
        "StartedAtUTC"
    )
    .size()
)


timestamp_tie_groups_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


if timestamp_tie_groups_count != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Project 17 timestamp-tie count differs from the frozen selection contract."
    )


timestamp_tie_groups = []
timestamp_tie_group_records = []

tied_rows = chronology.loc[
    chronology["StartedAtUTC"].duplicated(keep=False)
].copy()

for tie_group_number, (started_at, group) in enumerate(
    tied_rows.groupby("StartedAtUTC", sort=True),
    start=1,
):
    baseline_builds = (
        group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"]
        .astype(int)
        .tolist()
    )

    permutation_count = math.factorial(len(baseline_builds))
    if permutation_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A timestamp-tie group is too large for exact enumeration.\n"
            f"StartedAtUTC={started_at}; builds={baseline_builds}; "
            f"permutations={permutation_count}"
        )

    options = [tuple(int(value) for value in order) for order in permutations(baseline_builds)]
    timestamp_tie_groups.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at,
        "BuildIDs": tuple(baseline_builds),
        "Options": options,
    })

    timestamp_tie_group_records.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at.isoformat(),
        "BuildCount": len(baseline_builds),
        "BuildIDsJSON": json.dumps(baseline_builds),
        "PermutationCount": permutation_count,
    })


timestamp_tie_groups_frame = pd.DataFrame(
    timestamp_tie_group_records,
    columns=[
        "TieGroup",
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsJSON",
        "PermutationCount",
    ],
)


build_chronology_map = chronology.set_index(
    "BuildID"
)[
    "ChronologyOrder"
].astype(
    int
).to_dict()


build_timestamp_map = chronology.set_index(
    "BuildID"
)[
    "StartedAtUTC"
].to_dict()


dataset_header = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    nrows=0,
).columns.tolist()


model_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

model_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

model_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    usecols=[
        model_build_column,
        model_test_column,
        model_verdict_column,
    ] + REC_FEATURES,
    low_memory=False,
)


dataset[
    model_build_column
] = parse_int(
    dataset[
        model_build_column
    ],
    "dataset.Build",
)


dataset[
    model_test_column
] = parse_int(
    dataset[
        model_test_column
    ],
    "dataset.Test",
)


dataset[
    model_verdict_column
] = parse_int(
    dataset[
        model_verdict_column
    ],
    "dataset.Verdict",
)


dataset = dataset.rename(
    columns={
        model_build_column:
            "Build",

        model_test_column:
            "Test",

        model_verdict_column:
            "Verdict",
    }
).reset_index(
    drop=True
)


dataset[
    "_ModelRow"
] = np.arange(
    len(
        dataset
    ),
    dtype=np.int64,
)


print(
    "Loading the 58,101-row clean execution history."
)


exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.verdict",
)


exe[
    exe_job_column
] = pd.to_numeric(
    exe[
        exe_job_column
    ],
    errors="coerce",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


if exe[
    exe_job_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing/non-numeric job values."
    )


if not np.isfinite(
    exe[
        exe_duration_column
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )


if exe[
    exe_duration_column
].lt(
    0
).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )


observed_verdict_codes = sorted(
    int(
        value
    )
    for value in exe[
        exe_verdict_column
    ].unique().tolist()
)


if not set(
    observed_verdict_codes
).issubset({
    0,
    1,
    2,
    3,
}):
    raise RuntimeError(
        "exe.csv contains an unsupported verdict code.\n"
        f"Observed codes: {observed_verdict_codes}"
    )


exe = exe.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "ChronologyOrder"
] = exe[
    "Build"
].map(
    build_chronology_map
)


if exe[
    "ChronologyOrder"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to frozen chronology."
    )


exe[
    "ChronologyOrder"
] = exe[
    "ChronologyOrder"
].astype(
    np.int64
)


raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_pairs != 0
    or model_duplicate_pairs != 0
):
    raise RuntimeError(
        "Duplicate Build-Test pairs prevent exact REC reconstruction."
    )


raw_build_ids = set(
    exe[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)


global_build_sequence = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            raw_build_ids
        )
    ]
    .sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )[
        "BuildID"
    ]
    .astype(
        int
    )
    .tolist()
)


global_build_position = {
    int(
        build_id
    ):
        position
    for position, build_id in enumerate(
        global_build_sequence
    )
}


exe[
    "GlobalBuildPosition"
] = exe[
    "Build"
].map(
    global_build_position
)


if exe[
    "GlobalBuildPosition"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to global first-appearance order."
    )


exe[
    "GlobalBuildPosition"
] = exe[
    "GlobalBuildPosition"
].astype(
    np.int64
)


print(
    "Sorting raw execution history by Test and frozen chronology."
)


sort_started = time.perf_counter()


exe = (
    exe.sort_values(
        [
            "Test",
            "ChronologyOrder",
            "Build",
            "Job",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exe[
    "InferredTestOrder"
] = (
    exe.groupby(
        "Test",
        sort=False,
    )
    .cumcount()
    .astype(
        np.int64
    )
)


sort_seconds = float(
    time.perf_counter()
    - sort_started
)


commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)


build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)


mapping_incomplete_source = pd.read_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    low_memory=False,
)


commit_audit[
    "BuildID"
] = parse_int(
    commit_audit[
        "BuildID"
    ],
    "commit_audit.BuildID",
)


build_entity[
    "BuildID"
] = parse_int(
    build_entity[
        "BuildID"
    ],
    "build_entity.BuildID",
)


build_entity[
    "EntityId"
] = parse_int(
    build_entity[
        "EntityId"
    ],
    "build_entity.EntityId",
)


mapping_incomplete_source[
    "BuildID"
] = parse_int(
    mapping_incomplete_source[
        "BuildID"
    ],
    "mapping_incomplete.BuildID",
)


mapping_incomplete_source_partitions = sorted(
    set(
        mapping_incomplete_source[
            "Partition"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )
)


mapping_incomplete_source_rows_with_entities = int(
    mapping_incomplete_source[
        "HasMappedEntities"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
    .sum()
)


normalised_match_type = (
    commit_audit[
        "MatchType"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


exact_match_mask = normalised_match_type.eq(
    "EXACT"
)

prefix_match_mask = normalised_match_type.eq(
    "UNIQUE_PREFIX"
)

unmatched_mask = normalised_match_type.eq(
    "UNMATCHED"
)

ambiguous_mask = normalised_match_type.eq(
    "AMBIGUOUS_PREFIX"
)


unknown_match_type_rows = int(
    (
        ~(
            exact_match_mask
            | prefix_match_mask
            | unmatched_mask
            | ambiguous_mask
        )
    ).sum()
)


if unknown_match_type_rows != 0:
    raise RuntimeError(
        "Commit audit contains unknown MatchType rows."
    )


exact_matches = int(
    exact_match_mask.sum()
)

prefix_matches = int(
    prefix_match_mask.sum()
)

unmatched_tokens = int(
    unmatched_mask.sum()
)

ambiguous_tokens = int(
    ambiguous_mask.sum()
)


unmatched_token_builds = sorted(
    commit_audit.loc[
        unmatched_mask,
        "BuildID",
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


mapping_incomplete_builds = sorted(
    set(
        unmatched_token_builds
    )
    | set(
        builds_without_entities
    )
)


changed_entities_by_build = {
    int(
        build_id
    ):
        frozenset(
            int(
                entity_id
            )
            for entity_id in values
        )
    for build_id, values in build_entity.groupby(
        "BuildID",
        sort=False,
    )[
        "EntityId"
    ]
}


entity_changed_builds_accumulator = defaultdict(
    set
)


for row in build_entity[
    [
        "BuildID",
        "EntityId",
    ]
].itertuples(
    index=False
):
    entity_changed_builds_accumulator[
        int(
            row.EntityId
        )
    ].add(
        int(
            row.BuildID
        )
    )


entity_changed_builds = {
    entity_id:
        frozenset(
            build_ids
        )
    for entity_id, build_ids in entity_changed_builds_accumulator.items()
}


del entity_changed_builds_accumulator
gc.collect()


raw_build_counts = exe.groupby(
    "Build",
    sort=False,
).size()


raw_build_failures = (
    exe[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        exe[
            "Build"
        ]
    )
    .sum()
)


model_build_counts = dataset.groupby(
    "Build",
    sort=False,
).size()


model_build_failures = (
    dataset[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        dataset[
            "Build"
        ]
    )
    .sum()
)


unmatched_mapping_audit = (
    mapping_incomplete_source.copy()
    .sort_values(
        "BuildID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


unmatched_mapping_audit[
    "RawExecutionRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "RawFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_failures
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelReadyRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_failures
).fillna(
    0
).astype(
    int
)


print(
    "\nTimestamp tie groups:"
)

display(
    timestamp_tie_groups_frame
)


print(
    "\nUnmatched mapping audit:"
)

display(
    unmatched_mapping_audit
)


# --------------------------------------------------------------------------------------------------
# 6. EXACT PER-TEST TIE-ORDER INFERENCE
# --------------------------------------------------------------------------------------------------

order_search_started = time.perf_counter()

model_group_indices = dataset.groupby("Test", sort=False).indices
raw_group_indices = exe.groupby("Test", sort=False).indices

source_rec_arrays = {
    feature: dataset[feature].to_numpy(dtype=float)
    for feature in REC_FEATURES
}

build_timestamp_ns = {
    int(build_id): int(pd.Timestamp(timestamp).value)
    for build_id, timestamp in build_timestamp_map.items()
}


def ordered_raw_indices_for_choice(raw_indices, tie_choice):
    rows = exe.loc[raw_indices, ["Build", "Job"]].copy()
    rows["_OriginalIndex"] = np.asarray(raw_indices, dtype=np.int64)
    rows["_TimestampNS"] = rows["Build"].map(build_timestamp_ns).astype(np.int64)
    rows["_TieRank"] = 0

    for group_number, selected_order in tie_choice.items():
        rank = {int(build_id): position for position, build_id in enumerate(selected_order)}
        mask = rows["Build"].isin(rank)
        rows.loc[mask, "_TieRank"] = rows.loc[mask, "Build"].map(rank).astype(int)

    rows = rows.sort_values(
        ["_TimestampNS", "_TieRank", "Build", "Job"],
        kind="mergesort",
    )
    return rows["_OriginalIndex"].to_numpy(dtype=np.int64)


def tie_options_for_test(build_ids):
    build_set = set(int(value) for value in build_ids)
    touched = []
    for tie_group in timestamp_tie_groups:
        present = [value for value in tie_group["BuildIDs"] if value in build_set]
        if len(present) > 1:
            options = [
                tuple(value for value in option if value in build_set)
                for option in tie_group["Options"]
            ]
            options = list(dict.fromkeys(options))
            touched.append((int(tie_group["TieGroup"]), options))
    return touched


test_order_search_records = []
inferred_raw_indices_by_test = {}

total_tests = len(raw_group_indices)

for test_number, (test_id_raw, raw_indices_raw) in enumerate(raw_group_indices.items(), start=1):
    test_id = int(test_id_raw)
    raw_indices = np.asarray(raw_indices_raw, dtype=np.int64)
    group_builds_baseline = exe.loc[raw_indices, "Build"].to_numpy(dtype=np.int64)
    touched_groups = tie_options_for_test(group_builds_baseline)
    model_rows = model_group_indices.get(test_id)

    if touched_groups:
        combination_count = int(np.prod([len(options) for _, options in touched_groups]))
    else:
        combination_count = 1

    if combination_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A test requires too many exact tie-order combinations.\n"
            f"Test={test_id}; combinations={combination_count}"
        )

    choice_records = []
    choice_product = product(*[options for _, options in touched_groups]) if touched_groups else [tuple()]

    for candidate_number, selected_orders in enumerate(choice_product, start=1):
        tie_choice = {
            group_number: selected_order
            for (group_number, _), selected_order in zip(touched_groups, selected_orders)
        }
        candidate_indices = ordered_raw_indices_for_choice(raw_indices, tie_choice)

        if model_rows is None:
            mismatch_counts = {}
            mismatch_values = 0
        else:
            model_rows_array = np.asarray(model_rows, dtype=np.int64)
            requested_builds = dataset.loc[model_rows_array, "Build"].to_numpy(dtype=np.int64)
            candidate_builds = exe.loc[candidate_indices, "Build"].to_numpy(dtype=np.int64)
            position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
            missing_requested = [int(build_id) for build_id in requested_builds if int(build_id) not in position_by_build]
            if missing_requested:
                raise RuntimeError(
                    "A model-ready test contains builds missing from raw history.\n"
                    f"Test={test_id}; sample={missing_requested[:20]}"
                )
            requested_positions = np.asarray(
                [position_by_build[int(build_id)] for build_id in requested_builds],
                dtype=np.int64,
            )
            provisional_global = np.asarray(
                [global_build_position[int(build_id)] for build_id in candidate_builds],
                dtype=np.int64,
            )
            reconstructed_candidate, _ = reconstruct_requested_group_features(
                builds=candidate_builds,
                verdicts=exe.loc[candidate_indices, "Verdict"].to_numpy(dtype=np.int64),
                durations=exe.loc[candidate_indices, "Duration"].to_numpy(dtype=np.float64),
                global_positions=provisional_global,
                requested_positions=requested_positions,
                changed_entities_by_build=changed_entities_by_build,
                entity_changed_builds=entity_changed_builds,
            )
            mismatch_counts = {}
            for feature in TIE_INFERENCE_FEATURES:
                source_values = source_rec_arrays[feature][model_rows_array]
                reconstructed_values = reconstructed_candidate[feature]
                mismatch_counts[feature] = int((~np.isclose(
                    source_values,
                    reconstructed_values,
                    rtol=DIRECT_RTOL,
                    atol=DIRECT_ATOL,
                    equal_nan=False,
                )).sum())
            mismatch_values = int(sum(mismatch_counts.values()))

        choice_records.append({
            "Candidate": candidate_number,
            "TieChoice": tie_choice,
            "OrderedIndices": candidate_indices,
            "MismatchCounts": mismatch_counts,
            "MismatchValues": mismatch_values,
        })

    minimum_mismatch = min(record["MismatchValues"] for record in choice_records)
    best_records = [record for record in choice_records if record["MismatchValues"] == minimum_mismatch]
    selected_record = best_records[0]
    inferred_raw_indices_by_test[test_id] = selected_record["OrderedIndices"]

    search_mode = (
        "RAW_ONLY_TEST_FROZEN_TIE_ORDER"
        if model_rows is None and touched_groups
        else "RAW_ONLY_TEST_DIRECT_ORDER"
        if model_rows is None
        else "MODEL_READY_TEST_EXACT_TIE_SEARCH"
        if touched_groups
        else "MODEL_READY_TEST_DIRECT_ORDER"
    )

    test_order_search_records.append({
        "Test": test_id,
        "RawExecutionRows": len(raw_indices),
        "ModelReadyRows": 0 if model_rows is None else len(model_rows),
        "TimestampTieGroupsForTest": len(touched_groups),
        "CandidateOrderCombinations": combination_count,
        "MinimumMismatchValues": minimum_mismatch,
        "ZeroMismatchCandidates": int(sum(record["MismatchValues"] == 0 for record in choice_records)),
        "BestMismatchCountsJSON": json.dumps(selected_record["MismatchCounts"], sort_keys=True),
        "SelectedTieOrdersJSON": json.dumps(
            [list(selected_record["TieChoice"].get(group_number, tuple())) for group_number, _ in touched_groups]
        ),
        "SearchMode": search_mode,
    })

    if test_number % 100 == 0 or test_number == total_tests:
        print("Per-test tie-order inference progress:", test_number, "/", total_tests, "tests")


test_order_search_audit = pd.DataFrame(test_order_search_records)
model_ready_tests = int(test_order_search_audit["ModelReadyRows"].gt(0).sum())
raw_only_tests = int(test_order_search_audit["ModelReadyRows"].eq(0).sum())
tests_with_timestamp_ties = int(test_order_search_audit["TimestampTieGroupsForTest"].gt(0).sum())
tests_with_nonzero_order_mismatches = int(test_order_search_audit["MinimumMismatchValues"].gt(0).sum())
tests_with_ambiguous_zero_orders = int(test_order_search_audit["ZeroMismatchCandidates"].gt(1).sum())
total_test_order_mismatch_values = int(test_order_search_audit["MinimumMismatchValues"].sum())

order_search_seconds = float(time.perf_counter() - order_search_started)

print("\nPer-test tie-order inference summary:")
display(pd.DataFrame([
    {"Metric": "Tests", "Value": total_tests},
    {"Metric": "Model-ready tests", "Value": model_ready_tests},
    {"Metric": "Raw-only tests", "Value": raw_only_tests},
    {"Metric": "Tests touching timestamp ties", "Value": tests_with_timestamp_ties},
    {"Metric": "Tests with non-zero minimum mismatch", "Value": tests_with_nonzero_order_mismatches},
    {"Metric": "Total minimum mismatch values", "Value": total_test_order_mismatch_values},
    {"Metric": "Tests with multiple zero-mismatch orders", "Value": tests_with_ambiguous_zero_orders},
    {"Metric": "Inference seconds", "Value": order_search_seconds},
]))


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL REC_AGE ORDER SEARCH AND FULL CLEAN RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

global_order_search_records = []
global_tie_options = [tie_group["Options"] for tie_group in timestamp_tie_groups]
global_choice_product = product(*global_tie_options) if global_tie_options else [tuple()]

for candidate_number, selected_orders in enumerate(global_choice_product, start=1):
    selected_by_timestamp = {
        int(pd.Timestamp(tie_group["StartedAtUTC"]).value): tuple(int(value) for value in selected_order)
        for tie_group, selected_order in zip(timestamp_tie_groups, selected_orders)
    }

    candidate_sequence = []
    for started_at, group in chronology.groupby("StartedAtUTC", sort=True):
        timestamp_ns = int(pd.Timestamp(started_at).value)
        group_builds = [
            int(value)
            for value in group["BuildID"].astype(int).tolist()
            if int(value) in raw_build_ids
        ]
        if not group_builds:
            continue
        if timestamp_ns in selected_by_timestamp:
            order = [value for value in selected_by_timestamp[timestamp_ns] if value in set(group_builds)]
        else:
            order = [
                int(value)
                for value in group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"].astype(int).tolist()
                if int(value) in raw_build_ids
            ]
        candidate_sequence.extend(order)

    candidate_position = {int(build_id): position for position, build_id in enumerate(candidate_sequence)}
    first_build_by_test = {
        int(test_id): int(exe.loc[indices, "Build"].iloc[0])
        for test_id, indices in inferred_raw_indices_by_test.items()
    }
    source_age = dataset["REC_Age"].to_numpy(dtype=float)
    reconstructed_age = np.asarray([
        candidate_position[int(build_id)] - candidate_position[first_build_by_test[int(test_id)]]
        for build_id, test_id in dataset[["Build", "Test"]].itertuples(index=False, name=None)
    ], dtype=float)
    age_mismatches = int((~np.isclose(
        source_age,
        reconstructed_age,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )).sum())

    global_order_search_records.append({
        "Candidate": candidate_number,
        "AgeMismatchRows": age_mismatches,
        "BuildOrderSHA256": hashlib.sha256(
            ",".join(str(build_id) for build_id in candidate_sequence).encode("utf-8")
        ).hexdigest(),
        "TieOrdersJSON": json.dumps([list(order) for order in selected_orders]),
        "BuildSequence": candidate_sequence,
        "BuildPosition": candidate_position,
    })

best_age_mismatches = min(record["AgeMismatchRows"] for record in global_order_search_records)
best_global_records = [record for record in global_order_search_records if record["AgeMismatchRows"] == best_age_mismatches]
selected_global_record = best_global_records[0]
global_build_sequence = selected_global_record["BuildSequence"]
global_build_position = selected_global_record["BuildPosition"]
global_age_combination_count = len(global_order_search_records)
zero_age_candidates = int(sum(record["AgeMismatchRows"] == 0 for record in global_order_search_records))

global_age_order_search = pd.DataFrame([
    {key: value for key, value in record.items() if key not in {"BuildSequence", "BuildPosition"}}
    for record in global_order_search_records
])

print("\nGlobal REC_Age tie-order search:")
display(global_age_order_search)

reconstruction_started = time.perf_counter()
model_group_indices = dataset.groupby("Test", sort=False).indices
model_build_array = dataset["Build"].to_numpy(dtype=np.int64)
result_arrays = {
    feature: np.full(len(dataset), np.nan, dtype=np.float64)
    for feature in REC_FEATURES
}
filled_model_rows = np.zeros(len(dataset), dtype=bool)
inferred_order_lookup = {}

for test_number, (test_id_raw, ordered_indices) in enumerate(inferred_raw_indices_by_test.items(), start=1):
    test_id = int(test_id_raw)
    ordered_indices = np.asarray(ordered_indices, dtype=np.int64)
    candidate_builds = exe.loc[ordered_indices, "Build"].to_numpy(dtype=np.int64)
    for position, build_id in enumerate(candidate_builds):
        inferred_order_lookup[(test_id, int(build_id))] = position

    model_rows = model_group_indices.get(test_id)
    if model_rows is None:
        continue
    model_rows = np.asarray(model_rows, dtype=np.int64)
    requested_builds = model_build_array[model_rows]
    position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
    requested_positions = np.asarray([position_by_build[int(build_id)] for build_id in requested_builds], dtype=np.int64)
    candidate_global_positions = np.asarray([global_build_position[int(build_id)] for build_id in candidate_builds], dtype=np.int64)

    reconstructed_group, _ = reconstruct_requested_group_features(
        builds=candidate_builds,
        verdicts=exe.loc[ordered_indices, "Verdict"].to_numpy(dtype=np.int64),
        durations=exe.loc[ordered_indices, "Duration"].to_numpy(dtype=np.float64),
        global_positions=candidate_global_positions,
        requested_positions=requested_positions,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )

    for feature in REC_FEATURES:
        result_arrays[feature][model_rows] = reconstructed_group[feature]
    filled_model_rows[model_rows] = True

    if test_number % 100 == 0 or test_number == total_tests:
        print("Full REC reconstruction progress:", test_number, "/", total_tests, "tests | reconstructed rows:", int(filled_model_rows.sum()))

if not filled_model_rows.all():
    missing_model_rows = np.flatnonzero(~filled_model_rows)
    raise RuntimeError(
        "Clean REC reconstruction did not fill every model-ready row.\n"
        f"Missing rows: {len(missing_model_rows)}; sample={missing_model_rows[:20].tolist()}"
    )

exe["InferredTestOrder"] = np.asarray([
    inferred_order_lookup[(int(test_id), int(build_id))]
    for test_id, build_id in exe[["Test", "Build"]].itertuples(index=False, name=None)
], dtype=np.int64)
exe["GlobalBuildPosition"] = exe["Build"].map(global_build_position).astype(np.int64)
exe = exe.sort_values(["Test", "InferredTestOrder"], kind="mergesort").reset_index(drop=True)

clean_reconstructed = dataset[["Build", "Test"]].copy()
for feature in REC_FEATURES:
    clean_reconstructed[feature] = result_arrays[feature]

reconstruction_seconds = float(time.perf_counter() - reconstruction_started)

frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder": np.arange(1, len(global_build_sequence) + 1, dtype=np.int64),
    "BuildID": global_build_sequence,
})
frozen_global_build_order["StartedAtUTC"] = frozen_global_build_order["BuildID"].map(build_timestamp_map)

reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


missing_reconstructed_rows = int(
    clean_reconstructed[
        REC_FEATURES
    ].isna().any(
        axis=1
    ).sum()
)


if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )


if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction contains missing values."
    )


comparison_records = []
mismatch_examples = []


anchor_offsets = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)


rows_at_mapping_incomplete_build = dataset[
    "Build"
].isin(
    mapping_incomplete_build_set
).to_numpy()


for feature in REC_FEATURES:
    original_values = dataset[
        feature
    ].to_numpy(
        dtype=float
    )

    reconstructed_values = clean_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(
            original_values
        ).all()
        or not np.isfinite(
            reconstructed_values
        ).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = (
        ~direct_match_mask
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[
        feature
    ] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in FILE_HISTORY_REC,

        "Rows":
            len(
                dataset
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            int(
                direct_mismatch_mask.sum()
            ),

        "DirectMismatchesAtMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & rows_at_mapping_incomplete_build
                ).sum()
            ),

        "DirectMismatchesOutsideMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & (
                        ~rows_at_mapping_incomplete_build
                    )
                ).sum()
            ),

        "NonZeroAnchorOffsets":
            int(
                (
                    direct_difference
                    != 0
                ).sum()
            ),

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_match_mask
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MaximumAbsoluteAnchoredDifference":
            float(
                np.max(
                    np.abs(
                        original_values
                        - anchored_values
                    )
                )
            ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[
        :20
    ]

    for mismatch_index in mismatch_indices:
        mismatch_examples.append({
            "Build":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Build"
                    ]
                ),

            "Test":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Test"
                    ]
                ),

            "Feature":
                feature,

            "Original":
                float(
                    original_values[
                        mismatch_index
                    ]
                ),

            "Reconstructed":
                float(
                    reconstructed_values[
                        mismatch_index
                    ]
                ),

            "Difference":
                float(
                    direct_difference[
                        mismatch_index
                    ]
                ),

            "MappingIncompleteBuild":
                bool(
                    rows_at_mapping_incomplete_build[
                        mismatch_index
                    ]
                ),
        })


comparison_summary = pd.DataFrame(
    comparison_records
)


mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "Original",
        "Reconstructed",
        "Difference",
        "MappingIncompleteBuild",
    ],
)


anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows":
            "MatchingRows",

        "AnchoredMismatchingRows":
            "MismatchingRows",
    }
)


anchor_validation[
    "Pass"
] = anchor_validation[
    "MismatchingRows"
].eq(
    0
)


direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)


verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_DEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_INDEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


anchored_mismatch_values = int(
    anchor_validation[
        "MismatchingRows"
    ].sum()
)


nonzero_anchor_offset_values = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).sum()
)


rows_with_any_nonzero_anchor_offset = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).any(
        axis=1
    ).sum()
)


unmatched_mapping_effect_is_confined = bool(
    non_file_direct_mismatches == 0
    and file_mismatches_outside_mapping_incomplete_build == 0
)


zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)


age_mismatch_rows = int(
    comparison_summary.loc[
        comparison_summary["Feature"].eq("REC_Age"),
        "DirectMismatchingRows",
    ].iloc[0]
)

if age_mismatch_rows != best_age_mismatches:
    raise RuntimeError(
        "Final REC_Age mismatch count differs from the global-order search result."
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

raw_train_mask = exe[
    "Build"
].isin(
    training_builds
)


raw_eval_mask = exe[
    "Build"
].isin(
    evaluation_builds
)


model_train_mask = dataset[
    "Build"
].isin(
    training_builds
)


model_eval_mask = dataset[
    "Build"
].isin(
    evaluation_builds
)


validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)


add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get(
        "Status"
    ),
    step2a_status.get(
        "Status"
    )
    == EXPECTED_STEP2A_STATUS,
)


add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)


add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)


add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_groups_count,
    timestamp_tie_groups_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)


add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    int(
        raw_train_mask.sum()
    ),
    int(
        raw_train_mask.sum()
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    int(
        raw_eval_mask.sum()
    ),
    int(
        raw_eval_mask.sum()
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    int(
        model_train_mask.sum()
    ),
    int(
        model_train_mask.sum()
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    int(
        model_eval_mask.sum()
    ),
    int(
        model_eval_mask.sum()
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    )
    == EXPECTED_DATASET_COLUMNS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_pairs,
    raw_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_pairs,
    model_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Official assertion verdict code",
    2,
    ASSERTION_VERDICT_CODE,
    ASSERTION_VERDICT_CODE
    == 2,
)


add_check(
    validation_records,
    "Official exception verdict code",
    1,
    EXCEPTION_VERDICT_CODE,
    EXCEPTION_VERDICT_CODE
    == 1,
)


add_check(
    validation_records,
    "Per-test search accounting",
    total_tests,
    model_ready_tests
    + raw_only_tests,
    (
        model_ready_tests
        + raw_only_tests
    )
    == total_tests,
)


add_check(
    validation_records,
    "Tests touching timestamp ties",
    "> 0",
    tests_with_timestamp_ties,
    tests_with_timestamp_ties
    > 0,
)


add_check(
    validation_records,
    "Tests with non-zero order mismatches",
    0,
    tests_with_nonzero_order_mismatches,
    tests_with_nonzero_order_mismatches
    == 0,
)


add_check(
    validation_records,
    "Total order mismatch values",
    0,
    total_test_order_mismatch_values,
    total_test_order_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age mismatch rows",
    0,
    best_age_mismatches,
    best_age_mismatches
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age zero-match candidates",
    "> 0",
    zero_age_candidates,
    zero_age_candidates
    > 0,
)


add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(
        commit_audit
    ),
    len(
        commit_audit
    )
    == EXPECTED_COMMIT_TOKEN_ROWS,
)


add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unique-prefix matches",
    EXPECTED_PREFIX_COMMIT_MATCHES,
    prefix_matches,
    prefix_matches
    == EXPECTED_PREFIX_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(
        builds_with_entities
    ),
    len(
        builds_with_entities
    )
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(
        builds_without_entities
    ),
    len(
        builds_without_entities
    )
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Mapping-incomplete build identities",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    mapping_incomplete_builds,
    set(
        mapping_incomplete_builds
    )
    == EXPECTED_MAPPING_INCOMPLETE_BUILDS,
)


add_check(
    validation_records,
    "Mapping-incomplete source rows",
    len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    len(
        mapping_incomplete_source
    ),
    len(
        mapping_incomplete_source
    )
    == len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
)


add_check(
    validation_records,
    "Mapping-incomplete partitions",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    ),
    mapping_incomplete_source_partitions,
    set(
        mapping_incomplete_source_partitions
    )
    == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS,
)


add_check(
    validation_records,
    "Mapping-incomplete rows with mapped entities",
    EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
    mapping_incomplete_source_rows_with_entities,
    mapping_incomplete_source_rows_with_entities
    == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
)


add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(
        build_entity
    ),
    len(
        build_entity
    )
    == EXPECTED_BUILD_ENTITY_ROWS,
)


add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(
        clean_reconstructed
    ),
    len(
        clean_reconstructed
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows
    == 0,
)


add_check(
    validation_records,
    "Missing reconstructed values",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows
    == 0,
)


add_check(
    validation_records,
    "Non-file direct mismatch values",
    0,
    non_file_direct_mismatches,
    non_file_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build
    == 0,
)


add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)


add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features
    == 0,
)


add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection.get(
        "RuntimePriorityRule"
    ),
    selection.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    validation_records,
    "Project 17 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 17 Step 2B validation:"
)

display(
    validation
)


print(
    "\nClean REC comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor validation:"
)

display(
    anchor_validation
)


if not failed_validation.empty:
    print(
        "\nFailed Step 2B checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 2B PASS checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 17 STEP 2B VALIDATION FAILED. "
        "DO NOT START THE EXPERIMENT."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


exe[
    "StartedAtUTC"
] = exe[
    "Build"
].map(
    build_timestamp_map
)


inferred_execution_order_for_storage = (
    exe[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "StartedAtUTC",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)


atomic_csv(
    TIMESTAMP_TIE_GROUPS_PATH,
    timestamp_tie_groups_frame,
)


atomic_csv(
    TEST_ORDER_SEARCH_AUDIT_PATH,
    test_order_search_audit,
)


print(
    "\nWriting the frozen 58,101-row execution-order parquet."
)


atomic_parquet(
    INFERRED_EXECUTION_ORDER_PATH,
    inferred_execution_order_for_storage,
)


atomic_csv(
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    global_age_order_search,
)


atomic_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    frozen_global_build_order,
)


atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)


atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)


atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

execution_order_metadata = pq.ParquetFile(
    INFERRED_EXECUTION_ORDER_PATH
)


execution_order_readback_rows = int(
    execution_order_metadata.metadata.num_rows
)


execution_order_readback_columns = set(
    execution_order_metadata.schema.names
)


required_execution_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if (
    execution_order_readback_rows
    != EXPECTED_RAW_ROWS
    or not required_execution_order_columns.issubset(
        execution_order_readback_columns
    )
):
    raise RuntimeError(
        "Frozen execution-order parquet metadata readback failed."
    )


reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if len(
    reconstructed_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )


readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


readback_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original_values = readback_join[
        feature
    ].to_numpy(
        dtype=float
    )

    readback_mismatch_values += int(
        (
            ~np.isclose(
                reproduced_values,
                original_values,
                rtol=ANCHOR_RTOL,
                atol=ANCHOR_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH,
    TIMESTAMP_TIE_GROUPS_PATH,
    TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "OfficialVerdictSemantics": {
        "Success":
            SUCCESS_VERDICT_CODE,

        "Exception":
            EXCEPTION_VERDICT_CODE,

        "Assertion":
            ASSERTION_VERDICT_CODE,
    },

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "ModelReadyTests":
        model_ready_tests,

    "RawOnlyTests":
        raw_only_tests,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "TestsWithMultipleZeroMismatchOrders":
        tests_with_ambiguous_zero_orders,

    "GlobalAgeOrderCombinations":
        global_age_combination_count,

    "GlobalAgeZeroMismatchCandidates":
        zero_age_candidates,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "RawSortSeconds":
        sort_seconds,

    "RECReconstructionSeconds":
        reconstruction_seconds,

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "RawExecutionRows":
        len(
            inferred_execution_order_for_storage
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "ReconstructedRows":
        len(
            clean_reconstructed
        ),

    "GlobalBuildOrderRows":
        len(
            frozen_global_build_order
        ),

    "CommitTokenRows":
        len(
            commit_audit
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        mapping_incomplete_builds,

    "MappingIncompletePartitions":
        mapping_incomplete_source_partitions,

    "MappingIncompleteRowsWithMappedEntities":
        mapping_incomplete_source_rows_with_entities,

    "DirectMismatchValues":
        direct_mismatch_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_anchor_offset,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_offset_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ReadbackMismatchValues":
        readback_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        active_reservations,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_16_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        rec_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 17 Step 2B."
    )


final_source_manifest_records = []

for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_manifest_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 17 source changed during Step 2B."
    )


checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP2B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP2B_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP2B_STATUS
):
    raise RuntimeError(
        "Project 17 Step 2B checkpoint/status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 17 CELL 5 / STEP 2B RESULT ===")
print("=" * 136)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Active reservations:",
    active_reservations,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)


print(
    "\nOfficial verdict semantics:"
)

print(
    "Success:",
    SUCCESS_VERDICT_CODE,
)

print(
    "Exception:",
    EXCEPTION_VERDICT_CODE,
)

print(
    "Assertion:",
    ASSERTION_VERDICT_CODE,
)


print(
    "\nTie-aware execution-order freeze:"
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups_count,
)

print(
    "Raw execution-order rows:",
    execution_order_readback_rows,
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Tests:",
    total_tests,
)

print(
    "Model-ready tests:",
    model_ready_tests,
)

print(
    "Raw-only tests:",
    raw_only_tests,
)

print(
    "Tests with non-zero order mismatches:",
    tests_with_nonzero_order_mismatches,
)

print(
    "Global REC_Age mismatch rows:",
    best_age_mismatches,
)


print(
    "\nClean REC reconstruction:"
)

print(
    "Raw history rows:",
    len(
        inferred_execution_order_for_storage
    ),
)

print(
    "Model rows requested/reconstructed:",
    len(
        dataset
    ),
    "/",
    len(
        clean_reconstructed
    ),
)

print(
    "Direct mismatch values:",
    direct_mismatch_values,
)

print(
    "Non-file direct mismatch values:",
    non_file_direct_mismatches,
)

print(
    "File-history direct mismatch values:",
    file_history_direct_mismatches,
)

print(
    "File-history mismatches outside mapping-incomplete builds:",
    file_mismatches_outside_mapping_incomplete_build,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_anchor_offset,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_offset_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatch values:",
    anchored_mismatch_values,
)

print(
    "Readback mismatch values:",
    readback_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    zero_percent_clean_reproduced_exactly,
)


print(
    "\nMapping audit:"
)

print(
    "Commit-token rows:",
    len(
        commit_audit
    ),
)

print(
    "Exact / prefix / unmatched / ambiguous:",
    exact_matches,
    "/",
    prefix_matches,
    "/",
    unmatched_tokens,
    "/",
    ambiguous_tokens,
)

print(
    "Builds with / without mapped entities:",
    len(
        builds_with_entities
    ),
    "/",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Mapping-incomplete partitions:",
    mapping_incomplete_source_partitions,
)

print(
    "Mapping-incomplete rows with mapped entities:",
    mapping_incomplete_source_rows_with_entities,
)

print(
    "Unmatched mapping effect confined:",
    unmatched_mapping_effect_is_confined,
)


print(
    "\nRuntime:"
)

print(
    "Raw sort seconds:",
    round(
        sort_seconds,
        2,
    ),
)

print(
    "REC reconstruction seconds:",
    round(
        reconstruction_seconds,
        2,
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nREC reconstruction checkpoint:"
)

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    rec_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP2B_STATUS,
)

print("=" * 136)


=== PROJECT 17 CELL 5 / STEP 2B: EXACT TIE-AWARE CLEAN REC RECONSTRUCTION ===
Loading the 58,101-row clean execution history.
Sorting raw execution history by Test and frozen chronology.

Timestamp tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount
0,1,2019-04-01T16:54:45+00:00,2,"[514249525, 514249489]",2



Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows
0,349740857,41,TRAIN,1,0,False,108,0,0,0
1,368944017,101,TRAIN,1,19,True,113,0,0,0


Per-test tie-order inference progress: 100 / 160 tests
Per-test tie-order inference progress: 160 / 160 tests

Per-test tie-order inference summary:


...


Clean REC comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,DirectMismatchesAtMappingIncompleteBuild,DirectMismatchesOutsideMappingIncompleteBuild,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference,MaximumAbsoluteAnchoredDifference
0,REC_Age,VERDICT_INDEPENDENT,False,7533,7533,0,0,0,0,7533,0,0.000000e+00,0.000000e+00,0.0
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,7533,7533,0,0,0,0,7533,0,0.000000e+00,0.000000e+00,0.0
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,7533,7533,0,0,0,0,7533,0,0.000000e+00,0.000000e+00,0.0
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,7533,7533,0,0,0,1109,7533,0,1.455192e-11,1.080283e-13,0.0
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,7533,7533,0,0,0,0,7533,0,0.000000e+00,0.000000e+00,0.0
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,7533,7533,0,0,0,133,7533,0,5.551115e-17,9.800854e-19,0.0
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,7533,7533,0,0,0,37,7533,0,5.551115e-17,2.726553e-19,0.0
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,7533,7533,0,0,0,96,7533,0,5.551115e-17,7.074300e-19,0.0
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,7533,7533,0,0,0,92,7533,0,5.551115e-17,6.779538e-19,0.0
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,7533,7533,0,0,0,1190,7533,0,7.275958e-12,1.110326e-13,0.0



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,7533,7533,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,7533,7533,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,7533,7533,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,7533,7533,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,7533,7533,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,7533,7533,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,7533,7533,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,7533,7533,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,7533,7533,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,7533,7533,0,0.0,True



Writing the frozen 58,101-row execution-order parquet.


=== PROJECT 17 CELL 5 / STEP 2B RESULT ===
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Source root SHA-256: 64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2

Official verdict semantics:
Success: 0
Exception: 1
Assertion: 2

Tie-aware execution-order freeze:
Timestamp tie groups: 1
Raw execution-order rows: 58101
Global build-order rows: 504
Tests: 160
Model-ready tests: 152
Raw-only tests: 8
Tests with non-zero order mismatches: 0
Global REC_Age mismatch rows: 0

Clean REC reconstruction:
Raw history rows: 58101
Model rows reque

In [18]:
# ==================================================================================================
# PROJECT 17 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN AND COHORT FREEZE
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 17 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the Project 17 failure-subtype distribution;
# - generate deterministic project/seed random streams for label-noise injection;
# - prove nested masks across all noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the evaluation partition clean and immutable;
# - perform no model fitting and no registry write.
#
# SAFETY:
# - Projects 1–16 must remain COMPLETE_AND_FROZEN and unchanged;
# - Project 17 must remain absent from the completion registry;
# - no prior-project condition output is accessed or modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 17 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

SOURCE_DIR = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_17_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_17_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d69f27517f91bfac416"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "c6c3ce653908b7b407f6d87525c836bf4310857ff1e33f4d05e6f93ebf11e367"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 16

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 15_106_494

EXPECTED_BUILDS = 504
EXPECTED_TRAIN_BUILDS = 378
EXPECTED_EVAL_BUILDS = 126

EXPECTED_RAW_ROWS = 58_101
EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_RAW_TRAIN_FAILURES = 147
EXPECTED_RAW_EVAL_FAILURES = 45

EXPECTED_MODEL_ROWS = 7_533
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 10

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_17_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_17_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_17_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target_path = str(
        Path(
            path
        )
    )

    matches = [
        entry
        for entry in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            entry.get(
                "Path",
                "",
            )
        ) == target_path
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "The Project 17 REC checkpoint does not contain exactly "
            f"one manifest entry for {target_path}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR / "dataset.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 17 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 17 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 17 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 17 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 17 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


expected_rec_freeze_flags = {
    # The implementation label is frozen exactly as written by Project 17 Step 2B.
    # Step 2B retained the legacy PROJECT_16 checkpoint-type label; the project identity fields are 17.
    "ImplementationVersion":
        "PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING",

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_16_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    if rec_checkpoint.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "The Project 17 REC checkpoint does not match the frozen "
            f"Step 2B contract: {flag_name}={expected_value!r}."
        )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 17 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–16."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


selection_active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    None,
)


if selection_active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 17 selection checkpoint active reservations differ."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 17 selection checkpoint runtime-priority rule differs."
    )


if rec_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 17 REC checkpoint active reservations differ."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 17 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


expected_inferred_execution_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    INFERRED_EXECUTION_ORDER_PATH,
)

expected_global_build_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
)

expected_clean_reconstructed_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_RECONSTRUCTED_PATH,
)

expected_clean_anchor_offsets_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_ANCHOR_OFFSETS_PATH,
)

actual_inferred_execution_order_sha256 = sha256_file(
    INFERRED_EXECUTION_ORDER_PATH
)

actual_global_build_order_sha256 = sha256_file(
    FROZEN_GLOBAL_BUILD_ORDER_PATH
)

actual_clean_reconstructed_sha256 = sha256_file(
    CLEAN_RECONSTRUCTED_PATH
)

actual_clean_anchor_offsets_sha256 = sha256_file(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    actual_inferred_execution_order_sha256
    != expected_inferred_execution_order_sha256
    or actual_global_build_order_sha256
    != expected_global_build_order_sha256
    or actual_clean_reconstructed_sha256
    != expected_clean_reconstructed_sha256
    or actual_clean_anchor_offsets_sha256
    != expected_clean_anchor_offsets_sha256
):
    raise RuntimeError(
        "One or more frozen Project 17 Step 2B artifacts "
        "do not match the REC checkpoint manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, MODEL DATA, AND THE FROZEN V6 RAW EXECUTION ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


# V6 froze the exact per-test execution order required to reproduce all 19 REC features.
# This is the canonical raw-history cohort for every Project 17 noise condition.
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


missing_inferred_columns = (
    required_inferred_columns
    - set(
        inferred_execution_order.columns
    )
)


if missing_inferred_columns:
    raise RuntimeError(
        "The frozen V6 inferred execution-order file is missing columns:\n"
        + "\n".join(
            sorted(
                missing_inferred_columns
            )
        )
    )


inferred_execution_order[
    "Build"
] = parse_int(
    inferred_execution_order[
        "Build"
    ],
    "inferred_execution_order.Build",
)

inferred_execution_order[
    "Test"
] = parse_int(
    inferred_execution_order[
        "Test"
    ],
    "inferred_execution_order.Test",
)

inferred_execution_order[
    "Verdict"
] = parse_int(
    inferred_execution_order[
        "Verdict"
    ],
    "inferred_execution_order.Verdict",
)

inferred_execution_order[
    "InferredTestOrder"
] = parse_int(
    inferred_execution_order[
        "InferredTestOrder"
    ],
    "inferred_execution_order.InferredTestOrder",
)

inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)

inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "The frozen raw execution order contains missing/non-numeric "
        "job or duration values."
    )


if not np.isfinite(
    inferred_execution_order[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "The frozen raw execution order contains non-finite durations."
    )


if inferred_execution_order[
    "Duration"
].lt(
    0
).any():
    raise RuntimeError(
        "The frozen raw execution order contains negative durations."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_build_test_rows
    or raw_duplicate_test_order_rows
):
    raise RuntimeError(
        "The frozen V6 execution order contains duplicate keys."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "The frozen V6 global build-order file is missing required columns."
    )


frozen_global_build_order[
    "GlobalBuildOrder"
] = parse_int(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order[
    "BuildID"
] = parse_int(
    frozen_global_build_order[
        "BuildID"
    ],
    "frozen_global_build_order.BuildID",
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            1,
            EXPECTED_BUILDS
            + 1,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "The frozen V6 global build order is invalid."
    )


# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_valid = bool(
    failure_subtypes.astype(
        int
    ).tolist()
    == [
        1,
        2,
    ]
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 17 clean training failures do not use exactly "
        "the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 17 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "fixed chronological split: started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "RawHistoryOrder":
        (
            "Project 17 Step 2B frozen per-test execution order; "
            "timestamp-tie order inferred from exact clean REC reproduction"
        ),

    "GlobalRECAgeBuildOrder":
        (
            "Project 17 Step 2B frozen global build first-appearance order"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint implementation",
    "PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING",
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == "PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING",
)

add_check(
    validation_records,
    "REC checkpoint schema version",
    1,
    rec_checkpoint.get(
        "CheckpointVersion"
    ),
    rec_checkpoint.get(
        "CheckpointVersion"
    )
    == 1,
)

add_check(
    validation_records,
    "Frozen inferred execution-order SHA-256",
    expected_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256
    == expected_inferred_execution_order_sha256,
)

add_check(
    validation_records,
    "Frozen global build-order SHA-256",
    expected_global_build_order_sha256,
    actual_global_build_order_sha256,
    actual_global_build_order_sha256
    == expected_global_build_order_sha256,
)

add_check(
    validation_records,
    "Frozen clean reconstruction SHA-256",
    expected_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256
    == expected_clean_reconstructed_sha256,
)

add_check(
    validation_records,
    "Frozen clean anchor-offset SHA-256",
    expected_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256
    == expected_clean_anchor_offsets_sha256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Failure subtype values",
    [
        1,
        2,
    ],
    failure_subtypes.astype(
        int
    ).tolist(),
    failure_subtype_values_valid,
)

add_check(
    validation_records,
    "Failure subtype profile sum",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    ) == EXPECTED_REGISTERED_PROJECTS,
)

for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project == required_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_active_reservations,
    selection_active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Project 17 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 17 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 17 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        "Project 17 Step 2B exact per-test inferred order",

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 17 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 17 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 17 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 17 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 17 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print("\nFrozen clean-history order:")

print(
    "Inferred execution-order rows:",
    len(
        exe
    ),
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Raw duplicate Test-order rows:",
    raw_duplicate_test_order_rows,
)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 17 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)


=== PROJECT 17 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 17 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_17_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_17_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d...,7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d...,True
3,REC checkpoint SHA-256,c6c3ce653908b7b407f6d87525c836bf4310857ff1e33f...,c6c3ce653908b7b407f6d87525c836bf4310857ff1e33f...,True
4,REC checkpoint implementation,PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_...,PROJECT_17_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_...,True
...,...,...,...,...
67,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
68,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
69,Active reservations,[],[],True
70,Project 17 registry rows,0,0,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,97,0.659864
1,2,50,0.340136



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,3213219829,3782039437,42277,0,...,0,147,147,6452,0,145,145,ffda9bca5c0f8caafca764df7dd27b289ce7624c1d730d...,b12c91a19045da050d02b0356652b9896e423445eef28e...,0c307d89fbca2b5f2b95ce78ab4c31431020ef9ed4aceb...
1,2,noise_05__seed_01,1,2,5,1,3213219829,3782039437,42277,2104,...,9,147,2233,6452,320,145,447,e7de1df49110c19e76a27de393eb4d9a335587cc44bfc7...,c1482b53f0c3b3268197dd9c08273b9388be50c932b397...,445dd0172548ea702dc84d676b8c7f5ffa0b0232fb38b9...
2,3,noise_10__seed_01,1,3,10,1,3213219829,3782039437,42277,4207,...,18,147,4318,6452,661,145,770,24110d1aaf50c176e8a08c9d146cbcfbf50ad3c155f670...,41f8b6a12673a2b3d81c923805da72aaa6f9e62bea1c5a...,c8873ddf6ca121b4177e954a81f3e0144cf3886f3a380e...
3,4,noise_15__seed_01,1,4,15,1,3213219829,3782039437,42277,6280,...,26,147,6375,6452,976,145,1069,2bb19e200a5c8f5e1bff67c192c5ae44d71139552b6872...,8c53c033adb3cd95a868df03f04df340862d0d0294f20c...,e870d51dd3218de247b58651e4d95a8df5b65bc8e9e906...
4,5,noise_20__seed_01,1,5,20,1,3213219829,3782039437,42277,8408,...,34,147,8487,6452,1282,145,1359,59ccbf8ff17e40bff27eea76193a379be87b165c35b77b...,75830322fca097abe31b908b3aaa80fafc4f841fb3851f...,a39ada9f38cd81669430ac9cbe3631b009db5aa1713bc3...
5,6,noise_25__seed_01,1,6,25,1,3213219829,3782039437,42277,10498,...,41,147,10563,6452,1601,145,1666,966a1f0df824171a3d2ba29206aa91e5c9f76473efce5f...,a050432f50c7c402d98555a26f010aec7bf88f45ffe9b3...,1821c7fcbbf5ad14513c7587e799270291a6147d4e7195...
6,7,noise_30__seed_01,1,7,30,1,3213219829,3782039437,42277,12651,...,49,147,12700,6452,1944,145,1993,846591346487aaf08b05bcef61ce7847e951b38312c222...,e128f368d01e70885133bdbf6ea32bbeb1b58af03f24ff...,d5e5d1012b144eabd837a0cc1be00c2e28d680fbafe7d2...
7,8,noise_40__seed_01,1,8,40,1,3213219829,3782039437,42277,16848,...,60,147,16875,6452,2580,145,2607,cbbfefcdc891bb698841431da9b0ca93b34b4e11cf5d5d...,d657d812e257e0d566d968a1dc141ef4679087d1ec3085...,feaaabe80065aac2936bcb9e663392a7c57e95784e5f67...
8,9,noise_50__seed_01,1,9,50,1,3213219829,3782039437,42277,21032,...,80,147,21019,6452,3221,145,3210,b6b8dbf155921507e58dd82f6c713019a42b09698beddf...,9251b8e7b74920a87f1a89e73dc38f4ce465241451ca81...,820b5347013c16597fa59fc2891f9607b7d858ad9a3220...
9,262,noise_00__seed_30,30,1,0,30,3495884253,1372587638,42277,0,...,0,147,147,6452,0,145,145,ffda9bca5c0f8caafca764df7dd27b289ce7624c1d730d...,b12c91a19045da050d02b0356652b9896e423445eef28e...,0c307d89fbca2b5f2b95ce78ab4c31431020ef9ed4aceb...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 17 CELL 6 / STEP 3A RESULT ===

Project:
yamcs@Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen clean-history order:
Inferred execution-order rows: 58101
Global build-order rows: 504
Raw duplicate Build-Test rows: 0
Raw duplicate Test-order rows: 0

Fixed cohorts:
Raw training rows: 42277
Raw evaluation rows: 15824
Raw training failures: 147
Raw evaluation failures: 45
Model training rows: 6452
Model evaluation rows: 1081
Model training failures: 145
Model evaluation failures: 45
Model failing evaluation builds: 10

Noise plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
RNG-manifest 

In [19]:
# ==================================================================================================
# PROJECT 17 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 17 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 17 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 17 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_17_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_17_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "a52f9156732c2667d62702ff100c4178e3002ad127225d7195e5f6c0398e24db"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 10

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_17_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 17 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 17 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 17 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 17 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 17 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != 16
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            17,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–16."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 17 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 17 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 17 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    16,
    len(
        registry
    ),
    len(
        registry
    )
    == 16,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 17 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 17 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 17 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 17 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project17ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project17ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 17 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 17 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 17 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 17 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 17 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 17 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–16 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 17 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 17 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===

Project 17 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_17_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_17_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,a52f9156732c2667d62702ff100c4178e3002ad127225d...,a52f9156732c2667d62702ff100c4178e3002ad127225d...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,64cf358897e6363470ffb08f3239dd46044867f510a094...,64cf358897e6363470ffb08f3239dd46044867f510a094...,True
4,Raw training rows,42277,42277,True
5,Raw evaluation rows,15824,15824,True
6,Model training rows,6452,6452,True
7,Model evaluation rows,1081,1081,True
8,Model training failures,145,145,True
9,Model evaluation failures,45,45,True



Runtime versions:


,Component,Version,ExpectedVersion,Pass
0,Python,3.12.13,3.12.13,True
1,numpy,2.0.2,2.0.2,True
2,pandas,2.2.2,2.2.2,True
3,scikit-learn,1.6.1,1.6.1,True
4,xgboost,3.3.0,3.3.0,True
5,lightgbm,4.6.0,4.6.0,True
6,pyarrow,18.1.0,18.1.0,True



Predictor contract summary:


...


Metric self-tests:


,Check,Expected,Actual,Pass
0,Manual APFD,0.8,0.8,True
1,APFDc rewards quick failing test first,True,True,True
2,All-pass APFD is NaN,True,True,True
3,All-pass APFDc is NaN,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,256407,256407,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,95107,95107,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,1041949,1041949,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,300483,300483,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,49011,49011,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,12012,12012,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,93,93,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5243,5243,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,17868060,17868060,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,84393,84393,True




=== PROJECT 17 CELL 7 / STEP 4A RESULT ===

Project:
yamcs@Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen experiment contract:
Predictors: 151
REC features: 19
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD
Positive class: failure = 1
Median imputation: condition-training medians
Ranking tie-break: Test ascending
Rolling retraining: False

Fixed cohorts:
Model training rows: 6452
Model evaluation rows: 1081
Training failures: 145
Evaluation failures: 45
Failing evaluation builds: 10

Runtime validation:
Step 3A output-manifest

In [20]:
# ==================================================================================================
# PROJECT 17 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_17.ipynb.
#
# SMOKE CONDITIONS:
# - 0% noise, repetition seed 1
# - 50% noise, repetition seed 1
#
# THIS CELL:
# - verifies the frozen Step 4A runtime/model contract;
# - reconstructs condition-specific dependent REC features;
# - preserves all six verdict-independent REC features;
# - applies the frozen clean-anchor offsets;
# - trains all four ML techniques once per smoke condition;
# - evaluates ML plus Random, LatestFail, and QTF-Avg;
# - validates APFDc/APFD outputs and baseline invariance;
# - writes only Project 17 smoke-test outputs and checkpoint/status files;
# - does not modify the registry or full 270-condition raw-result root.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 17 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_17_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_17_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_17_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

STEP4B_STATUS = (
    "PASS_PROJECT_17_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)

CONDITION_STATUS = (
    "PASS_PROJECT_17_SMOKE_CONDITION"
)

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "500ac6cf30f5267feb31fb9cd86db3a136220921f9d5e3338a0da13919dbd657"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "a52f9156732c2667d62702ff100c4178e3002ad127225d7195e5f6c0398e24db"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "c6c3ce653908b7b407f6d87525c836bf4310857ff1e33f4d05e6f93ebf11e367"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d69f27517f91bfac416"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)

EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_BUILDS = 504
EXPECTED_RAW_ROWS = 58_101
EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_ROWS = 7_533
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45
EXPECTED_FAILING_EVAL_BUILDS = 10
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 126
EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS
EXPECTED_RNG_MANIFEST_ROWS = 1_268_310
EXPECTED_REGISTERED_PROJECTS = 16

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

SMOKE_CONDITION_IDS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

SMOKE_REPETITION_SEED = 1
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/yamcs@Yamcs"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_17_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_17_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_17_fixed_chronological_builds.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_selection_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

BUILD_ENTITY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_smoke_test"
)

SMOKE_CONDITION_INVENTORY_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_condition_inventory.csv"
)

SMOKE_COMBINED_CONDITION_AUDIT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
)

SMOKE_COMBINED_PROJECT_RUNS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"
)

SMOKE_COMBINED_BUILD_METRICS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
)

SMOKE_COMBINED_MODEL_FITS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_report.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4b_status.json"
)

SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_17_smoke_test_checkpoint.json"
)

# The smoke test must never write to this future full-run root.
FULL_RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                "inferred_test_order",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)


def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def audit_checkpoint_manifest(
    payload,
    manifest_key,
    label,
):
    manifest = payload.get(
        manifest_key,
        [],
    )

    if not isinstance(
        manifest,
        list,
    ) or not manifest:
        raise RuntimeError(
            f"{label} contains no {manifest_key}."
        )

    records = []

    for item in manifest:
        path = Path(
            item[
                "Path"
            ]
        )

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha256 = str(
            item[
                "SHA256"
            ]
        ).lower()

        exists = path.is_file()

        actual_bytes = (
            int(
                path.stat().st_size
            )
            if exists
            else -1
        )

        actual_sha256 = (
            sha256_file(
                path
            )
            if exists
            else "MISSING"
        )

        records.append({
            "Checkpoint":
                label,

            "Path":
                str(
                    path
                ),

            "ExpectedBytes":
                expected_bytes,

            "ActualBytes":
                actual_bytes,

            "ExpectedSHA256":
                expected_sha256,

            "ActualSHA256":
                actual_sha256,

            "Pass":
                bool(
                    exists
                    and actual_bytes
                    == expected_bytes
                    and actual_sha256
                    == expected_sha256
                ),
        })

    audit = pd.DataFrame(
        records
    )

    failures = int(
        (
            ~audit[
                "Pass"
            ]
        ).sum()
    )

    return (
        audit,
        failures,
    )


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 17 Step 4B inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 runtime-contract checkpoint SHA-256 differs."
    )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)

if selection_checkpoint.get(
    "Status"
) != "PASS_PROJECT_17_SELECTION_AND_SOURCE_FROZEN":
    raise RuntimeError(
        "Selection checkpoint does not contain the frozen Step 1B PASS status."
    )

if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise RuntimeError(
        "REC checkpoint does not contain the frozen Step 2B PASS status."
    )

if noise_plan_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise RuntimeError(
        "Noise-plan checkpoint does not contain the frozen Step 3A PASS status."
    )

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active-reservation state differs."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 4B."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–16."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 17 is already present in the completion registry."
    )

active_reservations = []

if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 17 freeze."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 17 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 17 source root differs before Step 4B."
    )

rec_manifest_audit, rec_manifest_failures = (
    audit_checkpoint_manifest(
        rec_checkpoint,
        "OutputManifest",
        "REC checkpoint",
    )
)

noise_manifest_audit, noise_manifest_failures = (
    audit_checkpoint_manifest(
        noise_plan_checkpoint,
        "OutputManifest",
        "Noise-plan checkpoint",
    )
)

if rec_manifest_failures != 0:
    print(
        "\nFailed REC output-manifest checks:"
    )

    display(
        rec_manifest_audit.loc[
            ~rec_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen REC outputs changed."
    )

if noise_manifest_failures != 0:
    print(
        "\nFailed noise-plan output-manifest checks:"
    )

    display(
        noise_manifest_audit.loc[
            ~noise_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen noise-plan outputs changed."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 17 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)
frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")
if len(inferred_execution_order) != EXPECTED_RAW_ROWS:
    raise RuntimeError("Frozen inferred execution-order row count differs.")
if len(frozen_global_build_order) != EXPECTED_BUILDS:
    raise RuntimeError("Frozen global build-order row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

smoke_plan = (
    condition_plan.loc[
        condition_plan["ConditionID"].isin(
            SMOKE_CONDITION_IDS
        )
    ]
    .copy()
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(smoke_plan) != EXPECTED_SMOKE_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain exactly the two smoke conditions."
    )

if smoke_plan["ConditionID"].tolist() != SMOKE_CONDITION_IDS:
    raise RuntimeError(
        "Smoke-condition order differs from the frozen contract."
    )

if smoke_plan["NoisePercent"].astype(int).tolist() != SMOKE_NOISE_LEVELS:
    raise RuntimeError(
        "Smoke noise levels differ from the frozen contract."
    )

if not smoke_plan["RepetitionSeed"].astype(int).eq(
    SMOKE_REPETITION_SEED
).all():
    raise RuntimeError(
        "Smoke repetition seed differs from the frozen contract."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_MANIFEST_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

rng_seed = pd.read_parquet(
    RNG_MANIFEST_PATH,
    filters=[
        (
            "RepetitionSeed",
            "==",
            SMOKE_REPETITION_SEED,
        ),
    ],
)

rng_seed[raw_order_column] = parse_int(
    rng_seed[raw_order_column],
    "rng_seed.RawTrainingRowOrder",
)

rng_seed = (
    rng_seed.sort_values(
        raw_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(rng_seed) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Seed-1 RNG stream has the wrong row count."
    )

if not np.array_equal(
    rng_seed[raw_order_column].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_RAW_TRAIN_ROWS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Seed-1 RNG stream row order differs."
    )

flip_uniform = rng_seed[
    "FlipUniform"
].to_numpy(dtype=np.float64)
sampled_failure_subtype = rng_seed[
    "SampledFailureSubtype"
].to_numpy(dtype=np.int16)

if not np.isfinite(flip_uniform).all():
    raise RuntimeError(
        "Seed-1 flip-uniform stream contains non-finite values."
    )

if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
    raise RuntimeError(
        "Seed-1 flip-uniform values are outside [0,1)."
    )

if sorted(np.unique(sampled_failure_subtype).tolist()) != [1, 2]:
    raise RuntimeError(
        "Seed-1 failure-subtype stream differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

required_inferred_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "InferredTestOrder",
}

missing_inferred_order_columns = (
    required_inferred_order_columns
    - set(inferred_execution_order.columns)
)

if missing_inferred_order_columns:
    raise RuntimeError(
        "Frozen inferred execution order is missing columns: "
        f"{sorted(missing_inferred_order_columns)}"
    )

for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[column] = parse_int(
        inferred_execution_order[column],
        f"inferred_execution_order.{column}",
    )

inferred_execution_order["Job"] = pd.to_numeric(
    inferred_execution_order["Job"],
    errors="coerce",
)

inferred_execution_order["Duration"] = pd.to_numeric(
    inferred_execution_order["Duration"],
    errors="coerce",
)

if not np.isfinite(
    inferred_execution_order["Job"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite jobs."
    )

if not np.isfinite(
    inferred_execution_order["Duration"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite durations."
    )

if inferred_execution_order["Duration"].lt(0).any():
    raise RuntimeError(
        "Frozen inferred execution order contains negative durations."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Build",
        "Test",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate Build-Test rows."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Test",
        "InferredTestOrder",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate per-test order rows."
    )

required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}

missing_global_order_columns = (
    required_global_order_columns
    - set(frozen_global_build_order.columns)
)

if missing_global_order_columns:
    raise RuntimeError(
        "Frozen global build order is missing columns: "
        f"{sorted(missing_global_order_columns)}"
    )

frozen_global_build_order["GlobalBuildOrder"] = parse_int(
    frozen_global_build_order["GlobalBuildOrder"],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order["BuildID"] = parse_int(
    frozen_global_build_order["BuildID"],
    "frozen_global_build_order.BuildID",
)

frozen_global_build_order = (
    frozen_global_build_order.sort_values(
        "GlobalBuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not np.array_equal(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_BUILDS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen global build-order sequence is not canonical."
    )

if frozen_global_build_order["BuildID"].nunique() != EXPECTED_BUILDS:
    raise RuntimeError(
        "Frozen global build order contains duplicate build IDs."
    )

ordered_builds = (
    frozen_global_build_order[
        "BuildID"
    ]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing InferredTestOrder."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

combined_raw_order = (
    pd.concat(
        [
            raw_training[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
            raw_evaluation[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

inferred_order_reference = (
    inferred_execution_order[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

raw_order_key_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            combined_raw_order[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
            != inferred_order_reference[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
        ).sum()
    )
)

raw_order_numeric_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            ~np.isclose(
                combined_raw_order[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                inferred_order_reference[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                rtol=0,
                atol=0,
                equal_nan=False,
            )
        ).sum()
    )
)

if (
    raw_order_key_mismatches != 0
    or raw_order_numeric_mismatches != 0
):
    raise RuntimeError(
        "The fixed raw cohorts no longer reproduce the frozen V6 "
        "inferred execution order."
    )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

random_scores = np.empty(
    EXPECTED_MODEL_EVAL_ROWS,
    dtype=np.float64,
)

for build_id in sorted(
    evaluation_meta_base["Build"].unique()
):
    build_indices = np.flatnonzero(
        evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        == int(build_id)
    )

    random_scores[build_indices] = np.random.default_rng(
        deterministic_random_build_seed(
            SMOKE_REPETITION_SEED,
            int(build_id),
        )
    ).random(len(build_indices))

if not np.isfinite(random_scores).all():
    raise RuntimeError(
        "Random baseline produced non-finite scores."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN THE TWO END-TO-END SMOKE CONDITIONS
# --------------------------------------------------------------------------------------------------

# Remove only incomplete/previous Project 17 smoke-test outputs.
# Frozen Steps 0–4A and the future full-result root are untouched.
if SMOKE_ROOT.exists():
    shutil.rmtree(
        SMOKE_ROOT
    )

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

raw_training_hash_before = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_before = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_before = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_before = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)

condition_inventory_records = []
all_condition_audits = []
all_project_runs = []
all_build_metrics = []
all_model_fits = []
all_rankings_for_invariance = []

smoke_execution_started = time.perf_counter()

for smoke_index, plan_row in enumerate(
    smoke_plan.itertuples(index=False),
    start=1,
):
    condition_started = time.perf_counter()
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)

    print("\n" + "-" * 136)
    print(
        f"[{smoke_index}/{EXPECTED_SMOKE_CONDITIONS}] "
        f"Running {condition_key}"
    )
    print("-" * 136)

    condition_dir = (
        SMOKE_ROOT
        / condition_key
    )
    condition_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ranking_path = (
        condition_dir
        / "rankings.csv.gz"
    )
    build_metrics_path = (
        condition_dir
        / "build_metrics.csv"
    )
    project_runs_path = (
        condition_dir
        / "project_runs.csv"
    )
    model_fits_path = (
        condition_dir
        / "model_fits.csv"
    )
    training_medians_path = (
        condition_dir
        / "training_medians.csv"
    )
    condition_audit_path = (
        condition_dir
        / "condition_audit.csv"
    )
    condition_summary_path = (
        condition_dir
        / "condition_summary.json"
    )
    completion_marker_path = (
        condition_dir
        / "COMPLETE.json"
    )

    flip_mask = (
        flip_uniform
        < (noise_percent / 100.0)
    )

    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = (
        flip_mask
        & (clean_raw_training_verdict == 0)
    )
    failure_to_pass_mask = (
        flip_mask
        & (clean_raw_training_verdict != 0)
    )

    noisy_raw_training_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtype[
        pass_to_failure_mask
    ]
    noisy_raw_training_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(
        plan_row.FlipMaskSHA256
    )
    expected_noisy_raw_sha256 = str(
        plan_row.NoisyRawVerdictSHA256
    )
    expected_noisy_model_sha256 = str(
        plan_row.NoisyModelVerdictSHA256
    )

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )

    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )

    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (
            noisy_model_training_verdict
            != clean_model_training_verdict
        ).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(
        noisy_model_training_binary.sum()
    )

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    print(
        "  Reconstructing REC features from the condition-specific history."
    )

    train_history = pd.DataFrame({
        "build": raw_training["Build"].to_numpy(dtype=np.int64),
        "test": raw_training["Test"].to_numpy(dtype=np.int64),
        "job": raw_training["Job"].to_numpy(),
        "verdict": noisy_raw_training_verdict.astype(np.int16),
        "duration": raw_training["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_training[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    evaluation_history = pd.DataFrame({
        "build": raw_evaluation["Build"].to_numpy(dtype=np.int64),
        "test": raw_evaluation["Test"].to_numpy(dtype=np.int64),
        "job": raw_evaluation["Job"].to_numpy(),
        "verdict": clean_raw_evaluation_verdict.astype(np.int16),
        "duration": raw_evaluation["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_evaluation[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    execution_history = pd.concat(
        [
            train_history,
            evaluation_history,
        ],
        ignore_index=True,
    )

    if execution_history.duplicated(
        subset=[
            "build",
            "test",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate Build-Test rows."
        )

    if execution_history.duplicated(
        subset=[
            "test",
            "inferred_test_order",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate per-test order rows."
        )

    execution_history = (
        execution_history.sort_values(
            [
                "test",
                "inferred_test_order",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    rec_started = time.perf_counter()

    reconstructed = reconstruct_rec_features(
        execution_history=execution_history,
        requested_rows=model_all[["Build", "Test"]],
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
        recent_window=RECENT_WINDOW,
    )

    rec_seconds = time.perf_counter() - rec_started

    if len(reconstructed) != EXPECTED_MODEL_ROWS:
        raise RuntimeError(
            f"{condition_key}: reconstructed REC row count differs."
        )

    if reconstructed.duplicated(
        subset=["Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: reconstructed REC contains duplicate keys."
        )

    reconstructed["Build"] = parse_int(
        reconstructed["Build"],
        f"{condition_key}.reconstructed.Build",
    )
    reconstructed["Test"] = parse_int(
        reconstructed["Test"],
        f"{condition_key}.reconstructed.Test",
    )

    reconstructed_indexed = reconstructed.set_index(
        [
            "Build",
            "Test",
        ]
    )

    missing_reconstructed_keys = model_key_index.difference(
        reconstructed_indexed.index
    )

    if len(missing_reconstructed_keys) != 0:
        raise RuntimeError(
            f"{condition_key}: reconstruction does not cover all model rows."
        )

    reconstructed_values_all = reconstructed_indexed.loc[
        model_key_index,
        REC_FEATURES,
    ].to_numpy(dtype=np.float64)

    anchored_values_all = (
        reconstructed_values_all
        + anchor_values_all
    )

    independent_reconstruction_mismatches = int(
        (~np.isclose(
            anchored_values_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            clean_original_rec_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            rtol=0,
            atol=1e-12,
        )).sum()
    )

    if independent_reconstruction_mismatches != 0:
        raise RuntimeError(
            f"{condition_key}: verdict-independent REC reconstruction changed."
        )

    condition_numeric_all = all_base_numeric.copy()

    dependent_rec_values_all = anchored_values_all[:, [
        REC_FEATURES.index(feature)
        for feature in VERDICT_DEPENDENT_REC
    ]]

    condition_numeric_all[:, dependent_predictor_indices] = (
        dependent_rec_values_all
    )

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    condition_original_dependent_all = all_base_numeric[
        :, dependent_predictor_indices
    ]

    dependent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, dependent_predictor_indices],
            condition_original_dependent_all,
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        )).sum()
    )

    independent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, independent_predictor_indices],
            all_base_numeric[:, independent_predictor_indices],
            rtol=0,
            atol=0,
            equal_nan=True,
        )).sum()
    )

    if independent_rec_changes != 0:
        raise RuntimeError(
            f"{condition_key}: preserved independent REC predictors changed."
        )

    if noise_percent == 0:
        zero_rec_mismatches = int(
            (~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )).sum()
        )

        if zero_rec_mismatches != 0:
            raise RuntimeError(
                "0% smoke condition did not reproduce the clean REC cohort."
            )

        if number_flipped != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly flipped raw labels."
            )

        if model_label_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed model labels."
            )

        if dependent_rec_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed dependent REC values."
            )

    if noise_percent > 0:
        if number_flipped <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no dependent REC values."
            )

    medians = np.nanmedian(
        condition_training_numeric,
        axis=0,
    )

    nonfinite_median_indices = np.flatnonzero(
        ~np.isfinite(medians)
    )

    if len(nonfinite_median_indices) != 0:
        bad_features = [
            predictor_columns[index]
            for index in nonfinite_median_indices
        ]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(
        condition_training_numeric
    )
    evaluation_missing_mask = ~np.isfinite(
        condition_evaluation_numeric
    )

    if training_missing_mask.any():
        row_indices, column_indices = np.where(
            training_missing_mask
        )
        condition_training_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(
            evaluation_missing_mask
        )
        condition_evaluation_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite after imputation."
        )

    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite after imputation."
        )

    # Keep float64 throughout the smoke test. This matches the frozen Step 4A
    # numeric/imputation contract and avoids changing ranking/model behaviour
    # through an unapproved dtype conversion.

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()
        fit_status = "PASS_MODEL_FIT"
        fit_error = ""

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )

            fit_seconds = time.perf_counter() - fit_started
            scores = positive_probability(
                model,
                condition_evaluation_numeric,
            )

            technique_scores[technique] = scores

        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)

            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })

            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps(
                [
                    int(value)
                    for value in np.asarray(model.classes_).tolist()
                ]
            ),
            "Status": fit_status,
            "Error": fit_error,
        })

        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )

    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :,
        predictor_index["REC_LastFailureAge"],
    ].astype(float)

    condition_eval_qtf = condition_evaluation_numeric[
        :,
        predictor_index["REC_TotalAvgExeTime"],
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = (
        -condition_eval_last_failure_age
    )
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []

    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )

    ranking_techniques = sorted(
        rankings["Technique"].unique().tolist()
    )

    if ranking_techniques != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )

    ranking_rows_per_technique = rankings.groupby(
        "Technique"
    ).size()

    if not ranking_rows_per_technique.eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )

    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_ranking_rows != 0:
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(
        rankings
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )

    if sorted(project_runs["Technique"].tolist()) != sorted(
        ALL_TECHNIQUES
    ):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    metric_columns = [
        "APFDc",
        "APFD",
    ]

    build_metric_values = build_metrics[
        metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )

    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    project_metric_values = project_runs[
        project_metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )

    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(len(reconstructed)),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    atomic_csv(
        ranking_path,
        rankings,
        compression="gzip",
    )
    atomic_csv(
        build_metrics_path,
        build_metrics,
    )
    atomic_csv(
        project_runs_path,
        project_runs,
    )
    atomic_csv(
        model_fits_path,
        model_fits,
    )
    atomic_csv(
        training_medians_path,
        training_medians,
    )
    atomic_csv(
        condition_audit_path,
        condition_audit,
    )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]

    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "OutputManifest": condition_output_manifest,
    }

    atomic_json(
        condition_summary_path,
        condition_summary,
    )

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json(
        completion_marker_path,
        completion_marker,
    )

    condition_manifest = directory_manifest(
        condition_dir
    )
    condition_root_sha256 = directory_root_hash(
        condition_manifest
    )

    condition_inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": smoke_index,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": condition_root_sha256,
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "ModelFitRows": int(len(model_fits)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
    })

    all_condition_audits.append(condition_audit)
    all_project_runs.append(project_runs)
    all_build_metrics.append(build_metrics)
    all_model_fits.append(model_fits)
    all_rankings_for_invariance.append(
        rankings.loc[
            rankings["Technique"].isin(
                [
                    "Random",
                    "QTF-Avg",
                ]
            )
        ].copy()
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del train_history
    del evaluation_history
    del execution_history
    del reconstructed
    del reconstructed_indexed
    del reconstructed_values_all
    del anchored_values_all
    del condition_numeric_all
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. COMBINE SMOKE OUTPUTS AND VERIFY BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

condition_inventory = pd.DataFrame(
    condition_inventory_records
)
combined_condition_audit = pd.concat(
    all_condition_audits,
    ignore_index=True,
)
combined_project_runs = pd.concat(
    all_project_runs,
    ignore_index=True,
)
combined_build_metrics = pd.concat(
    all_build_metrics,
    ignore_index=True,
)
combined_model_fits = pd.concat(
    all_model_fits,
    ignore_index=True,
)
combined_invariance_rankings = pd.concat(
    all_rankings_for_invariance,
    ignore_index=True,
)

baseline_invariance_records = []

for technique in [
    "Random",
    "QTF-Avg",
]:
    zero_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(0)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    noisy_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(50)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    same_keys = bool(
        zero_rows[["Build", "Test"]].equals(
            noisy_rows[["Build", "Test"]]
        )
    )

    score_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (~np.isclose(
                zero_rows["Score"].to_numpy(dtype=float),
                noisy_rows["Score"].to_numpy(dtype=float),
                rtol=0,
                atol=0,
            )).sum()
        )
    )

    rank_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (
                zero_rows["Rank"].to_numpy(dtype=np.int64)
                != noisy_rows["Rank"].to_numpy(dtype=np.int64)
            ).sum()
        )
    )

    baseline_invariance_records.append({
        "Technique": technique,
        "Rows": int(len(zero_rows)),
        "SameBuildTestKeys": same_keys,
        "ScoreMismatches": score_mismatches,
        "RankMismatches": rank_mismatches,
        "Pass": (
            len(zero_rows) == EXPECTED_MODEL_EVAL_ROWS
            and len(noisy_rows) == EXPECTED_MODEL_EVAL_ROWS
            and same_keys
            and score_mismatches == 0
            and rank_mismatches == 0
        ),
    })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)
baseline_invariance_failures = int(
    (~baseline_invariance["Pass"]).sum()
)

if baseline_invariance_failures != 0:
    print("\nBaseline invariance failures:")
    display(
        baseline_invariance.loc[
            ~baseline_invariance["Pass"]
        ]
    )
    raise RuntimeError(
        "Random or QTF-Avg changed across the two smoke noise levels."
    )

atomic_csv(
    SMOKE_CONDITION_INVENTORY_PATH,
    condition_inventory,
)
atomic_csv(
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)
atomic_csv(
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)
atomic_csv(
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)
atomic_csv(
    SMOKE_COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)
atomic_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_hash_after = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_after = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_after = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_after = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)
registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

current_source_rows_after = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

source_root_sha256_after = source_root_hash(
    pd.DataFrame(current_source_rows_after)
)

full_raw_result_root_exists_after = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_after = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_after = directory_root_hash(
    full_raw_result_manifest_after
)

full_raw_result_unchanged = bool(
    full_raw_result_root_existed_before
    == full_raw_result_root_exists_after
    and full_raw_result_root_hash_before
    == full_raw_result_root_hash_after
    and len(full_raw_result_manifest_before)
    == len(full_raw_result_manifest_after)
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
].iloc[0]

positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(50)
].iloc[0]

validation_records = []

add_check(
    validation_records,
    "Step 4A status",
    EXPECTED_STEP4A_STATUS,
    step4a_status.get("Status"),
    step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
)
add_check(
    validation_records,
    "Runtime checkpoint SHA-256",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    runtime_checkpoint_sha256,
    runtime_checkpoint_sha256 == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_plan_checkpoint_sha256,
    noise_plan_checkpoint_sha256 == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256 == EXPECTED_REC_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256 == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC output-manifest failures",
    0,
    rec_manifest_failures,
    rec_manifest_failures == 0,
)
add_check(
    validation_records,
    "Noise-plan output-manifest failures",
    0,
    noise_manifest_failures,
    noise_manifest_failures == 0,
)
add_check(
    validation_records,
    "Step 4A output-manifest failures",
    0,
    runtime_manifest_failures,
    runtime_manifest_failures == 0,
)
add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_sha256_after,
    source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256,
)
add_check(
    validation_records,
    "Smoke conditions",
    EXPECTED_SMOKE_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Smoke condition keys",
    SMOKE_CONDITION_IDS,
    condition_inventory["ConditionKey"].tolist(),
    condition_inventory["ConditionKey"].tolist() == SMOKE_CONDITION_IDS,
)
add_check(
    validation_records,
    "Condition statuses",
    CONDITION_STATUS,
    sorted(condition_inventory["Status"].unique().tolist()),
    condition_inventory["Status"].eq(CONDITION_STATUS).all(),
)
add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_SMOKE_CONDITIONS,
    len(combined_condition_audit),
    len(combined_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Total ML fits",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
)
add_check(
    validation_records,
    "Model-fit failures",
    0,
    int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()),
    combined_model_fits["Status"].eq("PASS_MODEL_FIT").all(),
)
add_check(
    validation_records,
    "Total project-run rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Total build-metric rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Technique set",
    sorted(ALL_TECHNIQUES),
    sorted(combined_project_runs["Technique"].unique().tolist()),
    sorted(combined_project_runs["Technique"].unique().tolist())
    == sorted(ALL_TECHNIQUES),
)
add_check(
    validation_records,
    "Project-run rows per condition violations",
    0,
    int((
        combined_project_runs.groupby("ConditionKey").size()
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_project_runs.groupby("ConditionKey").size()
        == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Build-metric rows per condition violations",
    0,
    int((
        combined_build_metrics.groupby("ConditionKey").size()
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_build_metrics.groupby("ConditionKey").size()
        == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Zero-noise raw flips",
    0,
    int(zero_audit["NumberFlipped"]),
    int(zero_audit["NumberFlipped"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise model-label changes",
    0,
    int(zero_audit["ModelLabelChanges"]),
    int(zero_audit["ModelLabelChanges"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise dependent REC changes",
    0,
    int(zero_audit["DependentRECChanges"]),
    int(zero_audit["DependentRECChanges"]) == 0,
)
add_check(
    validation_records,
    "Positive-noise raw flips",
    "> 0",
    int(positive_audit["NumberFlipped"]),
    int(positive_audit["NumberFlipped"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise model-label changes",
    "> 0",
    int(positive_audit["ModelLabelChanges"]),
    int(positive_audit["ModelLabelChanges"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise dependent REC changes",
    "> 0",
    int(positive_audit["DependentRECChanges"]),
    int(positive_audit["DependentRECChanges"]) > 0,
)
add_check(
    validation_records,
    "Independent REC changes",
    0,
    int(combined_condition_audit["IndependentRECChanges"].sum()),
    int(combined_condition_audit["IndependentRECChanges"].sum()) == 0,
)
add_check(
    validation_records,
    "Frozen raw-order key mismatches",
    0,
    raw_order_key_mismatches,
    raw_order_key_mismatches == 0,
)
add_check(
    validation_records,
    "Frozen raw-order numeric mismatches",
    0,
    raw_order_numeric_mismatches,
    raw_order_numeric_mismatches == 0,
)
add_check(
    validation_records,
    "Independent REC reconstruction mismatches",
    0,
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()),
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()) == 0,
)
add_check(
    validation_records,
    "Noise-plan hash mismatches",
    0,
    int((
        combined_condition_audit["ExpectedFlipMaskSHA256"]
        != combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
        != combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
        != combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()),
    bool(
        (
            combined_condition_audit["ExpectedFlipMaskSHA256"]
            == combined_condition_audit["ActualFlipMaskSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
            == combined_condition_audit["ActualNoisyRawVerdictSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
            == combined_condition_audit["ActualNoisyModelVerdictSHA256"]
        ).all()
    ),
)
add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)
add_check(
    validation_records,
    "Project metrics non-finite",
    0,
    int((~np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    )).sum()),
    bool(np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    ).all()),
)
add_check(
    validation_records,
    "Project metrics outside [0,1]",
    0,
    int((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            < 0
        )
        | (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            > 1
        )
    ).sum()),
    bool((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            >= 0
        )
        & (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            <= 1
        )
    ).all()),
)
add_check(
    validation_records,
    "Raw training cohort unchanged",
    raw_training_hash_before,
    raw_training_hash_after,
    raw_training_hash_after == raw_training_hash_before,
)
add_check(
    validation_records,
    "Raw evaluation cohort unchanged",
    raw_evaluation_hash_before,
    raw_evaluation_hash_after,
    raw_evaluation_hash_after == raw_evaluation_hash_before,
)
add_check(
    validation_records,
    "Model training cohort unchanged",
    model_training_hash_before,
    model_training_hash_after,
    model_training_hash_after == model_training_hash_before,
)
add_check(
    validation_records,
    "Model evaluation cohort unchanged",
    model_evaluation_hash_before,
    model_evaluation_hash_after,
    model_evaluation_hash_after == model_evaluation_hash_before,
)
add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after,
    registry_sha256_after == registry_sha256_before,
)
add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Registry Project 17 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)
add_check(
    validation_records,
    "Full raw-result root unchanged",
    True,
    full_raw_result_unchanged,
    full_raw_result_unchanged,
)

validation = pd.DataFrame(
    validation_records
)
failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 17 Step 4B validation:")
display(validation)

print("\nBaseline invariance audit:")
display(baseline_invariance)

print("\nSmoke project-run results:")
display(
    combined_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

if not failed_validation.empty:
    print("\nFailed Project 17 Step 4B checks:")
    display(failed_validation)
    print("\nNo Step 4B PASS status or checkpoint was written.")
    raise RuntimeError(
        "PROJECT 17 STEP 4B VALIDATION FAILED. DO NOT START THE FULL EXPERIMENT."
    )

atomic_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, STATUS, AND FINAL READBACK
# --------------------------------------------------------------------------------------------------

smoke_execution_seconds = (
    time.perf_counter()
    - smoke_execution_started
)
completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

smoke_output_paths = [
    SMOKE_CONDITION_INVENTORY_PATH,
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    SMOKE_COMBINED_MODEL_FITS_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]

for condition_key in SMOKE_CONDITION_IDS:
    condition_dir = SMOKE_ROOT / condition_key
    smoke_output_paths.extend([
        path
        for path in condition_dir.rglob("*")
        if path.is_file()
    ])

smoke_output_paths = sorted(
    set(smoke_output_paths),
    key=lambda path: str(path),
)

smoke_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in smoke_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "SmokeConditionKeys": SMOKE_CONDITION_IDS,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": int(len(combined_build_metrics)),
    "ProjectRunRows": int(len(combined_project_runs)),
    "TrainingMedianRows": int(
        condition_inventory["TrainingMedianRows"].sum()
    ),
    "ZeroNoiseRawFlips": int(zero_audit["NumberFlipped"]),
    "ZeroNoiseModelLabelChanges": int(
        zero_audit["ModelLabelChanges"]
    ),
    "ZeroNoiseDependentRECChanges": int(
        zero_audit["DependentRECChanges"]
    ),
    "PositiveNoiseRawFlips": int(
        positive_audit["NumberFlipped"]
    ),
    "PositiveNoiseModelLabelChanges": int(
        positive_audit["ModelLabelChanges"]
    ),
    "PositiveNoiseDependentRECChanges": int(
        positive_audit["DependentRECChanges"]
    ),
    "IndependentRECChanges": int(
        combined_condition_audit["IndependentRECChanges"].sum()
    ),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": int(len(validation)),
    "FailedValidationChecks": int(len(failed_validation)),
    "SmokeExecutionSeconds": float(smoke_execution_seconds),
    "OutputManifest": smoke_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To16Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "FullRawResultRootModified": False,
    "FullExperimentStarted": False,
}

atomic_json(
    SMOKE_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "SmokeTestPassed": True,
    "RuntimeContractFrozen": True,
    "NoisePlanFrozen": True,
    "EvaluationCohortImmutable": True,
    "ReadyForFull270ConditionExperiment": True,
}

atomic_json(
    SMOKE_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "FailedValidationChecks": int(len(failed_validation)),
    "Checkpoint": str(SMOKE_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(SMOKE_CHECKPOINT_PATH),
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "FullExperimentStarted": False,
}

atomic_json(
    STEP4B_STATUS_PATH,
    status_payload,
)

checkpoint_readback = load_json(
    SMOKE_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP4B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B checkpoint readback failed."
    )

if status_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B status readback failed."
    )

if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Step 4B finalisation."
    )

final_source_rows = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    final_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

if source_root_hash(pd.DataFrame(final_source_rows)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 17 source changed during Step 4B finalisation."
    )

print("\n" + "=" * 136)
print("=== PROJECT 17 CELL 8 / STEP 4B RESULT ===")
print("=" * 136)
print()
print("Project:")
print(PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)
print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)
print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Two-condition end-to-end smoke test:")
print("Conditions:", SMOKE_CONDITION_IDS)
print("Conditions passed:", len(condition_inventory), "/", EXPECTED_SMOKE_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Noise and REC audit:")
print("0% raw flips:", int(zero_audit["NumberFlipped"]))
print("0% model-label changes:", int(zero_audit["ModelLabelChanges"]))
print("0% dependent REC changes:", int(zero_audit["DependentRECChanges"]))
print("50% raw flips:", int(positive_audit["NumberFlipped"]))
print("50% model-label changes:", int(positive_audit["ModelLabelChanges"]))
print("50% dependent REC changes:", int(positive_audit["DependentRECChanges"]))
print("Independent REC changes:", int(combined_condition_audit["IndependentRECChanges"].sum()))
print()
print("Baselines and metrics:")
print("Random/QTF-Avg invariance failures:", baseline_invariance_failures)
print("Techniques:", ALL_TECHNIQUES)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 17 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–16 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print("Full experiment raw-result root modified:", False)
print("Full 270-condition experiment started:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Smoke-test checkpoint:")
print(SMOKE_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(SMOKE_CHECKPOINT_PATH))
print()
print("Runtime seconds:", round(smoke_execution_seconds, 2))
print()
print("STATUS:", STEP4B_STATUS)
print("=" * 136)


=== PROJECT 17 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 17 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.

----------------------------------------------------------------------------------------------------------------------------------------
[1/2] Running noise_00__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 160 tests | reconstructed rows: 6157
    REC reconstruction progress: 160 / 160 tests | reconstructed rows: 7533
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 145 | condition seconds: 73.72

----------------------------------------------------------------------------------------------------------------------------------------
[2/2] Running noise_50__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 160 tests | reconstructed rows: 6157
    REC reconstruction progress: 160 / 160 tests | reconstructed rows: 7533
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 21032 | model-label changes: 3221 | dependent REC changes: 84644
  Training failures: 3210 | condition seconds: 41.61

Project 17 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,1081,True,0,0,True
1,QTF-Avg,1081,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,LatestFail,126,10,1081,45,0.652900,0.725899,0.346198,0.292187
1,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,LightGBM,126,10,1081,45,0.635972,0.758041,0.843627,0.955444
2,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,NaiveBayes,126,10,1081,45,0.486442,0.518601,0.753321,0.870085
3,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,QTF-Avg,126,10,1081,45,0.476298,0.387841,0.184283,0.048913
4,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,Random,126,10,1081,45,0.491602,0.472629,0.494133,0.467391
5,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,RandomForest,126,10,1081,45,0.633711,0.754689,0.789549,0.954762
6,17,yamcs@Yamcs,yamcs__Yamcs,noise_00__seed_01,0,1,XGBoost,126,10,1081,45,0.645013,0.715524,0.791886,0.901426
7,17,yamcs@Yamcs,yamcs__Yamcs,noise_50__seed_01,50,1,LatestFail,126,10,1081,45,0.647122,0.737407,0.732612,0.856687
8,17,yamcs@Yamcs,yamcs__Yamcs,noise_50__seed_01,50,1,LightGBM,126,10,1081,45,0.393039,0.379023,0.431808,0.376812
9,17,yamcs@Yamcs,yamcs__Yamcs,noise_50__seed_01,50,1,NaiveBayes,126,10,1081,45,0.402896,0.398869,0.183787,0.083100



=== PROJECT 17 CELL 8 / STEP 4B RESULT ===

Project:
yamcs@Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 15134
Build-metric rows: 140
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 21032
50% model-label changes: 3221
50% dependent REC changes: 84644
Independent REC changes: 0

Baselines and metrics:
Random/QTF-Avg invariance failures: 0
Techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'Naive

In [21]:
# ==================================================================================================
# PROJECT 17 — CELL 9 / STEP 5A V2 CHECKPOINT-SCHEMA-COMPATIBLE ACCELERATED
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT WITH VECTORIZED REC ENGINE
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 17 Step 4B smoke-test checkpoint and every upstream contract;
# - validate the accelerated REC engine against the exact frozen Step 4B smoke outputs;
# - execute all 270 noise/seed conditions with scientifically identical inputs and outputs;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze the complete Project 17 raw-result root for independent Step 5B revalidation.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–16;
# - no prior-project condition-output access;
# - clean evaluation data remain immutable;
# - incomplete condition outputs are preserved in quarantine before rerun.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)

print("=" * 136)
print("=== PROJECT 17 CELL 9 / STEP 5A V2: CHECKPOINT-SCHEMA-COMPATIBLE FULL 270-CONDITION EXPERIMENT ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_17_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_17_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_17_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "500ac6cf30f5267feb31fb9cd86db3a136220921f9d5e3338a0da13919dbd657"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "a844057c5869a914036c7d3df655a396fa3817ef706bb2f14e39219eb2f50fb5"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "a52f9156732c2667d62702ff100c4178e3002ad127225d7195e5f6c0398e24db"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "c6c3ce653908b7b407f6d87525c836bf4310857ff1e33f4d05e6f93ebf11e367"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7579c4d6376cdb64cf745fd19c04bc646e7ecc98c1515d69f27517f91bfac416"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
)
EXPECTED_REGISTRY_SHA256 = (
    "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
)

EXPECTED_REGISTERED_PROJECTS = 16

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 42_277
EXPECTED_RAW_EVAL_ROWS = 15_824
EXPECTED_MODEL_TRAIN_ROWS = 6_452
EXPECTED_MODEL_EVAL_ROWS = 1_081
EXPECTED_MODEL_ROWS = 7_533
EXPECTED_MODEL_TRAIN_FAILURES = 145
EXPECTED_MODEL_EVAL_FAILURES = 45
EXPECTED_FAILING_EVAL_BUILDS = 10
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 126
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 1_268_310
ACCELERATED_ENGINE_VERSION = "PROJECT_17_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/yamcs@Yamcs")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_17_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_17_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_17_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_17_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_17_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_17_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_17_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_17_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
INCOMPLETE_BACKUP_ROOT = FULL_EXPERIMENT_ROOT / "incomplete_condition_backups"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_17_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)

def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)

def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)

def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()

def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")

def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })

def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )

def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )

def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )

def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )

def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)

def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the exact Step 2B-frozen InferredTestOrder is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result

def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))

def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    audit_columns = [
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "RawTrainingRows",
        "NumberFlipped",
        "ExpectedNumberFlipped",
        "PassToFailure",
        "FailureToPass",
        "ModelTrainingRows",
        "ModelLabelChanges",
        "ExpectedModelLabelChanges",
        "TrainingFailures",
        "ExpectedTrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ReconstructedRows",
        "Predictors",
        "MLFits",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ]
    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)

def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]

def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs

DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]

def directory_manifest(root):
    root = Path(root)
    rows = []

    if root.exists():
        for path in sorted(
            [
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ],
            key=lambda candidate: candidate.relative_to(root).as_posix(),
        ):
            rows.append({
                "RelativePath": path.relative_to(root).as_posix(),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            })

    return pd.DataFrame(
        rows,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )

def directory_root_hash(manifest):
    if manifest is None:
        raise TypeError(
            "Directory manifest cannot be None."
        )

    missing_columns = [
        column
        for column in DIRECTORY_MANIFEST_COLUMNS
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Directory manifest is missing required columns: "
            + ", ".join(missing_columns)
        )

    digest = hashlib.sha256()

    if manifest.empty:
        return digest.hexdigest()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()

# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 17 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 17 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 17 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 17 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

if smoke_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Smoke-test checkpoint active reservations differ."
    )

# Step 4B validates the runtime-priority rule against the frozen Step 4A
# runtime checkpoint, but its checkpoint schema does not duplicate that field.
# Therefore, validate the frozen linkage instead of requiring an absent key.
if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke-test checkpoint does not link to the frozen runtime contract."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )

# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–16."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–16 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 17 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 17 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 17 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)

# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 17 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )["BuildID"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing the V6-frozen InferredTestOrder column."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "InferredTestOrder": np.concatenate((
        raw_training["InferredTestOrder"].to_numpy(dtype=np.int64),
        raw_evaluation["InferredTestOrder"].to_numpy(dtype=np.int64),
    )),
})

if combined_history.duplicated(
    subset=["Test", "InferredTestOrder"],
    keep=False,
).any():
    raise RuntimeError(
        "Accelerated REC history contains duplicate V6 per-test order keys."
    )

# Project 17 must use the exact per-test execution order frozen by Step 2B.
# The fixed chronological Build-ID tie-break is not the REC history order for this project.
combined_history = (
    combined_history.sort_values(
        ["Test", "InferredTestOrder"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "BuildID",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)

# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND FULL CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}

def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }

def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination

print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending conditions:", EXPECTED_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The frozen 0% and 50% seed-1 smoke conditions are executed/validated first.
equivalence_records_by_key = {}
for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
    equivalence_row = condition_plan.loc[
        condition_plan["ConditionID"].eq(equivalence_key)
    ]
    if len(equivalence_row) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
        )
    equivalence_plan_row = next(equivalence_row.itertuples(index=False))
    equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
    if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
        equivalence_record = compare_full_condition_to_smoke(
            equivalence_key,
            equivalence_dir,
        )
        if not equivalence_record["Pass"]:
            raise RuntimeError(
                f"Existing accelerated condition {equivalence_key} differs from Step 4B."
            )
        equivalence_records_by_key[equivalence_key] = equivalence_record

smoke_first_plan = condition_plan.loc[
    condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].copy()
smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
    key: index
    for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
})
smoke_first_plan = smoke_first_plan.sort_values(
    "__SmokeOrder",
    kind="mergesort",
).drop(columns="__SmokeOrder")
remaining_plan = condition_plan.loc[
    ~condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].sort_values("ConditionOrder", kind="mergesort")
execution_plan = pd.concat(
    [smoke_first_plan, remaining_plan],
    ignore_index=True,
)

if len(execution_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError("Accelerated execution plan does not contain 270 conditions.")

if equivalence_records_by_key:
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for plan_row in execution_plan.itertuples(index=False):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[{condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = pd.read_parquet(
            RNG_MANIFEST_PATH,
            filters=[("RepetitionSeed", "==", repetition_seed)],
        )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(eval_build_array == int(build_id))
            random_scores[build_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(build_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[{condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical full-history reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        atomic_csv(
            ACCELERATED_EQUIVALENCE_PATH,
            pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                "ConditionKey",
                kind="mergesort",
            ),
        )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": "PROJECT_11_FULL_EXPERIMENT_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "PriorProjectConditionOutputsAccessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()

# --------------------------------------------------------------------------------------------------
# 9. FINAL 270-CONDITION REVALIDATION
# --------------------------------------------------------------------------------------------------

print("\nValidating all 270 completed conditions.")

inventory_records = []
condition_audits = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for plan_row in condition_plan.itertuples(index=False):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Final validation failed for {plan_row.ConditionID}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    project_run_frames.append(pd.read_csv(condition_dir / "project_runs.csv"))
    build_metric_frames.append(pd.read_csv(condition_dir / "build_metrics.csv"))
    model_fit_frames.append(pd.read_csv(condition_dir / "model_fits.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

condition_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)

baseline_invariance_records = []
for repetition_seed in REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~baseline_invariance["Pass"]).sum())
qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

raw_training_hash_after = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_after = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_after = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_after = sha256_file(MODEL_EVALUATION_COHORT_PATH)
registry_sha256_after = sha256_file(REGISTRY_PATH)

current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]

noise_plan_hash_mismatches = int(
    combined_condition_audit[
        "ExpectedFlipMaskSHA256"
    ].ne(combined_condition_audit["ActualFlipMaskSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyRawVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyRawVerdictSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyModelVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyModelVerdictSHA256"]).sum()
)

project_metric_columns = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
project_metric_values = combined_project_runs[
    project_metric_columns
].to_numpy(dtype=float)

if not ACCELERATED_EQUIVALENCE_PATH.is_file():
    raise FileNotFoundError(
        "Accelerated-engine equivalence audit is missing."
    )
accelerated_equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)
accelerated_equivalence_passes = int(
    accelerated_equivalence["Pass"].astype(bool).sum()
)

validation_records = []
add_check(validation_records, "Step 4B passed", EXPECTED_STEP4B_STATUS, smoke_checkpoint.get("Status"), smoke_checkpoint.get("Status") == EXPECTED_STEP4B_STATUS)
add_check(validation_records, "Smoke checkpoint SHA-256", EXPECTED_SMOKE_CHECKPOINT_SHA256, smoke_checkpoint_sha256, smoke_checkpoint_sha256 == EXPECTED_SMOKE_CHECKPOINT_SHA256)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Accelerated smoke-equivalence rows", 2, len(accelerated_equivalence), len(accelerated_equivalence) == 2)
add_check(validation_records, "Accelerated smoke-equivalence keys", sorted(SMOKE_EQUIVALENCE_KEYS), sorted(accelerated_equivalence["ConditionKey"].tolist()), sorted(accelerated_equivalence["ConditionKey"].tolist()) == sorted(SMOKE_EQUIVALENCE_KEYS))
add_check(validation_records, "Accelerated smoke-equivalence failures", 0, int((~accelerated_equivalence["Pass"].astype(bool)).sum()), accelerated_equivalence["Pass"].astype(bool).all())
add_check(validation_records, "Completed conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation_records, "Noise levels", NOISE_LEVELS, sorted(condition_inventory["NoisePercent"].unique().tolist()), sorted(condition_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Repetition seeds", REPETITION_SEEDS, sorted(condition_inventory["RepetitionSeed"].unique().tolist()), sorted(condition_inventory["RepetitionSeed"].unique().tolist()) == REPETITION_SEEDS)
add_check(validation_records, "Duplicate condition keys", 0, int(condition_inventory["ConditionKey"].duplicated(keep=False).sum()), not condition_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Duplicate condition coordinates", 0, int(condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).sum()), not condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).any())
add_check(validation_records, "Condition-order sequence", list(range(1, EXPECTED_CONDITIONS + 1)), condition_inventory["ConditionOrder"].tolist(), condition_inventory["ConditionOrder"].tolist() == list(range(1, EXPECTED_CONDITIONS + 1)))
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(condition_inventory["Files"].unique().tolist()), condition_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation_records, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation_records, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation_records, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation_records, "Model fits", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation_records, "Condition-audit rows", EXPECTED_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_CONDITIONS)
add_check(validation_records, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Active predictors", EXPECTED_PREDICTORS, sorted(combined_condition_audit["Predictors"].unique().tolist()), combined_condition_audit["Predictors"].eq(EXPECTED_PREDICTORS).all())
add_check(validation_records, "Condition statuses", [CONDITION_STATUS], sorted(combined_condition_audit["Status"].unique().tolist()), combined_condition_audit["Status"].eq(CONDITION_STATUS).all())
add_check(validation_records, "Model-fit failures", 0, int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()), combined_model_fits["Status"].eq("PASS_MODEL_FIT").all())
add_check(validation_records, "Project-run technique set", sorted(ALL_TECHNIQUES), sorted(combined_project_runs["Technique"].unique().tolist()), sorted(combined_project_runs["Technique"].unique().tolist()) == sorted(ALL_TECHNIQUES))
add_check(validation_records, "Model-fit technique set", sorted(ML_TECHNIQUES), sorted(combined_model_fits["Technique"].unique().tolist()), sorted(combined_model_fits["Technique"].unique().tolist()) == sorted(ML_TECHNIQUES))
add_check(validation_records, "Zero-noise conditions", len(REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "QTF global score variants", 1, qtf_global_score_variants, qtf_global_score_variants == 1)
add_check(validation_records, "QTF global rank variants", 1, qtf_global_rank_variants, qtf_global_rank_variants == 1)
add_check(validation_records, "Project metrics non-finite", 0, int((~np.isfinite(project_metric_values)).sum()), np.isfinite(project_metric_values).all())
add_check(validation_records, "Project metrics outside [0,1]", 0, int(((project_metric_values < 0) | (project_metric_values > 1)).sum()), bool(((project_metric_values >= 0) & (project_metric_values <= 1)).all()))
add_check(validation_records, "Raw training cohort unchanged", raw_training_hash_before, raw_training_hash_after, raw_training_hash_after == raw_training_hash_before)
add_check(validation_records, "Raw evaluation cohort unchanged", raw_evaluation_hash_before, raw_evaluation_hash_after, raw_evaluation_hash_after == raw_evaluation_hash_before)
add_check(validation_records, "Model training cohort unchanged", model_training_hash_before, model_training_hash_after, model_training_hash_after == model_training_hash_before)
add_check(validation_records, "Model evaluation cohort unchanged", model_evaluation_hash_before, model_evaluation_hash_after, model_evaluation_hash_after == model_evaluation_hash_before)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Registry rows", EXPECTED_REGISTERED_PROJECTS, len(registry), len(registry) == EXPECTED_REGISTERED_PROJECTS)
for predecessor_number, predecessor_project in required_registered_identities.items():
    predecessor_actual = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            project_column,
        ].iloc[0]
    )
    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        predecessor_actual,
        predecessor_actual == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    runtime_checkpoint.get("ActiveReservations"),
    runtime_checkpoint.get("ActiveReservations") == EXPECTED_ACTIVE_RESERVATIONS,
)
add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)
add_check(
    validation_records,
    "Smoke checkpoint runtime-contract linkage",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    ),
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    )
    == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(validation_records, "Registry Project 17 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

validation = pd.DataFrame(validation_records)
failed_validation = validation.loc[~validation["Pass"]]

print("\nStep 5A validation:")
display(validation)

if not failed_validation.empty:
    print("\nFailed Step 5A checks:")
    display(failed_validation)
    print("\nCompleted condition checkpoints remain resume-safe.")
    raise RuntimeError(
        "PROJECT 17 STEP 5A FINAL VALIDATION FAILED."
    )

# --------------------------------------------------------------------------------------------------
# 10. FREEZE FULL RAW ROOT AND STEP 5A CHECKPOINT
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(STEP5A_VALIDATION_PATH, validation)

full_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    STEP5A_VALIDATION_PATH,
]
aggregate_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in aggregate_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "AcceleratedSmokeEquivalenceRows": len(accelerated_equivalence),
    "AcceleratedSmokeEquivalenceFailures": int((~accelerated_equivalence["Pass"].astype(bool)).sum()),
    "AcceleratedEquivalenceAudit": str(ACCELERATED_EQUIVALENCE_PATH),
    "CompletedAtUTC": completed_at_utc,
    "SmokeCheckpointSHA256": smoke_checkpoint_sha256,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "RawRoot": str(FULL_RAW_RESULT_ROOT),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "RawManifest": str(RAW_MANIFEST_PATH),
    "RawManifestSHA256": sha256_file(RAW_MANIFEST_PATH),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisRun": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "FullExecutionSecondsThisInvocation": float(full_execution_seconds),
    "AggregateOutputManifest": aggregate_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To16Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "EvaluationCohortImmutable": True,
    "ResumeSafe": True,
}
atomic_json(STEP5A_REPORT_PATH, report_payload)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint_payload)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "Conditions": len(condition_inventory),
    "MLFits": len(combined_model_fits),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "FailedValidationChecks": len(failed_validation),
    "RegistryModified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5A_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_CONDITIONS,
        "ExpectedConditions": EXPECTED_CONDITIONS,
        "RawRootSHA256": raw_root_sha256,
        "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "RegistryModified": False,
        "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

        "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,
    },
)

checkpoint_readback = load_json(STEP5A_CHECKPOINT_PATH)
status_readback = load_json(STEP5A_STATUS_PATH)
if checkpoint_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A checkpoint readback failed.")
if status_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A status readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during Step 5A finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 17 CELL 9 / STEP 5A ACCELERATED RESULT ===")
print("=" * 136)
print()
print("Project:", PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[14],
)
print(
    "Project 15 identity:",
    required_registered_identities[15],
)
print(
    "Project 16 identity:",
    required_registered_identities[16],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Accelerated engine:")
print("Engine version:", ACCELERATED_ENGINE_VERSION)
print("Clean REC mismatches:", accelerated_clean_mismatch_values)
print("Frozen smoke-equivalence failures:", int((~accelerated_equivalence["Pass"].astype(bool)).sum()))
print()
print("Full experiment:")
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Raw result freeze:")
print("Raw files:", raw_files)
print("Raw bytes:", raw_bytes)
print("Raw root SHA-256:", raw_root_sha256)
print()
print("Checkpoint/resume:")
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Resume safe:", True)
print()
print("Baselines and metrics:")
print("Baseline invariance failures:", baseline_invariance_failures)
print("QTF global score variants:", qtf_global_score_variants)
print("QTF global rank variants:", qtf_global_rank_variants)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 17 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–16 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Step 5A checkpoint:")
print(STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print()
print("Runtime seconds this invocation:", round(full_execution_seconds, 2))
print()
print("STATUS:", STEP5A_STATUS)
print("=" * 136)


=== PROJECT 17 CELL 9 / STEP 5A V2: CHECKPOINT-SCHEMA-COMPATIBLE FULL 270-CONDITION EXPERIMENT ===

Loading frozen Project 17 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 160 / 7533 / 3341

Scanning existing condition checkpoints...
Valid completed conditions: 0
Incomplete/invalid condition directories: 0
Pending conditions: 270

Loading deterministic RNG stream for seed 1.

--------------------------------------------------------------------------------------------------------------
[1/270] Running noise_00__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_00__seed_01
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 145 | condition seconds: 20.15

--------------------------------------------------------------------------------------------------------------
[9/270] Running noise_50__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_50__seed_01
  Completed: noise_50__seed_01
  Raw flips: 21032 | model-label changes: 3221 | dependent REC changes: 84644
  Training failures: 3210 | condition seconds: 24.38

--------------------------------------------------------------------------------------------------------------
[2/270] Running noise_05__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_30
  Raw flips: 16898 | model-label changes: 2536 | dependent REC changes: 82794
  Training failures: 2551 | condition seconds: 8.36

--------------------------------------------------------------------------------------------------------------
[270/270] Running noise_50__seed_30
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_30
  Raw flips: 21182 | model-label changes: 3219 | dependent REC changes: 84987
  Training failures: 3210 | condition seconds: 11.19

Validating all 270 completed conditions.

Step 5A validation:


,Check,Expected,Actual,Pass
0,Step 4B passed,PASS_PROJECT_17_TWO_CONDITION_END_TO_END_SMOKE...,PASS_PROJECT_17_TWO_CONDITION_END_TO_END_SMOKE...,True
1,Smoke checkpoint SHA-256,a844057c5869a914036c7d3df655a396fa3817ef706bb2...,a844057c5869a914036c7d3df655a396fa3817ef706bb2...,True
2,Accelerated clean REC mismatches,0,0,True
3,Accelerated smoke-equivalence rows,2,2,True
4,Accelerated smoke-equivalence keys,"[noise_00__seed_01, noise_50__seed_01]","[noise_00__seed_01, noise_50__seed_01]",True
5,Accelerated smoke-equivalence failures,0,0,True
6,Completed conditions,270,270,True
7,Noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
8,Repetition seeds,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
9,Duplicate condition keys,0,0,True



=== PROJECT 17 CELL 9 / STEP 5A ACCELERATED RESULT ===

Project: yamcs@Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Accelerated engine:
Engine version: PROJECT_17_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER
Clean REC mismatches: 0
Frozen smoke-equivalence failures: 0

Full experiment:
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 2043090
Build-metric rows: 18900
Project-run rows: 1890
Condition-audit rows: 270
Training-median rows: 40770

Raw result freeze:
Raw files: 2160
Raw bytes: 38425581
Raw root SHA-256: b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72

Checkpoint/resume:
Completed thi

In [22]:
# ==================================================================================================
# PROJECT 17 — CELL 10 / STEP 5B
# CORRECTED PROJECT-SPECIFIC COUNT CONTRACT, RAW REVALIDATION, AND COMPACT AGGREGATION
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# CURRENT REGISTRY CONTRACT:
# - Projects 1–16 must be present exactly once and COMPLETE_AND_FROZEN.
# - Project 11 must be apache@shardingsphere.
# - Project 12 must be zolyfarkas@spf4j.
# - Project 13 must be jcabi@jcabi-github.
# - Project 14 must be JMRI@JMRI.
# - Project 15 must be eclipse@steady.
# - Project 16 must be apache@rocketmq.
# - Project 17 must still be absent.
#
# THIS CELL:
# - independently hashes all 2,160 Project 17 raw files;
# - validates every condition checkpoint and compact output;
# - recounts all 2,043,090 compressed ranking rows;
# - independently validates noise hashes, REC invariance, metrics, and baselines;
# - creates analysis-ready aggregates across all 30 seeds;
# - writes the Project 17 Step 5B checkpoint;
# - does not rerun conditions or fit models;
# - does not access or modify prior-project condition outputs;
# - does not register Project 17.
#
# BASELINE-INVARIANCE CONTRACT:
# - Random and QTF-Avg are compared independently within each metric;
# - APFDc and APFD are never compared against one another.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gzip, hashlib, json, os, time
import numpy as np
import pandas as pd

print('=' * 136)
print('=== PROJECT 17 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===')
print('=' * 136)

PROJECT_NUMBER = 17
PROJECT_NAME = 'yamcs@Yamcs'
PROJECT_SLUG = 'yamcs__Yamcs'
PROJECT_SHORT = 'YAMCS'
STEP5A_STATUS = 'PASS_PROJECT_17_FULL_270_CONDITION_EXPERIMENT_COMPLETE'
CONDITION_STATUS = 'PASS_FULL_CONDITION'
STEP5B_STATUS = 'PASS_PROJECT_17_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN'

EXPECTED_STEP5A_SHA = 'd559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653ea152c344ab7727b5b'
EXPECTED_RAW_ROOT_SHA = 'b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72'
EXPECTED_REGISTRY_SHA = 'd0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a'
EXPECTED_SOURCE_ROOT_SHA = '64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2'

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
ML_TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
INVARIANT_BASELINES = ['Random', 'QTF-Avg']
PROJECT_METRICS = ['MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']
BUILD_METRICS = ['APFDc', 'APFD']
EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    'ModelTrainingRows ascending',
    'ModelEvaluationRows ascending',
    'RawExecutionRows ascending',
    'Project ascending',
]

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 38_425_581
EXPECTED_RANKING_ROWS_PER_CONDITION = 1_081 * 7
EXPECTED_BUILD_ROWS_PER_CONDITION = 10 * 7
EXPECTED_PROJECT_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_TOTAL_RANKING_ROWS = 2_043_090
EXPECTED_TOTAL_BUILD_ROWS = 18_900
EXPECTED_TOTAL_PROJECT_ROWS = 1_890
EXPECTED_TOTAL_FIT_ROWS = 1_080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40_770

# Project 17 fixed evaluation counts.
# These are validated in Step 1A/1B, Step 2A, Step 4A, Step 4B, and Step 5A.
EXPECTED_SCORED_FAILING_BUILDS = 10
EXPECTED_EVALUATION_BUILDS = 126
EXPECTED_EVALUATION_FAILURES = 45

EXPECTED_CONDITION_FILES = {
    'rankings.csv.gz', 'build_metrics.csv', 'project_runs.csv', 'model_fits.csv',
    'training_medians.csv', 'condition_audit.csv', 'condition_summary.json', 'COMPLETE.json'
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {'condition_summary.json', 'COMPLETE.json'}

# Mount only Drive. No source re-extraction is needed for Step 5B.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Thesis_Experiment')
NOTES = ROOT / 'Notes'
RESULTS = ROOT / 'Results'
REGISTRY = NOTES / 'completed_project_registry.csv'
PROJECT_ROOT = RESULTS / 'Aggregated' / PROJECT_SLUG
RAW_ROOT = RESULTS / 'Raw' / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f'{PROJECT_SHORT}_full_experiment'
PLAN = PROJECT_ROOT / f'{PROJECT_SHORT}_noise_plan' / f'{PROJECT_SHORT}_condition_plan.csv'
STEP5A_CHECKPOINT = NOTES / 'project_17_step5a_checkpoint.json'
STEP5A_STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5a_status.json'
STEP5A_REPORT = FULL_ROOT / f'{PROJECT_SHORT}_step5a_report.json'
STEP5A_RAW_MANIFEST = FULL_ROOT / f'{PROJECT_SHORT}_raw_manifest.csv'
STEP5A_BASELINE = FULL_ROOT / f'{PROJECT_SHORT}_baseline_invariance.csv'

OUT = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b'
CURRENT_MANIFEST = OUT / f'{PROJECT_SHORT}_independent_raw_manifest.csv'
CONDITION_INVENTORY = OUT / f'{PROJECT_SHORT}_independent_condition_inventory.csv'
REVALIDATED_PROJECT_RUNS = OUT / f'{PROJECT_SHORT}_revalidated_project_runs.csv'
REVALIDATED_BUILD_METRICS = OUT / f'{PROJECT_SHORT}_revalidated_build_metrics.csv'
REVALIDATED_MODEL_FITS = OUT / f'{PROJECT_SHORT}_revalidated_model_fits.csv'
REVALIDATED_CONDITION_AUDIT = OUT / f'{PROJECT_SHORT}_revalidated_condition_audit.csv'
REVALIDATED_MEDIANS = OUT / f'{PROJECT_SHORT}_revalidated_training_medians.csv'
NOISE_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_technique_summary.csv'
SEED_DELTAS = OUT / f'{PROJECT_SHORT}_seed_level_noise_deltas.csv'
DELTA_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_delta_summary.csv'
VALIDATION_PATH = OUT / f'{PROJECT_SHORT}_step5b_validation.csv'
REPORT_PATH = OUT / f'{PROJECT_SHORT}_step5b_report.json'
STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b_status.json'
CHECKPOINT_PATH = NOTES / 'project_17_step5b_checkpoint.json'


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write('\n')
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    df.to_csv(tmp, index=False, lineterminator='\n')
    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise RuntimeError(f'Could not resolve {label}; columns={list(columns)}')


def normalize_manifest(df, label):
    p = resolve_col(df.columns, ['RelativePath'], f'{label} path')
    b = resolve_col(df.columns, ['Bytes', 'SizeBytes'], f'{label} bytes')
    s = resolve_col(df.columns, ['SHA256'], f'{label} sha')
    out = df[[p, b, s]].copy(); out.columns = ['RelativePath', 'Bytes', 'SHA256']
    out['RelativePath'] = out['RelativePath'].astype(str).str.replace('\\', '/', regex=False)
    out['Bytes'] = pd.to_numeric(out['Bytes'], errors='raise').astype('int64')
    out['SHA256'] = out['SHA256'].astype(str).str.lower()
    return out.sort_values('RelativePath', kind='mergesort').reset_index(drop=True)


def root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values('RelativePath', kind='mergesort').itertuples(index=False):
        h.update(f'{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n'.encode('utf-8'))
    return h.hexdigest()


def gzip_rows(path):
    n = 0
    with gzip.open(path, 'rb') as f:
        for _ in f:
            n += 1
    return max(0, n - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Pass': bool(passed)})


def metric_nonfinite(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int((~np.isfinite(arr)).sum())


def metric_outside(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int(((arr < 0) | (arr > 1)).sum())


# Required Drive inputs.
required = [REGISTRY, PLAN, STEP5A_CHECKPOINT, STEP5A_STATUS_PATH, STEP5A_REPORT,
            STEP5A_RAW_MANIFEST, STEP5A_BASELINE, RAW_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Project 17 Step 5B inputs:\n' + '\n'.join(missing))

# Frozen Step 5A and registry.
step5a_sha = sha256_file(STEP5A_CHECKPOINT)
step5a = load_json(STEP5A_CHECKPOINT)
step5a_status = load_json(STEP5A_STATUS_PATH)
step5a_report = load_json(STEP5A_REPORT)
if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(f'Step 5A checkpoint SHA differs. Expected={EXPECTED_STEP5A_SHA}; actual={step5a_sha}')
for label, payload in [('checkpoint', step5a), ('status', step5a_status), ('report', step5a_report)]:
    if payload.get('Status') != STEP5A_STATUS:
        raise RuntimeError(f'Step 5A {label} is not in PASS state.')
if step5a.get('SourceRootSHA256') != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError('Step 5A source-root SHA differs.')
if step5a.get('RawRootSHA256') != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError('Step 5A frozen raw-root SHA differs.')
if step5a.get('ActiveReservations') != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError('Step 5A active-reservation state differs.')
if step5a.get('RuntimePriorityRule') != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError('Step 5A runtime-priority rule differs.')

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f'Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA}; actual={registry_sha_before}')
registry = pd.read_csv(REGISTRY, dtype=str).fillna('')
pn_col = resolve_col(registry.columns, ['ProjectNumber'], 'registry ProjectNumber')
project_col = resolve_col(registry.columns, ['Project'], 'registry Project')
st_col = resolve_col(registry.columns, ['Status'], 'registry Status')
pnums = pd.to_numeric(registry[pn_col], errors='raise').astype(int)

if len(registry) != 16 or sorted(pnums.tolist()) != list(range(1, 17)):
    raise RuntimeError(
        'Registry must contain exactly Projects 1–16 before Project 17 Step 5B.'
    )

if not registry[st_col].eq('COMPLETE_AND_FROZEN').all():
    raise RuntimeError(
        'Projects 1–16 are not all COMPLETE_AND_FROZEN.'
    )

required_registered_identities = {
    11: 'apache@shardingsphere',
    12: 'zolyfarkas@spf4j',
    13: 'jcabi@jcabi-github',
    14: 'JMRI@JMRI',
    15: 'eclipse@steady',
    16: 'apache@rocketmq',
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(required_number)
    ]

    if (
        len(matching_rows) != 1
        or matching_rows.iloc[0][project_col] != required_project
    ):
        raise RuntimeError(
            'A required frozen predecessor has a different registry identity.\n'
            f'Project number: {required_number}\n'
            f'Expected project: {required_project}'
        )

if pnums.eq(17).any() or registry[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError(
        'Project 17 is unexpectedly already present in the completion registry.'
    )

# Validate Step 5A aggregate-output manifest.
agg_manifest = step5a.get('AggregateOutputManifest', [])
if not isinstance(agg_manifest, list) or not agg_manifest:
    raise RuntimeError('Step 5A checkpoint has no AggregateOutputManifest.')
agg_failures = 0
for item in agg_manifest:
    p = Path(item['Path'])
    ok = p.is_file() and p.stat().st_size == int(item['Bytes']) and sha256_file(p) == str(item['SHA256'])
    agg_failures += int(not ok)
if agg_failures:
    raise RuntimeError(f'{agg_failures} Step 5A aggregate outputs changed.')

# Independently hash all raw files.
print('\nIndependently hashing all 2,160 raw files.')
hash_start = time.perf_counter()
paths = sorted([p for p in RAW_ROOT.rglob('*') if p.is_file()], key=lambda p: p.relative_to(RAW_ROOT).as_posix())
manifest_rows = []
for i, p in enumerate(paths, 1):
    manifest_rows.append({'RelativePath': p.relative_to(RAW_ROOT).as_posix(),
                          'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)})
    if i % 200 == 0 or i == len(paths):
        print(f'  Raw hashing progress: {i} / {len(paths)} files')
current_manifest = normalize_manifest(pd.DataFrame(manifest_rows), 'current manifest')
hash_seconds = time.perf_counter() - hash_start
frozen_manifest = normalize_manifest(pd.read_csv(STEP5A_RAW_MANIFEST), 'frozen manifest')
current_raw_sha = root_hash(current_manifest)
current_raw_bytes = int(current_manifest['Bytes'].sum())
merged_manifest = frozen_manifest.merge(current_manifest, on='RelativePath', how='outer',
                                        suffixes=('_frozen', '_current'), indicator=True)
missing_raw = int(merged_manifest['_merge'].eq('left_only').sum())
unexpected_raw = int(merged_manifest['_merge'].eq('right_only').sum())
size_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['Bytes_frozen'].ne(merged_manifest['Bytes_current'])).sum())
hash_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['SHA256_frozen'].ne(merged_manifest['SHA256_current'])).sum())

# Condition-by-condition independent validation and compact reload.
plan = pd.read_csv(PLAN, low_memory=False)
id_col = resolve_col(plan.columns, ['ConditionID', 'ConditionKey'], 'condition identifier')
order_col = resolve_col(plan.columns, ['ConditionOrder'], 'condition order')
noise_col = resolve_col(plan.columns, ['NoisePercent'], 'noise percent')
seed_col = resolve_col(plan.columns, ['RepetitionSeed'], 'repetition seed')
for col in [order_col, noise_col, seed_col]:
    plan[col] = pd.to_numeric(plan[col], errors='raise').astype(int)
plan = plan.sort_values(order_col, kind='mergesort').reset_index(drop=True)

inventory_rows, project_frames, build_frames, fit_frames, audit_frames, median_frames = [], [], [], [], [], []
marker_fail = summary_fail = file_set_fail = embedded_fail = ranking_count_fail = 0
print('\nRevalidating all 270 condition directories.')
condition_start = time.perf_counter()
for i, row in enumerate(plan.itertuples(index=False), 1):
    key = str(getattr(row, id_col)); order = int(getattr(row, order_col))
    noise = int(getattr(row, noise_col)); seed = int(getattr(row, seed_col))
    d = RAW_ROOT / key
    if not d.is_dir():
        raise FileNotFoundError(f'Missing condition directory: {d}')
    actual_files = {p.name for p in d.iterdir() if p.is_file()}
    file_ok = actual_files == EXPECTED_CONDITION_FILES
    file_set_fail += int(not file_ok)
    complete_path, summary_path = d / 'COMPLETE.json', d / 'condition_summary.json'
    complete, summary = load_json(complete_path), load_json(summary_path)
    complete_ok = (complete.get('Status') == CONDITION_STATUS and complete.get('ConditionKey') == key and
                   str(complete.get('ConditionSummaryPath')) == str(summary_path) and
                   str(complete.get('ConditionSummarySHA256')).lower() == sha256_file(summary_path))
    summary_ok = (summary.get('Status') == CONDITION_STATUS and summary.get('ConditionKey') == key and
                  int(summary.get('NoisePercent', -1)) == noise and int(summary.get('RepetitionSeed', -1)) == seed)
    marker_fail += int(not complete_ok); summary_fail += int(not summary_ok)
    output_manifest = summary.get('OutputManifest', [])
    local_embedded_fail = 0
    names = set()
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        local_embedded_fail += 1
    else:
        for item in output_manifest:
            p = Path(item.get('Path', '')); names.add(p.name)
            ok = (p.parent == d and p.is_file() and p.stat().st_size == int(item.get('Bytes', -1)) and
                  sha256_file(p) == str(item.get('SHA256', '')).lower())
            local_embedded_fail += int(not ok)
        local_embedded_fail += int(names != CONDITION_OUTPUT_FILES)
    embedded_fail += local_embedded_fail
    ranking_rows = gzip_rows(d / 'rankings.csv.gz')
    ranking_count_fail += int(ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION)
    build = pd.read_csv(d / 'build_metrics.csv', low_memory=False)
    project = pd.read_csv(d / 'project_runs.csv', low_memory=False)
    fits = pd.read_csv(d / 'model_fits.csv', low_memory=False)
    audit = pd.read_csv(d / 'condition_audit.csv', low_memory=False)
    medians = pd.read_csv(d / 'training_medians.csv', low_memory=False)
    expected_counts = [EXPECTED_BUILD_ROWS_PER_CONDITION, EXPECTED_PROJECT_ROWS_PER_CONDITION,
                       EXPECTED_FIT_ROWS_PER_CONDITION, 1, EXPECTED_MEDIAN_ROWS_PER_CONDITION]
    actual_counts = [len(build), len(project), len(fits), len(audit), len(medians)]
    if actual_counts != expected_counts:
        raise RuntimeError(f'{key}: compact output counts differ. expected={expected_counts}; actual={actual_counts}')
    for field, count in [('RankingRows', ranking_rows), ('BuildMetricRows', len(build)),
                         ('ProjectRunRows', len(project)), ('MLFits', len(fits)),
                         ('TrainingMedianRows', len(medians))]:
        if int(summary.get(field, -1)) != count:
            raise RuntimeError(f'{key}: condition_summary {field} differs.')
    project_frames.append(project); build_frames.append(build); fit_frames.append(fits)
    audit_frames.append(audit); median_frames.append(medians)
    inventory_rows.append({
        'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
        'ConditionOrder': order, 'ConditionKey': key, 'NoisePercent': noise, 'RepetitionSeed': seed,
        'ConditionDirectory': str(d), 'CompletionStatus': complete.get('Status'),
        'SummaryStatus': summary.get('Status'), 'Files': len(actual_files),
        'ConditionBytes': int(sum(p.stat().st_size for p in d.iterdir() if p.is_file())),
        'RankingRows': ranking_rows, 'BuildMetricRows': len(build), 'ProjectRunRows': len(project),
        'ModelFits': len(fits), 'ConditionAuditRows': len(audit), 'TrainingMedianRows': len(medians),
        'FileSetPass': file_ok, 'CompletionMarkerPass': complete_ok, 'ConditionSummaryPass': summary_ok,
        'EmbeddedManifestFailures': local_embedded_fail,
        'CompletionMarkerSHA256': sha256_file(complete_path), 'ConditionSummarySHA256': sha256_file(summary_path),
    })
    if i % 30 == 0 or i == len(plan):
        print(f'  Condition revalidation progress: {i} / {len(plan)}')
condition_seconds = time.perf_counter() - condition_start

inventory = pd.DataFrame(inventory_rows).sort_values('ConditionOrder', kind='mergesort').reset_index(drop=True)
project_runs = pd.concat(project_frames, ignore_index=True)
build_metrics = pd.concat(build_frames, ignore_index=True)
model_fits = pd.concat(fit_frames, ignore_index=True)
condition_audit = pd.concat(audit_frames, ignore_index=True)
training_medians = pd.concat(median_frames, ignore_index=True)

# Contract audits.
coordinate_count = len(inventory[['NoisePercent', 'RepetitionSeed']].drop_duplicates())
dup_keys = int(inventory.duplicated(['ConditionKey'], keep=False).sum())
dup_coords = int(inventory.duplicated(['NoisePercent', 'RepetitionSeed'], keep=False).sum())
order_viol = int((inventory['ConditionOrder'].to_numpy(int) != np.arange(1, 271)).sum())
files_viol = int(inventory['Files'].ne(8).sum())
ranking_viol = int(inventory['RankingRows'].ne(EXPECTED_RANKING_ROWS_PER_CONDITION).sum())
small_per_condition_viol = int(inventory['BuildMetricRows'].ne(EXPECTED_BUILD_ROWS_PER_CONDITION).sum() +
                               inventory['ProjectRunRows'].ne(7).sum() + inventory['ModelFits'].ne(4).sum() +
                               inventory['ConditionAuditRows'].ne(1).sum() + inventory['TrainingMedianRows'].ne(151).sum())
project_techniques = sorted(project_runs['Technique'].astype(str).unique().tolist())
fit_techniques = sorted(model_fits['Technique'].astype(str).unique().tolist())
dup_project = int(project_runs.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_build = int(build_metrics.duplicated(['ConditionKey', 'Technique', 'Build'], keep=False).sum())
dup_fit = int(model_fits.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_audit = int(condition_audit.duplicated(['ConditionKey'], keep=False).sum())
dup_median = int(training_medians.duplicated(['ConditionKey', 'PredictorOrder'], keep=False).sum())
fit_fail = int((~model_fits['Status'].astype(str).eq('PASS_MODEL_FIT')).sum())
fit_errors = int(model_fits['Error'].fillna('').astype(str).str.len().gt(0).sum())
project_nonfinite = metric_nonfinite(project_runs, PROJECT_METRICS)
project_outside = metric_outside(project_runs, PROJECT_METRICS)
build_nonfinite = metric_nonfinite(build_metrics, BUILD_METRICS)
build_outside = metric_outside(build_metrics, BUILD_METRICS)
median_nonfinite = int((~np.isfinite(pd.to_numeric(training_medians['TrainingMedian'], errors='coerce').to_numpy(float))).sum())
median_predictor_viol = int(training_medians.groupby('ConditionKey')['Predictor'].nunique().ne(151).sum())
scored_build_viol = int(
    project_runs[
        'ScoredFailingBuilds'
    ].ne(
        EXPECTED_SCORED_FAILING_BUILDS
    ).sum()
)

eval_build_viol = int(
    project_runs[
        'EvaluationBuilds'
    ].ne(
        EXPECTED_EVALUATION_BUILDS
    ).sum()
)

eval_failure_viol = int(
    project_runs[
        'EvaluationFailures'
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)
zero = condition_audit[condition_audit['NoisePercent'].eq(0)]
positive = condition_audit[condition_audit['NoisePercent'].gt(0)]
zero_flip_viol = int(zero['NumberFlipped'].ne(0).sum())
zero_model_viol = int(zero['ModelLabelChanges'].ne(0).sum())
zero_rec_viol = int(zero['DependentRECChanges'].ne(0).sum())
pos_raw_viol = int(positive['NumberFlipped'].le(0).sum())
pos_model_viol = int(positive['ModelLabelChanges'].le(0).sum())
pos_rec_viol = int(positive['DependentRECChanges'].le(0).sum())
independent_viol = int(condition_audit['IndependentRECChanges'].ne(0).sum())
independent_recon_viol = int(condition_audit['IndependentReconstructionMismatches'].ne(0).sum())
noise_hash_mismatch = 0
for e, a in [('ExpectedFlipMaskSHA256', 'ActualFlipMaskSHA256'),
             ('ExpectedNoisyRawVerdictSHA256', 'ActualNoisyRawVerdictSHA256'),
             ('ExpectedNoisyModelVerdictSHA256', 'ActualNoisyModelVerdictSHA256')]:
    noise_hash_mismatch += int((condition_audit[e].astype(str) != condition_audit[a].astype(str)).sum())

baseline = pd.read_csv(STEP5A_BASELINE, low_memory=False)
if 'Pass' in baseline.columns:
    bpass = baseline['Pass'].astype(str).str.strip().str.lower().isin({'true', '1'})
    ranking_baseline_fail = int((~bpass).sum())
else:
    mismatch_cols = [c for c in baseline.columns if 'mismatch' in c.lower()]
    ranking_baseline_fail = int(baseline[mismatch_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(float).sum())
# Project-metric invariance must be evaluated independently for each metric.
# The previous V1 expression compared the maximum of one metric with the
# minimum of another metric. Because APFDc and APFD naturally have different
# values, that incorrectly marked all 60 seed/baseline groups as failures even
# though each individual metric was invariant across noise.
metric_baseline_fail = 0
metric_baseline_max_range = 0.0

for _, g in project_runs[
    project_runs['Technique'].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        'RepetitionSeed',
        'Technique',
    ],
    sort=False,
):
    arr = g[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            arr,
            axis=0,
        )
        - np.min(
            arr,
            axis=0,
        )
    )

    metric_baseline_max_range = max(
        metric_baseline_max_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    metric_baseline_fail += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )

# Compact aggregates.
agg_start = time.perf_counter()
noise_summary = project_runs.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Runs=('ConditionKey', 'count'), Seeds=('RepetitionSeed', 'nunique'),
    Mean_MeanAPFDc=('MeanAPFDc', 'mean'), SD_MeanAPFDc=('MeanAPFDc', 'std'), Median_MeanAPFDc=('MeanAPFDc', 'median'),
    Mean_MedianAPFDc=('MedianAPFDc', 'mean'), SD_MedianAPFDc=('MedianAPFDc', 'std'), Median_MedianAPFDc=('MedianAPFDc', 'median'),
    Mean_MeanAPFD=('MeanAPFD', 'mean'), SD_MeanAPFD=('MeanAPFD', 'std'), Median_MeanAPFD=('MeanAPFD', 'median'),
    Mean_MedianAPFD=('MedianAPFD', 'mean'), SD_MedianAPFD=('MedianAPFD', 'std'), Median_MedianAPFD=('MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
clean = project_runs[project_runs['NoisePercent'].eq(0)][['RepetitionSeed', 'Technique'] + PROJECT_METRICS].rename(
    columns={c: f'Clean_{c}' for c in PROJECT_METRICS})
seed_deltas = project_runs.merge(clean, on=['RepetitionSeed', 'Technique'], how='left', validate='many_to_one')
for c in PROJECT_METRICS:
    seed_deltas[f'Delta_{c}'] = seed_deltas[c] - seed_deltas[f'Clean_{c}']
delta_cols = [f'Delta_{c}' for c in PROJECT_METRICS]
seed_deltas = seed_deltas[['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent',
                           'RepetitionSeed', 'Technique'] + PROJECT_METRICS +
                          [f'Clean_{c}' for c in PROJECT_METRICS] + delta_cols].sort_values(
                              ['NoisePercent', 'Technique', 'RepetitionSeed'], kind='mergesort').reset_index(drop=True)
delta_summary = seed_deltas.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Seeds=('RepetitionSeed', 'nunique'),
    Mean_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'mean'), SD_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'std'), Median_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'median'),
    Mean_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'mean'), SD_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'std'), Median_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'median'),
    Mean_Delta_MeanAPFD=('Delta_MeanAPFD', 'mean'), SD_Delta_MeanAPFD=('Delta_MeanAPFD', 'std'), Median_Delta_MeanAPFD=('Delta_MeanAPFD', 'median'),
    Mean_Delta_MedianAPFD=('Delta_MedianAPFD', 'mean'), SD_Delta_MedianAPFD=('Delta_MedianAPFD', 'std'), Median_Delta_MedianAPFD=('Delta_MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
agg_seconds = time.perf_counter() - agg_start
summary_nonfinite = metric_nonfinite(noise_summary, [c for c in noise_summary.columns if c not in {'NoisePercent', 'Technique'}])
delta_nonfinite = metric_nonfinite(delta_summary, [c for c in delta_summary.columns if c not in {'NoisePercent', 'Technique'}])
clean_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['NoisePercent'].eq(0)][delta_cols].to_numpy(float)) > 1e-15).sum())
baseline_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['Technique'].isin(INVARIANT_BASELINES)][delta_cols].to_numpy(float)) > 1e-15).sum())

# Validation table.
checks = []
add_check(checks, 'Step 5A status', STEP5A_STATUS, step5a.get('Status'), step5a.get('Status') == STEP5A_STATUS)
add_check(checks, 'Step 5A checkpoint SHA-256', EXPECTED_STEP5A_SHA, step5a_sha, step5a_sha == EXPECTED_STEP5A_SHA)
add_check(checks, 'Frozen raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, step5a.get('RawRootSHA256'), step5a.get('RawRootSHA256') == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Independent current raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, current_raw_sha, current_raw_sha == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Step 5A aggregate-manifest failures', 0, agg_failures, agg_failures == 0)
add_check(checks, 'Condition marker failures', 0, marker_fail, marker_fail == 0)
add_check(checks, 'Condition summary failures', 0, summary_fail, summary_fail == 0)
add_check(checks, 'Condition file-set failures', 0, file_set_fail, file_set_fail == 0)
add_check(checks, 'Embedded output-manifest failures', 0, embedded_fail, embedded_fail == 0)
add_check(checks, 'Conditions', 270, len(inventory), len(inventory) == 270)
add_check(checks, 'Condition coordinates', 270, coordinate_count, coordinate_count == 270)
add_check(checks, 'Duplicate condition keys', 0, dup_keys, dup_keys == 0)
add_check(checks, 'Duplicate condition coordinates', 0, dup_coords, dup_coords == 0)
add_check(checks, 'Condition-order violations', 0, order_viol, order_viol == 0)
add_check(checks, 'Files-per-condition violations', 0, files_viol, files_viol == 0)
add_check(checks, 'Raw files', EXPECTED_RAW_FILES, len(current_manifest), len(current_manifest) == EXPECTED_RAW_FILES)
add_check(checks, 'Raw bytes', EXPECTED_RAW_BYTES, current_raw_bytes, current_raw_bytes == EXPECTED_RAW_BYTES)
add_check(checks, 'Missing raw files', 0, missing_raw, missing_raw == 0)
add_check(checks, 'Unexpected raw files', 0, unexpected_raw, unexpected_raw == 0)
add_check(checks, 'Raw size mismatches', 0, size_mismatch, size_mismatch == 0)
add_check(checks, 'Raw SHA-256 mismatches', 0, hash_mismatch, hash_mismatch == 0)
add_check(checks, 'Ranking rows', EXPECTED_TOTAL_RANKING_ROWS, int(inventory['RankingRows'].sum()), int(inventory['RankingRows'].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(checks, 'Ranking row-count failures', 0, ranking_count_fail + ranking_viol, ranking_count_fail + ranking_viol == 0)
add_check(checks, 'Project-run rows', EXPECTED_TOTAL_PROJECT_ROWS, len(project_runs), len(project_runs) == EXPECTED_TOTAL_PROJECT_ROWS)
add_check(checks, 'Build-metric rows', EXPECTED_TOTAL_BUILD_ROWS, len(build_metrics), len(build_metrics) == EXPECTED_TOTAL_BUILD_ROWS)
add_check(checks, 'Model-fit rows', EXPECTED_TOTAL_FIT_ROWS, len(model_fits), len(model_fits) == EXPECTED_TOTAL_FIT_ROWS)
add_check(checks, 'Condition-audit rows', EXPECTED_TOTAL_AUDIT_ROWS, len(condition_audit), len(condition_audit) == EXPECTED_TOTAL_AUDIT_ROWS)
add_check(checks, 'Training-median rows', EXPECTED_TOTAL_MEDIAN_ROWS, len(training_medians), len(training_medians) == EXPECTED_TOTAL_MEDIAN_ROWS)
add_check(checks, 'Small rows-per-condition violations', 0, small_per_condition_viol, small_per_condition_viol == 0)
add_check(checks, 'Project-run technique set', sorted(TECHNIQUES), project_techniques, project_techniques == sorted(TECHNIQUES))
add_check(checks, 'Model-fit technique set', sorted(ML_TECHNIQUES), fit_techniques, fit_techniques == sorted(ML_TECHNIQUES))
add_check(checks, 'Duplicate project/build/fit/audit/median rows', 0, dup_project + dup_build + dup_fit + dup_audit + dup_median, dup_project + dup_build + dup_fit + dup_audit + dup_median == 0)
add_check(checks, 'Model-fit failures', 0, fit_fail + fit_errors, fit_fail + fit_errors == 0)
add_check(
    checks,
    'Scored-failing-build count violations',
    0,
    scored_build_viol,
    scored_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-build count violations',
    0,
    eval_build_viol,
    eval_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-failure count violations',
    0,
    eval_failure_viol,
    eval_failure_viol == 0,
)

add_check(
    checks,
    'Combined scored/evaluated/failure count violations',
    0,
    scored_build_viol + eval_build_viol + eval_failure_viol,
    scored_build_viol + eval_build_viol + eval_failure_viol == 0,
)
add_check(checks, 'Project metric invalid values', 0, project_nonfinite + project_outside, project_nonfinite + project_outside == 0)
add_check(checks, 'Build metric invalid values', 0, build_nonfinite + build_outside, build_nonfinite + build_outside == 0)
add_check(checks, 'Training-median invalid values', 0, median_nonfinite + median_predictor_viol, median_nonfinite + median_predictor_viol == 0)
add_check(checks, 'Zero-noise conditions', 30, len(zero), len(zero) == 30)
add_check(checks, 'Zero-noise violations', 0, zero_flip_viol + zero_model_viol + zero_rec_viol, zero_flip_viol + zero_model_viol + zero_rec_viol == 0)
add_check(checks, 'Positive-noise violations', 0, pos_raw_viol + pos_model_viol + pos_rec_viol, pos_raw_viol + pos_model_viol + pos_rec_viol == 0)
add_check(checks, 'Independent REC violations', 0, independent_viol + independent_recon_viol, independent_viol + independent_recon_viol == 0)
add_check(checks, 'Noise-plan hash mismatches', 0, noise_hash_mismatch, noise_hash_mismatch == 0)
add_check(checks, 'Ranking-level baseline-invariance failures', 0, ranking_baseline_fail, ranking_baseline_fail == 0)
add_check(checks, 'Project-metric baseline-invariance failures', 0, metric_baseline_fail, metric_baseline_fail == 0)
add_check(checks, 'Noise-technique summary rows', 63, len(noise_summary), len(noise_summary) == 63)
add_check(checks, 'Noise-technique summary count/nonfinite violations', 0, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite == 0)
add_check(checks, 'Seed-level delta rows', 1890, len(seed_deltas), len(seed_deltas) == 1890)
add_check(checks, 'Clean delta non-zero values', 0, clean_delta_nonzero, clean_delta_nonzero == 0)
add_check(checks, 'Invariant-baseline delta non-zero values', 0, baseline_delta_nonzero, baseline_delta_nonzero == 0)
add_check(checks, 'Noise-delta summary rows', 63, len(delta_summary), len(delta_summary) == 63)
add_check(checks, 'Noise-delta summary count/nonfinite violations', 0, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite == 0)
add_check(checks, 'Registry rows', 16, len(registry), len(registry) == 16)
add_check(checks, 'Active reservations', EXPECTED_ACTIVE_RESERVATIONS, step5a.get('ActiveReservations'), step5a.get('ActiveReservations') == EXPECTED_ACTIVE_RESERVATIONS)
add_check(checks, 'Runtime-priority ranking rule', EXPECTED_RUNTIME_PRIORITY_RULE, step5a.get('RuntimePriorityRule'), step5a.get('RuntimePriorityRule') == EXPECTED_RUNTIME_PRIORITY_RULE)

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(
            required_number
        )
    ]

    add_check(
        checks,
        f'Registry Project {required_number} rows',
        1,
        len(
            matching_rows
        ),
        len(
            matching_rows
        )
        == 1,
    )

    add_check(
        checks,
        f'Project {required_number} frozen identity',
        required_project,
        (
            str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            if len(
                matching_rows
            )
            == 1
            else None
        ),
        (
            len(
                matching_rows
            )
            == 1
            and str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            == required_project
        ),
    )

add_check(
    checks,
    'Registry Project 17 rows',
    0,
    int(
        pnums.eq(
            17
        ).sum()
    ),
    int(
        pnums.eq(
            17
        ).sum()
    )
    == 0,
)

validation = pd.DataFrame(checks)
failed = validation[~validation['Pass']]
print('\nProject 17 Step 5B validation:')
display(validation)
if not failed.empty:
    print('\nFailed checks:'); display(failed)
    raise RuntimeError('PROJECT 17 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.')

# Freeze outputs.
OUT.mkdir(parents=True, exist_ok=True)
for path, frame in [
    (CURRENT_MANIFEST, current_manifest), (CONDITION_INVENTORY, inventory),
    (REVALIDATED_PROJECT_RUNS, project_runs), (REVALIDATED_BUILD_METRICS, build_metrics),
    (REVALIDATED_MODEL_FITS, model_fits), (REVALIDATED_CONDITION_AUDIT, condition_audit),
    (REVALIDATED_MEDIANS, training_medians), (NOISE_SUMMARY, noise_summary),
    (SEED_DELTAS, seed_deltas), (DELTA_SUMMARY, delta_summary), (VALIDATION_PATH, validation),
]:
    atomic_csv(path, frame)
output_paths = [CURRENT_MANIFEST, CONDITION_INVENTORY, REVALIDATED_PROJECT_RUNS,
                REVALIDATED_BUILD_METRICS, REVALIDATED_MODEL_FITS, REVALIDATED_CONDITION_AUDIT,
                REVALIDATED_MEDIANS, NOISE_SUMMARY, SEED_DELTAS, DELTA_SUMMARY, VALIDATION_PATH]
output_manifest = [{'Path': str(p), 'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)} for p in output_paths]
completed = datetime.now(timezone.utc).isoformat()
report = {
    'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
    'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
    'FrozenRawRootSHA256': EXPECTED_RAW_ROOT_SHA, 'IndependentRawRootSHA256': current_raw_sha,
    'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes, 'Conditions': len(inventory),
    'ExpectedScoredFailingBuilds': EXPECTED_SCORED_FAILING_BUILDS,
    'ExpectedEvaluationBuilds': EXPECTED_EVALUATION_BUILDS,
    'ExpectedEvaluationFailures': EXPECTED_EVALUATION_FAILURES,
    'MLFits': len(model_fits), 'RankingRows': int(inventory['RankingRows'].sum()),
    'BuildMetricRows': len(build_metrics), 'ProjectRunRows': len(project_runs),
    'ConditionAuditRows': len(condition_audit), 'TrainingMedianRows': len(training_medians),
    'NoiseTechniqueSummaryRows': len(noise_summary), 'SeedLevelNoiseDeltaRows': len(seed_deltas),
    'NoiseDeltaSummaryRows': len(delta_summary),
    'StandardDeviationDefinition': 'Sample SD across 30 seeds; pandas std, ddof=1',
    'RankingLevelBaselineInvarianceFailures': int(ranking_baseline_fail),
    'ProjectMetricBaselineInvarianceFailures': int(metric_baseline_fail),
    'ProjectMetricBaselineMaximumWithinMetricRange': float(metric_baseline_max_range),
    'RawHashingSeconds': float(hash_seconds), 'ConditionRevalidationSeconds': float(condition_seconds),
    'AggregationSeconds': float(agg_seconds), 'OutputManifest': output_manifest,
    'ValidationChecks': len(validation), 'FailedValidationChecks': len(failed),
    'RegistrySHA256': registry_sha_before,
    'RegistryModified': False,
    'Projects1To16Modified': False,
    'Project16RegistryIdentity': required_registered_identities[16],
    'Project16ConditionOutputsAccessed': False,
    'Project16ConditionOutputsModified': False,
    'ActiveReservations': EXPECTED_ACTIVE_RESERVATIONS,
    'RuntimePriorityRule': EXPECTED_RUNTIME_PRIORITY_RULE,
    'PriorProjectConditionOutputsAccessed': False,
    'PriorProjectWriteAttempted': False,
    'ModelsFitted': False,
    'ConditionsRerun': False,
}
atomic_json(REPORT_PATH, report)
checkpoint = {**report, 'CheckpointVersion': 1,
              'CheckpointType': 'PROJECT_17_RAW_REVALIDATION_AND_COMPACT_AGGREGATION',
              'RawResultsRevalidated': True, 'CompactAggregatesFrozen': True,
              'ReadyForFinalPackageAndRegistration': True}
atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
status = {'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
          'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
          'RawRootSHA256': current_raw_sha, 'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes,
          'Conditions': len(inventory), 'MLFits': len(model_fits), 'Checkpoint': str(CHECKPOINT_PATH),
          'CheckpointSHA256': checkpoint_sha,
          'ReadyForFinalPackageAndRegistration': True,
          'RegistryModified': False,
          'Projects1To16Modified': False,
          'Project16ConditionOutputsAccessed': False,
          'Project16ConditionOutputsModified': False,
          'ActiveReservations': EXPECTED_ACTIVE_RESERVATIONS,
          'RuntimePriorityRule': EXPECTED_RUNTIME_PRIORITY_RULE,
          'PriorProjectConditionOutputsAccessed': False}
atomic_json(STATUS_PATH, status)

# Readback and immutability.
if load_json(CHECKPOINT_PATH).get('Status') != STEP5B_STATUS or load_json(STATUS_PATH).get('Status') != STEP5B_STATUS:
    raise RuntimeError('Step 5B checkpoint/status readback failed.')
for item in output_manifest:
    p = Path(item['Path'])
    if not p.is_file() or p.stat().st_size != item['Bytes'] or sha256_file(p) != item['SHA256']:
        raise RuntimeError(f'Step 5B output readback failed: {p}')
registry_sha_after = sha256_file(REGISTRY)
if registry_sha_after != registry_sha_before:
    raise RuntimeError('Registry changed during Project 17 Step 5B.')
if sha256_file(STEP5A_CHECKPOINT) != EXPECTED_STEP5A_SHA:
    raise RuntimeError('Step 5A checkpoint changed during Step 5B.')

print('\nNoise-technique summary:')
display(noise_summary)
print('\nNoise-delta summary:')
display(delta_summary)
print('\n' + '=' * 136)
print('=== PROJECT 17 CELL 10 / STEP 5B RESULT ===')
print('=' * 136)

print('Project:', PROJECT_NAME)
print('Project slug:', PROJECT_SLUG)
print('Step 5A checkpoint SHA-256:', step5a_sha)
print('Frozen raw-root SHA-256:', EXPECTED_RAW_ROOT_SHA)
print('Independent current raw-root SHA-256:', current_raw_sha)

print('\nRaw-output revalidation:')
print('Conditions:', len(inventory), '/', EXPECTED_CONDITIONS)
print('Raw files:', len(current_manifest), '/', EXPECTED_RAW_FILES)
print('Raw bytes:', current_raw_bytes, '/', EXPECTED_RAW_BYTES)
print(
    'Missing / unexpected / size / SHA mismatches:',
    missing_raw,
    '/',
    unexpected_raw,
    '/',
    size_mismatch,
    '/',
    hash_mismatch,
)
print('Embedded output-manifest failures:', embedded_fail)

print('\nExperiment totals:')
print('ML fits:', len(model_fits), '/', EXPECTED_TOTAL_FIT_ROWS)
print(
    'Ranking rows:',
    int(
        inventory[
            'RankingRows'
        ].sum()
    ),
    '/',
    EXPECTED_TOTAL_RANKING_ROWS,
)
print('Build-metric rows:', len(build_metrics), '/', EXPECTED_TOTAL_BUILD_ROWS)
print('Project-run rows:', len(project_runs), '/', EXPECTED_TOTAL_PROJECT_ROWS)
print('Condition-audit rows:', len(condition_audit), '/', EXPECTED_TOTAL_AUDIT_ROWS)
print('Training-median rows:', len(training_medians), '/', EXPECTED_TOTAL_MEDIAN_ROWS)

print('\nAnalysis-ready aggregates:')
print('Noise-technique summary rows:', len(noise_summary))
print('Seed-level noise-delta rows:', len(seed_deltas))
print('Noise-delta summary rows:', len(delta_summary))
print('Sample SD calculated with ddof=1:', True)
print('Ranking-level baseline-invariance failures:', ranking_baseline_fail)
print('Project-metric baseline-invariance failures:', metric_baseline_fail)
print('Maximum within-metric baseline range:', metric_baseline_max_range)

print('\nImmutability and isolation:')
print('Completion registry unchanged:', registry_sha_after == registry_sha_before)
print('Registry Project 11 rows:', int(pnums.eq(11).sum()))
print('Registry Project 12 rows:', int(pnums.eq(12).sum()))
print('Registry Project 13 rows:', int(pnums.eq(13).sum()))
print('Registry Project 14 rows:', int(pnums.eq(14).sum()))
print('Registry Project 15 rows:', int(pnums.eq(15).sum()))
print('Registry Project 16 rows:', int(pnums.eq(16).sum()))
print('Registry Project 17 rows:', int(pnums.eq(17).sum()))
print('Project 11 identity:', required_registered_identities[11])
print('Project 12 identity:', required_registered_identities[12])
print('Project 13 identity:', required_registered_identities[13])
print('Project 14 identity:', required_registered_identities[14])
print('Project 15 identity:', required_registered_identities[15])
print('Project 16 identity:', required_registered_identities[16])
print('Active reservations:', EXPECTED_ACTIVE_RESERVATIONS)
print('Runtime-priority rule:', EXPECTED_RUNTIME_PRIORITY_RULE)
print('Projects 1–16 modified:', 0)
print('Project 16 condition outputs accessed:', False)
print('Project 16 condition outputs modified:', False)
print('Prior project condition outputs accessed:', False)
print('Prior project write attempted:', False)
print('Conditions rerun:', False)
print('Models fitted:', False)

print('\nRuntime:')
print('Raw hashing seconds:', round(hash_seconds, 2))
print('Condition revalidation seconds:', round(condition_seconds, 2))
print('Compact aggregation seconds:', round(agg_seconds, 2))

print('\nValidation:')
print('Checks:', len(validation))
print('Failed checks:', len(failed))

print('\nProject 17 Step 5B checkpoint:')
print(CHECKPOINT_PATH)
print('Checkpoint SHA-256:', checkpoint_sha)

print('\nSTATUS:', STEP5B_STATUS)
print('=' * 136)


=== PROJECT 17 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Condition revalidatio

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_17_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_17_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653...,d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653...,True
2,Frozen raw-root SHA-256,b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a7...,b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a7...,True
3,Independent current raw-root SHA-256,b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a7...,b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a7...,True
4,Step 5A aggregate-manifest failures,0,0,True
...,...,...,...,...
65,Registry Project 15 rows,1,1,True
66,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
67,Registry Project 16 rows,1,1,True
68,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.652900,0.000000,0.652900,0.725899,0.000000,0.725899,0.346198,0.000000,0.346198,0.292187,0.000000,0.292187
1,0,LightGBM,30,30,0.635972,0.000000,0.635972,0.758041,0.000000,0.758041,0.843627,0.000000,0.843627,0.955444,0.000000,0.955444
2,0,NaiveBayes,30,30,0.486442,0.000000,0.486442,0.518601,0.000000,0.518601,0.753321,0.000000,0.753321,0.870085,0.000000,0.870085
3,0,QTF-Avg,30,30,0.476298,0.000000,0.476298,0.387841,0.000000,0.387841,0.184283,0.000000,0.184283,0.048913,0.000000,0.048913
4,0,Random,30,30,0.493134,0.075014,0.492417,0.492520,0.088711,0.500388,0.500145,0.065431,0.491245,0.510515,0.079857,0.506212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.520567,0.081817,0.526899,0.519285,0.102908,0.520543,0.535448,0.213061,0.600931,0.557565,0.304023,0.650677
59,50,QTF-Avg,30,30,0.476298,0.000000,0.476298,0.387841,0.000000,0.387841,0.184283,0.000000,0.184283,0.048913,0.000000,0.048913
60,50,Random,30,30,0.493134,0.075014,0.492417,0.492520,0.088711,0.500388,0.500145,0.065431,0.491245,0.510515,0.079857,0.506212
61,50,RandomForest,30,30,0.496692,0.086725,0.493001,0.474544,0.091874,0.471464,0.442688,0.093539,0.423114,0.434156,0.113864,0.421645



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,0.034126,0.081817,0.040457,0.000684,0.102908,0.001942,-0.217874,0.213061,-0.152390,-0.312520,0.304023,-0.219408
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.135398,0.090247,-0.145774,-0.276652,0.101858,-0.261137,-0.343271,0.095575,-0.358339,-0.524195,0.115543,-0.539628



=== PROJECT 17 CELL 10 / STEP 5B RESULT ===
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Step 5A checkpoint SHA-256: d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653ea152c344ab7727b5b
Frozen raw-root SHA-256: b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72
Independent current raw-root SHA-256: b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 38425581 / 38425581
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0
Embedded output-manifest failures: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 2043090 / 2043090
Build-metric rows: 18900 / 18900
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Sample SD calculated with ddof=1: True
Ranking-level baseline-invariance failures: 0
Project-m

In [23]:
# ==================================================================================================
# PROJECT 17 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   yamcs@Yamcs
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 17 Step 5A checkpoint SHA-256:
#   d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653ea152c344ab7727b5b
# - Project 17 Step 5B checkpoint SHA-256:
#   a49289356d33f801f115581f4b8778b1776910a46fc9d6b0efb90856a83da180
# - Project 17 raw-root SHA-256:
#   b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72
# - Registry before registration:
#   exactly Projects 1–16, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 17 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 17
PROJECT_NAME = "yamcs@Yamcs"
PROJECT_SLUG = "yamcs__Yamcs"
PROJECT_SHORT = "YAMCS"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_17_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_17_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_17_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "d0d3c32bc2374014357113905ed42282bc22f1d63888eb73ccac802f99bb188a"
STEP5A_SHA_EXPECTED = "d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653ea152c344ab7727b5b"
STEP5B_SHA_EXPECTED = "a49289356d33f801f115581f4b8778b1776910a46fc9d6b0efb90856a83da180"
SOURCE_ROOT_SHA = "64cf358897e6363470ffb08f3239dd46044867f510a094925a82cd3b53c239f2"
RAW_ROOT_SHA = "b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 38425581, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 2043090, "BuildMetricRows": 18900, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 504,
    "TrainingBuilds": 378, "EvaluationBuilds": 126, "RawRows": 58101,
    "RawTrainingRows": 42277, "RawEvaluationRows": 15824,
    "RawTrainingFailures": 147, "RawEvaluationFailures": 45, "ModelRows": 7533,
    "ModelTrainingRows": 6452, "ModelEvaluationRows": 1081,
    "ModelTrainingFailures": 145, "ModelEvaluationFailures": 45,
    "ModelFailingEvaluationBuilds": 10, "Predictors": 151, "RECFeatures": 19,
}
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_17_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_17.csv"

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)
STEP5A_CP = NOTES / "project_17_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_17_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_17_selection_checkpoint.json",
    NOTES / "project_17_rec_reconstruction_checkpoint.json",
    NOTES / "project_17_noise_plan_checkpoint.json",
    NOTES / "project_17_runtime_contract_checkpoint.json",
    NOTES / "project_17_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 17 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "Project16ConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to Project 16 condition outputs."
    )

if bool(
    step5b.get(
        "Project16ConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of Project 16 condition outputs."
    )

step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must contain exactly completed Projects 1–16, with Project 17 absent.
registry_sha_before = sha(REGISTRY)

if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(
        "Registry SHA differs before Project 17 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna("")

pn_col = resolve(reg_before.columns, "ProjectNumber")
project_col = resolve(reg_before.columns, "Project")
status_col = resolve(reg_before.columns, "Status")

pnums = pd.to_numeric(
    reg_before[pn_col],
    errors="raise",
).astype(int)

if len(reg_before) != 16 or sorted(pnums.tolist()) != list(range(1, 17)):
    raise RuntimeError("Registry must contain exactly Projects 1–16.")

if not reg_before[status_col].eq(COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–16 are not all COMPLETE_AND_FROZEN.")

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = reg_before.loc[pnums.eq(required_number)]
    if len(matching_rows) != 1 or matching_rows.iloc[0][project_col] != required_project:
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if pnums.eq(PROJECT_NUMBER).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Project 17 is already present in the completion registry.")

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-17 registry backup does not match the live registry."
    )

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

# Use the frozen Step 5B completion timestamp so the package is deterministic
# across safe reruns.
created_at = str(
    step5b.get(
        "CompletedAtUTC",
        ""
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project17_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 17 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
        "Project16ConditionOutputsAccessed": False,
        "Project16ConditionOutputsModified": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 17 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 17 final package already exists and "
                "was not modified."
            )

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems. A direct
        # os.replace(tmp_root, FINAL_ROOT) therefore raises EXDEV. Publish in
        # two stages:
        #   1. copy the completed local package to a sibling staging directory
        #      on Google Drive;
        #   2. verify every staged payload file;
        #   3. rename the staging directory to FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root,
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + comparison.loc[
                    (
                        comparison["_merge"].ne(
                            "both"
                        )
                        | comparison[
                            "Bytes_candidate"
                        ].ne(
                            comparison[
                                "Bytes_staged"
                            ]
                        )
                        | comparison[
                            "SHA256_candidate"
                        ].ne(
                            comparison[
                                "SHA256_staged"
                            ]
                        )
                    )
                ].head(
                    20
                ).to_string(
                    index=False
                )
            )

        # This rename is within the Google Drive filesystem, so it does not
        # cross a device boundary.
        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 17 package failed readback validation.")

# Build a complete Project 17 registry row.
#
# The registry has evolved across Projects 1–16. Some columns are protocol
# descriptors, some are project-specific counts, and some are paths to frozen
# audit artefacts. V2 deliberately stopped because it did not map every
# variable column. V3 handles the complete observed schema explicitly.
#
# For protocol fields whose textual formatting has varied historically
# (Seeds, NoiseLevels, Techniques, DoNotRerun), use the exact frozen
# Project 16 representation. Project 17 uses the same protocol.
project_16_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        16
    )
]

if len(
    project_16_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 16 registry template row."
    )

project_16_template = project_16_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_16_template[
                registry_column
            ]
        ).strip()


protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}


for protocol_key, fallback_value in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        ""
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds": protocol_template_values["seeds"],
    "noiselevels": protocol_template_values["noiselevels"],
    "techniques": protocol_template_values["techniques"],
    "evaluationrows": COUNTS["ModelEvaluationRows"],
    "evaluationfailures": COUNTS["ModelEvaluationFailures"],
    "finaldirectory": str(FINAL_ROOT),
    "finalauditreport": str(REPORT_PATH),
    "donotrerun": protocol_template_values["donotrerun"],
    "freezerecord": str(CHECKPOINT_PATH),
    "rawresultsmanifest": str(STEP5B_RAW_MANIFEST_PATH),
    "finalpackagemanifest": str(MANIFEST_PATH),
    "rawresultsrootsha256": RAW_ROOT_SHA,
    "finalauditstatus": STEP5C_STATUS,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )


new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; no registry write "
        "was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [reg_before, pd.DataFrame([new_row])],
    ignore_index=True,
)
reg_candidate[pn_col] = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int).astype(str)
candidate_nums = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int)

project17_candidate = reg_candidate.loc[candidate_nums.eq(17)]

if len(reg_candidate) != 17 or sorted(candidate_nums.tolist()) != list(range(1, 18)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–17.")

if (
    not reg_candidate[status_col].eq(COMPLETE_STATUS).all()
    or len(project17_candidate) != 1
    or project17_candidate.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError("Candidate Project 17 registry row failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 16, len(reg_before), len(reg_before) == 16)
check(rows, "Registry rows candidate", 17, len(reg_candidate), len(reg_candidate) == 17)
check(rows, "Candidate Project 17 rows", 1, len(project17_candidate), len(project17_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}


registry_field_validation_failures = 0

for expected_column_name, expected_value in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project17_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )


check(
    rows,
    "Explicit Project 17 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(rows)
print("\nProject 17 Step 5C pre-write validation:"); display(pre)
print("\nProject 17 registry row candidate:"); display(project17_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 17 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project17_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if (
    len(tmp_read) != 17
    or sorted(tmp_nums.tolist()) != list(range(1, 18))
    or not tmp_read[status_col].eq(COMPLETE_STATUS).all()
    or int(tmp_nums.eq(17).sum()) != 1
):
    tmp_reg.unlink(missing_ok=True)
    raise RuntimeError("Temporary Project 17 registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
project12_after = reg_after.loc[after_nums.eq(12)]
project13_after = reg_after.loc[after_nums.eq(13)]
project14_after = reg_after.loc[after_nums.eq(14)]
project15_after = reg_after.loc[after_nums.eq(15)]
project16_after = reg_after.loc[after_nums.eq(16)]
project17_after = reg_after.loc[after_nums.eq(17)]
registry_sha_after = sha(REGISTRY)

if (
    len(reg_after) != 17
    or sorted(after_nums.tolist()) != list(range(1, 18))
    or not reg_after[status_col].eq(COMPLETE_STATUS).all()
    or len(project11_after) != 1
    or project11_after.iloc[0][project_col] != "apache@shardingsphere"
    or len(project12_after) != 1
    or project12_after.iloc[0][project_col] != "zolyfarkas@spf4j"
    or len(project13_after) != 1
    or project13_after.iloc[0][project_col] != "jcabi@jcabi-github"
    or len(project14_after) != 1
    or project14_after.iloc[0][project_col] != "JMRI@JMRI"
    or len(project15_after) != 1
    or project15_after.iloc[0][project_col] != "eclipse@steady"
    or len(project16_after) != 1
    or project16_after.iloc[0][project_col] != "apache@rocketmq"
    or len(project17_after) != 1
    or project17_after.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed Project 17 post-write validation. "
        f"Backup: {BACKUP_PATH}"
    )

check(rows, "Registry rows after", 17, len(reg_after), len(reg_after) == 17)
check(rows, "COMPLETE_AND_FROZEN projects after", 17, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), int(reg_after[status_col].eq(COMPLETE_STATUS).sum()) == 17)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry Project 12 rows after", 1, len(project12_after), len(project12_after) == 1)
check(rows, "Registry Project 13 rows after", 1, len(project13_after), len(project13_after) == 1)
check(rows, "Registry Project 14 rows after", 1, len(project14_after), len(project14_after) == 1)
check(rows, "Registry Project 15 rows after", 1, len(project15_after), len(project15_after) == 1)
check(rows, "Registry Project 16 rows after", 1, len(project16_after), len(project16_after) == 1)
check(rows, "Registry Project 17 rows after", 1, len(project17_after), len(project17_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed)
    raise RuntimeError("PROJECT 17 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after),
    "Project12RegistryRowsAfter": len(project12_after),
    "Project13RegistryRowsAfter": len(project13_after),
    "Project14RegistryRowsAfter": len(project14_after),
    "Project15RegistryRowsAfter": len(project15_after),
    "Project16RegistryRowsAfter": len(project16_after),
    "Project17RegistryRowsAfter": len(project17_after),
    "Project16ConditionOutputsAccessed": False,
    "Project16ConditionOutputsModified": False,
    "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectWriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "CheckpointType": "PROJECT_17_FINAL_PACKAGE_AND_REGISTRY", "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha,
    "ProjectCompleteAndFrozen": True,
    "Project16ConditionOutputsAccessed": False,
    "Project16ConditionOutputsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 17 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Project 11 identity:", required_registered_identities[11])
print("Project 12 identity:", required_registered_identities[12])
print("Project 13 identity:", required_registered_identities[13])
print("Project 14 identity:", required_registered_identities[14])
print("Project 15 identity:", required_registered_identities[15])
print("Project 16 identity:", required_registered_identities[16])
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Project 12 registry rows:", len(project12_after))
print("Project 13 registry rows:", len(project13_after))
print("Project 14 registry rows:", len(project14_after))
print("Project 15 registry rows:", len(project15_after))
print("Project 16 registry rows:", len(project16_after))
print("Project 17 registry rows:", len(project17_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 16 condition outputs accessed:", False)
print("Project 16 condition outputs modified:", False)
print("Prior project condition outputs accessed:", False)
print("Prior project write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 17 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print("Explicit registry-schema fields validated:", len(required_registry_field_expectations))
print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)


=== PROJECT 17 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project 17 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653...,d559c781a1655e0a2ea39d3c37e2e1e7e4b19f37f62653...,True
1,Step 5B checkpoint SHA-256,a49289356d33f801f115581f4b8778b1776910a46fc9d6...,a49289356d33f801f115581f4b8778b1776910a46fc9d6...,True
2,Step 5A manifest failures,0,0,True
3,Step 5B manifest failures,0,0,True
4,Package missing files,0,0,True
5,Package unexpected files,0,0,True
6,Package size mismatches,0,0,True
7,Package SHA-256 mismatches,0,0,True
8,Registry rows before,16,16,True
9,Registry rows candidate,17,17,True



Project 17 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
16,17,yamcs@Yamcs,yamcs__Yamcs,COMPLETE_AND_FROZEN,270,30,9,7,126,1081,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a7...,95e8b8847dbb01a6ea23b3ff3dca9217c3fd68808ce3be...,1080,5358150.0,PASS_PROJECT_17_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 17 CELL 11 / STEP 5C RESULT ===
Project number: 17
Project: yamcs@Yamcs
Project slug: yamcs__Yamcs
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 38425581
Raw root SHA-256: b6987da63d1dddeaee1ba4c6f17a3a9ac6ee3bea8d47a729b502a387313a8e72

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/yamcs__Yamcs
Package files: 32
Package bytes: 10635933
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: 95e8b8847dbb01a6ea23b3ff3dca9217c3fd68808ce3bed8dbc2ee2189c9b9a5

Completion registry:
Registry rows: 17
COMPLETE_AND_FROZEN projects: 17
Project 11 registry rows: 1
Project 12 registry rows: 1
Project 13 regi

In [24]:
# ==================================================================================================
# PROJECT 18 — CELL 1 / STEP 0
# SAME-NOTEBOOK POST-PROJECT-17 BOOTSTRAP AND CANDIDATE DISCOVERY
#
# RUN THIS AS THE NEXT NEW CELL IN THE EXISTING:
#   Thesis_project_16.ipynb
#
# PROJECT 17 IS COMPLETE_AND_FROZEN AND MUST NOT BE RERUN.
#
# SAFETY:
# - validates the frozen 17-project completion registry and Project 17 completion checkpoint;
# - reads but never modifies the completion registry;
# - writes only Project 18 bootstrap/selection files;
# - never reads or modifies any prior-project condition-output files;
# - does not inject noise, reconstruct REC features, fit models, or start an experiment;
# - prepares the eight remaining projects for runtime-prioritized selection in Step 1A.
# ==================================================================================================

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil
import tarfile

import pandas as pd


print("=" * 136)
print("=== PROJECT 18 CELL 1 / STEP 0: SAME-NOTEBOOK POST-PROJECT-17 BOOTSTRAP ===")
print("=" * 136)


PROJECT_NUMBER = 18

STEP0_STATUS = (
    "PASS_PROJECT_18_SAME_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_REGISTERED_PROJECTS = 17
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

ACTIVE_RESERVED_PROJECTS = {}

EXPECTED_CANDIDATES = 8

REQUIRED_PROJECT_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
}

RUNTIME_PRIORITY_POLICY = {
    "purpose":
        "processing order only; protocol eligibility and final project set are unchanged",
    "primary":
        "ModelTrainingRows ascending",
    "secondary":
        "ModelEvaluationRows ascending",
    "tertiary":
        "RawExecutionRows ascending",
    "final_tie_break":
        "Project ascending",
    "scientific_effect":
        "none when all protocol-eligible projects are completed",
}


drive.mount(
    "/content/drive",
    force_remount=False,
)

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

PROJECT_17_STEP5C_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_17_step5c_checkpoint.json"
)

EXPECTED_PROJECT_17_STEP5C_SHA256 = (
    "60fc763dcdfe8e957a8aa62fb96eab10c7b94b0bf8e410974466004f9911fb72"
)

LOCAL_EXTRACTION_ROOT = Path(
    "/content/datasets"
)

LOCAL_DATASET_ROOT = (
    LOCAL_EXTRACTION_ROOT
    / "datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_18_selection"
)

BOOTSTRAP_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_18_bootstrap_candidate_inventory.csv"
)

BOOTSTRAP_REPORT_PATH = (
    SELECTION_ROOT
    / "project_18_step0_report.json"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step0_status.json"
)


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def resolve_column(
    columns,
    *candidates,
):
    normalized = {
        str(column).strip().lower():
            column
        for column in columns
    }

    for candidate in candidates:
        key = str(
            candidate
        ).strip().lower()

        if key in normalized:
            return normalized[
                key
            ]

    raise RuntimeError(
        "Could not resolve any of these columns: "
        + ", ".join(
            candidates
        )
    )


def atomic_write_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_json(
    path,
    payload,
):
    atomic_write_text(
        path,
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
    )


def atomic_write_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def extract_archive_safely(
    archive_path,
    extraction_root,
):
    extraction_root = Path(
        extraction_root
    )

    extraction_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolved_root = extraction_root.resolve()
    extracted_files = 0

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace(
                    "\\",
                    "/",
                )
                .lstrip(
                    "/"
                )
            )

            target_path = (
                extraction_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target
                != resolved_root
                and resolved_root
                not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open(
                        "wb"
                    ) as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    return extracted_files


required_drive_paths = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    PROJECT_17_STEP5C_CHECKPOINT_PATH,
]

missing_drive_paths = [
    str(
        path
    )
    for path in required_drive_paths
    if not path.is_file()
]

if missing_drive_paths:
    raise FileNotFoundError(
        "Required Project 18 bootstrap inputs are missing:\n"
        + "\n".join(
            missing_drive_paths
        )
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Frozen TCP-CI archive SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


project_17_step5c_sha256 = sha256_file(
    PROJECT_17_STEP5C_CHECKPOINT_PATH
)

if project_17_step5c_sha256 != EXPECTED_PROJECT_17_STEP5C_SHA256:
    raise RuntimeError(
        "Project 17 completion checkpoint SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_PROJECT_17_STEP5C_SHA256}\n"
        f"Actual:   {project_17_step5c_sha256}"
    )

project_17_step5c_checkpoint = json.loads(
    PROJECT_17_STEP5C_CHECKPOINT_PATH.read_text(encoding="utf-8")
)

if project_17_step5c_checkpoint.get("Status") != (
    "PASS_PROJECT_17_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
):
    raise RuntimeError(
        "Project 17 completion checkpoint is not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs from the frozen Projects 1–17 state.\n"
        "Do not continue Project 18 until the unexpected registry change is investigated.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "Project Number",
    "Project_Number",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly frozen Projects 1–17."
    )


if not registry[
    registry_status_column
].astype(
    str
).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Not every registered predecessor is COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(
        str
    )
)


if ACTIVE_RESERVED_PROJECTS:
    raise RuntimeError(
        "Project 18 bootstrap expects no active project reservations."
    )


EXPECTED_PREDECESSOR_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}

for predecessor_number, expected_project in EXPECTED_PREDECESSOR_IDENTITIES.items():
    matches = registry.loc[
        registry_project_numbers.eq(predecessor_number),
        registry_project_column,
    ].astype(str).tolist()

    if matches != [expected_project]:
        raise RuntimeError(
            f"Frozen Project {predecessor_number} identity mismatch.\n"
            f"Expected: {expected_project}\n"
            f"Actual:   {matches}"
        )


def local_dataset_looks_complete():
    if not LOCAL_DATASET_ROOT.is_dir():
        return False

    project_directories = [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ]

    return bool(
        len(
            project_directories
        )
        == 25
    )


if local_dataset_looks_complete():
    extraction_performed = False
    extracted_files = 0

    print(
        "\nA complete-looking local TCP-CI dataset is already present."
    )

else:
    extraction_performed = True

    print(
        "\nRestoring the frozen TCP-CI archive into the Project 18 runtime."
    )

    if LOCAL_EXTRACTION_ROOT.exists():
        shutil.rmtree(
            LOCAL_EXTRACTION_ROOT
        )

    extracted_files = extract_archive_safely(
        ARCHIVE_PATH,
        LOCAL_EXTRACTION_ROOT,
    )


if not LOCAL_DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Archive extraction did not create the expected dataset root:\n"
        f"{LOCAL_DATASET_ROOT}"
    )


all_project_directories = sorted(
    [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path:
        path.name,
)


if len(all_project_directories) != 25:
    raise RuntimeError(
        "Unexpected number of TCP-CI project directories.\n"
        f"Expected: 25\n"
        f"Actual:   {len(all_project_directories)}"
    )


reserved_projects = set(
    ACTIVE_RESERVED_PROJECTS.values()
)

candidate_rows = []

for source_directory in all_project_directories:
    project = source_directory.name

    source_files = {
        path.name
        for path in source_directory.iterdir()
        if path.is_file()
    }

    missing_required_files = sorted(
        REQUIRED_PROJECT_FILES
        - source_files
    )

    excluded_registered = (
        project in registered_projects
    )

    excluded_reserved = (
        project in reserved_projects
    )

    candidate_eligible_for_scan = (
        not excluded_registered
        and not excluded_reserved
        and not missing_required_files
    )

    candidate_rows.append({
        "Project":
            project,
        "ProjectSlug":
            project.replace(
                "@",
                "__",
            ),
        "SourceDirectory":
            str(
                source_directory
            ),
        "ExcludedRegistered":
            bool(
                excluded_registered
            ),
        "ExcludedReserved":
            bool(
                excluded_reserved
            ),
        "MissingRequiredFiles":
            "; ".join(
                missing_required_files
            ),
        "CandidateForProject18Scan":
            bool(
                candidate_eligible_for_scan
            ),
    })


inventory = pd.DataFrame(
    candidate_rows
)


project_18_candidates = (
    inventory.loc[
        inventory[
            "CandidateForProject18Scan"
        ]
    ]
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    project_18_candidates
) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected number of Project 18 candidates after excluding "
        "frozen Projects 1–17.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(project_18_candidates)}"
    )


if (
    project_18_candidates[
        "Project"
    ].isin(
        registered_projects
        | reserved_projects
    ).any()
):
    raise RuntimeError(
        "A registered identity leaked into the Project 18 candidate set."
    )


SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    BOOTSTRAP_INVENTORY_PATH,
    project_18_candidates,
)


created_at_utc = datetime.now(
    timezone.utc
).isoformat()


report = {
    "ProjectNumber":
        PROJECT_NUMBER,
    "Status":
        STEP0_STATUS,
    "CreatedAtUTC":
        created_at_utc,
    "ArchivePath":
        str(
            ARCHIVE_PATH
        ),
    "ArchiveSHA256":
        archive_sha256,
    "RegistryPath":
        str(
            REGISTRY_PATH
        ),
    "RegistrySHA256":
        registry_sha256_before,
    "Project17Step5CCheckpoint":
        str(
            PROJECT_17_STEP5C_CHECKPOINT_PATH
        ),
    "Project17Step5CCheckpointSHA256":
        project_17_step5c_sha256,
    "RegisteredProjects":
        EXPECTED_REGISTERED_PROJECTS,
    "RegisteredStatuses":
        sorted(
            registry[
                registry_status_column
            ].astype(
                str
            ).unique().tolist()
        ),
    "ActiveReservations":
        {
            str(
                key
            ):
                value
            for key, value in ACTIVE_RESERVED_PROJECTS.items()
        },
    "FrozenPredecessorIdentities":
        {
            str(key): value
            for key, value in EXPECTED_PREDECESSOR_IDENTITIES.items()
        },
    "DatasetRoot":
        str(
            LOCAL_DATASET_ROOT
        ),
    "SourceProjectDirectories":
        len(
            all_project_directories
        ),
    "Project18CandidateCount":
        len(
            project_18_candidates
        ),
    "CandidateInventory":
        str(
            BOOTSTRAP_INVENTORY_PATH
        ),
    "RuntimePriorityPolicy":
        RUNTIME_PRIORITY_POLICY,
    "ExtractionPerformed":
        bool(
            extraction_performed
        ),
    "ArchiveFilesExtracted":
        int(
            extracted_files
        ),
    "RegistryModified":
        False,
    "PriorProjectConditionOutputsAccessed":
        False,
    "PriorProjectConditionOutputsModified":
        False,
    "NoiseInjected":
        False,
    "ModelsFitted":
        False,
}


atomic_write_json(
    BOOTSTRAP_REPORT_PATH,
    report,
)

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,
        "Status":
            STEP0_STATUS,
        "CreatedAtUTC":
            created_at_utc,
        "Report":
            str(
                BOOTSTRAP_REPORT_PATH
            ),
    },
)


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during the Project 18 bootstrap."
    )


print("\nProject 18 candidates after excluding registered identities:")
print(
    project_18_candidates[
        [
            "Project",
            "ProjectSlug",
            "SourceDirectory",
        ]
    ].to_string(
        index=False
    )
)


print("\n")
print("=" * 136)
print("=== PROJECT 18 CELL 1 / STEP 0 RESULT ===")
print("=" * 136)

print(
    "Registered and frozen projects:",
    EXPECTED_REGISTERED_PROJECTS,
)

print(
    "Active reservations:",
    [],
)

print(
    "TCP-CI source directories:",
    len(
        all_project_directories
    ),
)

print(
    "Project 18 candidates:",
    len(
        project_18_candidates
    ),
)

print(
    "Runtime-priority policy:",
    RUNTIME_PRIORITY_POLICY,
)

print(
    "Candidate inventory:",
    BOOTSTRAP_INVENTORY_PATH,
)

print(
    "Completion registry modified:",
    False,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "\nSTATUS:",
    STEP0_STATUS,
)

print("=" * 136)


=== PROJECT 18 CELL 1 / STEP 0: SAME-NOTEBOOK POST-PROJECT-17 BOOTSTRAP ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

A complete-looking local TCP-CI dataset is already present.

Project 18 candidates after excluding registered identities:
                      Project                    ProjectSlug                                          SourceDirectory
         EMResearch@EvoMaster          EMResearch__EvoMaster          /content/datasets/datasets/EMResearch@EvoMaster
     Graylog2@graylog2-server      Graylog2__graylog2-server      /content/datasets/datasets/Graylog2@graylog2-server
        SonarSource@sonarqube         SonarSource__sonarqube         /content/datasets/datasets/SonarSource@sonarqube
               apache@curator                apache__curator                /content/datasets/datasets/apache@curator
        apache@logging-log4j2         apache__logging-log4j2         /content/data

In [25]:
# ==================================================================================================
# PROJECT 18 — CELL 2 / STEP 1A
# ROBUST CANDIDATE DISCOVERY, PROTOCOL ELIGIBILITY, RUNTIME-PRIORITIZED RANKING,
# AND PROVISIONAL PROJECT 17 SELECTION
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# THIS CELL:
# - inspects all 8 candidates frozen by Project 18 Step 0;
# - validates the chronological 75/25 split and raw/model cohort viability;
# - deterministically ranks eligible candidates by estimated experiment cost (smallest first);
# - changes processing order only, not protocol eligibility or the intended final project set;
# - freezes only a provisional Project 18 selection for Step 1B;
# - does not run experiment conditions or fit models;
# - does not modify the completion registry or Projects 1–17;
# - writes only Project 18 selection artifacts;
# - does not access prior-project condition outputs.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 18 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_18_SAME_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_18_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 17
EXPECTED_CANDIDATES = 8

RESERVED_ACTIVE_PROJECTS = set()

RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

# These three files are sufficient for deterministic selection.
# id_map.csv and entity_change_history.csv are checked and frozen later in Step 1B/2A.
REQUIRED_SELECTION_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_SOURCE_ROOT = Path(
    "/content/datasets/datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_18_selection"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step0_status.json"
)

BOOTSTRAP_CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_18_bootstrap_candidate_inventory.csv"
)

SCAN_PROGRESS_PATH = (
    SELECTION_ROOT
    / "project_18_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_18_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_18_candidate_inventory.csv"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_18_eligible_candidates_ranked.csv"
)

INELIGIBLE_PATH = (
    SELECTION_ROOT
    / "project_18_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    SELECTION_ROOT
    / "project_18_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_18_provisional_selection.json"
)

STEP1A_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_18_step1a_validation.csv"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_18_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
        1,
    )


def count_partitioned_rows(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0
    verdict_values = set()

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk_build = parse_integer_series(
            chunk[build_column],
            f"{label}.{build_column}",
        )

        chunk_verdict = parse_integer_series(
            chunk[verdict_column],
            f"{label}.{verdict_column}",
        )

        training_mask = chunk_build.isin(
            training_build_ids
        )

        evaluation_mask = chunk_build.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = chunk_verdict.ne(0)

        total_rows += len(chunk)

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            chunk_build.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            chunk_build.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (~linked_mask).sum()
        )

        verdict_values.update(
            int(value)
            for value in chunk_verdict.unique().tolist()
        )

    return {
        "Rows":
            int(total_rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(len(failing_training_builds)),

        "FailingEvaluationBuilds":
            int(len(failing_evaluation_builds)),

        "UnlinkedRows":
            int(unlinked_rows),

        "VerdictValuesJSON":
            json.dumps(
                sorted(verdict_values)
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


def reusable_scan_row_is_valid(
    row,
):
    required_fields = [
        "Project",
        "ProjectSlug",
        "SourceDirectory",
        "InspectionStatus",
        "InspectionError",
        "BuildIDColumn",
        "StartedAtColumn",
        "ExecutionBuildColumn",
        "ExecutionVerdictColumn",
        "DatasetBuildColumn",
        "DatasetVerdictColumn",
        "Builds",
        "TrainingBuilds",
        "EvaluationBuilds",
        "RawExecutionRows",
        "RawTrainingRows",
        "RawEvaluationRows",
        "RawTrainFailures",
        "RawEvaluationFailures",
        "RawFailingTrainingBuilds",
        "RawFailingEvaluationBuilds",
        "RawUnlinkedRows",
        "ModelReadyRows",
        "ModelTrainingRows",
        "ModelEvaluationRows",
        "ModelTrainFailures",
        "ModelEvaluationFailures",
        "ModelFailingTrainingBuilds",
        "ModelFailingEvaluationBuilds",
        "ModelUnlinkedRows",
    ]

    if any(
        field not in row
        for field in required_fields
    ):
        return False

    status = str(
        row.get(
            "InspectionStatus",
            "",
        )
    ).strip()

    error = str(
        row.get(
            "InspectionError",
            "",
        )
    ).strip().lower()

    return (
        status in {
            "ELIGIBLE",
            "INELIGIBLE",
        }
        and error in {
            "",
            "nan",
            "none",
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 0, REGISTRY, ARCHIVE, AND LOCAL SOURCE
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 18 Step 1A V2 inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not LOCAL_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        "The local Project 18 dataset source is missing:\n"
        f"{LOCAL_SOURCE_ROOT}"
    )


bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

if bootstrap_status.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 18 Step 0 is not in the expected PASS state."
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 18))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(str).tolist()
)


if registered_projects & RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "A reserved active-project identity is unexpectedly present in the completion registry."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


bootstrap_candidates = pd.read_csv(
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)


candidate_project_column = resolve_column(
    bootstrap_candidates.columns,
    "Project",
    "bootstrap candidate Project",
)

candidate_source_column = resolve_column(
    bootstrap_candidates.columns,
    "SourceDirectory",
    "bootstrap candidate SourceDirectory",
)


candidate_records = (
    bootstrap_candidates[
        [
            candidate_project_column,
            candidate_source_column,
        ]
    ]
    .rename(
        columns={
            candidate_project_column:
                "Project",

            candidate_source_column:
                "SourceDirectory",
        }
    )
    .copy()
)


candidate_records[
    "Project"
] = candidate_records[
    "Project"
].astype(str)


candidate_records[
    "SourceDirectory"
] = candidate_records[
    "SourceDirectory"
].astype(str)


if len(candidate_records) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 18 candidate count.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(candidate_records)}"
    )


if candidate_records[
    "Project"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Project 18 bootstrap candidate inventory contains duplicates."
    )


forbidden_candidates = (
    set(
        candidate_records[
            "Project"
        ]
    )
    & (
        registered_projects
        | RESERVED_ACTIVE_PROJECTS
    )
)


if forbidden_candidates:
    raise RuntimeError(
        "Project 18 inventory contains registered/reserved projects:\n"
        + "\n".join(
            sorted(forbidden_candidates)
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REUSE ANY VALID COMPLETED PROJECT 17 SCANS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


reusable_rows = {}


if SCAN_PROGRESS_PATH.is_file():
    try:
        previous_progress = pd.read_csv(
            SCAN_PROGRESS_PATH,
            low_memory=False,
        )

        valid_candidate_names = set(
            candidate_records[
                "Project"
            ]
        )

        for row in previous_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            if (
                project in valid_candidate_names
                and reusable_scan_row_is_valid(
                    row
                )
            ):
                reusable_rows[
                    project
                ] = row

        print(
            "\nReusable completed candidate scans:",
            len(reusable_rows),
        )

    except Exception as error:
        print(
            "\nPrevious scan progress was ignored:",
            type(error).__name__,
            str(error),
        )


# --------------------------------------------------------------------------------------------------
# 6. INSPECT ALL 11 CANDIDATES
# --------------------------------------------------------------------------------------------------

scan_rows = []


for candidate_index, candidate in enumerate(
    candidate_records.itertuples(
        index=False
    ),
    start=1,
):
    project = str(
        candidate.Project
    )

    source_directory = Path(
        candidate.SourceDirectory
    )

    print("-" * 132)
    print(
        f"[{candidate_index:02d}/{EXPECTED_CANDIDATES:02d}] "
        f"Inspecting: {project}"
    )


    if project in reusable_rows:
        row = dict(
            reusable_rows[
                project
            ]
        )

        row[
            "CandidateInspectionOrder"
        ] = candidate_index

        row[
            "ProtocolEligible"
        ] = (
            str(
                row[
                    "InspectionStatus"
                ]
            )
            == "ELIGIBLE"
        )

        row[
            "InspectionError"
        ] = ""

        scan_rows.append(
            row
        )

        print(
            "    Reused:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            int(
                row[
                    "Builds"
                ]
            ),
            "| Model eval failures:",
            int(
                row[
                    "ModelEvaluationFailures"
                ]
            ),
        )

        continue


    started = time.perf_counter()

    row = {
        "CandidateInspectionOrder":
            candidate_index,

        "Project":
            project,

        "ProjectSlug":
            project_slug(
                project
            ),

        "SourceDirectory":
            str(
                source_directory
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",
    }


    try:
        missing_files = [
            filename
            for filename in REQUIRED_SELECTION_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            raise FileNotFoundError(
                "Missing selection files: "
                + ", ".join(
                    missing_files
                )
            )


        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )


        build_columns = pd.read_csv(
            builds_path,
            nrows=0,
        ).columns.tolist()

        exe_columns = pd.read_csv(
            exe_path,
            nrows=0,
        ).columns.tolist()

        dataset_columns = pd.read_csv(
            dataset_path,
            nrows=0,
        ).columns.tolist()


        build_id_column = resolve_column(
            build_columns,
            "id",
            f"{project} builds.csv ID",
        )

        started_at_column = resolve_column(
            build_columns,
            "started_at",
            f"{project} builds.csv started_at",
        )

        execution_build_column = resolve_column(
            exe_columns,
            "build",
            f"{project} exe.csv build",
        )

        execution_verdict_column = resolve_column(
            exe_columns,
            "verdict",
            f"{project} exe.csv verdict",
        )

        dataset_build_column = resolve_column(
            dataset_columns,
            "Build",
            f"{project} dataset.csv Build",
        )

        dataset_verdict_column = resolve_column(
            dataset_columns,
            "Verdict",
            f"{project} dataset.csv Verdict",
        )


        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )


        builds[
            build_id_column
        ] = parse_integer_series(
            builds[
                build_id_column
            ],
            f"{project}.builds.id",
        )


        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )


        invalid_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )


        duplicate_build_id_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )


        if invalid_timestamps != 0:
            raise RuntimeError(
                f"Invalid build timestamps: {invalid_timestamps}"
            )


        if duplicate_build_id_rows != 0:
            raise RuntimeError(
                f"Duplicate build-ID rows: {duplicate_build_id_rows}"
            )


        ordered_builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        number_of_builds = len(
            ordered_builds
        )


        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )


        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )


        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "Chronological 75/25 split has an empty partition."
            )


        training_build_ids = set(
            ordered_builds.iloc[
                :training_build_count
            ][
                build_id_column
            ].astype(int).tolist()
        )


        evaluation_build_ids = set(
            ordered_builds.iloc[
                training_build_count:
            ][
                build_id_column
            ].astype(int).tolist()
        )


        if training_build_ids & evaluation_build_ids:
            raise RuntimeError(
                "Training/evaluation build partitions overlap."
            )


        raw_profile = count_partitioned_rows(
            csv_path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.exe",
            chunksize=500_000,
        )


        model_profile = count_partitioned_rows(
            csv_path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.dataset",
            chunksize=250_000,
        )


        eligibility_reasons = []


        eligibility_tests = [
            (
                raw_profile[
                    "TrainingRows"
                ] > 0,
                "No raw training rows",
            ),

            (
                raw_profile[
                    "EvaluationRows"
                ] > 0,
                "No raw evaluation rows",
            ),

            (
                raw_profile[
                    "TrainingFailures"
                ] > 0,
                "No raw training failures",
            ),

            (
                raw_profile[
                    "EvaluationFailures"
                ] > 0,
                "No raw evaluation failures",
            ),

            (
                model_profile[
                    "TrainingRows"
                ] > 0,
                "No model training rows",
            ),

            (
                model_profile[
                    "EvaluationRows"
                ] > 0,
                "No model evaluation rows",
            ),

            (
                model_profile[
                    "TrainingFailures"
                ] > 0,
                "No model training failures",
            ),

            (
                model_profile[
                    "EvaluationFailures"
                ] > 0,
                "No model evaluation failures",
            ),

            (
                raw_profile[
                    "UnlinkedRows"
                ] == 0,
                "Raw rows reference unknown builds",
            ),

            (
                model_profile[
                    "UnlinkedRows"
                ] == 0,
                "Model rows reference unknown builds",
            ),
        ]


        for passed, failure_reason in eligibility_tests:
            if not passed:
                eligibility_reasons.append(
                    failure_reason
                )


        protocol_eligible = (
            len(
                eligibility_reasons
            )
            == 0
        )


        row.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "RawExecutionRows":
                raw_profile[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_profile[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_profile[
                    "EvaluationRows"
                ],

            "RawTrainFailures":
                raw_profile[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_profile[
                    "EvaluationFailures"
                ],

            "RawFailingTrainingBuilds":
                raw_profile[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_profile[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_profile[
                    "UnlinkedRows"
                ],

            "RawVerdictValuesJSON":
                raw_profile[
                    "VerdictValuesJSON"
                ],

            "ModelReadyRows":
                model_profile[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_profile[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_profile[
                    "EvaluationRows"
                ],

            "ModelTrainFailures":
                model_profile[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_profile[
                    "EvaluationFailures"
                ],

            "ModelFailingTrainingBuilds":
                model_profile[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_profile[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_profile[
                    "UnlinkedRows"
                ],

            "ModelVerdictValuesJSON":
                model_profile[
                    "VerdictValuesJSON"
                ],

            "ProtocolEligible":
                protocol_eligible,

            "EligibilityReason":
                (
                    ""
                    if protocol_eligible
                    else "; ".join(
                        eligibility_reasons
                    )
                ),

            "InspectionStatus":
                (
                    "ELIGIBLE"
                    if protocol_eligible
                    else "INELIGIBLE"
                ),

            "InspectionError":
                "",
        })


        print(
            "    Status:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            number_of_builds,
            "| Model rows:",
            model_profile[
                "Rows"
            ],
            "| Model eval failures:",
            model_profile[
                "EvaluationFailures"
            ],
        )


    except Exception as error:
        row.update({
            "ProtocolEligible":
                False,

            "EligibilityReason":
                "Inspection error",

            "InspectionStatus":
                "ERROR",

            "InspectionError":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })

        print(
            "    ERROR:",
            row[
                "InspectionError"
            ],
        )


    row[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )


    scan_rows.append(
        row
    )


    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        pd.DataFrame(
            scan_rows
        ).sort_values(
            "CandidateInspectionOrder",
            kind="mergesort",
        ),
    )


scan_progress = (
    pd.DataFrame(
        scan_rows
    )
    .sort_values(
        "CandidateInspectionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DETERMINISTIC RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ERROR"
    )
].copy()


eligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ELIGIBLE"
    )
].copy()


ineligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "INELIGIBLE"
    )
].copy()


if not inspection_errors.empty:
    print(
        "\nCandidate inspection errors:"
    )

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 18 candidates could not be inspected. "
        "No provisional selection was frozen."
    )


if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 18 candidate was found."
    )


eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelTrainingRows",
            "ModelEvaluationRows",
            "RawExecutionRows",
            "Project",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        )
        + 1,
        dtype=np.int64,
    ),
)


top_candidate = eligible_candidates.iloc[
    0
]


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Projects 1–17 COMPLETE_AND_FROZEN",
    EXPECTED_REGISTERED_PROJECTS,
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ) == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Candidates inspected",
    EXPECTED_CANDIDATES,
    len(scan_progress),
    len(scan_progress)
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Unique candidate identities",
    EXPECTED_CANDIDATES,
    int(
        scan_progress[
            "Project"
        ].nunique()
    ),
    int(
        scan_progress[
            "Project"
        ].nunique()
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Registered/reserved candidates",
    0,
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ),
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Inspection errors",
    0,
    len(inspection_errors),
    len(inspection_errors)
    == 0,
)

add_check(
    validation_records,
    "Candidate accounting",
    EXPECTED_CANDIDATES,
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ),
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "At least one eligible candidate",
    "> 0",
    len(eligible_candidates),
    len(eligible_candidates)
    > 0,
)

add_check(
    validation_records,
    "Candidate ranks unique",
    len(eligible_candidates),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ) == len(
        eligible_candidates
    ),
)

add_check(
    validation_records,
    "Top rank",
    1,
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
    int(
        top_candidate[
            "CandidateRank"
        ]
    ) == 1,
)

add_check(
    validation_records,
    "Top candidate eligible",
    True,
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
)

add_check(
    validation_records,
    "Top candidate raw unlinked rows",
    0,
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model unlinked rows",
    0,
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model training failures",
    "> 0",
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ) > 0,
)

add_check(
    validation_records,
    "Top candidate model evaluation failures",
    "> 0",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ) > 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 18 Step 1A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed validation checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 17 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE AUTHORITATIVE STEP 1A OUTPUTS
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]


source_schema_audit = scan_progress[
    schema_columns
].copy()


atomic_write_csv(
    SCAN_PROGRESS_PATH,
    scan_progress,
)

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    scan_progress,
)

atomic_write_csv(
    ELIGIBLE_RANKED_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_PATH,
    ineligible_candidates,
)

atomic_write_csv(
    INSPECTION_ERRORS_PATH,
    inspection_errors,
)

atomic_write_csv(
    STEP1A_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


dimension_fields = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]


provisional_selection_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionState":
        "PROVISIONAL_PENDING_STEP_1B_FREEZE",

    "CandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "SourceDirectory":
        str(
            top_candidate[
                "SourceDirectory"
            ]
        ),

    "BuildIDColumn":
        str(
            top_candidate[
                "BuildIDColumn"
            ]
        ),

    "StartedAtColumn":
        str(
            top_candidate[
                "StartedAtColumn"
            ]
        ),

    "ExecutionBuildColumn":
        str(
            top_candidate[
                "ExecutionBuildColumn"
            ]
        ),

    "ExecutionVerdictColumn":
        str(
            top_candidate[
                "ExecutionVerdictColumn"
            ]
        ),

    "DatasetBuildColumn":
        str(
            top_candidate[
                "DatasetBuildColumn"
            ]
        ),

    "DatasetVerdictColumn":
        str(
            top_candidate[
                "DatasetVerdictColumn"
            ]
        ),

    "Dimensions": {
        field:
            int(
                top_candidate[
                    field
                ]
            )
        for field in dimension_fields
    },

    "RankingRule":
        RUNTIME_PRIORITY_RULE,

    "RankingPurpose":
        "Runtime-prioritized processing order only; protocol eligibility and final project set are unchanged",

    "EligibleCandidateCount":
        len(
            eligible_candidates
        ),

    "IneligibleCandidateCount":
        len(
            ineligible_candidates
        ),

    "ReservedActiveProjectsExcluded":
        sorted(
            RESERVED_ACTIVE_PROJECTS
        ),

    "CompletedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_selection_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_18_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalSelection":
        provisional_selection_payload,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_18_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project18ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL ISOLATION CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 18 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print(
    "\nRanked eligible Project 18 candidates:"
)

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


print(
    "\nProtocol-ineligible candidates:"
)

if ineligible_candidates.empty:
    print(
        "None"
    )

else:
    display(
        ineligible_candidates[
            [
                "Project",
                "Builds",
                "RawTrainFailures",
                "RawEvaluationFailures",
                "ModelTrainFailures",
                "ModelEvaluationFailures",
                "EligibilityReason",
            ]
        ]
    )


print("\n")
print("=" * 132)
print("=== PROJECT 18 CELL 2 / STEP 1A RESULT ===")
print("=" * 132)


print(
    "Registered projects:",
    len(
        registry
    ),
)

for required_number in sorted(
    required_registered_identities
):
    print(
        f"Project {required_number} identity:",
        required_registered_identities[
            required_number
        ],
    )


print(
    "Candidates inspected:",
    len(
        scan_progress
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)

print(
    "Runtime-priority ranking rule:",
    RUNTIME_PRIORITY_RULE,
)


print(
    "\nProvisional Project 18 candidate:"
)

print(
    "Candidate rank:",
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    str(
        top_candidate[
            "Project"
        ]
    ),
)

print(
    "Project slug:",
    str(
        top_candidate[
            "ProjectSlug"
        ]
    ),
)

print(
    "Source directory:",
    str(
        top_candidate[
            "SourceDirectory"
        ]
    ),
)


print(
    "\nCandidate dimensions:"
)

for field in dimension_fields:
    print(
        f"{field}:",
        int(
            top_candidate[
                field
            ]
        ),
    )


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 18 experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 18 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===
------------------------------------------------------------------------------------------------------------------------------------
[01/08] Inspecting: EMResearch@EvoMaster
    Status: ELIGIBLE | Builds: 583 | Model rows: 14460 | Model eval failures: 68
------------------------------------------------------------------------------------------------------------------------------------
[02/08] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE | Builds: 3668 | Model rows: 4822 | Model eval failures: 0
------------------------------------------------------------------------------------------------------------------------------------
[03/08] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE | Builds: 4286 | Model rows: 224550 | Model eval failures: 20
------------------------------------------------------------------------------------------------------------------------------------
[04/0

,Check,Expected,Actual,Pass
0,Completion registry rows,17,17,True
1,Projects 1–17 COMPLETE_AND_FROZEN,17,17,True
2,Project 18 registry rows,0,0,True
3,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
4,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True
5,Project 13 frozen identity,jcabi@jcabi-github,jcabi@jcabi-github,True
6,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
7,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
8,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
9,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True



Ranked eligible Project 18 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,450,337,113,67108,144,31,12,10580,8232,2348,142,31,12,0,0
1,2,EMResearch@EvoMaster,EMResearch__EvoMaster,583,437,146,59155,286,68,41,14460,9907,4553,284,68,41,0,0
2,3,apache@curator,apache__curator,517,387,130,59697,124,2,2,10509,10403,106,123,2,2,0,0
3,4,facebook@buck,facebook__buck,846,634,212,561294,1120,8,7,80898,75643,5255,1119,8,7,0,0
4,5,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0
5,6,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
6,7,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,5635027,1778,20,17,224550,205696,18854,1777,20,17,0,0



Protocol-ineligible candidates:


,Project,Builds,RawTrainFailures,RawEvaluationFailures,ModelTrainFailures,ModelEvaluationFailures,EligibilityReason
1,Graylog2@graylog2-server,3668,280,0,279,0,No raw evaluation failures; No model evaluatio...




=== PROJECT 18 CELL 2 / STEP 1A RESULT ===
Registered projects: 17
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Candidates inspected: 8
Protocol-eligible candidates: 7
Protocol-ineligible candidates: 1
Inspection errors: 0
Runtime-priority ranking rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Provisional Project 18 candidate:
Candidate rank: 1
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Source directory: /content/datasets/datasets/cantaloupe-project@cantaloupe

Candidate dimensions:
Builds: 450
TrainingBuilds: 337
EvaluationBuilds: 113
RawExecutionRows: 67108
RawTrainingRows: 45140
RawEvaluationRows: 21968
RawTrainFailures: 144
RawEvaluationFailures: 31
RawFai

In [26]:
# ==================================================================================================
# PROJECT 18 — CELL 3 / STEP 1B
# FINAL SELECTION, CHRONOLOGY FREEZE, SOURCE MANIFEST, AND CHECKPOINT
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# SAFETY:
# - freezes the Project 18 identity selected by Step 1A;
# - freezes the complete source manifest and source-root SHA-256;
# - freezes the chronological 75/25 build split;
# - validates the exact raw/model dimensions discovered in Step 1A;
# - writes no completion-registry changes;
# - does not access or modify prior-project condition outputs;
# - does not start the Project 18 experiment.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 18 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
CANDIDATE_RANK = 1

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_18_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_DIMENSIONS = {
    "Builds": 450,
    "TrainingBuilds": 337,
    "EvaluationBuilds": 113,

    "RawExecutionRows": 67_108,
    "RawTrainingRows": 45_140,
    "RawEvaluationRows": 21_968,
    "RawTrainFailures": 144,
    "RawEvaluationFailures": 31,
    "RawFailingTrainingBuilds": 59,
    "RawFailingEvaluationBuilds": 12,
    "RawUnlinkedRows": 0,

    "ModelReadyRows": 10_580,
    "ModelTrainingRows": 8_232,
    "ModelEvaluationRows": 2_348,
    "ModelTrainFailures": 142,
    "ModelEvaluationFailures": 31,
    "ModelFailingTrainingBuilds": 58,
    "ModelFailingEvaluationBuilds": 12,
    "ModelUnlinkedRows": 0,
}

REQUIRED_SOURCE_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_18_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_18_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_18_provisional_selection.json"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_18_eligible_candidates_ranked.csv"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

SOURCE_SCHEMA_SNAPSHOT_PATH = (
    SELECTION_ROOT
    / "project_18_selected_source_schema_snapshot.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_18_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def canonical_root_hash(
    manifest,
):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }

    missing_columns = (
        required_columns
        - set(manifest.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Source manifest is missing columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )

    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def profile_partition(
    frame,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
):
    training_mask = frame[
        build_column
    ].isin(
        training_build_ids
    )

    evaluation_mask = frame[
        build_column
    ].isin(
        evaluation_build_ids
    )

    linked_mask = (
        training_mask
        | evaluation_mask
    )

    failure_mask = frame[
        verdict_column
    ].ne(0)

    return {
        "Rows":
            int(len(frame)),

        "TrainingRows":
            int(training_mask.sum()),

        "EvaluationRows":
            int(evaluation_mask.sum()),

        "TrainingFailures":
            int(
                (
                    training_mask
                    & failure_mask
                ).sum()
            ),

        "EvaluationFailures":
            int(
                (
                    evaluation_mask
                    & failure_mask
                ).sum()
            ),

        "FailingTrainingBuilds":
            int(
                frame.loc[
                    training_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "FailingEvaluationBuilds":
            int(
                frame.loc[
                    evaluation_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "UnlinkedRows":
            int(
                (
                    ~linked_mask
                ).sum()
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_RANKED_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 18 Step 1B inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Selected Project 18 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


source_file_names = {
    path.name
    for path in SOURCE_DIRECTORY.iterdir()
    if path.is_file()
}


missing_required_source_files = sorted(
    REQUIRED_SOURCE_FILES
    - source_file_names
)


if missing_required_source_files:
    raise FileNotFoundError(
        "Selected Project 18 source is missing required files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1A, ARCHIVE, AND REGISTRY
# --------------------------------------------------------------------------------------------------

step1a_status_sha256 = sha256_file(
    STEP1A_STATUS_PATH
)

step1a_report_sha256 = sha256_file(
    STEP1A_REPORT_PATH
)

provisional_selection_sha256 = sha256_file(
    PROVISIONAL_SELECTION_PATH
)

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)


if step1a_status.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 18 Step 1A status is not PASS."
    )


if step1a_report.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 18 Step 1A report is not PASS."
    )


if provisional_selection.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 18 provisional selection state differs."
    )


if provisional_selection.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "Project 18 provisional project differs.\n"
        f"Expected: {PROJECT_NAME}\n"
        f"Actual:   {provisional_selection.get('Project')}"
    )


if provisional_selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Project 18 provisional slug differs."
    )


if Path(
    provisional_selection.get(
        "SourceDirectory",
        "",
    )
) != SOURCE_DIRECTORY:
    raise RuntimeError(
        "Project 18 provisional source directory differs."
    )


if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != CANDIDATE_RANK:
    raise RuntimeError(
        "Project 18 provisional candidate rank differs."
    )


if provisional_selection.get(
    "RankingRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 18 runtime-priority ranking rule differs."
    )


step1a_dimensions = {
    key: int(value)
    for key, value in provisional_selection.get(
        "Dimensions",
        {},
    ).items()
}

if step1a_dimensions != EXPECTED_DIMENSIONS:
    raise RuntimeError(
        "Project 18 Step 1A dimensions differ from the frozen Step 1B contract.\n"
        f"Expected: {EXPECTED_DIMENSIONS}\n"
        f"Actual:   {step1a_dimensions}"
    )


reserved_in_step1a = sorted(
    provisional_selection.get(
        "ReservedActiveProjectsExcluded",
        [],
    )
)

if reserved_in_step1a != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 18 Step 1A active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {reserved_in_step1a}"
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 18))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


if registry[
    registry_project_column
].eq(
    PROJECT_NAME
).any():
    raise RuntimeError(
        "The selected Project 18 identity is already registered."
    )




required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


ranked_candidates = pd.read_csv(
    ELIGIBLE_RANKED_PATH,
    low_memory=False,
)


rank_one_rows = ranked_candidates.loc[
    pd.to_numeric(
        ranked_candidates[
            "CandidateRank"
        ],
        errors="coerce",
    ).eq(
        CANDIDATE_RANK
    )
]


if len(rank_one_rows) != 1:
    raise RuntimeError(
        "Step 1A ranked candidates do not contain exactly one rank-1 row."
    )


rank_one = rank_one_rows.iloc[0]


if (
    str(
        rank_one[
            "Project"
        ]
    ) != PROJECT_NAME
    or str(
        rank_one[
            "ProjectSlug"
        ]
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 1A rank-1 identity differs."
    )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE COMPLETE SOURCE MANIFEST
# --------------------------------------------------------------------------------------------------

source_files = sorted(
    [
        path
        for path in SOURCE_DIRECTORY.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            SOURCE_DIRECTORY
        ).as_posix(),
)


if not source_files:
    raise RuntimeError(
        "Selected Project 18 source directory contains no files."
    )


source_manifest_records = []


for source_path in source_files:
    relative_path = source_path.relative_to(
        SOURCE_DIRECTORY
    ).as_posix()

    source_manifest_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


source_manifest = pd.DataFrame(
    source_manifest_records
)


source_root_sha256 = canonical_root_hash(
    source_manifest
)

source_file_count = len(
    source_manifest
)

source_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 7. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

source_paths = {
    "builds.csv":
        SOURCE_DIRECTORY
        / "builds.csv",

    "exe.csv":
        SOURCE_DIRECTORY
        / "exe.csv",

    "dataset.csv":
        SOURCE_DIRECTORY
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIRECTORY
        / "entity_change_history.csv",

    "id_map.csv":
        SOURCE_DIRECTORY
        / "id_map.csv",
}


if (
    SOURCE_DIRECTORY
    / "contributors.csv"
).is_file():
    source_paths[
        "contributors.csv"
    ] = (
        SOURCE_DIRECTORY
        / "contributors.csv"
    )


schema_snapshot_records = []


for filename, file_path in source_paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    schema_snapshot_records.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(columns),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_snapshot = pd.DataFrame(
    schema_snapshot_records
)


build_columns = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    nrows=0,
).columns.tolist()


build_id_column = resolve_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_timestamp_column = resolve_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)

exe_build_column = resolve_column(
    exe_columns,
    "build",
    "exe.csv build",
)

exe_verdict_column = resolve_column(
    exe_columns,
    "verdict",
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset_columns,
    "Build",
    "dataset.csv Build",
)

dataset_verdict_column = resolve_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. FREEZE CHRONOLOGY AND 75/25 SPLIT
# --------------------------------------------------------------------------------------------------

builds = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_series(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


invalid_timestamp_rows = int(
    builds[
        build_timestamp_column
    ].isna().sum()
)


duplicate_build_id_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)


if invalid_timestamp_rows != 0:
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


if duplicate_build_id_rows != 0:
    raise RuntimeError(
        "builds.csv contains duplicate build IDs."
    )


timestamp_group_sizes = builds.groupby(
    build_timestamp_column
).size()


timestamp_tie_groups = int(
    timestamp_group_sizes.gt(1).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(1)
    ].sum()
)


ordered_builds = (
    builds.sort_values(
        [
            build_timestamp_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


number_of_builds = len(
    ordered_builds
)


training_build_count = int(
    math.floor(
        0.75
        * number_of_builds
    )
)


evaluation_build_count = int(
    number_of_builds
    - training_build_count
)


ordered_builds[
    "ChronologyOrder"
] = np.arange(
    1,
    number_of_builds
    + 1,
    dtype=np.int64,
)


ordered_builds[
    "Partition"
] = np.where(
    ordered_builds[
        "ChronologyOrder"
    ].le(
        training_build_count
    ),
    "TRAIN",
    "EVALUATION",
)


ordered_builds[
    "PartitionOrder"
] = (
    ordered_builds.groupby(
        "Partition",
        sort=False,
    ).cumcount()
    + 1
)


fixed_chronology = ordered_builds[
    [
        "ChronologyOrder",
        build_id_column,
        build_timestamp_column,
        "Partition",
        "PartitionOrder",
    ]
].rename(
    columns={
        build_id_column:
            "BuildID",

        build_timestamp_column:
            "StartedAtUTC",
    }
)


training_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(int).tolist()
)


evaluation_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(int).tolist()
)


partition_overlap = len(
    training_build_ids
    & evaluation_build_ids
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE RAW AND MODEL DIMENSIONS
# --------------------------------------------------------------------------------------------------

exe = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=[
        exe_build_column,
        exe_verdict_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_series(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_verdict_column
] = parse_integer_series(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


dataset = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=[
        dataset_build_column,
        dataset_verdict_column,
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_series(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_verdict_column
] = parse_integer_series(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


raw_profile = profile_partition(
    frame=exe,
    build_column=exe_build_column,
    verdict_column=exe_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


model_profile = profile_partition(
    frame=dataset,
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


actual_dimensions = {
    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "RawExecutionRows":
        raw_profile[
            "Rows"
        ],

    "RawTrainingRows":
        raw_profile[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_profile[
            "EvaluationRows"
        ],

    "RawTrainFailures":
        raw_profile[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_profile[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_profile[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_profile[
            "FailingEvaluationBuilds"
        ],

    "RawUnlinkedRows":
        raw_profile[
            "UnlinkedRows"
        ],

    "ModelReadyRows":
        model_profile[
            "Rows"
        ],

    "ModelTrainingRows":
        model_profile[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_profile[
            "EvaluationRows"
        ],

    "ModelTrainFailures":
        model_profile[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_profile[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_profile[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_profile[
            "FailingEvaluationBuilds"
        ],

    "ModelUnlinkedRows":
        model_profile[
            "UnlinkedRows"
        ],
}


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1A status",
    STEP1A_PASS_STATUS,
    step1a_status.get(
        "Status"
    ),
    step1a_status.get(
        "Status"
    ) == STEP1A_PASS_STATUS,
)

add_check(
    validation_records,
    "Candidate rank",
    CANDIDATE_RANK,
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ),
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ) == CANDIDATE_RANK,
)

add_check(
    validation_records,
    "Selected project",
    PROJECT_NAME,
    provisional_selection[
        "Project"
    ],
    provisional_selection[
        "Project"
    ] == PROJECT_NAME,
)

add_check(
    validation_records,
    "Selected project slug",
    PROJECT_SLUG,
    provisional_selection[
        "ProjectSlug"
    ],
    provisional_selection[
        "ProjectSlug"
    ] == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Archive SHA-256",
    EXPECTED_ARCHIVE_SHA256,
    archive_sha256,
    archive_sha256
    == EXPECTED_ARCHIVE_SHA256,
)

add_check(
    validation_records,
    "Registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha256_before,
    registry_sha256_before
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_predecessor = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            registry_project_column,
        ].iloc[0]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_predecessor,
        actual_predecessor == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    reserved_in_step1a,
    reserved_in_step1a == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    provisional_selection.get(
        "RankingRule"
    ),
    provisional_selection.get(
        "RankingRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Source files",
    "> 0",
    source_file_count,
    source_file_count > 0,
)

add_check(
    validation_records,
    "Source bytes",
    "> 0",
    source_bytes,
    source_bytes > 0,
)

add_check(
    validation_records,
    "Invalid timestamp rows",
    0,
    invalid_timestamp_rows,
    invalid_timestamp_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build-ID rows",
    0,
    duplicate_build_id_rows,
    duplicate_build_id_rows == 0,
)

add_check(
    validation_records,
    "Partition overlap",
    0,
    partition_overlap,
    partition_overlap == 0,
)


for metric, expected_value in EXPECTED_DIMENSIONS.items():
    actual_value = int(
        actual_dimensions[
            metric
        ]
    )

    add_check(
        validation_records,
        metric,
        expected_value,
        actual_value,
        actual_value
        == expected_value,
    )


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 18 Step 1B validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Project 18 Step 1B checks:")

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 18 STEP 1B VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE FROZEN OUTPUTS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    fixed_chronology,
)

atomic_write_csv(
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    source_schema_snapshot,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RuntimePriorityPurpose":
        (
            "Processing order only; protocol eligibility "
            "and final project set are unchanged"
        ),

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "Step1AStatusSHA256":
        step1a_status_sha256,

    "Step1AReportSHA256":
        step1a_report_sha256,

    "ProvisionalSelectionSHA256":
        provisional_selection_sha256,

    "ChronologyRule":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "Dimensions":
        actual_dimensions,

    "FrozenSourceManifest":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronology":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "SourceSchemaSnapshot":
        str(
            SOURCE_SCHEMA_SNAPSHOT_PATH
        ),

    "Validation":
        str(
            STEP1B_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "Projects1To17Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "SelectionCheckpoint":
        True,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceManifest":
        True,

    "DoNotChangeChronology":
        True,

    "DoNotChangeBuildPartitions":
        True,
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project18ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK AND IMMUTABILITY CHECKS
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 18 Step 1B."
    )


final_manifest_records = []


for row in source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise RuntimeError(
            "A frozen Project 18 source file disappeared:\n"
            f"{source_path}"
        )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_manifest_records
)


final_source_root_sha256 = canonical_root_hash(
    final_source_manifest
)


if final_source_root_sha256 != source_root_sha256:
    raise RuntimeError(
        "Project 18 source changed during Step 1B."
    )


checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 18 selection checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 18 Step 1B status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\nFrozen Project 18 source manifest:")

display(
    source_manifest
)


print("\nFixed Project 18 chronology sample:")

display(
    pd.concat(
        [
            fixed_chronology.head(10),
            fixed_chronology.tail(10),
        ],
        ignore_index=True,
    )
)


print("\n")
print("=" * 132)
print("=== PROJECT 18 CELL 3 / STEP 1B RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    CANDIDATE_RANK,
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[predecessor_number],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)


print("\nFrozen source:")

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nChronology:")

print(
    "Rule: started_at ascending; "
    "Build ID descending for timestamp ties"
)

print(
    "Builds:",
    number_of_builds,
)

print(
    "Training / evaluation builds:",
    len(
        training_build_ids
    ),
    "/",
    len(
        evaluation_build_ids
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Partition overlap:",
    partition_overlap,
)


print("\nRaw and model dimensions:")

for metric in EXPECTED_DIMENSIONS:
    print(
        f"{metric}:",
        actual_dimensions[
            metric
        ],
    )


print("\nIsolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 18 experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    selection_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 18 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 18 Step 1B validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_18_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_18_CANDIDATE_DISCOVERY_COMPLETE,True
1,Candidate rank,1,1,True
2,Selected project,cantaloupe-project@cantaloupe,cantaloupe-project@cantaloupe,True
3,Selected project slug,cantaloupe-project__cantaloupe,cantaloupe-project__cantaloupe,True
4,Archive SHA-256,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,True
5,Registry SHA-256,f804412b0c2ac005418762a7892d7aeb74b4c543877b29...,f804412b0c2ac005418762a7892d7aeb74b4c543877b29...,True
6,Registry rows,17,17,True
7,Project 18 registry rows,0,0,True
8,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
9,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True



Frozen Project 18 source manifest:


,RelativePath,SizeBytes,SHA256
0,builds.csv,59354,5c9834a38378d7d0bbbde0b46fb57e0c90dcf0e7d7158b...
1,contributors.csv,1552,f3343e8e0ede8facabf7ef7f4546482b9e73eac990aacf...
2,dataset.csv,9329880,72c193ff16ebe0229fd5345b4957a7c8ec5e7461fca42e...
3,entity_change_history.csv,2785373,9efe0160013baeafb56d9a193e798c9da4f39436d313c9...
4,exe.csv,2099476,f9555fbc628c61f4f418a3d570dfb684f8ec4d8877ed09...
5,id_map.csv,260290,5208ddf5a5ea3d6e5b6a604765987d543852fe9afeaf90...



Fixed Project 18 chronology sample:


,ChronologyOrder,BuildID,StartedAtUTC,Partition,PartitionOrder
0,1,265744307,2017-08-17 21:12:44+00:00,TRAIN,1
1,2,265749988,2017-08-17 21:30:01+00:00,TRAIN,2
2,3,265766677,2017-08-17 22:22:45+00:00,TRAIN,3
3,4,265818865,2017-08-18 03:10:47+00:00,TRAIN,4
4,5,265820926,2017-08-18 03:22:53+00:00,TRAIN,5
5,6,266033623,2017-08-18 16:21:37+00:00,TRAIN,6
6,7,266033717,2017-08-18 16:22:32+00:00,TRAIN,7
7,8,266059618,2017-08-18 17:45:18+00:00,TRAIN,8
8,9,266171424,2017-08-19 00:37:55+00:00,TRAIN,9
9,10,266935775,2017-08-21 20:08:17+00:00,TRAIN,10




=== PROJECT 18 CELL 3 / STEP 1B RESULT ===

Project identity:
Project number: 18
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Candidate rank: 1
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Selection state: FINAL_AND_FROZEN

Frozen source:
Source directory: /content/datasets/datasets/cantaloupe-project@cantaloupe
Source files: 6
Source bytes: 14535925
Source root SHA-256: d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8

Chronology:
Rule: started_at ascending; Build ID descending for timestamp ties
Builds: 450
Training / evaluation builds: 337 / 113
Timestamp ti

In [27]:
# ==================================================================================================
# PROJECT 18 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 18 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commits to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 18 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 18 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_18_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 14_535_925

EXPECTED_BUILDS = 450
EXPECTED_TRAIN_BUILDS = 337
EXPECTED_EVAL_BUILDS = 113

EXPECTED_RAW_ROWS = 67_108
EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_RAW_TRAIN_FAILURES = 144
EXPECTED_RAW_EVAL_FAILURES = 31

EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )

            result.append(
                token
            )

    return result


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(
        path
    ).is_file()
]

missing_inputs.extend(
    str(
        SOURCE_DIR
        / filename
    )
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIR
        / filename
    ).is_file()
)


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 18 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN SELECTION, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 18 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 18 identity differs."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 18 runtime-priority rule differs."
    )


active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 18 active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {active_reservations}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 17
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            18,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


frozen_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []


for row in frozen_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 18 source file is missing:\n"
            f"{source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)

current_source_root = source_root_hash(
    current_manifest
)

current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 18 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


build_order = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN VALIDATION
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        [
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        [
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(
        0
    ).sum()
)


if (
    raw_duplicate_pairs
    or model_duplicate_pairs
):
    raise RuntimeError(
        "Duplicate Build-Test pairs were found.\n"
        f"Raw duplicate rows: {raw_duplicate_pairs}\n"
        f"Model duplicate rows: {model_duplicate_pairs}"
    )


raw_pairs = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pairs = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


joined = model_pairs.merge(
    raw_pairs,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


verdict_mismatches = int(
    joined[
        "ModelVerdict"
    ].ne(
        joined[
            "RawVerdict"
        ]
    ).sum()
)


raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)


raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)


raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)


model_training_mask = joined[
    "Build"
].isin(
    training_builds
)


model_evaluation_mask = joined[
    "Build"
].isin(
    evaluation_builds
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


join_audit[
    "Pass"
] = (
    join_audit[
        "Expected"
    ].astype(
        str
    )
    == join_audit[
        "Actual"
    ].astype(
        str
    )
)


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,

        "PresentInDataset":
            feature
            in dataset.columns,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

token_rows = []
builds_without_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    tokens = extract_commit_tokens(
        raw_commits
    )

    if not tokens:
        builds_without_tokens += 1

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        token_rows.append({
            "BuildID":
                int(
                    build_id
                ),

            "ChronologyOrder":
                int(
                    build_order[
                        int(
                            build_id
                        )
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    token_rows
)


if build_tokens.empty:
    raise RuntimeError(
        "No build commit tokens could be extracted."
    )


build_tokens = (
    build_tokens.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ENTITY HISTORY AND ALIAS-AWARE ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


entity_history = (
    entity_history.loc[
        entity_history[
            "NormalisedCommit"
        ].ne(
            ""
        ),
        [
            entity_id_column,
            "NormalisedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    )
)


history_commits = sorted(
    entity_history[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commits
)


id_raw = pd.read_csv(
    paths[
        "id_map.csv"
    ],
    usecols=[
        id_key_column,
        id_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


orientation_rows = []


for column in [
    id_key_column,
    id_value_column,
]:
    numeric = pd.to_numeric(
        id_raw[
            column
        ],
        errors="coerce",
    )

    numeric_filled = numeric.fillna(
        0
    )

    valid_integral = (
        numeric.notna()
        & np.isclose(
            numeric_filled,
            np.floor(
                numeric_filled
            ),
            rtol=0,
            atol=0,
        )
    )

    parsed_ids = set(
        numeric.loc[
            valid_integral
        ].astype(
            "int64"
        )
    )

    overlap = len(
        parsed_ids
        & history_entity_ids
    )

    orientation_rows.append({
        "Column":
            column,

        "Rows":
            len(
                id_raw
            ),

        "IntegralNumericRows":
            int(
                valid_integral.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~valid_integral
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(
                parsed_ids
            ),

        "MatchingHistoryEntityIDs":
            overlap,

        "HistoryEntityCoveragePercent":
            (
                100.0
                * overlap
                / len(
                    history_entity_ids
                )
                if history_entity_ids
                else 0.0
            ),
    })


id_orientation = pd.DataFrame(
    orientation_rows
)


best_orientation = (
    id_orientation.sort_values(
        [
            "MatchingHistoryEntityIDs",
            "IntegralNumericRows",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    best_orientation
) < 2:
    raise RuntimeError(
        "id_map orientation audit is incomplete."
    )


if (
    int(
        best_orientation.loc[
            0,
            "MatchingHistoryEntityIDs",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "MatchingHistoryEntityIDs",
        ]
    )
    and int(
        best_orientation.loc[
            0,
            "IntegralNumericRows",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "IntegralNumericRows",
        ]
    )
):
    raise RuntimeError(
        "Could not uniquely resolve the EntityId column in id_map.csv."
    )


resolved_id_column = str(
    best_orientation.loc[
        0,
        "Column",
    ]
)


resolved_path_column = (
    id_value_column
    if resolved_id_column
    == id_key_column
    else id_key_column
)


resolved_numeric = pd.to_numeric(
    id_raw[
        resolved_id_column
    ],
    errors="coerce",
)


resolved_numeric_filled = resolved_numeric.fillna(
    0
)


valid_resolved = (
    resolved_numeric.notna()
    & np.isclose(
        resolved_numeric_filled,
        np.floor(
            resolved_numeric_filled
        ),
        rtol=0,
        atol=0,
    )
)


invalid_resolved_rows = int(
    (
        ~valid_resolved
    ).sum()
)


if invalid_resolved_rows:
    raise RuntimeError(
        "Resolved id_map EntityId column contains "
        f"{invalid_resolved_rows} invalid rows."
    )


resolved_id_map = pd.DataFrame({
    "EntityPath":
        id_raw[
            resolved_path_column
        ].astype(
            str
        ).str.strip(),

    "EntityId":
        resolved_numeric.astype(
            "int64"
        ),
})


empty_path_rows = int(
    resolved_id_map[
        "EntityPath"
    ].eq(
        ""
    ).sum()
)


exact_duplicate_rows = int(
    len(
        resolved_id_map
    )
    - len(
        resolved_id_map.drop_duplicates(
            [
                "EntityPath",
                "EntityId",
            ]
        )
    )
)


resolved_id_map = (
    resolved_id_map.drop_duplicates(
        [
            "EntityPath",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "EntityId",
            "EntityPath",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


duplicate_entity_id_rows = int(
    resolved_id_map.duplicated(
        "EntityId",
        keep=False,
    ).sum()
)


entity_ids_with_multiple_paths = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().gt(
        1
    ).sum()
)


paths_with_multiple_ids = int(
    resolved_id_map.groupby(
        "EntityPath"
    )[
        "EntityId"
    ].nunique().gt(
        1
    ).sum()
)


maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().max()
)


id_map_entity_ids = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# 12. COMMIT MATCHING AND BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

match_started = time.perf_counter()

match_rows = []


for row in build_tokens.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    matched_commit = None


    if token in history_commit_set:
        match_type = "EXACT"
        matched_commit = token
        candidate_count = 1

    else:
        candidates = [
            commit
            for commit in history_commits
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]

        if len(
            candidates
        ) == 1:
            match_type = (
                "UNIQUE_PREFIX"
            )

            matched_commit = candidates[
                0
            ]

            candidate_count = 1

        elif len(
            candidates
        ) == 0:
            match_type = (
                "UNMATCHED"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX"
            )

            candidate_count = len(
                candidates
            )


    match_rows.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_audit = (
    pd.DataFrame(
        match_rows
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


commit_matching_seconds = float(
    time.perf_counter()
    - match_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)


prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX"
    ).sum()
)


unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)


ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX"
    ).sum()
)


matched_token_rows = int(
    exact_matches
    + prefix_matches
)


commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        commit_audit
    )
)


matched_build_commits = (
    commit_audit.loc[
        commit_audit[
            "MatchedCommit"
        ].notna(),
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


entity_for_join = (
    entity_history.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity = (
    matched_build_commits.merge(
        entity_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity[
    "EntityId"
] = build_entity[
    "EntityId"
].astype(
    "int64"
)


build_entity = (
    build_entity[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


unmatched_commit_builds = sorted(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        ),
        "BuildID",
    ].astype(
        int
    ).unique().tolist()
)


mapping_incomplete_builds = sorted(
    set(
        builds_without_entities
    )
    | set(
        unmatched_commit_builds
    )
)


mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    )
)


mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


alias_summary = (
    resolved_id_map.groupby(
        "EntityId",
        as_index=False,
    )
    .agg(
        EntityPathAliasCount=(
            "EntityPath",
            "nunique",
        ),

        CanonicalEntityPath=(
            "EntityPath",
            "min",
        ),
    )
)


entity_id_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(
                mapped_entity_ids
            )
    })
    .merge(
        alias_summary,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


entity_id_audit[
    "PresentInIDMap"
] = entity_id_audit[
    "EntityPathAliasCount"
].notna()


entity_id_audit[
    "EntityPathAliasCount"
] = entity_id_audit[
    "EntityPathAliasCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            mapping_incomplete_builds
        ),
        [
            "BuildID",
            "ChronologyOrder",
            "Partition",
        ],
    ]
    .copy()
)


unmatched_counts = (
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        )
    ]
    .groupby(
        "BuildID"
    )
    .size()
    .rename(
        "UnmatchedCommitTokens"
    )
)


mapped_entity_counts = (
    build_entity.groupby(
        "BuildID"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "MappedEntityCount"
    )
)


mapping_incomplete_frame = (
    mapping_incomplete_frame.merge(
        unmatched_counts,
        on="BuildID",
        how="left",
    )
    .merge(
        mapped_entity_counts,
        on="BuildID",
        how="left",
    )
)


mapping_incomplete_frame[
    "UnmatchedCommitTokens"
] = mapping_incomplete_frame[
    "UnmatchedCommitTokens"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "MappedEntityCount"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "HasMappedEntities"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].gt(
    0
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    checks,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    checks,
    "Dataset key columns",
    3,
    3,
    (
        dataset_build_column
        in dataset.columns
        and dataset_test_column
        in dataset.columns
        and dataset_verdict_column
        in dataset.columns
    ),
)

add_check(
    checks,
    "Predictor count consistency",
    len(
        dataset.columns
    )
    - 3,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == (
        len(
            dataset.columns
        )
        - 3
    ),
)

add_check(
    checks,
    "REC features",
    19,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == 19
        and not missing_rec_features
    ),
)

for row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            row.Metric
        ),
        row.Expected,
        row.Actual,
        bool(
            row.Pass
        ),
    )

add_check(
    checks,
    "Invalid resolved id_map IDs",
    0,
    invalid_resolved_rows,
    invalid_resolved_rows
    == 0,
)

add_check(
    checks,
    "Empty id_map paths",
    0,
    empty_path_rows,
    empty_path_rows
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped entity IDs missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    17,
    len(
        registry
    ),
    len(
        registry
    ) == 17,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    checks,
    "Project 18 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 18 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 18 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 18 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


report_payload = {
    **summary_payload,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To17Modified":
        False,

    "ActiveReservations":
        active_reservations,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 18 Step 2A."
    )


final_manifest_records = []


for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),
            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 132
)

print(
    "=== PROJECT 18 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 132
)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

for predecessor_number in sorted(
    required_registered_identities
):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )


print(
    "Active reservations:",
    active_reservations,
)


print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)


print(
    "\nBuild-Test joins:"
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print(
    "\nid_map.csv resolution:"
)

print(
    "Resolved EntityId column:",
    resolved_id_column,
)

print(
    "Resolved path column:",
    resolved_path_column,
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)


print(
    "\nCommit and entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact commit matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Step 2B mapping audit required:",
    bool(
        unmatched_tokens
        or builds_without_entities
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 132
)


=== PROJECT 18 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===

Project 18 Step 2A validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,True
1,Source root SHA-256,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,True
2,Source files,6,6,True
3,Source bytes,14535925,14535925,True
4,Builds,450,450,True
5,Training builds,337,337,True
6,Evaluation builds,113,113,True
7,Raw rows,67108,67108,True
8,Model rows,10580,10580,True
9,Dataset key columns,3,3,True



Build-Test join audit:


,Metric,Expected,Actual,Pass
0,RawRows,67108,67108,True
1,RawTrainingRows,45140,45140,True
2,RawEvaluationRows,21968,21968,True
3,RawTrainingFailures,144,144,True
4,RawEvaluationFailures,31,31,True
5,ModelRows,10580,10580,True
6,ModelTrainingRows,8232,8232,True
7,ModelEvaluationRows,2348,2348,True
8,ModelTrainingFailures,142,142,True
9,ModelEvaluationFailures,31,31,True



id_map orientation audit:


...


Mapping-incomplete builds:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities
0,287126088,66,TRAIN,1,0,False
1,296980042,70,TRAIN,1,0,False
2,313788532,113,TRAIN,1,0,False
3,563813699,385,EVALUATION,1,4,True



Build-entity sample:


,BuildID,ChronologyOrder,MatchedCommit,EntityId
0,265744307,1,1b546ae7906c8964b938fa9a46fafc302e64dda4,23
1,265744307,1,1b546ae7906c8964b938fa9a46fafc302e64dda4,2002
2,265749988,2,3b6d4168e7a93631ffb571a42f04e70e20e8aae6,25
3,265749988,2,3b6d4168e7a93631ffb571a42f04e70e20e8aae6,499
4,265766677,3,dc6578adcc3f20d26f3aea7882eae43620f84827,2002
5,265818865,4,6e3b1950520815dd11d990d1776f07ec4d9736bf,1
6,265818865,4,6e3b1950520815dd11d990d1776f07ec4d9736bf,2006
7,265818865,4,6e3b1950520815dd11d990d1776f07ec4d9736bf,2007
8,265818865,4,6e3b1950520815dd11d990d1776f07ec4d9736bf,2008
9,265820926,5,7b62ef982c11fff2c40321e88a662bd3bc88ad49,1



=== PROJECT 18 CELL 4 / STEP 2A RESULT ===
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Builds: 450
Training / evaluation builds: 337 / 113
Raw execution rows: 67108
Model-ready rows: 10580
Dataset columns: 154
Predictor columns: 151
REC features: 19

Build-Test joins:
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links: 0
Model/raw verdict mismatches: 0
Non-finite duration rows: 0
Negative duration rows: 0

id_map.csv resolution:
Resolved EntityId column: value
Resolved path column: key
Dup

In [28]:
# ==================================================================================================
# PROJECT 18 — CELL 5 / STEP 2B
# DETERMINISTIC CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# WHY THIS IMPLEMENTATION IS SAFE:
# - Project 18 has no frozen timestamp-tie groups; source chronology is globally deterministic.
# - Each raw Build-Test pair is unique.
# - The generic tie-aware path remains active but resolves to one direct order for every test.
# - The 16 non-file history features validate the direct per-test order independently of file mapping.
# - REC_Age validates the single compatible global build order.
# - The 16 non-file history features are reconstructed with vectorized cumulative calculations.
# - The two file-history features are reconstructed from the Step 2A build-entity map.
# - Clean anchor offsets preserve any accepted source-level file-mapping residuals exactly.
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
# DO NOT RERUN PROJECTS 1–17 OR PROJECT 18 STEPS 0–2A.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from collections import defaultdict
from itertools import permutations, product
import math

import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 136)
print("=== PROJECT 18 CELL 5 / STEP 2B: DETERMINISTIC CLEAN REC RECONSTRUCTION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_18_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

IMPLEMENTATION_VERSION = (
    "PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPING_BOUNDARY_AUDIT"
)

EXPECTED_SELECTION_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 14_535_925

EXPECTED_BUILDS = 450
EXPECTED_TRAIN_BUILDS = 337
EXPECTED_EVAL_BUILDS = 113
EXPECTED_TIMESTAMP_TIE_GROUPS = 0

EXPECTED_RAW_ROWS = 67_108
EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_RAW_TRAIN_FAILURES = 144
EXPECTED_RAW_EVAL_FAILURES = 31

EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 1_052
EXPECTED_EXACT_COMMIT_MATCHES = 1_048
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 4
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 447
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 3
EXPECTED_BUILD_ENTITY_ROWS = 5_704

EXPECTED_MAPPING_INCOMPLETE_BUILDS = {
    287126088,
    296980042,
    313788532,
    563813699,
}
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = {
    "TRAIN",
    "EVALUATION",
}
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 1

RECENT_WINDOW = 6

SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2

DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9

ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

TIE_INFERENCE_FEATURES = [
    feature
    for feature in REC_FEATURES
    if feature != "REC_Age"
    and feature not in FILE_HISTORY_REC
]

MAX_TIE_ORDER_COMBINATIONS = 1_024

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

TIMESTAMP_TIE_GROUPS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
)

TEST_ORDER_SEARCH_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_test_order_search_audit.csv"
)

INFERRED_EXECUTION_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

GLOBAL_AGE_ORDER_SEARCH_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_global_age_order_search.csv"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def prefix_sum(
    values,
):
    values = np.asarray(
        values
    )

    dtype = (
        np.float64
        if values.dtype.kind == "f"
        else np.int64
    )

    result = np.empty(
        len(values) + 1,
        dtype=dtype,
    )

    result[0] = 0

    np.cumsum(
        values,
        out=result[1:],
    )

    return result


def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.full(
        len(denominator),
        -1.0,
        dtype=float,
    )

    valid = denominator > 0

    result[
        valid
    ] = (
        numerator[
            valid
        ]
        / denominator[
            valid
        ]
    )

    return result


def calculate_file_rate(
    target_builds,
    current_changed_entities,
    entity_changed_builds,
):
    if not target_builds:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(
                entity_id
            )
        )

        if not changed_builds:
            continue

        overlap_count = len(
            target_builds.intersection(
                changed_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_requested_group_features(
    builds,
    verdicts,
    durations,
    global_positions,
    requested_positions,
    changed_entities_by_build,
    entity_changed_builds,
):
    builds = np.asarray(
        builds,
        dtype=np.int64,
    )

    verdicts = np.asarray(
        verdicts,
        dtype=np.int64,
    )

    durations = np.asarray(
        durations,
        dtype=np.float64,
    )

    global_positions = np.asarray(
        global_positions,
        dtype=np.int64,
    )

    requested_positions = np.asarray(
        requested_positions,
        dtype=np.int64,
    )

    n = len(
        builds
    )

    all_positions = np.arange(
        n,
        dtype=np.int64,
    )

    failure = (
        verdicts
        != SUCCESS_VERDICT_CODE
    ).astype(
        np.int64
    )

    assertion = (
        verdicts
        == ASSERTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    exception = (
        verdicts
        == EXCEPTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    transition = np.zeros(
        n,
        dtype=np.int64,
    )

    if n > 1:
        transition[
            1:
        ] = (
            verdicts[
                1:
            ]
            != verdicts[
                :-1
            ]
        ).astype(
            np.int64
        )

    duration_prefix = prefix_sum(
        durations
    )

    failure_prefix = prefix_sum(
        failure
    )

    assertion_prefix = prefix_sum(
        assertion
    )

    exception_prefix = prefix_sum(
        exception
    )

    transition_prefix = prefix_sum(
        transition
    )

    positions = requested_positions

    history_length = positions.astype(
        float
    )

    recent_start = np.maximum(
        0,
        positions - RECENT_WINDOW,
    )

    recent_length = (
        positions
        - recent_start
    ).astype(
        float
    )

    last_failure_inclusive = np.maximum.accumulate(
        np.where(
            failure > 0,
            all_positions,
            -1,
        )
    )

    last_transition_inclusive = np.maximum.accumulate(
        np.where(
            transition > 0,
            all_positions,
            -1,
        )
    )

    prior_failure_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    prior_transition_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    positive_history = positions > 0

    prior_failure_position[
        positive_history
    ] = last_failure_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    prior_transition_position[
        positive_history
    ] = last_transition_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    recent_max = np.full(
        n,
        np.nan,
        dtype=float,
    )

    for offset in range(
        1,
        RECENT_WINDOW + 1,
    ):
        if n <= offset:
            continue

        recent_max[
            offset:
        ] = np.fmax(
            recent_max[
                offset:
            ],
            durations[
                :-offset
            ],
        )

    total_max_inclusive = np.maximum.accumulate(
        durations
    )

    previous_indices = np.maximum(
        positions - 1,
        0,
    )

    reconstructed = {
        "REC_Age":
            (
                global_positions[
                    positions
                ]
                - global_positions[
                    0
                ]
            ).astype(
                float
            ),

        "REC_LastFailureAge":
            np.where(
                prior_failure_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_failure_position
                ).astype(
                    float
                ),
            ),

        "REC_LastTransitionAge":
            np.where(
                prior_transition_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_transition_position
                ).astype(
                    float
                ),
            ),

        "REC_RecentAvgExeTime":
            safe_divide(
                (
                    duration_prefix[
                        positions
                    ]
                    - duration_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentMaxExeTime":
            np.where(
                positive_history,
                recent_max[
                    positions
                ],
                -1.0,
            ),

        "REC_RecentFailRate":
            safe_divide(
                (
                    failure_prefix[
                        positions
                    ]
                    - failure_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentAssertRate":
            safe_divide(
                (
                    assertion_prefix[
                        positions
                    ]
                    - assertion_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentExcRate":
            safe_divide(
                (
                    exception_prefix[
                        positions
                    ]
                    - exception_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentTransitionRate":
            safe_divide(
                (
                    transition_prefix[
                        positions
                    ]
                    - transition_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_TotalAvgExeTime":
            safe_divide(
                duration_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalMaxExeTime":
            np.where(
                positive_history,
                total_max_inclusive[
                    previous_indices
                ],
                -1.0,
            ),

        "REC_TotalFailRate":
            safe_divide(
                failure_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalAssertRate":
            safe_divide(
                assertion_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalExcRate":
            safe_divide(
                exception_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalTransitionRate":
            safe_divide(
                transition_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_LastVerdict":
            np.where(
                positive_history,
                verdicts[
                    previous_indices
                ],
                -1,
            ).astype(
                float
            ),

        "REC_LastExeTime":
            np.where(
                positive_history,
                durations[
                    previous_indices
                ],
                -1.0,
            ),
    }

    file_failure_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    file_transition_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    failure_event_positions = np.flatnonzero(
        failure > 0
    )

    transition_event_positions = np.flatnonzero(
        transition > 0
    )

    failure_pointer = 0
    transition_pointer = 0

    prior_failure_builds = set()
    prior_transition_builds = set()

    requested_order = np.argsort(
        positions,
        kind="mergesort",
    )

    for requested_index in requested_order:
        current_position = int(
            positions[
                requested_index
            ]
        )

        while (
            failure_pointer
            < len(
                failure_event_positions
            )
            and int(
                failure_event_positions[
                    failure_pointer
                ]
            )
            < current_position
        ):
            prior_failure_builds.add(
                int(
                    builds[
                        failure_event_positions[
                            failure_pointer
                        ]
                    ]
                )
            )

            failure_pointer += 1

        while (
            transition_pointer
            < len(
                transition_event_positions
            )
            and int(
                transition_event_positions[
                    transition_pointer
                ]
            )
            < current_position
        ):
            prior_transition_builds.add(
                int(
                    builds[
                        transition_event_positions[
                            transition_pointer
                        ]
                    ]
                )
            )

            transition_pointer += 1

        current_build = int(
            builds[
                current_position
            ]
        )

        current_entities = changed_entities_by_build.get(
            current_build,
            frozenset(),
        )

        file_failure_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_failure_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

        file_transition_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_transition_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

    reconstructed[
        "REC_MaxTestFileFailRate"
    ] = file_failure_rate

    reconstructed[
        "REC_MaxTestFileTransitionRate"
    ] = file_transition_rate

    return (
        reconstructed,
        transition,
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2a_status = load_json(
    STEP2A_STATUS_PATH
)

step2a_report = load_json(
    STEP2A_REPORT_PATH
)

entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    selection.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 18 Step 1B is not frozen successfully."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or step2a_report.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or entity_mapping_summary.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):
    raise RuntimeError(
        "Project 18 Step 2A outputs are not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


if selection.get(
    "Project"
) != PROJECT_NAME or selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Frozen Project 18 identity differs."
    )


active_reservations = selection.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 18 active-reservation state differs."
    )


if selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 18 runtime-priority rule differs."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 18 source file is missing:\n"
            f"{source_path}"
        )

    current_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_manifest_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The local Project 18 source does not match "
        "the frozen source manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


chronology[
    "ChronologyOrder"
] = parse_int(
    chronology[
        "ChronologyOrder"
    ],
    "chronology.ChronologyOrder",
)


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "The frozen chronology contains invalid timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


timestamp_group_sizes = (
    chronology.groupby(
        "StartedAtUTC"
    )
    .size()
)


timestamp_tie_groups_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


if timestamp_tie_groups_count != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Project 18 timestamp-tie count differs from the frozen selection contract."
    )


timestamp_tie_groups = []
timestamp_tie_group_records = []

tied_rows = chronology.loc[
    chronology["StartedAtUTC"].duplicated(keep=False)
].copy()

for tie_group_number, (started_at, group) in enumerate(
    tied_rows.groupby("StartedAtUTC", sort=True),
    start=1,
):
    baseline_builds = (
        group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"]
        .astype(int)
        .tolist()
    )

    permutation_count = math.factorial(len(baseline_builds))
    if permutation_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A timestamp-tie group is too large for exact enumeration.\n"
            f"StartedAtUTC={started_at}; builds={baseline_builds}; "
            f"permutations={permutation_count}"
        )

    options = [tuple(int(value) for value in order) for order in permutations(baseline_builds)]
    timestamp_tie_groups.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at,
        "BuildIDs": tuple(baseline_builds),
        "Options": options,
    })

    timestamp_tie_group_records.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at.isoformat(),
        "BuildCount": len(baseline_builds),
        "BuildIDsJSON": json.dumps(baseline_builds),
        "PermutationCount": permutation_count,
    })


timestamp_tie_groups_frame = pd.DataFrame(
    timestamp_tie_group_records,
    columns=[
        "TieGroup",
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsJSON",
        "PermutationCount",
    ],
)


build_chronology_map = chronology.set_index(
    "BuildID"
)[
    "ChronologyOrder"
].astype(
    int
).to_dict()


build_timestamp_map = chronology.set_index(
    "BuildID"
)[
    "StartedAtUTC"
].to_dict()


dataset_header = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    nrows=0,
).columns.tolist()


model_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

model_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

model_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    usecols=[
        model_build_column,
        model_test_column,
        model_verdict_column,
    ] + REC_FEATURES,
    low_memory=False,
)


dataset[
    model_build_column
] = parse_int(
    dataset[
        model_build_column
    ],
    "dataset.Build",
)


dataset[
    model_test_column
] = parse_int(
    dataset[
        model_test_column
    ],
    "dataset.Test",
)


dataset[
    model_verdict_column
] = parse_int(
    dataset[
        model_verdict_column
    ],
    "dataset.Verdict",
)


dataset = dataset.rename(
    columns={
        model_build_column:
            "Build",

        model_test_column:
            "Test",

        model_verdict_column:
            "Verdict",
    }
).reset_index(
    drop=True
)


dataset[
    "_ModelRow"
] = np.arange(
    len(
        dataset
    ),
    dtype=np.int64,
)


print(
    "Loading the 58,101-row clean execution history."
)


exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.verdict",
)


exe[
    exe_job_column
] = pd.to_numeric(
    exe[
        exe_job_column
    ],
    errors="coerce",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


if exe[
    exe_job_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing/non-numeric job values."
    )


if not np.isfinite(
    exe[
        exe_duration_column
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )


if exe[
    exe_duration_column
].lt(
    0
).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )


observed_verdict_codes = sorted(
    int(
        value
    )
    for value in exe[
        exe_verdict_column
    ].unique().tolist()
)


if not set(
    observed_verdict_codes
).issubset({
    0,
    1,
    2,
    3,
}):
    raise RuntimeError(
        "exe.csv contains an unsupported verdict code.\n"
        f"Observed codes: {observed_verdict_codes}"
    )


exe = exe.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "ChronologyOrder"
] = exe[
    "Build"
].map(
    build_chronology_map
)


if exe[
    "ChronologyOrder"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to frozen chronology."
    )


exe[
    "ChronologyOrder"
] = exe[
    "ChronologyOrder"
].astype(
    np.int64
)


raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_pairs != 0
    or model_duplicate_pairs != 0
):
    raise RuntimeError(
        "Duplicate Build-Test pairs prevent exact REC reconstruction."
    )


raw_build_ids = set(
    exe[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)


global_build_sequence = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            raw_build_ids
        )
    ]
    .sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )[
        "BuildID"
    ]
    .astype(
        int
    )
    .tolist()
)


global_build_position = {
    int(
        build_id
    ):
        position
    for position, build_id in enumerate(
        global_build_sequence
    )
}


exe[
    "GlobalBuildPosition"
] = exe[
    "Build"
].map(
    global_build_position
)


if exe[
    "GlobalBuildPosition"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to global first-appearance order."
    )


exe[
    "GlobalBuildPosition"
] = exe[
    "GlobalBuildPosition"
].astype(
    np.int64
)


print(
    "Sorting raw execution history by Test and frozen chronology."
)


sort_started = time.perf_counter()


exe = (
    exe.sort_values(
        [
            "Test",
            "ChronologyOrder",
            "Build",
            "Job",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exe[
    "InferredTestOrder"
] = (
    exe.groupby(
        "Test",
        sort=False,
    )
    .cumcount()
    .astype(
        np.int64
    )
)


sort_seconds = float(
    time.perf_counter()
    - sort_started
)


commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)


build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)


mapping_incomplete_source = pd.read_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    low_memory=False,
)


commit_audit[
    "BuildID"
] = parse_int(
    commit_audit[
        "BuildID"
    ],
    "commit_audit.BuildID",
)


build_entity[
    "BuildID"
] = parse_int(
    build_entity[
        "BuildID"
    ],
    "build_entity.BuildID",
)


build_entity[
    "EntityId"
] = parse_int(
    build_entity[
        "EntityId"
    ],
    "build_entity.EntityId",
)


mapping_incomplete_source[
    "BuildID"
] = parse_int(
    mapping_incomplete_source[
        "BuildID"
    ],
    "mapping_incomplete.BuildID",
)


mapping_incomplete_source_partitions = sorted(
    set(
        mapping_incomplete_source[
            "Partition"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )
)


mapping_incomplete_source_rows_with_entities = int(
    mapping_incomplete_source[
        "HasMappedEntities"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
    .sum()
)


normalised_match_type = (
    commit_audit[
        "MatchType"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


exact_match_mask = normalised_match_type.eq(
    "EXACT"
)

prefix_match_mask = normalised_match_type.eq(
    "UNIQUE_PREFIX"
)

unmatched_mask = normalised_match_type.eq(
    "UNMATCHED"
)

ambiguous_mask = normalised_match_type.eq(
    "AMBIGUOUS_PREFIX"
)


unknown_match_type_rows = int(
    (
        ~(
            exact_match_mask
            | prefix_match_mask
            | unmatched_mask
            | ambiguous_mask
        )
    ).sum()
)


if unknown_match_type_rows != 0:
    raise RuntimeError(
        "Commit audit contains unknown MatchType rows."
    )


exact_matches = int(
    exact_match_mask.sum()
)

prefix_matches = int(
    prefix_match_mask.sum()
)

unmatched_tokens = int(
    unmatched_mask.sum()
)

ambiguous_tokens = int(
    ambiguous_mask.sum()
)


unmatched_token_builds = sorted(
    commit_audit.loc[
        unmatched_mask,
        "BuildID",
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


mapping_incomplete_builds = sorted(
    set(
        unmatched_token_builds
    )
    | set(
        builds_without_entities
    )
)


changed_entities_by_build = {
    int(
        build_id
    ):
        frozenset(
            int(
                entity_id
            )
            for entity_id in values
        )
    for build_id, values in build_entity.groupby(
        "BuildID",
        sort=False,
    )[
        "EntityId"
    ]
}


entity_changed_builds_accumulator = defaultdict(
    set
)


for row in build_entity[
    [
        "BuildID",
        "EntityId",
    ]
].itertuples(
    index=False
):
    entity_changed_builds_accumulator[
        int(
            row.EntityId
        )
    ].add(
        int(
            row.BuildID
        )
    )


entity_changed_builds = {
    entity_id:
        frozenset(
            build_ids
        )
    for entity_id, build_ids in entity_changed_builds_accumulator.items()
}


del entity_changed_builds_accumulator
gc.collect()


raw_build_counts = exe.groupby(
    "Build",
    sort=False,
).size()


raw_build_failures = (
    exe[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        exe[
            "Build"
        ]
    )
    .sum()
)


model_build_counts = dataset.groupby(
    "Build",
    sort=False,
).size()


model_build_failures = (
    dataset[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        dataset[
            "Build"
        ]
    )
    .sum()
)


unmatched_mapping_audit = (
    mapping_incomplete_source.copy()
    .sort_values(
        "BuildID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


unmatched_mapping_audit[
    "RawExecutionRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "RawFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_failures
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelReadyRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_failures
).fillna(
    0
).astype(
    int
)


print(
    "\nTimestamp tie groups:"
)

display(
    timestamp_tie_groups_frame
)


print(
    "\nUnmatched mapping audit:"
)

display(
    unmatched_mapping_audit
)


# --------------------------------------------------------------------------------------------------
# 6. EXACT PER-TEST TIE-ORDER INFERENCE
# --------------------------------------------------------------------------------------------------

order_search_started = time.perf_counter()

model_group_indices = dataset.groupby("Test", sort=False).indices
raw_group_indices = exe.groupby("Test", sort=False).indices

source_rec_arrays = {
    feature: dataset[feature].to_numpy(dtype=float)
    for feature in REC_FEATURES
}

build_timestamp_ns = {
    int(build_id): int(pd.Timestamp(timestamp).value)
    for build_id, timestamp in build_timestamp_map.items()
}


def ordered_raw_indices_for_choice(raw_indices, tie_choice):
    rows = exe.loc[raw_indices, ["Build", "Job"]].copy()
    rows["_OriginalIndex"] = np.asarray(raw_indices, dtype=np.int64)
    rows["_TimestampNS"] = rows["Build"].map(build_timestamp_ns).astype(np.int64)
    rows["_TieRank"] = 0

    for group_number, selected_order in tie_choice.items():
        rank = {int(build_id): position for position, build_id in enumerate(selected_order)}
        mask = rows["Build"].isin(rank)
        rows.loc[mask, "_TieRank"] = rows.loc[mask, "Build"].map(rank).astype(int)

    rows = rows.sort_values(
        ["_TimestampNS", "_TieRank", "Build", "Job"],
        kind="mergesort",
    )
    return rows["_OriginalIndex"].to_numpy(dtype=np.int64)


def tie_options_for_test(build_ids):
    build_set = set(int(value) for value in build_ids)
    touched = []
    for tie_group in timestamp_tie_groups:
        present = [value for value in tie_group["BuildIDs"] if value in build_set]
        if len(present) > 1:
            options = [
                tuple(value for value in option if value in build_set)
                for option in tie_group["Options"]
            ]
            options = list(dict.fromkeys(options))
            touched.append((int(tie_group["TieGroup"]), options))
    return touched


test_order_search_records = []
inferred_raw_indices_by_test = {}

total_tests = len(raw_group_indices)

for test_number, (test_id_raw, raw_indices_raw) in enumerate(raw_group_indices.items(), start=1):
    test_id = int(test_id_raw)
    raw_indices = np.asarray(raw_indices_raw, dtype=np.int64)
    group_builds_baseline = exe.loc[raw_indices, "Build"].to_numpy(dtype=np.int64)
    touched_groups = tie_options_for_test(group_builds_baseline)
    model_rows = model_group_indices.get(test_id)

    if touched_groups:
        combination_count = int(np.prod([len(options) for _, options in touched_groups]))
    else:
        combination_count = 1

    if combination_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A test requires too many exact tie-order combinations.\n"
            f"Test={test_id}; combinations={combination_count}"
        )

    choice_records = []
    choice_product = product(*[options for _, options in touched_groups]) if touched_groups else [tuple()]

    for candidate_number, selected_orders in enumerate(choice_product, start=1):
        tie_choice = {
            group_number: selected_order
            for (group_number, _), selected_order in zip(touched_groups, selected_orders)
        }
        candidate_indices = ordered_raw_indices_for_choice(raw_indices, tie_choice)

        if model_rows is None:
            mismatch_counts = {}
            mismatch_values = 0
        else:
            model_rows_array = np.asarray(model_rows, dtype=np.int64)
            requested_builds = dataset.loc[model_rows_array, "Build"].to_numpy(dtype=np.int64)
            candidate_builds = exe.loc[candidate_indices, "Build"].to_numpy(dtype=np.int64)
            position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
            missing_requested = [int(build_id) for build_id in requested_builds if int(build_id) not in position_by_build]
            if missing_requested:
                raise RuntimeError(
                    "A model-ready test contains builds missing from raw history.\n"
                    f"Test={test_id}; sample={missing_requested[:20]}"
                )
            requested_positions = np.asarray(
                [position_by_build[int(build_id)] for build_id in requested_builds],
                dtype=np.int64,
            )
            provisional_global = np.asarray(
                [global_build_position[int(build_id)] for build_id in candidate_builds],
                dtype=np.int64,
            )
            reconstructed_candidate, _ = reconstruct_requested_group_features(
                builds=candidate_builds,
                verdicts=exe.loc[candidate_indices, "Verdict"].to_numpy(dtype=np.int64),
                durations=exe.loc[candidate_indices, "Duration"].to_numpy(dtype=np.float64),
                global_positions=provisional_global,
                requested_positions=requested_positions,
                changed_entities_by_build=changed_entities_by_build,
                entity_changed_builds=entity_changed_builds,
            )
            mismatch_counts = {}
            for feature in TIE_INFERENCE_FEATURES:
                source_values = source_rec_arrays[feature][model_rows_array]
                reconstructed_values = reconstructed_candidate[feature]
                mismatch_counts[feature] = int((~np.isclose(
                    source_values,
                    reconstructed_values,
                    rtol=DIRECT_RTOL,
                    atol=DIRECT_ATOL,
                    equal_nan=False,
                )).sum())
            mismatch_values = int(sum(mismatch_counts.values()))

        choice_records.append({
            "Candidate": candidate_number,
            "TieChoice": tie_choice,
            "OrderedIndices": candidate_indices,
            "MismatchCounts": mismatch_counts,
            "MismatchValues": mismatch_values,
        })

    minimum_mismatch = min(record["MismatchValues"] for record in choice_records)
    best_records = [record for record in choice_records if record["MismatchValues"] == minimum_mismatch]
    selected_record = best_records[0]
    inferred_raw_indices_by_test[test_id] = selected_record["OrderedIndices"]

    search_mode = (
        "RAW_ONLY_TEST_FROZEN_TIE_ORDER"
        if model_rows is None and touched_groups
        else "RAW_ONLY_TEST_DIRECT_ORDER"
        if model_rows is None
        else "MODEL_READY_TEST_EXACT_TIE_SEARCH"
        if touched_groups
        else "MODEL_READY_TEST_DIRECT_ORDER"
    )

    test_order_search_records.append({
        "Test": test_id,
        "RawExecutionRows": len(raw_indices),
        "ModelReadyRows": 0 if model_rows is None else len(model_rows),
        "TimestampTieGroupsForTest": len(touched_groups),
        "CandidateOrderCombinations": combination_count,
        "MinimumMismatchValues": minimum_mismatch,
        "ZeroMismatchCandidates": int(sum(record["MismatchValues"] == 0 for record in choice_records)),
        "BestMismatchCountsJSON": json.dumps(selected_record["MismatchCounts"], sort_keys=True),
        "SelectedTieOrdersJSON": json.dumps(
            [list(selected_record["TieChoice"].get(group_number, tuple())) for group_number, _ in touched_groups]
        ),
        "SearchMode": search_mode,
    })

    if test_number % 100 == 0 or test_number == total_tests:
        print("Per-test tie-order inference progress:", test_number, "/", total_tests, "tests")


test_order_search_audit = pd.DataFrame(test_order_search_records)
model_ready_tests = int(test_order_search_audit["ModelReadyRows"].gt(0).sum())
raw_only_tests = int(test_order_search_audit["ModelReadyRows"].eq(0).sum())
tests_with_timestamp_ties = int(test_order_search_audit["TimestampTieGroupsForTest"].gt(0).sum())
tests_with_nonzero_order_mismatches = int(test_order_search_audit["MinimumMismatchValues"].gt(0).sum())
tests_with_ambiguous_zero_orders = int(test_order_search_audit["ZeroMismatchCandidates"].gt(1).sum())
total_test_order_mismatch_values = int(test_order_search_audit["MinimumMismatchValues"].sum())

order_search_seconds = float(time.perf_counter() - order_search_started)

print("\nPer-test tie-order inference summary:")
display(pd.DataFrame([
    {"Metric": "Tests", "Value": total_tests},
    {"Metric": "Model-ready tests", "Value": model_ready_tests},
    {"Metric": "Raw-only tests", "Value": raw_only_tests},
    {"Metric": "Tests touching timestamp ties", "Value": tests_with_timestamp_ties},
    {"Metric": "Tests with non-zero minimum mismatch", "Value": tests_with_nonzero_order_mismatches},
    {"Metric": "Total minimum mismatch values", "Value": total_test_order_mismatch_values},
    {"Metric": "Tests with multiple zero-mismatch orders", "Value": tests_with_ambiguous_zero_orders},
    {"Metric": "Inference seconds", "Value": order_search_seconds},
]))


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL REC_AGE ORDER SEARCH AND FULL CLEAN RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

global_order_search_records = []
global_tie_options = [tie_group["Options"] for tie_group in timestamp_tie_groups]
global_choice_product = product(*global_tie_options) if global_tie_options else [tuple()]

for candidate_number, selected_orders in enumerate(global_choice_product, start=1):
    selected_by_timestamp = {
        int(pd.Timestamp(tie_group["StartedAtUTC"]).value): tuple(int(value) for value in selected_order)
        for tie_group, selected_order in zip(timestamp_tie_groups, selected_orders)
    }

    candidate_sequence = []
    for started_at, group in chronology.groupby("StartedAtUTC", sort=True):
        timestamp_ns = int(pd.Timestamp(started_at).value)
        group_builds = [
            int(value)
            for value in group["BuildID"].astype(int).tolist()
            if int(value) in raw_build_ids
        ]
        if not group_builds:
            continue
        if timestamp_ns in selected_by_timestamp:
            order = [value for value in selected_by_timestamp[timestamp_ns] if value in set(group_builds)]
        else:
            order = [
                int(value)
                for value in group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"].astype(int).tolist()
                if int(value) in raw_build_ids
            ]
        candidate_sequence.extend(order)

    candidate_position = {int(build_id): position for position, build_id in enumerate(candidate_sequence)}
    first_build_by_test = {
        int(test_id): int(exe.loc[indices, "Build"].iloc[0])
        for test_id, indices in inferred_raw_indices_by_test.items()
    }
    source_age = dataset["REC_Age"].to_numpy(dtype=float)
    reconstructed_age = np.asarray([
        candidate_position[int(build_id)] - candidate_position[first_build_by_test[int(test_id)]]
        for build_id, test_id in dataset[["Build", "Test"]].itertuples(index=False, name=None)
    ], dtype=float)
    age_mismatches = int((~np.isclose(
        source_age,
        reconstructed_age,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )).sum())

    global_order_search_records.append({
        "Candidate": candidate_number,
        "AgeMismatchRows": age_mismatches,
        "BuildOrderSHA256": hashlib.sha256(
            ",".join(str(build_id) for build_id in candidate_sequence).encode("utf-8")
        ).hexdigest(),
        "TieOrdersJSON": json.dumps([list(order) for order in selected_orders]),
        "BuildSequence": candidate_sequence,
        "BuildPosition": candidate_position,
    })

best_age_mismatches = min(record["AgeMismatchRows"] for record in global_order_search_records)
best_global_records = [record for record in global_order_search_records if record["AgeMismatchRows"] == best_age_mismatches]
selected_global_record = best_global_records[0]
global_build_sequence = selected_global_record["BuildSequence"]
global_build_position = selected_global_record["BuildPosition"]
global_age_combination_count = len(global_order_search_records)
zero_age_candidates = int(sum(record["AgeMismatchRows"] == 0 for record in global_order_search_records))

global_age_order_search = pd.DataFrame([
    {key: value for key, value in record.items() if key not in {"BuildSequence", "BuildPosition"}}
    for record in global_order_search_records
])

print("\nGlobal REC_Age tie-order search:")
display(global_age_order_search)

reconstruction_started = time.perf_counter()
model_group_indices = dataset.groupby("Test", sort=False).indices
model_build_array = dataset["Build"].to_numpy(dtype=np.int64)
result_arrays = {
    feature: np.full(len(dataset), np.nan, dtype=np.float64)
    for feature in REC_FEATURES
}
filled_model_rows = np.zeros(len(dataset), dtype=bool)
inferred_order_lookup = {}

for test_number, (test_id_raw, ordered_indices) in enumerate(inferred_raw_indices_by_test.items(), start=1):
    test_id = int(test_id_raw)
    ordered_indices = np.asarray(ordered_indices, dtype=np.int64)
    candidate_builds = exe.loc[ordered_indices, "Build"].to_numpy(dtype=np.int64)
    for position, build_id in enumerate(candidate_builds):
        inferred_order_lookup[(test_id, int(build_id))] = position

    model_rows = model_group_indices.get(test_id)
    if model_rows is None:
        continue
    model_rows = np.asarray(model_rows, dtype=np.int64)
    requested_builds = model_build_array[model_rows]
    position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
    requested_positions = np.asarray([position_by_build[int(build_id)] for build_id in requested_builds], dtype=np.int64)
    candidate_global_positions = np.asarray([global_build_position[int(build_id)] for build_id in candidate_builds], dtype=np.int64)

    reconstructed_group, _ = reconstruct_requested_group_features(
        builds=candidate_builds,
        verdicts=exe.loc[ordered_indices, "Verdict"].to_numpy(dtype=np.int64),
        durations=exe.loc[ordered_indices, "Duration"].to_numpy(dtype=np.float64),
        global_positions=candidate_global_positions,
        requested_positions=requested_positions,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )

    for feature in REC_FEATURES:
        result_arrays[feature][model_rows] = reconstructed_group[feature]
    filled_model_rows[model_rows] = True

    if test_number % 100 == 0 or test_number == total_tests:
        print("Full REC reconstruction progress:", test_number, "/", total_tests, "tests | reconstructed rows:", int(filled_model_rows.sum()))

if not filled_model_rows.all():
    missing_model_rows = np.flatnonzero(~filled_model_rows)
    raise RuntimeError(
        "Clean REC reconstruction did not fill every model-ready row.\n"
        f"Missing rows: {len(missing_model_rows)}; sample={missing_model_rows[:20].tolist()}"
    )

exe["InferredTestOrder"] = np.asarray([
    inferred_order_lookup[(int(test_id), int(build_id))]
    for test_id, build_id in exe[["Test", "Build"]].itertuples(index=False, name=None)
], dtype=np.int64)
exe["GlobalBuildPosition"] = exe["Build"].map(global_build_position).astype(np.int64)
exe = exe.sort_values(["Test", "InferredTestOrder"], kind="mergesort").reset_index(drop=True)

clean_reconstructed = dataset[["Build", "Test"]].copy()
for feature in REC_FEATURES:
    clean_reconstructed[feature] = result_arrays[feature]

reconstruction_seconds = float(time.perf_counter() - reconstruction_started)

frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder": np.arange(1, len(global_build_sequence) + 1, dtype=np.int64),
    "BuildID": global_build_sequence,
})
frozen_global_build_order["StartedAtUTC"] = frozen_global_build_order["BuildID"].map(build_timestamp_map)

reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


missing_reconstructed_rows = int(
    clean_reconstructed[
        REC_FEATURES
    ].isna().any(
        axis=1
    ).sum()
)


if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )


if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction contains missing values."
    )


comparison_records = []
mismatch_examples = []


anchor_offsets = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)


rows_at_mapping_incomplete_build = dataset[
    "Build"
].isin(
    mapping_incomplete_build_set
).to_numpy()


for feature in REC_FEATURES:
    original_values = dataset[
        feature
    ].to_numpy(
        dtype=float
    )

    reconstructed_values = clean_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(
            original_values
        ).all()
        or not np.isfinite(
            reconstructed_values
        ).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = (
        ~direct_match_mask
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[
        feature
    ] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in FILE_HISTORY_REC,

        "Rows":
            len(
                dataset
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            int(
                direct_mismatch_mask.sum()
            ),

        "DirectMismatchesAtMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & rows_at_mapping_incomplete_build
                ).sum()
            ),

        "DirectMismatchesOutsideMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & (
                        ~rows_at_mapping_incomplete_build
                    )
                ).sum()
            ),

        "NonZeroAnchorOffsets":
            int(
                (
                    direct_difference
                    != 0
                ).sum()
            ),

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_match_mask
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MaximumAbsoluteAnchoredDifference":
            float(
                np.max(
                    np.abs(
                        original_values
                        - anchored_values
                    )
                )
            ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[
        :20
    ]

    for mismatch_index in mismatch_indices:
        mismatch_examples.append({
            "Build":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Build"
                    ]
                ),

            "Test":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Test"
                    ]
                ),

            "Feature":
                feature,

            "Original":
                float(
                    original_values[
                        mismatch_index
                    ]
                ),

            "Reconstructed":
                float(
                    reconstructed_values[
                        mismatch_index
                    ]
                ),

            "Difference":
                float(
                    direct_difference[
                        mismatch_index
                    ]
                ),

            "MappingIncompleteBuild":
                bool(
                    rows_at_mapping_incomplete_build[
                        mismatch_index
                    ]
                ),
        })


comparison_summary = pd.DataFrame(
    comparison_records
)


mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "Original",
        "Reconstructed",
        "Difference",
        "MappingIncompleteBuild",
    ],
)


anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows":
            "MatchingRows",

        "AnchoredMismatchingRows":
            "MismatchingRows",
    }
)


anchor_validation[
    "Pass"
] = anchor_validation[
    "MismatchingRows"
].eq(
    0
)


direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)


verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_DEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_INDEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


anchored_mismatch_values = int(
    anchor_validation[
        "MismatchingRows"
    ].sum()
)


nonzero_anchor_offset_values = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).sum()
)


rows_with_any_nonzero_anchor_offset = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).any(
        axis=1
    ).sum()
)


unmatched_mapping_effect_is_confined = bool(
    non_file_direct_mismatches == 0
    and file_mismatches_outside_mapping_incomplete_build == 0
)


zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)


age_mismatch_rows = int(
    comparison_summary.loc[
        comparison_summary["Feature"].eq("REC_Age"),
        "DirectMismatchingRows",
    ].iloc[0]
)

if age_mismatch_rows != best_age_mismatches:
    raise RuntimeError(
        "Final REC_Age mismatch count differs from the global-order search result."
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

raw_train_mask = exe[
    "Build"
].isin(
    training_builds
)


raw_eval_mask = exe[
    "Build"
].isin(
    evaluation_builds
)


model_train_mask = dataset[
    "Build"
].isin(
    training_builds
)


model_eval_mask = dataset[
    "Build"
].isin(
    evaluation_builds
)


validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)


add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get(
        "Status"
    ),
    step2a_status.get(
        "Status"
    )
    == EXPECTED_STEP2A_STATUS,
)


add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)


add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)


add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_groups_count,
    timestamp_tie_groups_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)


add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    int(
        raw_train_mask.sum()
    ),
    int(
        raw_train_mask.sum()
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    int(
        raw_eval_mask.sum()
    ),
    int(
        raw_eval_mask.sum()
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    int(
        model_train_mask.sum()
    ),
    int(
        model_train_mask.sum()
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    int(
        model_eval_mask.sum()
    ),
    int(
        model_eval_mask.sum()
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    )
    == EXPECTED_DATASET_COLUMNS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_pairs,
    raw_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_pairs,
    model_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Official assertion verdict code",
    2,
    ASSERTION_VERDICT_CODE,
    ASSERTION_VERDICT_CODE
    == 2,
)


add_check(
    validation_records,
    "Official exception verdict code",
    1,
    EXCEPTION_VERDICT_CODE,
    EXCEPTION_VERDICT_CODE
    == 1,
)


add_check(
    validation_records,
    "Per-test search accounting",
    total_tests,
    model_ready_tests
    + raw_only_tests,
    (
        model_ready_tests
        + raw_only_tests
    )
    == total_tests,
)


add_check(
    validation_records,
    "Tests touching timestamp ties",
    0,
    tests_with_timestamp_ties,
    tests_with_timestamp_ties
    == 0,
)


add_check(
    validation_records,
    "Tests with non-zero order mismatches",
    0,
    tests_with_nonzero_order_mismatches,
    tests_with_nonzero_order_mismatches
    == 0,
)


add_check(
    validation_records,
    "Total order mismatch values",
    0,
    total_test_order_mismatch_values,
    total_test_order_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age mismatch rows",
    0,
    best_age_mismatches,
    best_age_mismatches
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age zero-match candidates",
    "> 0",
    zero_age_candidates,
    zero_age_candidates
    > 0,
)


add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(
        commit_audit
    ),
    len(
        commit_audit
    )
    == EXPECTED_COMMIT_TOKEN_ROWS,
)


add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unique-prefix matches",
    EXPECTED_PREFIX_COMMIT_MATCHES,
    prefix_matches,
    prefix_matches
    == EXPECTED_PREFIX_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(
        builds_with_entities
    ),
    len(
        builds_with_entities
    )
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(
        builds_without_entities
    ),
    len(
        builds_without_entities
    )
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Mapping-incomplete build identities",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    mapping_incomplete_builds,
    set(
        mapping_incomplete_builds
    )
    == EXPECTED_MAPPING_INCOMPLETE_BUILDS,
)


add_check(
    validation_records,
    "Mapping-incomplete source rows",
    len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    len(
        mapping_incomplete_source
    ),
    len(
        mapping_incomplete_source
    )
    == len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
)


add_check(
    validation_records,
    "Mapping-incomplete partitions",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    ),
    mapping_incomplete_source_partitions,
    set(
        mapping_incomplete_source_partitions
    )
    == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS,
)


add_check(
    validation_records,
    "Mapping-incomplete rows with mapped entities",
    EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
    mapping_incomplete_source_rows_with_entities,
    mapping_incomplete_source_rows_with_entities
    == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
)


add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(
        build_entity
    ),
    len(
        build_entity
    )
    == EXPECTED_BUILD_ENTITY_ROWS,
)


add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(
        clean_reconstructed
    ),
    len(
        clean_reconstructed
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows
    == 0,
)


add_check(
    validation_records,
    "Missing reconstructed values",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows
    == 0,
)


add_check(
    validation_records,
    "Non-file direct mismatch values",
    0,
    non_file_direct_mismatches,
    non_file_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build
    == 0,
)


add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)


add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features
    == 0,
)


add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection.get(
        "RuntimePriorityRule"
    ),
    selection.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 18 Step 2B validation:"
)

display(
    validation
)


print(
    "\nClean REC comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor validation:"
)

display(
    anchor_validation
)


if not failed_validation.empty:
    print(
        "\nFailed Step 2B checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 2B PASS checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 17 STEP 2B VALIDATION FAILED. "
        "DO NOT START THE EXPERIMENT."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


exe[
    "StartedAtUTC"
] = exe[
    "Build"
].map(
    build_timestamp_map
)


inferred_execution_order_for_storage = (
    exe[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "StartedAtUTC",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)


atomic_csv(
    TIMESTAMP_TIE_GROUPS_PATH,
    timestamp_tie_groups_frame,
)


atomic_csv(
    TEST_ORDER_SEARCH_AUDIT_PATH,
    test_order_search_audit,
)


print(
    "\nWriting the frozen 58,101-row execution-order parquet."
)


atomic_parquet(
    INFERRED_EXECUTION_ORDER_PATH,
    inferred_execution_order_for_storage,
)


atomic_csv(
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    global_age_order_search,
)


atomic_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    frozen_global_build_order,
)


atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)


atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)


atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

execution_order_metadata = pq.ParquetFile(
    INFERRED_EXECUTION_ORDER_PATH
)


execution_order_readback_rows = int(
    execution_order_metadata.metadata.num_rows
)


execution_order_readback_columns = set(
    execution_order_metadata.schema.names
)


required_execution_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if (
    execution_order_readback_rows
    != EXPECTED_RAW_ROWS
    or not required_execution_order_columns.issubset(
        execution_order_readback_columns
    )
):
    raise RuntimeError(
        "Frozen execution-order parquet metadata readback failed."
    )


reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if len(
    reconstructed_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )


readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


readback_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original_values = readback_join[
        feature
    ].to_numpy(
        dtype=float
    )

    readback_mismatch_values += int(
        (
            ~np.isclose(
                reproduced_values,
                original_values,
                rtol=ANCHOR_RTOL,
                atol=ANCHOR_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH,
    TIMESTAMP_TIE_GROUPS_PATH,
    TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "OfficialVerdictSemantics": {
        "Success":
            SUCCESS_VERDICT_CODE,

        "Exception":
            EXCEPTION_VERDICT_CODE,

        "Assertion":
            ASSERTION_VERDICT_CODE,
    },

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "ModelReadyTests":
        model_ready_tests,

    "RawOnlyTests":
        raw_only_tests,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "TestsWithMultipleZeroMismatchOrders":
        tests_with_ambiguous_zero_orders,

    "GlobalAgeOrderCombinations":
        global_age_combination_count,

    "GlobalAgeZeroMismatchCandidates":
        zero_age_candidates,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "RawSortSeconds":
        sort_seconds,

    "RECReconstructionSeconds":
        reconstruction_seconds,

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "RawExecutionRows":
        len(
            inferred_execution_order_for_storage
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "ReconstructedRows":
        len(
            clean_reconstructed
        ),

    "GlobalBuildOrderRows":
        len(
            frozen_global_build_order
        ),

    "CommitTokenRows":
        len(
            commit_audit
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        mapping_incomplete_builds,

    "MappingIncompletePartitions":
        mapping_incomplete_source_partitions,

    "MappingIncompleteRowsWithMappedEntities":
        mapping_incomplete_source_rows_with_entities,

    "DirectMismatchValues":
        direct_mismatch_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_anchor_offset,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_offset_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ReadbackMismatchValues":
        readback_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        active_reservations,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To17Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_18_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        rec_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 18 Step 2B."
    )


final_source_manifest_records = []

for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_manifest_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source changed during Step 2B."
    )


checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP2B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP2B_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP2B_STATUS
):
    raise RuntimeError(
        "Project 18 Step 2B checkpoint/status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 18 CELL 5 / STEP 2B RESULT ===")
print("=" * 136)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)

print(
    "Active reservations:",
    active_reservations,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)


print(
    "\nOfficial verdict semantics:"
)

print(
    "Success:",
    SUCCESS_VERDICT_CODE,
)

print(
    "Exception:",
    EXCEPTION_VERDICT_CODE,
)

print(
    "Assertion:",
    ASSERTION_VERDICT_CODE,
)


print(
    "\nDeterministic execution-order freeze:"
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups_count,
)

print(
    "Raw execution-order rows:",
    execution_order_readback_rows,
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Tests:",
    total_tests,
)

print(
    "Model-ready tests:",
    model_ready_tests,
)

print(
    "Raw-only tests:",
    raw_only_tests,
)

print(
    "Tests with non-zero order mismatches:",
    tests_with_nonzero_order_mismatches,
)

print(
    "Global REC_Age mismatch rows:",
    best_age_mismatches,
)


print(
    "\nClean REC reconstruction:"
)

print(
    "Raw history rows:",
    len(
        inferred_execution_order_for_storage
    ),
)

print(
    "Model rows requested/reconstructed:",
    len(
        dataset
    ),
    "/",
    len(
        clean_reconstructed
    ),
)

print(
    "Direct mismatch values:",
    direct_mismatch_values,
)

print(
    "Non-file direct mismatch values:",
    non_file_direct_mismatches,
)

print(
    "File-history direct mismatch values:",
    file_history_direct_mismatches,
)

print(
    "File-history mismatches outside mapping-incomplete builds:",
    file_mismatches_outside_mapping_incomplete_build,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_anchor_offset,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_offset_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatch values:",
    anchored_mismatch_values,
)

print(
    "Readback mismatch values:",
    readback_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    zero_percent_clean_reproduced_exactly,
)


print(
    "\nMapping audit:"
)

print(
    "Commit-token rows:",
    len(
        commit_audit
    ),
)

print(
    "Exact / prefix / unmatched / ambiguous:",
    exact_matches,
    "/",
    prefix_matches,
    "/",
    unmatched_tokens,
    "/",
    ambiguous_tokens,
)

print(
    "Builds with / without mapped entities:",
    len(
        builds_with_entities
    ),
    "/",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Mapping-incomplete partitions:",
    mapping_incomplete_source_partitions,
)

print(
    "Mapping-incomplete rows with mapped entities:",
    mapping_incomplete_source_rows_with_entities,
)

print(
    "Unmatched mapping effect confined:",
    unmatched_mapping_effect_is_confined,
)


print(
    "\nRuntime:"
)

print(
    "Raw sort seconds:",
    round(
        sort_seconds,
        2,
    ),
)

print(
    "REC reconstruction seconds:",
    round(
        reconstruction_seconds,
        2,
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nREC reconstruction checkpoint:"
)

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    rec_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP2B_STATUS,
)

print("=" * 136)


=== PROJECT 18 CELL 5 / STEP 2B: DETERMINISTIC CLEAN REC RECONSTRUCTION ===
Loading the 58,101-row clean execution history.
Sorting raw execution history by Test and frozen chronology.

Timestamp tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount



Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows
0,287126088,66,TRAIN,1,0,False,110,1,110,1
1,296980042,70,TRAIN,1,0,False,89,1,89,1
2,313788532,113,TRAIN,1,0,False,129,0,0,0
3,563813699,385,EVALUATION,1,4,True,197,0,0,0


Per-test tie-order inference progress: 100 / 241 tests
Per-test tie-order inference progress: 200 / 241 tests
Per-test tie-order inference progress: 241 / 241 tests

Per-test tie-order inference summary:


...


Clean REC comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,DirectMismatchesAtMappingIncompleteBuild,DirectMismatchesOutsideMappingIncompleteBuild,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference,MaximumAbsoluteAnchoredDifference
0,REC_Age,VERDICT_INDEPENDENT,False,10580,10580,0,0,0,0,10580,0,0.000000e+00,0.000000e+00,0.0
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,10580,10580,0,0,0,0,10580,0,0.000000e+00,0.000000e+00,0.0
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,10580,10580,0,0,0,0,10580,0,0.000000e+00,0.000000e+00,0.0
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,10580,10580,0,0,0,809,10580,0,7.275958e-12,1.062674e-13,0.0
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,10580,10580,0,0,0,0,10580,0,0.000000e+00,0.000000e+00,0.0
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,10580,10580,0,0,0,159,10580,0,5.551115e-17,8.342413e-19,0.0
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,10580,10580,0,0,0,61,10580,0,5.551115e-17,3.200548e-19,0.0
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,10580,10580,0,0,0,98,10580,0,5.551115e-17,5.141865e-19,0.0
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,10580,10580,0,0,0,74,10580,0,5.551115e-17,3.882633e-19,0.0
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,10580,10580,0,0,0,1573,10580,0,7.275958e-12,9.200051e-14,0.0



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,10580,10580,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,10580,10580,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,10580,10580,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,10580,10580,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,10580,10580,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,10580,10580,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,10580,10580,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,10580,10580,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,10580,10580,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,10580,10580,0,0.0,True



Writing the frozen 58,101-row execution-order parquet.


=== PROJECT 18 CELL 5 / STEP 2B RESULT ===
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Source root SHA-256: d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8

Official verdict semantics:
Success: 0
Exception: 1
Assertion: 2

Deterministic execution-order freeze:
Timestamp tie groups: 0
Raw execution-order rows: 67108
Global build-order rows: 450
Tests: 241
Model-ready tests: 239
Raw-only tests: 2
Tests with non-zero order mismatches: 0
Global REC_Age mismatch ro

In [29]:
# ==================================================================================================
# PROJECT 18 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN AND COHORT FREEZE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 18 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the Project 18 failure-subtype distribution;
# - generate deterministic project/seed random streams for label-noise injection;
# - prove nested masks across all noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the evaluation partition clean and immutable;
# - perform no model fitting and no registry write.
#
# SAFETY:
# - Projects 1–17 must remain COMPLETE_AND_FROZEN and unchanged;
# - Project 18 must remain absent from the completion registry;
# - no prior-project condition output is accessed or modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 18 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0e575f7841f005f5d39"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 14_535_925

EXPECTED_BUILDS = 450
EXPECTED_TRAIN_BUILDS = 337
EXPECTED_EVAL_BUILDS = 113

EXPECTED_RAW_ROWS = 67_108
EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_RAW_TRAIN_FAILURES = 144
EXPECTED_RAW_EVAL_FAILURES = 31

EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 12

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target_path = str(
        Path(
            path
        )
    )

    matches = [
        entry
        for entry in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            entry.get(
                "Path",
                "",
            )
        ) == target_path
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "The Project 18 REC checkpoint does not contain exactly "
            f"one manifest entry for {target_path}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR / "dataset.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 18 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 18 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 18 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


expected_rec_freeze_flags = {
    "ImplementationVersion":
        "PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPING_BOUNDARY_AUDIT",

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_18_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    if rec_checkpoint.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "The Project 18 REC checkpoint does not match the frozen "
            f"Step 2B contract: {flag_name}={expected_value!r}."
        )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 18 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


selection_active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    None,
)


if selection_active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 18 selection checkpoint active reservations differ."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 18 selection checkpoint runtime-priority rule differs."
    )


if rec_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 18 REC checkpoint active reservations differ."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


expected_inferred_execution_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    INFERRED_EXECUTION_ORDER_PATH,
)

expected_global_build_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
)

expected_clean_reconstructed_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_RECONSTRUCTED_PATH,
)

expected_clean_anchor_offsets_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_ANCHOR_OFFSETS_PATH,
)

actual_inferred_execution_order_sha256 = sha256_file(
    INFERRED_EXECUTION_ORDER_PATH
)

actual_global_build_order_sha256 = sha256_file(
    FROZEN_GLOBAL_BUILD_ORDER_PATH
)

actual_clean_reconstructed_sha256 = sha256_file(
    CLEAN_RECONSTRUCTED_PATH
)

actual_clean_anchor_offsets_sha256 = sha256_file(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    actual_inferred_execution_order_sha256
    != expected_inferred_execution_order_sha256
    or actual_global_build_order_sha256
    != expected_global_build_order_sha256
    or actual_clean_reconstructed_sha256
    != expected_clean_reconstructed_sha256
    or actual_clean_anchor_offsets_sha256
    != expected_clean_anchor_offsets_sha256
):
    raise RuntimeError(
        "One or more frozen Project 18 Step 2B artifacts "
        "do not match the REC checkpoint manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, MODEL DATA, AND THE FROZEN V6 RAW EXECUTION ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


# V6 froze the exact per-test execution order required to reproduce all 19 REC features.
# This is the canonical raw-history cohort for every Project 18 noise condition.
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


missing_inferred_columns = (
    required_inferred_columns
    - set(
        inferred_execution_order.columns
    )
)


if missing_inferred_columns:
    raise RuntimeError(
        "The frozen V6 inferred execution-order file is missing columns:\n"
        + "\n".join(
            sorted(
                missing_inferred_columns
            )
        )
    )


inferred_execution_order[
    "Build"
] = parse_int(
    inferred_execution_order[
        "Build"
    ],
    "inferred_execution_order.Build",
)

inferred_execution_order[
    "Test"
] = parse_int(
    inferred_execution_order[
        "Test"
    ],
    "inferred_execution_order.Test",
)

inferred_execution_order[
    "Verdict"
] = parse_int(
    inferred_execution_order[
        "Verdict"
    ],
    "inferred_execution_order.Verdict",
)

inferred_execution_order[
    "InferredTestOrder"
] = parse_int(
    inferred_execution_order[
        "InferredTestOrder"
    ],
    "inferred_execution_order.InferredTestOrder",
)

inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)

inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "The frozen raw execution order contains missing/non-numeric "
        "job or duration values."
    )


if not np.isfinite(
    inferred_execution_order[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "The frozen raw execution order contains non-finite durations."
    )


if inferred_execution_order[
    "Duration"
].lt(
    0
).any():
    raise RuntimeError(
        "The frozen raw execution order contains negative durations."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_build_test_rows
    or raw_duplicate_test_order_rows
):
    raise RuntimeError(
        "The frozen V6 execution order contains duplicate keys."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "The frozen V6 global build-order file is missing required columns."
    )


frozen_global_build_order[
    "GlobalBuildOrder"
] = parse_int(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order[
    "BuildID"
] = parse_int(
    frozen_global_build_order[
        "BuildID"
    ],
    "frozen_global_build_order.BuildID",
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            1,
            EXPECTED_BUILDS
            + 1,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "The frozen V6 global build order is invalid."
    )


# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_valid = bool(
    failure_subtypes.astype(
        int
    ).tolist()
    == [
        1,
        2,
    ]
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 18 clean training failures do not use exactly "
        "the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 18 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "fixed chronological split: started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "RawHistoryOrder":
        (
            "Project 18 Step 2B frozen per-test execution order; "
            "timestamp-tie order inferred from exact clean REC reproduction"
        ),

    "GlobalRECAgeBuildOrder":
        (
            "Project 18 Step 2B frozen global build first-appearance order"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint implementation",
    "PROJECT_18_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING",
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == "PROJECT_18_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_ONLY_TEST_HANDLING",
)

add_check(
    validation_records,
    "REC checkpoint schema version",
    1,
    rec_checkpoint.get(
        "CheckpointVersion"
    ),
    rec_checkpoint.get(
        "CheckpointVersion"
    )
    == 1,
)

add_check(
    validation_records,
    "Frozen inferred execution-order SHA-256",
    expected_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256
    == expected_inferred_execution_order_sha256,
)

add_check(
    validation_records,
    "Frozen global build-order SHA-256",
    expected_global_build_order_sha256,
    actual_global_build_order_sha256,
    actual_global_build_order_sha256
    == expected_global_build_order_sha256,
)

add_check(
    validation_records,
    "Frozen clean reconstruction SHA-256",
    expected_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256
    == expected_clean_reconstructed_sha256,
)

add_check(
    validation_records,
    "Frozen clean anchor-offset SHA-256",
    expected_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256
    == expected_clean_anchor_offsets_sha256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Failure subtype values",
    [
        1,
        2,
    ],
    failure_subtypes.astype(
        int
    ).tolist(),
    failure_subtype_values_valid,
)

add_check(
    validation_records,
    "Failure subtype profile sum",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    ) == EXPECTED_REGISTERED_PROJECTS,
)

for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project == required_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_active_reservations,
    selection_active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 18 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 18 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        "Project 18 Step 2B deterministic per-test execution order",

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 18 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 18 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 18 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 18 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print("\nFrozen clean-history order:")

print(
    "Inferred execution-order rows:",
    len(
        exe
    ),
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Raw duplicate Test-order rows:",
    raw_duplicate_test_order_rows,
)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 18 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)


=== PROJECT 18 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 18 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,True
3,REC checkpoint SHA-256,8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0...,8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0...,True
4,REC checkpoint implementation,PROJECT_18_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_...,PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPI...,False
...,...,...,...,...
68,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
69,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True
70,Active reservations,[],[],True
71,Project 18 registry rows,0,0,True



Failed Step 3A checks:


,Check,Expected,Actual,Pass
4,REC checkpoint implementation,PROJECT_18_V1_EXACT_SINGLE_TIE_GROUP_WITH_RAW_...,PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPI...,False



No Step 3A checkpoint or PASS status was written.


RuntimeError: PROJECT 18 STEP 3A VALIDATION FAILED.

In [30]:
# ==================================================================================================
# PROJECT 18 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN AND COHORT FREEZE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 18 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the Project 18 failure-subtype distribution;
# - generate deterministic project/seed random streams for label-noise injection;
# - prove nested masks across all noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the evaluation partition clean and immutable;
# - perform no model fitting and no registry write.
#
# SAFETY:
# - Projects 1–17 must remain COMPLETE_AND_FROZEN and unchanged;
# - Project 18 must remain absent from the completion registry;
# - no prior-project condition output is accessed or modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 18 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0e575f7841f005f5d39"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 14_535_925

EXPECTED_BUILDS = 450
EXPECTED_TRAIN_BUILDS = 337
EXPECTED_EVAL_BUILDS = 113

EXPECTED_RAW_ROWS = 67_108
EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_RAW_TRAIN_FAILURES = 144
EXPECTED_RAW_EVAL_FAILURES = 31

EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 12

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_18_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target_path = str(
        Path(
            path
        )
    )

    matches = [
        entry
        for entry in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            entry.get(
                "Path",
                "",
            )
        ) == target_path
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "The Project 18 REC checkpoint does not contain exactly "
            f"one manifest entry for {target_path}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR / "dataset.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 18 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 18 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 18 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


expected_rec_freeze_flags = {
    "ImplementationVersion":
        "PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPING_BOUNDARY_AUDIT",

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_18_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    if rec_checkpoint.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "The Project 18 REC checkpoint does not match the frozen "
            f"Step 2B contract: {flag_name}={expected_value!r}."
        )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 18 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


selection_active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    None,
)


if selection_active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 18 selection checkpoint active reservations differ."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 18 selection checkpoint runtime-priority rule differs."
    )


if rec_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 18 REC checkpoint active reservations differ."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


expected_inferred_execution_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    INFERRED_EXECUTION_ORDER_PATH,
)

expected_global_build_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
)

expected_clean_reconstructed_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_RECONSTRUCTED_PATH,
)

expected_clean_anchor_offsets_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_ANCHOR_OFFSETS_PATH,
)

actual_inferred_execution_order_sha256 = sha256_file(
    INFERRED_EXECUTION_ORDER_PATH
)

actual_global_build_order_sha256 = sha256_file(
    FROZEN_GLOBAL_BUILD_ORDER_PATH
)

actual_clean_reconstructed_sha256 = sha256_file(
    CLEAN_RECONSTRUCTED_PATH
)

actual_clean_anchor_offsets_sha256 = sha256_file(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    actual_inferred_execution_order_sha256
    != expected_inferred_execution_order_sha256
    or actual_global_build_order_sha256
    != expected_global_build_order_sha256
    or actual_clean_reconstructed_sha256
    != expected_clean_reconstructed_sha256
    or actual_clean_anchor_offsets_sha256
    != expected_clean_anchor_offsets_sha256
):
    raise RuntimeError(
        "One or more frozen Project 18 Step 2B artifacts "
        "do not match the REC checkpoint manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, MODEL DATA, AND THE FROZEN V6 RAW EXECUTION ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


# V6 froze the exact per-test execution order required to reproduce all 19 REC features.
# This is the canonical raw-history cohort for every Project 18 noise condition.
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


missing_inferred_columns = (
    required_inferred_columns
    - set(
        inferred_execution_order.columns
    )
)


if missing_inferred_columns:
    raise RuntimeError(
        "The frozen V6 inferred execution-order file is missing columns:\n"
        + "\n".join(
            sorted(
                missing_inferred_columns
            )
        )
    )


inferred_execution_order[
    "Build"
] = parse_int(
    inferred_execution_order[
        "Build"
    ],
    "inferred_execution_order.Build",
)

inferred_execution_order[
    "Test"
] = parse_int(
    inferred_execution_order[
        "Test"
    ],
    "inferred_execution_order.Test",
)

inferred_execution_order[
    "Verdict"
] = parse_int(
    inferred_execution_order[
        "Verdict"
    ],
    "inferred_execution_order.Verdict",
)

inferred_execution_order[
    "InferredTestOrder"
] = parse_int(
    inferred_execution_order[
        "InferredTestOrder"
    ],
    "inferred_execution_order.InferredTestOrder",
)

inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)

inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "The frozen raw execution order contains missing/non-numeric "
        "job or duration values."
    )


if not np.isfinite(
    inferred_execution_order[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "The frozen raw execution order contains non-finite durations."
    )


if inferred_execution_order[
    "Duration"
].lt(
    0
).any():
    raise RuntimeError(
        "The frozen raw execution order contains negative durations."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_build_test_rows
    or raw_duplicate_test_order_rows
):
    raise RuntimeError(
        "The frozen V6 execution order contains duplicate keys."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "The frozen V6 global build-order file is missing required columns."
    )


frozen_global_build_order[
    "GlobalBuildOrder"
] = parse_int(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order[
    "BuildID"
] = parse_int(
    frozen_global_build_order[
        "BuildID"
    ],
    "frozen_global_build_order.BuildID",
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            1,
            EXPECTED_BUILDS
            + 1,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "The frozen V6 global build order is invalid."
    )


# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_valid = bool(
    failure_subtypes.astype(
        int
    ).tolist()
    == [
        1,
        2,
    ]
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 18 clean training failures do not use exactly "
        "the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 18 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "fixed chronological split: started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "RawHistoryOrder":
        (
            "Project 18 Step 2B frozen per-test execution order; "
            "timestamp-tie order inferred from exact clean REC reproduction"
        ),

    "GlobalRECAgeBuildOrder":
        (
            "Project 18 Step 2B frozen global build first-appearance order"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint implementation",
    "PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPING_BOUNDARY_AUDIT",
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == "PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPING_BOUNDARY_AUDIT",
)

add_check(
    validation_records,
    "REC checkpoint schema version",
    1,
    rec_checkpoint.get(
        "CheckpointVersion"
    ),
    rec_checkpoint.get(
        "CheckpointVersion"
    )
    == 1,
)

add_check(
    validation_records,
    "Frozen inferred execution-order SHA-256",
    expected_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256
    == expected_inferred_execution_order_sha256,
)

add_check(
    validation_records,
    "Frozen global build-order SHA-256",
    expected_global_build_order_sha256,
    actual_global_build_order_sha256,
    actual_global_build_order_sha256
    == expected_global_build_order_sha256,
)

add_check(
    validation_records,
    "Frozen clean reconstruction SHA-256",
    expected_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256
    == expected_clean_reconstructed_sha256,
)

add_check(
    validation_records,
    "Frozen clean anchor-offset SHA-256",
    expected_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256
    == expected_clean_anchor_offsets_sha256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Failure subtype values",
    [
        1,
        2,
    ],
    failure_subtypes.astype(
        int
    ).tolist(),
    failure_subtype_values_valid,
)

add_check(
    validation_records,
    "Failure subtype profile sum",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    ) == EXPECTED_REGISTERED_PROJECTS,
)

for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project == required_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_active_reservations,
    selection_active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 18 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 18 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        "Project 18 Step 2B deterministic per-test execution order",

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To16Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 18 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 18 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 18 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 18 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print("\nFrozen clean-history order:")

print(
    "Inferred execution-order rows:",
    len(
        exe
    ),
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Raw duplicate Test-order rows:",
    raw_duplicate_test_order_rows,
)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 18 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)


=== PROJECT 18 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 18 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8a...,True
3,REC checkpoint SHA-256,8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0...,8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0...,True
4,REC checkpoint implementation,PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPI...,PROJECT_18_V1_ZERO_TIE_DIRECT_ORDER_WITH_MAPPI...,True
...,...,...,...,...
68,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
69,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True
70,Active reservations,[],[],True
71,Project 18 registry rows,0,0,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,91,0.631944
1,2,53,0.368056



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,15349271,1504423452,45140,0,...,0,144,144,8232,0,142,142,4e62a63e83ad664c7387d1077aea582bc51769bb722b84...,0ed7eb5f6f5f436bf832bbafe567ffc034b77706bf9903...,6a81dcf2046f3b8840440a2e2f9bfe86a03f699dbef742...
1,2,noise_05__seed_01,1,2,5,1,15349271,1504423452,45140,2221,...,6,144,2353,8232,425,142,555,fc67b9c959ed919684d0d54a10b693c8ca2ae9dc13b78a...,169423584530022bbe4d59875bc659ceb370e2493790a5...,c13f8461df4e62a4b9f2492cb404f0361155b0ff58ff2e...
2,3,noise_10__seed_01,1,3,10,1,15349271,1504423452,45140,4458,...,10,144,4582,8232,811,142,933,2e35c824f76cd489c126e3050affec4558ac0e114421df...,67d7cde8c6d82b548548e24ad2df7eaa1b4db4c285b16d...,c973a4e6065c224af1b4fda9bff472a723e638c3e8411b...
3,4,noise_15__seed_01,1,4,15,1,15349271,1504423452,45140,6751,...,19,144,6857,8232,1206,142,1310,47e813250aa27cfa8128330f46fc866f6aad1e70c44450...,1f82d05872414256a694b59abd83cb6cd2ebb8380a6953...,97d8a5932efd45097b43a9a81b7b91271980c92beb59db...
4,5,noise_20__seed_01,1,5,20,1,15349271,1504423452,45140,9005,...,23,144,9103,8232,1594,142,1690,1c818946d560394a36e0026af04455fa8f8f8753f9ffa2...,a4bbb47c73e352f3183d4533a7e099f88c51f77b2fe07a...,54414a8961737d0b7759cf0109663fcd4721cf8c681786...
5,6,noise_25__seed_01,1,6,25,1,15349271,1504423452,45140,11241,...,28,144,11329,8232,1985,142,2071,624283ba1a541c3c716fdc1eb47ec776830f65ac424c05...,bbffe18888690fa757033e556a6edd3426ec89f77e6cfd...,dd6a38368fe234c61a40b67cb6d77332d9c54771f2cdcb...
6,7,noise_30__seed_01,1,7,30,1,15349271,1504423452,45140,13523,...,39,144,13589,8232,2418,142,2482,67e5ba09859ac6551d82e18fe8f6c0bfaff8b3ef4a6572...,a52fe36b891be07ca99f70c1e57993576f78e32dc9f59e...,47975ac29bf422e344b140805a50910ab894fa0ab0f702...
7,8,noise_40__seed_01,1,8,40,1,15349271,1504423452,45140,18018,...,58,144,18046,8232,3263,142,3289,e93e98fff336a8a53d282e6f0085143de90f13f29ef39c...,5e07bcb07b25ea9e1257c1e8bf6384159b7cdd19d61445...,e6c093738375fa3343e4f319d9cb6bf7cabc7ac9775620...
8,9,noise_50__seed_01,1,9,50,1,15349271,1504423452,45140,22497,...,73,144,22495,8232,4069,142,4065,c30133a91d6e3f2c0545743ea0a290b434231e6add2df7...,fc46c1a625b9713ef21dd26a7cc63064ff1f8d176b3fc4...,40a791c831b379420f098bb0c9253520419d7dcc381c14...
9,262,noise_00__seed_30,30,1,0,30,3250434104,1898266564,45140,0,...,0,144,144,8232,0,142,142,4e62a63e83ad664c7387d1077aea582bc51769bb722b84...,0ed7eb5f6f5f436bf832bbafe567ffc034b77706bf9903...,6a81dcf2046f3b8840440a2e2f9bfe86a03f699dbef742...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 18 CELL 6 / STEP 3A RESULT ===

Project:
cantaloupe-project@cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen clean-history order:
Inferred execution-order rows: 67108
Global build-order rows: 450
Raw duplicate Build-Test rows: 0
Raw duplicate Test-order rows: 0

Fixed cohorts:
Raw training rows: 45140
Raw evaluation rows: 21968
Raw training failures: 144
Raw evaluation failures: 31
Model training rows: 8232
Model evaluation rows: 2348
Model training failures: 142
Model evaluation failures: 31
Model failing evaluation builds: 12

Noise plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions:

In [31]:
# ==================================================================================================
# PROJECT 18 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 18 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 18 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 18 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_18_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "ca33c05f49ed7c5cb359a137d4f57831be4a58816a04320fcbca6798ae88a49a"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 12

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 18 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 18 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 18 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 18 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != 17
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            18,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 18 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 18 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    16,
    len(
        registry
    ),
    len(
        registry
    )
    == 16,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 18 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 18 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 18 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To17Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 18 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 18 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 18 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 18 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 18 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 18 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===

Project 18 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,ca33c05f49ed7c5cb359a137d4f57831be4a58816a0432...,ca33c05f49ed7c5cb359a137d4f57831be4a58816a0432...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,True
4,Raw training rows,45140,45140,True
5,Raw evaluation rows,21968,21968,True
6,Model training rows,8232,8232,True
7,Model evaluation rows,2348,2348,True
8,Model training failures,142,142,True
9,Model evaluation failures,31,31,True



Failed Project 18 Step 4A checks:


,Check,Expected,Actual,Pass
28,Registry rows,16,17,False



No Step 4A PASS status or checkpoint was written.


RuntimeError: PROJECT 18 STEP 4A VALIDATION FAILED.

In [32]:
# ==================================================================================================
# PROJECT 18 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 18 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 18 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 18 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_18_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "ca33c05f49ed7c5cb359a137d4f57831be4a58816a04320fcbca6798ae88a49a"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 12

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 18 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 18 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 18 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 18 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != 17
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            18,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–17."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 18 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 18 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 18 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    17,
    len(
        registry
    ),
    len(
        registry
    )
    == 17,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 18 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 18 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 18 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 18 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To17Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project18ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 18 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 18 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 18 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 18 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 18 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–17 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 18 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 18 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===

Project 18 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,ca33c05f49ed7c5cb359a137d4f57831be4a58816a0432...,ca33c05f49ed7c5cb359a137d4f57831be4a58816a0432...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4...,True
4,Raw training rows,45140,45140,True
5,Raw evaluation rows,21968,21968,True
6,Model training rows,8232,8232,True
7,Model evaluation rows,2348,2348,True
8,Model training failures,142,142,True
9,Model evaluation failures,31,31,True



Runtime versions:


,Component,Version,ExpectedVersion,Pass
0,Python,3.12.13,3.12.13,True
1,numpy,2.0.2,2.0.2,True
2,pandas,2.2.2,2.2.2,True
3,scikit-learn,1.6.1,1.6.1,True
4,xgboost,3.3.0,3.3.0,True
5,lightgbm,4.6.0,4.6.0,True
6,pyarrow,18.1.0,18.1.0,True



Predictor contract summary:


...


Metric self-tests:


,Check,Expected,Actual,Pass
0,Manual APFD,0.8,0.8,True
1,APFDc rewards quick failing test first,True,True,True
2,All-pass APFD is NaN,True,True,True
3,All-pass APFDc is NaN,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,271896,271896,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,133616,133616,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,1335620,1335620,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,485788,485788,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,63229,63229,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,20253,20253,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,93,93,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5253,5253,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,19084685,19084685,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,84485,84485,True




=== PROJECT 18 CELL 7 / STEP 4A RESULT ===

Project:
cantaloupe-project@cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen experiment contract:
Predictors: 151
REC features: 19
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD
Positive class: failure = 1
Median imputation: condition-training medians
Ranking tie-break: Test ascending
Rolling retraining: False

Fixed cohorts:
Model training rows: 8232
Model evaluation rows: 2348
Training failures: 142
Evaluation failures: 31
Failing evaluation buil

In [33]:
# ==================================================================================================
# PROJECT 18 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# SMOKE CONDITIONS:
# - 0% noise, repetition seed 1
# - 50% noise, repetition seed 1
#
# THIS CELL:
# - verifies the frozen Step 4A runtime/model contract;
# - reconstructs condition-specific dependent REC features;
# - preserves all six verdict-independent REC features;
# - applies the frozen clean-anchor offsets;
# - trains all four ML techniques once per smoke condition;
# - evaluates ML plus Random, LatestFail, and QTF-Avg;
# - validates APFDc/APFD outputs and baseline invariance;
# - writes only Project 18 smoke-test outputs and checkpoint/status files;
# - does not modify the registry or full 270-condition raw-result root.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 18 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_18_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_18_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_18_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

STEP4B_STATUS = (
    "PASS_PROJECT_18_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)

CONDITION_STATUS = (
    "PASS_PROJECT_18_SMOKE_CONDITION"
)

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "35e5fdd559178e4b0e0573fcc9e1ed335975ec94d0ad13117d2f72aeb7bde0a5"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "ca33c05f49ed7c5cb359a137d4f57831be4a58816a04320fcbca6798ae88a49a"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0e575f7841f005f5d39"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)

EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_BUILDS = 450
EXPECTED_RAW_ROWS = 67_108
EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_FAILING_EVAL_BUILDS = 12
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 113
EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS
EXPECTED_RNG_MANIFEST_ROWS = 1_354_200
EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

SMOKE_CONDITION_IDS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

SMOKE_REPETITION_SEED = 1
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/cantaloupe-project@cantaloupe"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_18_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_18_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_18_fixed_chronological_builds.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_selection_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

BUILD_ENTITY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_smoke_test"
)

SMOKE_CONDITION_INVENTORY_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_condition_inventory.csv"
)

SMOKE_COMBINED_CONDITION_AUDIT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
)

SMOKE_COMBINED_PROJECT_RUNS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"
)

SMOKE_COMBINED_BUILD_METRICS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
)

SMOKE_COMBINED_MODEL_FITS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_report.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4b_status.json"
)

SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_18_smoke_test_checkpoint.json"
)

# The smoke test must never write to this future full-run root.
FULL_RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                "inferred_test_order",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)


def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def audit_checkpoint_manifest(
    payload,
    manifest_key,
    label,
):
    manifest = payload.get(
        manifest_key,
        [],
    )

    if not isinstance(
        manifest,
        list,
    ) or not manifest:
        raise RuntimeError(
            f"{label} contains no {manifest_key}."
        )

    records = []

    for item in manifest:
        path = Path(
            item[
                "Path"
            ]
        )

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha256 = str(
            item[
                "SHA256"
            ]
        ).lower()

        exists = path.is_file()

        actual_bytes = (
            int(
                path.stat().st_size
            )
            if exists
            else -1
        )

        actual_sha256 = (
            sha256_file(
                path
            )
            if exists
            else "MISSING"
        )

        records.append({
            "Checkpoint":
                label,

            "Path":
                str(
                    path
                ),

            "ExpectedBytes":
                expected_bytes,

            "ActualBytes":
                actual_bytes,

            "ExpectedSHA256":
                expected_sha256,

            "ActualSHA256":
                actual_sha256,

            "Pass":
                bool(
                    exists
                    and actual_bytes
                    == expected_bytes
                    and actual_sha256
                    == expected_sha256
                ),
        })

    audit = pd.DataFrame(
        records
    )

    failures = int(
        (
            ~audit[
                "Pass"
            ]
        ).sum()
    )

    return (
        audit,
        failures,
    )


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 4B inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 runtime-contract checkpoint SHA-256 differs."
    )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)

if selection_checkpoint.get(
    "Status"
) != "PASS_PROJECT_18_SELECTION_AND_SOURCE_FROZEN":
    raise RuntimeError(
        "Selection checkpoint does not contain the frozen Step 1B PASS status."
    )

if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise RuntimeError(
        "REC checkpoint does not contain the frozen Step 2B PASS status."
    )

if noise_plan_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise RuntimeError(
        "Noise-plan checkpoint does not contain the frozen Step 3A PASS status."
    )

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active-reservation state differs."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 4B."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–17."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is already present in the completion registry."
    )

active_reservations = []

if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 18 freeze."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 18 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source root differs before Step 4B."
    )

rec_manifest_audit, rec_manifest_failures = (
    audit_checkpoint_manifest(
        rec_checkpoint,
        "OutputManifest",
        "REC checkpoint",
    )
)

noise_manifest_audit, noise_manifest_failures = (
    audit_checkpoint_manifest(
        noise_plan_checkpoint,
        "OutputManifest",
        "Noise-plan checkpoint",
    )
)

if rec_manifest_failures != 0:
    print(
        "\nFailed REC output-manifest checks:"
    )

    display(
        rec_manifest_audit.loc[
            ~rec_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen REC outputs changed."
    )

if noise_manifest_failures != 0:
    print(
        "\nFailed noise-plan output-manifest checks:"
    )

    display(
        noise_manifest_audit.loc[
            ~noise_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen noise-plan outputs changed."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 18 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)
frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")
if len(inferred_execution_order) != EXPECTED_RAW_ROWS:
    raise RuntimeError("Frozen inferred execution-order row count differs.")
if len(frozen_global_build_order) != EXPECTED_BUILDS:
    raise RuntimeError("Frozen global build-order row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

smoke_plan = (
    condition_plan.loc[
        condition_plan["ConditionID"].isin(
            SMOKE_CONDITION_IDS
        )
    ]
    .copy()
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(smoke_plan) != EXPECTED_SMOKE_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain exactly the two smoke conditions."
    )

if smoke_plan["ConditionID"].tolist() != SMOKE_CONDITION_IDS:
    raise RuntimeError(
        "Smoke-condition order differs from the frozen contract."
    )

if smoke_plan["NoisePercent"].astype(int).tolist() != SMOKE_NOISE_LEVELS:
    raise RuntimeError(
        "Smoke noise levels differ from the frozen contract."
    )

if not smoke_plan["RepetitionSeed"].astype(int).eq(
    SMOKE_REPETITION_SEED
).all():
    raise RuntimeError(
        "Smoke repetition seed differs from the frozen contract."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_MANIFEST_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

rng_seed = pd.read_parquet(
    RNG_MANIFEST_PATH,
    filters=[
        (
            "RepetitionSeed",
            "==",
            SMOKE_REPETITION_SEED,
        ),
    ],
)

rng_seed[raw_order_column] = parse_int(
    rng_seed[raw_order_column],
    "rng_seed.RawTrainingRowOrder",
)

rng_seed = (
    rng_seed.sort_values(
        raw_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(rng_seed) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Seed-1 RNG stream has the wrong row count."
    )

if not np.array_equal(
    rng_seed[raw_order_column].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_RAW_TRAIN_ROWS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Seed-1 RNG stream row order differs."
    )

flip_uniform = rng_seed[
    "FlipUniform"
].to_numpy(dtype=np.float64)
sampled_failure_subtype = rng_seed[
    "SampledFailureSubtype"
].to_numpy(dtype=np.int16)

if not np.isfinite(flip_uniform).all():
    raise RuntimeError(
        "Seed-1 flip-uniform stream contains non-finite values."
    )

if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
    raise RuntimeError(
        "Seed-1 flip-uniform values are outside [0,1)."
    )

if sorted(np.unique(sampled_failure_subtype).tolist()) != [1, 2]:
    raise RuntimeError(
        "Seed-1 failure-subtype stream differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

required_inferred_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "InferredTestOrder",
}

missing_inferred_order_columns = (
    required_inferred_order_columns
    - set(inferred_execution_order.columns)
)

if missing_inferred_order_columns:
    raise RuntimeError(
        "Frozen inferred execution order is missing columns: "
        f"{sorted(missing_inferred_order_columns)}"
    )

for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[column] = parse_int(
        inferred_execution_order[column],
        f"inferred_execution_order.{column}",
    )

inferred_execution_order["Job"] = pd.to_numeric(
    inferred_execution_order["Job"],
    errors="coerce",
)

inferred_execution_order["Duration"] = pd.to_numeric(
    inferred_execution_order["Duration"],
    errors="coerce",
)

if not np.isfinite(
    inferred_execution_order["Job"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite jobs."
    )

if not np.isfinite(
    inferred_execution_order["Duration"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite durations."
    )

if inferred_execution_order["Duration"].lt(0).any():
    raise RuntimeError(
        "Frozen inferred execution order contains negative durations."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Build",
        "Test",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate Build-Test rows."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Test",
        "InferredTestOrder",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate per-test order rows."
    )

required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}

missing_global_order_columns = (
    required_global_order_columns
    - set(frozen_global_build_order.columns)
)

if missing_global_order_columns:
    raise RuntimeError(
        "Frozen global build order is missing columns: "
        f"{sorted(missing_global_order_columns)}"
    )

frozen_global_build_order["GlobalBuildOrder"] = parse_int(
    frozen_global_build_order["GlobalBuildOrder"],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order["BuildID"] = parse_int(
    frozen_global_build_order["BuildID"],
    "frozen_global_build_order.BuildID",
)

frozen_global_build_order = (
    frozen_global_build_order.sort_values(
        "GlobalBuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not np.array_equal(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_BUILDS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen global build-order sequence is not canonical."
    )

if frozen_global_build_order["BuildID"].nunique() != EXPECTED_BUILDS:
    raise RuntimeError(
        "Frozen global build order contains duplicate build IDs."
    )

ordered_builds = (
    frozen_global_build_order[
        "BuildID"
    ]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing InferredTestOrder."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

combined_raw_order = (
    pd.concat(
        [
            raw_training[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
            raw_evaluation[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

inferred_order_reference = (
    inferred_execution_order[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

raw_order_key_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            combined_raw_order[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
            != inferred_order_reference[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
        ).sum()
    )
)

raw_order_numeric_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            ~np.isclose(
                combined_raw_order[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                inferred_order_reference[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                rtol=0,
                atol=0,
                equal_nan=False,
            )
        ).sum()
    )
)

if (
    raw_order_key_mismatches != 0
    or raw_order_numeric_mismatches != 0
):
    raise RuntimeError(
        "The fixed raw cohorts no longer reproduce the frozen V6 "
        "inferred execution order."
    )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

random_scores = np.empty(
    EXPECTED_MODEL_EVAL_ROWS,
    dtype=np.float64,
)

for build_id in sorted(
    evaluation_meta_base["Build"].unique()
):
    build_indices = np.flatnonzero(
        evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        == int(build_id)
    )

    random_scores[build_indices] = np.random.default_rng(
        deterministic_random_build_seed(
            SMOKE_REPETITION_SEED,
            int(build_id),
        )
    ).random(len(build_indices))

if not np.isfinite(random_scores).all():
    raise RuntimeError(
        "Random baseline produced non-finite scores."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN THE TWO END-TO-END SMOKE CONDITIONS
# --------------------------------------------------------------------------------------------------

# Remove only incomplete/previous Project 18 smoke-test outputs.
# Frozen Steps 0–4A and the future full-result root are untouched.
if SMOKE_ROOT.exists():
    shutil.rmtree(
        SMOKE_ROOT
    )

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

raw_training_hash_before = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_before = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_before = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_before = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)

condition_inventory_records = []
all_condition_audits = []
all_project_runs = []
all_build_metrics = []
all_model_fits = []
all_rankings_for_invariance = []

smoke_execution_started = time.perf_counter()

for smoke_index, plan_row in enumerate(
    smoke_plan.itertuples(index=False),
    start=1,
):
    condition_started = time.perf_counter()
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)

    print("\n" + "-" * 136)
    print(
        f"[{smoke_index}/{EXPECTED_SMOKE_CONDITIONS}] "
        f"Running {condition_key}"
    )
    print("-" * 136)

    condition_dir = (
        SMOKE_ROOT
        / condition_key
    )
    condition_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ranking_path = (
        condition_dir
        / "rankings.csv.gz"
    )
    build_metrics_path = (
        condition_dir
        / "build_metrics.csv"
    )
    project_runs_path = (
        condition_dir
        / "project_runs.csv"
    )
    model_fits_path = (
        condition_dir
        / "model_fits.csv"
    )
    training_medians_path = (
        condition_dir
        / "training_medians.csv"
    )
    condition_audit_path = (
        condition_dir
        / "condition_audit.csv"
    )
    condition_summary_path = (
        condition_dir
        / "condition_summary.json"
    )
    completion_marker_path = (
        condition_dir
        / "COMPLETE.json"
    )

    flip_mask = (
        flip_uniform
        < (noise_percent / 100.0)
    )

    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = (
        flip_mask
        & (clean_raw_training_verdict == 0)
    )
    failure_to_pass_mask = (
        flip_mask
        & (clean_raw_training_verdict != 0)
    )

    noisy_raw_training_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtype[
        pass_to_failure_mask
    ]
    noisy_raw_training_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(
        plan_row.FlipMaskSHA256
    )
    expected_noisy_raw_sha256 = str(
        plan_row.NoisyRawVerdictSHA256
    )
    expected_noisy_model_sha256 = str(
        plan_row.NoisyModelVerdictSHA256
    )

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )

    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )

    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (
            noisy_model_training_verdict
            != clean_model_training_verdict
        ).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(
        noisy_model_training_binary.sum()
    )

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    print(
        "  Reconstructing REC features from the condition-specific history."
    )

    train_history = pd.DataFrame({
        "build": raw_training["Build"].to_numpy(dtype=np.int64),
        "test": raw_training["Test"].to_numpy(dtype=np.int64),
        "job": raw_training["Job"].to_numpy(),
        "verdict": noisy_raw_training_verdict.astype(np.int16),
        "duration": raw_training["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_training[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    evaluation_history = pd.DataFrame({
        "build": raw_evaluation["Build"].to_numpy(dtype=np.int64),
        "test": raw_evaluation["Test"].to_numpy(dtype=np.int64),
        "job": raw_evaluation["Job"].to_numpy(),
        "verdict": clean_raw_evaluation_verdict.astype(np.int16),
        "duration": raw_evaluation["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_evaluation[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    execution_history = pd.concat(
        [
            train_history,
            evaluation_history,
        ],
        ignore_index=True,
    )

    if execution_history.duplicated(
        subset=[
            "build",
            "test",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate Build-Test rows."
        )

    if execution_history.duplicated(
        subset=[
            "test",
            "inferred_test_order",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate per-test order rows."
        )

    execution_history = (
        execution_history.sort_values(
            [
                "test",
                "inferred_test_order",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    rec_started = time.perf_counter()

    reconstructed = reconstruct_rec_features(
        execution_history=execution_history,
        requested_rows=model_all[["Build", "Test"]],
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
        recent_window=RECENT_WINDOW,
    )

    rec_seconds = time.perf_counter() - rec_started

    if len(reconstructed) != EXPECTED_MODEL_ROWS:
        raise RuntimeError(
            f"{condition_key}: reconstructed REC row count differs."
        )

    if reconstructed.duplicated(
        subset=["Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: reconstructed REC contains duplicate keys."
        )

    reconstructed["Build"] = parse_int(
        reconstructed["Build"],
        f"{condition_key}.reconstructed.Build",
    )
    reconstructed["Test"] = parse_int(
        reconstructed["Test"],
        f"{condition_key}.reconstructed.Test",
    )

    reconstructed_indexed = reconstructed.set_index(
        [
            "Build",
            "Test",
        ]
    )

    missing_reconstructed_keys = model_key_index.difference(
        reconstructed_indexed.index
    )

    if len(missing_reconstructed_keys) != 0:
        raise RuntimeError(
            f"{condition_key}: reconstruction does not cover all model rows."
        )

    reconstructed_values_all = reconstructed_indexed.loc[
        model_key_index,
        REC_FEATURES,
    ].to_numpy(dtype=np.float64)

    anchored_values_all = (
        reconstructed_values_all
        + anchor_values_all
    )

    independent_reconstruction_mismatches = int(
        (~np.isclose(
            anchored_values_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            clean_original_rec_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            rtol=0,
            atol=1e-12,
        )).sum()
    )

    if independent_reconstruction_mismatches != 0:
        raise RuntimeError(
            f"{condition_key}: verdict-independent REC reconstruction changed."
        )

    condition_numeric_all = all_base_numeric.copy()

    dependent_rec_values_all = anchored_values_all[:, [
        REC_FEATURES.index(feature)
        for feature in VERDICT_DEPENDENT_REC
    ]]

    condition_numeric_all[:, dependent_predictor_indices] = (
        dependent_rec_values_all
    )

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    condition_original_dependent_all = all_base_numeric[
        :, dependent_predictor_indices
    ]

    dependent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, dependent_predictor_indices],
            condition_original_dependent_all,
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        )).sum()
    )

    independent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, independent_predictor_indices],
            all_base_numeric[:, independent_predictor_indices],
            rtol=0,
            atol=0,
            equal_nan=True,
        )).sum()
    )

    if independent_rec_changes != 0:
        raise RuntimeError(
            f"{condition_key}: preserved independent REC predictors changed."
        )

    if noise_percent == 0:
        zero_rec_mismatches = int(
            (~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )).sum()
        )

        if zero_rec_mismatches != 0:
            raise RuntimeError(
                "0% smoke condition did not reproduce the clean REC cohort."
            )

        if number_flipped != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly flipped raw labels."
            )

        if model_label_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed model labels."
            )

        if dependent_rec_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed dependent REC values."
            )

    if noise_percent > 0:
        if number_flipped <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no dependent REC values."
            )

    medians = np.nanmedian(
        condition_training_numeric,
        axis=0,
    )

    nonfinite_median_indices = np.flatnonzero(
        ~np.isfinite(medians)
    )

    if len(nonfinite_median_indices) != 0:
        bad_features = [
            predictor_columns[index]
            for index in nonfinite_median_indices
        ]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(
        condition_training_numeric
    )
    evaluation_missing_mask = ~np.isfinite(
        condition_evaluation_numeric
    )

    if training_missing_mask.any():
        row_indices, column_indices = np.where(
            training_missing_mask
        )
        condition_training_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(
            evaluation_missing_mask
        )
        condition_evaluation_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite after imputation."
        )

    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite after imputation."
        )

    # Keep float64 throughout the smoke test. This matches the frozen Step 4A
    # numeric/imputation contract and avoids changing ranking/model behaviour
    # through an unapproved dtype conversion.

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()
        fit_status = "PASS_MODEL_FIT"
        fit_error = ""

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )

            fit_seconds = time.perf_counter() - fit_started
            scores = positive_probability(
                model,
                condition_evaluation_numeric,
            )

            technique_scores[technique] = scores

        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)

            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })

            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps(
                [
                    int(value)
                    for value in np.asarray(model.classes_).tolist()
                ]
            ),
            "Status": fit_status,
            "Error": fit_error,
        })

        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )

    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :,
        predictor_index["REC_LastFailureAge"],
    ].astype(float)

    condition_eval_qtf = condition_evaluation_numeric[
        :,
        predictor_index["REC_TotalAvgExeTime"],
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = (
        -condition_eval_last_failure_age
    )
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []

    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )

    ranking_techniques = sorted(
        rankings["Technique"].unique().tolist()
    )

    if ranking_techniques != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )

    ranking_rows_per_technique = rankings.groupby(
        "Technique"
    ).size()

    if not ranking_rows_per_technique.eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )

    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_ranking_rows != 0:
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(
        rankings
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )

    if sorted(project_runs["Technique"].tolist()) != sorted(
        ALL_TECHNIQUES
    ):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    metric_columns = [
        "APFDc",
        "APFD",
    ]

    build_metric_values = build_metrics[
        metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )

    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    project_metric_values = project_runs[
        project_metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )

    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(len(reconstructed)),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    atomic_csv(
        ranking_path,
        rankings,
        compression="gzip",
    )
    atomic_csv(
        build_metrics_path,
        build_metrics,
    )
    atomic_csv(
        project_runs_path,
        project_runs,
    )
    atomic_csv(
        model_fits_path,
        model_fits,
    )
    atomic_csv(
        training_medians_path,
        training_medians,
    )
    atomic_csv(
        condition_audit_path,
        condition_audit,
    )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]

    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "OutputManifest": condition_output_manifest,
    }

    atomic_json(
        condition_summary_path,
        condition_summary,
    )

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json(
        completion_marker_path,
        completion_marker,
    )

    condition_manifest = directory_manifest(
        condition_dir
    )
    condition_root_sha256 = directory_root_hash(
        condition_manifest
    )

    condition_inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": smoke_index,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": condition_root_sha256,
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "ModelFitRows": int(len(model_fits)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
    })

    all_condition_audits.append(condition_audit)
    all_project_runs.append(project_runs)
    all_build_metrics.append(build_metrics)
    all_model_fits.append(model_fits)
    all_rankings_for_invariance.append(
        rankings.loc[
            rankings["Technique"].isin(
                [
                    "Random",
                    "QTF-Avg",
                ]
            )
        ].copy()
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del train_history
    del evaluation_history
    del execution_history
    del reconstructed
    del reconstructed_indexed
    del reconstructed_values_all
    del anchored_values_all
    del condition_numeric_all
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. COMBINE SMOKE OUTPUTS AND VERIFY BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

condition_inventory = pd.DataFrame(
    condition_inventory_records
)
combined_condition_audit = pd.concat(
    all_condition_audits,
    ignore_index=True,
)
combined_project_runs = pd.concat(
    all_project_runs,
    ignore_index=True,
)
combined_build_metrics = pd.concat(
    all_build_metrics,
    ignore_index=True,
)
combined_model_fits = pd.concat(
    all_model_fits,
    ignore_index=True,
)
combined_invariance_rankings = pd.concat(
    all_rankings_for_invariance,
    ignore_index=True,
)

baseline_invariance_records = []

for technique in [
    "Random",
    "QTF-Avg",
]:
    zero_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(0)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    noisy_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(50)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    same_keys = bool(
        zero_rows[["Build", "Test"]].equals(
            noisy_rows[["Build", "Test"]]
        )
    )

    score_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (~np.isclose(
                zero_rows["Score"].to_numpy(dtype=float),
                noisy_rows["Score"].to_numpy(dtype=float),
                rtol=0,
                atol=0,
            )).sum()
        )
    )

    rank_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (
                zero_rows["Rank"].to_numpy(dtype=np.int64)
                != noisy_rows["Rank"].to_numpy(dtype=np.int64)
            ).sum()
        )
    )

    baseline_invariance_records.append({
        "Technique": technique,
        "Rows": int(len(zero_rows)),
        "SameBuildTestKeys": same_keys,
        "ScoreMismatches": score_mismatches,
        "RankMismatches": rank_mismatches,
        "Pass": (
            len(zero_rows) == EXPECTED_MODEL_EVAL_ROWS
            and len(noisy_rows) == EXPECTED_MODEL_EVAL_ROWS
            and same_keys
            and score_mismatches == 0
            and rank_mismatches == 0
        ),
    })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)
baseline_invariance_failures = int(
    (~baseline_invariance["Pass"]).sum()
)

if baseline_invariance_failures != 0:
    print("\nBaseline invariance failures:")
    display(
        baseline_invariance.loc[
            ~baseline_invariance["Pass"]
        ]
    )
    raise RuntimeError(
        "Random or QTF-Avg changed across the two smoke noise levels."
    )

atomic_csv(
    SMOKE_CONDITION_INVENTORY_PATH,
    condition_inventory,
)
atomic_csv(
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)
atomic_csv(
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)
atomic_csv(
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)
atomic_csv(
    SMOKE_COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)
atomic_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_hash_after = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_after = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_after = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_after = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)
registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

current_source_rows_after = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

source_root_sha256_after = source_root_hash(
    pd.DataFrame(current_source_rows_after)
)

full_raw_result_root_exists_after = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_after = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_after = directory_root_hash(
    full_raw_result_manifest_after
)

full_raw_result_unchanged = bool(
    full_raw_result_root_existed_before
    == full_raw_result_root_exists_after
    and full_raw_result_root_hash_before
    == full_raw_result_root_hash_after
    and len(full_raw_result_manifest_before)
    == len(full_raw_result_manifest_after)
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
].iloc[0]

positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(50)
].iloc[0]

validation_records = []

add_check(
    validation_records,
    "Step 4A status",
    EXPECTED_STEP4A_STATUS,
    step4a_status.get("Status"),
    step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
)
add_check(
    validation_records,
    "Runtime checkpoint SHA-256",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    runtime_checkpoint_sha256,
    runtime_checkpoint_sha256 == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_plan_checkpoint_sha256,
    noise_plan_checkpoint_sha256 == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256 == EXPECTED_REC_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256 == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC output-manifest failures",
    0,
    rec_manifest_failures,
    rec_manifest_failures == 0,
)
add_check(
    validation_records,
    "Noise-plan output-manifest failures",
    0,
    noise_manifest_failures,
    noise_manifest_failures == 0,
)
add_check(
    validation_records,
    "Step 4A output-manifest failures",
    0,
    runtime_manifest_failures,
    runtime_manifest_failures == 0,
)
add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_sha256_after,
    source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256,
)
add_check(
    validation_records,
    "Smoke conditions",
    EXPECTED_SMOKE_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Smoke condition keys",
    SMOKE_CONDITION_IDS,
    condition_inventory["ConditionKey"].tolist(),
    condition_inventory["ConditionKey"].tolist() == SMOKE_CONDITION_IDS,
)
add_check(
    validation_records,
    "Condition statuses",
    CONDITION_STATUS,
    sorted(condition_inventory["Status"].unique().tolist()),
    condition_inventory["Status"].eq(CONDITION_STATUS).all(),
)
add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_SMOKE_CONDITIONS,
    len(combined_condition_audit),
    len(combined_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Total ML fits",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
)
add_check(
    validation_records,
    "Model-fit failures",
    0,
    int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()),
    combined_model_fits["Status"].eq("PASS_MODEL_FIT").all(),
)
add_check(
    validation_records,
    "Total project-run rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Total build-metric rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Technique set",
    sorted(ALL_TECHNIQUES),
    sorted(combined_project_runs["Technique"].unique().tolist()),
    sorted(combined_project_runs["Technique"].unique().tolist())
    == sorted(ALL_TECHNIQUES),
)
add_check(
    validation_records,
    "Project-run rows per condition violations",
    0,
    int((
        combined_project_runs.groupby("ConditionKey").size()
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_project_runs.groupby("ConditionKey").size()
        == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Build-metric rows per condition violations",
    0,
    int((
        combined_build_metrics.groupby("ConditionKey").size()
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_build_metrics.groupby("ConditionKey").size()
        == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Zero-noise raw flips",
    0,
    int(zero_audit["NumberFlipped"]),
    int(zero_audit["NumberFlipped"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise model-label changes",
    0,
    int(zero_audit["ModelLabelChanges"]),
    int(zero_audit["ModelLabelChanges"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise dependent REC changes",
    0,
    int(zero_audit["DependentRECChanges"]),
    int(zero_audit["DependentRECChanges"]) == 0,
)
add_check(
    validation_records,
    "Positive-noise raw flips",
    "> 0",
    int(positive_audit["NumberFlipped"]),
    int(positive_audit["NumberFlipped"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise model-label changes",
    "> 0",
    int(positive_audit["ModelLabelChanges"]),
    int(positive_audit["ModelLabelChanges"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise dependent REC changes",
    "> 0",
    int(positive_audit["DependentRECChanges"]),
    int(positive_audit["DependentRECChanges"]) > 0,
)
add_check(
    validation_records,
    "Independent REC changes",
    0,
    int(combined_condition_audit["IndependentRECChanges"].sum()),
    int(combined_condition_audit["IndependentRECChanges"].sum()) == 0,
)
add_check(
    validation_records,
    "Frozen raw-order key mismatches",
    0,
    raw_order_key_mismatches,
    raw_order_key_mismatches == 0,
)
add_check(
    validation_records,
    "Frozen raw-order numeric mismatches",
    0,
    raw_order_numeric_mismatches,
    raw_order_numeric_mismatches == 0,
)
add_check(
    validation_records,
    "Independent REC reconstruction mismatches",
    0,
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()),
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()) == 0,
)
add_check(
    validation_records,
    "Noise-plan hash mismatches",
    0,
    int((
        combined_condition_audit["ExpectedFlipMaskSHA256"]
        != combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
        != combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
        != combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()),
    bool(
        (
            combined_condition_audit["ExpectedFlipMaskSHA256"]
            == combined_condition_audit["ActualFlipMaskSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
            == combined_condition_audit["ActualNoisyRawVerdictSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
            == combined_condition_audit["ActualNoisyModelVerdictSHA256"]
        ).all()
    ),
)
add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)
add_check(
    validation_records,
    "Project metrics non-finite",
    0,
    int((~np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    )).sum()),
    bool(np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    ).all()),
)
add_check(
    validation_records,
    "Project metrics outside [0,1]",
    0,
    int((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            < 0
        )
        | (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            > 1
        )
    ).sum()),
    bool((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            >= 0
        )
        & (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            <= 1
        )
    ).all()),
)
add_check(
    validation_records,
    "Raw training cohort unchanged",
    raw_training_hash_before,
    raw_training_hash_after,
    raw_training_hash_after == raw_training_hash_before,
)
add_check(
    validation_records,
    "Raw evaluation cohort unchanged",
    raw_evaluation_hash_before,
    raw_evaluation_hash_after,
    raw_evaluation_hash_after == raw_evaluation_hash_before,
)
add_check(
    validation_records,
    "Model training cohort unchanged",
    model_training_hash_before,
    model_training_hash_after,
    model_training_hash_after == model_training_hash_before,
)
add_check(
    validation_records,
    "Model evaluation cohort unchanged",
    model_evaluation_hash_before,
    model_evaluation_hash_after,
    model_evaluation_hash_after == model_evaluation_hash_before,
)
add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after,
    registry_sha256_after == registry_sha256_before,
)
add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Registry Project 18 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)
add_check(
    validation_records,
    "Full raw-result root unchanged",
    True,
    full_raw_result_unchanged,
    full_raw_result_unchanged,
)

validation = pd.DataFrame(
    validation_records
)
failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 18 Step 4B validation:")
display(validation)

print("\nBaseline invariance audit:")
display(baseline_invariance)

print("\nSmoke project-run results:")
display(
    combined_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

if not failed_validation.empty:
    print("\nFailed Project 18 Step 4B checks:")
    display(failed_validation)
    print("\nNo Step 4B PASS status or checkpoint was written.")
    raise RuntimeError(
        "PROJECT 18 STEP 4B VALIDATION FAILED. DO NOT START THE FULL EXPERIMENT."
    )

atomic_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, STATUS, AND FINAL READBACK
# --------------------------------------------------------------------------------------------------

smoke_execution_seconds = (
    time.perf_counter()
    - smoke_execution_started
)
completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

smoke_output_paths = [
    SMOKE_CONDITION_INVENTORY_PATH,
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    SMOKE_COMBINED_MODEL_FITS_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]

for condition_key in SMOKE_CONDITION_IDS:
    condition_dir = SMOKE_ROOT / condition_key
    smoke_output_paths.extend([
        path
        for path in condition_dir.rglob("*")
        if path.is_file()
    ])

smoke_output_paths = sorted(
    set(smoke_output_paths),
    key=lambda path: str(path),
)

smoke_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in smoke_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "SmokeConditionKeys": SMOKE_CONDITION_IDS,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": int(len(combined_build_metrics)),
    "ProjectRunRows": int(len(combined_project_runs)),
    "TrainingMedianRows": int(
        condition_inventory["TrainingMedianRows"].sum()
    ),
    "ZeroNoiseRawFlips": int(zero_audit["NumberFlipped"]),
    "ZeroNoiseModelLabelChanges": int(
        zero_audit["ModelLabelChanges"]
    ),
    "ZeroNoiseDependentRECChanges": int(
        zero_audit["DependentRECChanges"]
    ),
    "PositiveNoiseRawFlips": int(
        positive_audit["NumberFlipped"]
    ),
    "PositiveNoiseModelLabelChanges": int(
        positive_audit["ModelLabelChanges"]
    ),
    "PositiveNoiseDependentRECChanges": int(
        positive_audit["DependentRECChanges"]
    ),
    "IndependentRECChanges": int(
        combined_condition_audit["IndependentRECChanges"].sum()
    ),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": int(len(validation)),
    "FailedValidationChecks": int(len(failed_validation)),
    "SmokeExecutionSeconds": float(smoke_execution_seconds),
    "OutputManifest": smoke_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To16Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "FullRawResultRootModified": False,
    "FullExperimentStarted": False,
}

atomic_json(
    SMOKE_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "SmokeTestPassed": True,
    "RuntimeContractFrozen": True,
    "NoisePlanFrozen": True,
    "EvaluationCohortImmutable": True,
    "ReadyForFull270ConditionExperiment": True,
}

atomic_json(
    SMOKE_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "FailedValidationChecks": int(len(failed_validation)),
    "Checkpoint": str(SMOKE_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(SMOKE_CHECKPOINT_PATH),
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "FullExperimentStarted": False,
}

atomic_json(
    STEP4B_STATUS_PATH,
    status_payload,
)

checkpoint_readback = load_json(
    SMOKE_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP4B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B checkpoint readback failed."
    )

if status_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B status readback failed."
    )

if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Step 4B finalisation."
    )

final_source_rows = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    final_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

if source_root_hash(pd.DataFrame(final_source_rows)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 18 source changed during Step 4B finalisation."
    )

print("\n" + "=" * 136)
print("=== PROJECT 18 CELL 8 / STEP 4B RESULT ===")
print("=" * 136)
print()
print("Project:")
print(PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)
print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)
print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)
print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Two-condition end-to-end smoke test:")
print("Conditions:", SMOKE_CONDITION_IDS)
print("Conditions passed:", len(condition_inventory), "/", EXPECTED_SMOKE_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Noise and REC audit:")
print("0% raw flips:", int(zero_audit["NumberFlipped"]))
print("0% model-label changes:", int(zero_audit["ModelLabelChanges"]))
print("0% dependent REC changes:", int(zero_audit["DependentRECChanges"]))
print("50% raw flips:", int(positive_audit["NumberFlipped"]))
print("50% model-label changes:", int(positive_audit["ModelLabelChanges"]))
print("50% dependent REC changes:", int(positive_audit["DependentRECChanges"]))
print("Independent REC changes:", int(combined_condition_audit["IndependentRECChanges"].sum()))
print()
print("Baselines and metrics:")
print("Random/QTF-Avg invariance failures:", baseline_invariance_failures)
print("Techniques:", ALL_TECHNIQUES)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 18 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–17 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print("Full experiment raw-result root modified:", False)
print("Full 270-condition experiment started:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Smoke-test checkpoint:")
print(SMOKE_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(SMOKE_CHECKPOINT_PATH))
print()
print("Runtime seconds:", round(smoke_execution_seconds, 2))
print()
print("STATUS:", STEP4B_STATUS)
print("=" * 136)


=== PROJECT 18 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 18 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.

----------------------------------------------------------------------------------------------------------------------------------------
[1/2] Running noise_00__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 241 tests | reconstructed rows: 6385
    REC reconstruction progress: 200 / 241 tests | reconstructed rows: 10238
    REC reconstruction progress: 241 / 241 tests | reconstructed rows: 10580
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 142 | condition seconds: 51.69

----------------------------------------------------------------------------------------------------------------------------------------
[2/2] Running noise_50__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 241 tests | reconstructed rows: 6385
    REC reconstruction progress: 200 / 241 tests | reconstructed rows: 10238
    REC reconstruction progress: 241 / 241 tests | reconstructed rows: 10580
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 22497 | model-label changes: 4069 | dependent REC changes: 115478
  Training failures: 4065 | condition seconds: 56.38

Project 18 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,2348,True,0,0,True
1,QTF-Avg,2348,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,LatestFail,113,12,2348,31,0.569503,0.642331,0.325883,0.322289
1,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,LightGBM,113,12,2348,31,0.670107,0.802702,0.808092,0.897815
2,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,NaiveBayes,113,12,2348,31,0.566164,0.583029,0.793760,0.833858
3,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,QTF-Avg,113,12,2348,31,0.603369,0.607914,0.394931,0.229846
4,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,Random,113,12,2348,31,0.496477,0.580741,0.492687,0.546572
5,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,RandomForest,113,12,2348,31,0.729348,0.722913,0.903338,0.920916
6,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_00__seed_01,0,1,XGBoost,113,12,2348,31,0.601287,0.653661,0.785787,0.853341
7,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_50__seed_01,50,1,LatestFail,113,12,2348,31,0.562028,0.503053,0.579372,0.609480
8,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_50__seed_01,50,1,LightGBM,113,12,2348,31,0.481299,0.354617,0.445883,0.348668
9,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,noise_50__seed_01,50,1,NaiveBayes,113,12,2348,31,0.570010,0.633068,0.631791,0.804122



=== PROJECT 18 CELL 8 / STEP 4B RESULT ===

Project:
cantaloupe-project@cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 32872
Build-metric rows: 168
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 22497
50% model-label changes: 4069
50% dependent REC changes: 115478
Independent REC changes: 0

Baselines and metrics:
Random/QTF-Avg invariance failures: 0
Techni

In [34]:
# ==================================================================================================
# PROJECT 18 — CELL 9 / STEP 5A RESUME-SAFE CHECKPOINT-SCHEMA-COMPATIBLE ACCELERATED
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT WITH VECTORIZED REC ENGINE
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_16.ipynb NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 18 Step 4B smoke-test checkpoint and every upstream contract;
# - validate the accelerated REC engine against the exact frozen Step 4B smoke outputs;
# - execute all 270 noise/seed conditions with scientifically identical inputs and outputs;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze the complete Project 18 raw-result root for independent Step 5B revalidation.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–17;
# - no prior-project condition-output access;
# - clean evaluation data remain immutable;
# - incomplete condition outputs are preserved in quarantine before rerun.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)

print("=" * 136)
print("=== PROJECT 18 CELL 9 / STEP 5A: RESUME-SAFE FULL 270-CONDITION EXPERIMENT ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_18_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_18_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_18_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "35e5fdd559178e4b0e0573fcc9e1ed335975ec94d0ad13117d2f72aeb7bde0a5"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "c558b8c3c02dfa5e9c83b56fa2143a051a9037db90b8796d3179e5add5be22b2"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "ca33c05f49ed7c5cb359a137d4f57831be4a58816a04320fcbca6798ae88a49a"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8045637a0a6d7bd3e9ec522559847407c9c39afb20f4b0e575f7841f005f5d39"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "7eff07150ccd7f05f3b711f72fbf9e96670401e2c41d8ad45245403d025679a2"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
)
EXPECTED_REGISTRY_SHA256 = (
    "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
)

EXPECTED_REGISTERED_PROJECTS = 17

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 45_140
EXPECTED_RAW_EVAL_ROWS = 21_968
EXPECTED_MODEL_TRAIN_ROWS = 8_232
EXPECTED_MODEL_EVAL_ROWS = 2_348
EXPECTED_MODEL_ROWS = 10_580
EXPECTED_MODEL_TRAIN_FAILURES = 142
EXPECTED_MODEL_EVAL_FAILURES = 31
EXPECTED_FAILING_EVAL_BUILDS = 12
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 113
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 1_354_200
ACCELERATED_ENGINE_VERSION = "PROJECT_18_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_ZERO_TIE_FROZEN_ORDER"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/cantaloupe-project@cantaloupe")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_18_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_18_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_18_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_18_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_18_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_18_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_18_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_18_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
INCOMPLETE_BACKUP_ROOT = FULL_EXPERIMENT_ROOT / "incomplete_condition_backups"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_18_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)

def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)

def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)

def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()

def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")

def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })

def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )

def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )

def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )

def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )

def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)

def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the exact Step 2B-frozen InferredTestOrder is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result

def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))

def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    audit_columns = [
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "RawTrainingRows",
        "NumberFlipped",
        "ExpectedNumberFlipped",
        "PassToFailure",
        "FailureToPass",
        "ModelTrainingRows",
        "ModelLabelChanges",
        "ExpectedModelLabelChanges",
        "TrainingFailures",
        "ExpectedTrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ReconstructedRows",
        "Predictors",
        "MLFits",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ]
    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)

def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]

def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs

DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]

def directory_manifest(root):
    root = Path(root)
    rows = []

    if root.exists():
        for path in sorted(
            [
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ],
            key=lambda candidate: candidate.relative_to(root).as_posix(),
        ):
            rows.append({
                "RelativePath": path.relative_to(root).as_posix(),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            })

    return pd.DataFrame(
        rows,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )

def directory_root_hash(manifest):
    if manifest is None:
        raise TypeError(
            "Directory manifest cannot be None."
        )

    missing_columns = [
        column
        for column in DIRECTORY_MANIFEST_COLUMNS
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Directory manifest is missing required columns: "
            + ", ".join(missing_columns)
        )

    digest = hashlib.sha256()

    if manifest.empty:
        return digest.hexdigest()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()

# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 18 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 18 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 18 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 18 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

if smoke_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Smoke-test checkpoint active reservations differ."
    )

# Step 4B validates the runtime-priority rule against the frozen Step 4A
# runtime checkpoint, but its checkpoint schema does not duplicate that field.
# Therefore, validate the frozen linkage instead of requiring an absent key.
if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke-test checkpoint does not link to the frozen runtime contract."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )

# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–17."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–17 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 18 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 18 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 18 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)

# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 18 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )["BuildID"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing the V6-frozen InferredTestOrder column."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "InferredTestOrder": np.concatenate((
        raw_training["InferredTestOrder"].to_numpy(dtype=np.int64),
        raw_evaluation["InferredTestOrder"].to_numpy(dtype=np.int64),
    )),
})

if combined_history.duplicated(
    subset=["Test", "InferredTestOrder"],
    keep=False,
).any():
    raise RuntimeError(
        "Accelerated REC history contains duplicate V6 per-test order keys."
    )

# Project 18 must use the exact per-test execution order frozen by Step 2B.
# The fixed chronological Build-ID tie-break is not the REC history order for this project.
combined_history = (
    combined_history.sort_values(
        ["Test", "InferredTestOrder"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "BuildID",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)

# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND FULL CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}

def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }

def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination

print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending conditions:", EXPECTED_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The frozen 0% and 50% seed-1 smoke conditions are executed/validated first.
equivalence_records_by_key = {}
for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
    equivalence_row = condition_plan.loc[
        condition_plan["ConditionID"].eq(equivalence_key)
    ]
    if len(equivalence_row) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
        )
    equivalence_plan_row = next(equivalence_row.itertuples(index=False))
    equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
    if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
        equivalence_record = compare_full_condition_to_smoke(
            equivalence_key,
            equivalence_dir,
        )
        if not equivalence_record["Pass"]:
            raise RuntimeError(
                f"Existing accelerated condition {equivalence_key} differs from Step 4B."
            )
        equivalence_records_by_key[equivalence_key] = equivalence_record

smoke_first_plan = condition_plan.loc[
    condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].copy()
smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
    key: index
    for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
})
smoke_first_plan = smoke_first_plan.sort_values(
    "__SmokeOrder",
    kind="mergesort",
).drop(columns="__SmokeOrder")
remaining_plan = condition_plan.loc[
    ~condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].sort_values("ConditionOrder", kind="mergesort")
execution_plan = pd.concat(
    [smoke_first_plan, remaining_plan],
    ignore_index=True,
)

if len(execution_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError("Accelerated execution plan does not contain 270 conditions.")

if equivalence_records_by_key:
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for plan_row in execution_plan.itertuples(index=False):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[{condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = pd.read_parquet(
            RNG_MANIFEST_PATH,
            filters=[("RepetitionSeed", "==", repetition_seed)],
        )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(eval_build_array == int(build_id))
            random_scores[build_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(build_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[{condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical full-history reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        atomic_csv(
            ACCELERATED_EQUIVALENCE_PATH,
            pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                "ConditionKey",
                kind="mergesort",
            ),
        )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": "PROJECT_11_FULL_EXPERIMENT_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "PriorProjectConditionOutputsAccessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()

# --------------------------------------------------------------------------------------------------
# 9. FINAL 270-CONDITION REVALIDATION
# --------------------------------------------------------------------------------------------------

print("\nValidating all 270 completed conditions.")

inventory_records = []
condition_audits = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for plan_row in condition_plan.itertuples(index=False):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Final validation failed for {plan_row.ConditionID}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    project_run_frames.append(pd.read_csv(condition_dir / "project_runs.csv"))
    build_metric_frames.append(pd.read_csv(condition_dir / "build_metrics.csv"))
    model_fit_frames.append(pd.read_csv(condition_dir / "model_fits.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

condition_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)

baseline_invariance_records = []
for repetition_seed in REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~baseline_invariance["Pass"]).sum())
qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

raw_training_hash_after = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_after = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_after = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_after = sha256_file(MODEL_EVALUATION_COHORT_PATH)
registry_sha256_after = sha256_file(REGISTRY_PATH)

current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]

noise_plan_hash_mismatches = int(
    combined_condition_audit[
        "ExpectedFlipMaskSHA256"
    ].ne(combined_condition_audit["ActualFlipMaskSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyRawVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyRawVerdictSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyModelVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyModelVerdictSHA256"]).sum()
)

project_metric_columns = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
project_metric_values = combined_project_runs[
    project_metric_columns
].to_numpy(dtype=float)

if not ACCELERATED_EQUIVALENCE_PATH.is_file():
    raise FileNotFoundError(
        "Accelerated-engine equivalence audit is missing."
    )
accelerated_equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)
accelerated_equivalence_passes = int(
    accelerated_equivalence["Pass"].astype(bool).sum()
)

validation_records = []
add_check(validation_records, "Step 4B passed", EXPECTED_STEP4B_STATUS, smoke_checkpoint.get("Status"), smoke_checkpoint.get("Status") == EXPECTED_STEP4B_STATUS)
add_check(validation_records, "Smoke checkpoint SHA-256", EXPECTED_SMOKE_CHECKPOINT_SHA256, smoke_checkpoint_sha256, smoke_checkpoint_sha256 == EXPECTED_SMOKE_CHECKPOINT_SHA256)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Accelerated smoke-equivalence rows", 2, len(accelerated_equivalence), len(accelerated_equivalence) == 2)
add_check(validation_records, "Accelerated smoke-equivalence keys", sorted(SMOKE_EQUIVALENCE_KEYS), sorted(accelerated_equivalence["ConditionKey"].tolist()), sorted(accelerated_equivalence["ConditionKey"].tolist()) == sorted(SMOKE_EQUIVALENCE_KEYS))
add_check(validation_records, "Accelerated smoke-equivalence failures", 0, int((~accelerated_equivalence["Pass"].astype(bool)).sum()), accelerated_equivalence["Pass"].astype(bool).all())
add_check(validation_records, "Completed conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation_records, "Noise levels", NOISE_LEVELS, sorted(condition_inventory["NoisePercent"].unique().tolist()), sorted(condition_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Repetition seeds", REPETITION_SEEDS, sorted(condition_inventory["RepetitionSeed"].unique().tolist()), sorted(condition_inventory["RepetitionSeed"].unique().tolist()) == REPETITION_SEEDS)
add_check(validation_records, "Duplicate condition keys", 0, int(condition_inventory["ConditionKey"].duplicated(keep=False).sum()), not condition_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Duplicate condition coordinates", 0, int(condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).sum()), not condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).any())
add_check(validation_records, "Condition-order sequence", list(range(1, EXPECTED_CONDITIONS + 1)), condition_inventory["ConditionOrder"].tolist(), condition_inventory["ConditionOrder"].tolist() == list(range(1, EXPECTED_CONDITIONS + 1)))
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(condition_inventory["Files"].unique().tolist()), condition_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation_records, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation_records, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation_records, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation_records, "Model fits", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation_records, "Condition-audit rows", EXPECTED_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_CONDITIONS)
add_check(validation_records, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Active predictors", EXPECTED_PREDICTORS, sorted(combined_condition_audit["Predictors"].unique().tolist()), combined_condition_audit["Predictors"].eq(EXPECTED_PREDICTORS).all())
add_check(validation_records, "Condition statuses", [CONDITION_STATUS], sorted(combined_condition_audit["Status"].unique().tolist()), combined_condition_audit["Status"].eq(CONDITION_STATUS).all())
add_check(validation_records, "Model-fit failures", 0, int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()), combined_model_fits["Status"].eq("PASS_MODEL_FIT").all())
add_check(validation_records, "Project-run technique set", sorted(ALL_TECHNIQUES), sorted(combined_project_runs["Technique"].unique().tolist()), sorted(combined_project_runs["Technique"].unique().tolist()) == sorted(ALL_TECHNIQUES))
add_check(validation_records, "Model-fit technique set", sorted(ML_TECHNIQUES), sorted(combined_model_fits["Technique"].unique().tolist()), sorted(combined_model_fits["Technique"].unique().tolist()) == sorted(ML_TECHNIQUES))
add_check(validation_records, "Zero-noise conditions", len(REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "QTF global score variants", 1, qtf_global_score_variants, qtf_global_score_variants == 1)
add_check(validation_records, "QTF global rank variants", 1, qtf_global_rank_variants, qtf_global_rank_variants == 1)
add_check(validation_records, "Project metrics non-finite", 0, int((~np.isfinite(project_metric_values)).sum()), np.isfinite(project_metric_values).all())
add_check(validation_records, "Project metrics outside [0,1]", 0, int(((project_metric_values < 0) | (project_metric_values > 1)).sum()), bool(((project_metric_values >= 0) & (project_metric_values <= 1)).all()))
add_check(validation_records, "Raw training cohort unchanged", raw_training_hash_before, raw_training_hash_after, raw_training_hash_after == raw_training_hash_before)
add_check(validation_records, "Raw evaluation cohort unchanged", raw_evaluation_hash_before, raw_evaluation_hash_after, raw_evaluation_hash_after == raw_evaluation_hash_before)
add_check(validation_records, "Model training cohort unchanged", model_training_hash_before, model_training_hash_after, model_training_hash_after == model_training_hash_before)
add_check(validation_records, "Model evaluation cohort unchanged", model_evaluation_hash_before, model_evaluation_hash_after, model_evaluation_hash_after == model_evaluation_hash_before)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Registry rows", EXPECTED_REGISTERED_PROJECTS, len(registry), len(registry) == EXPECTED_REGISTERED_PROJECTS)
for predecessor_number, predecessor_project in required_registered_identities.items():
    predecessor_actual = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            project_column,
        ].iloc[0]
    )
    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        predecessor_actual,
        predecessor_actual == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    runtime_checkpoint.get("ActiveReservations"),
    runtime_checkpoint.get("ActiveReservations") == EXPECTED_ACTIVE_RESERVATIONS,
)
add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)
add_check(
    validation_records,
    "Smoke checkpoint runtime-contract linkage",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    ),
    smoke_checkpoint.get(
        "RuntimeCheckpointSHA256"
    )
    == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(validation_records, "Registry Project 18 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

validation = pd.DataFrame(validation_records)
failed_validation = validation.loc[~validation["Pass"]]

print("\nStep 5A validation:")
display(validation)

if not failed_validation.empty:
    print("\nFailed Step 5A checks:")
    display(failed_validation)
    print("\nCompleted condition checkpoints remain resume-safe.")
    raise RuntimeError(
        "PROJECT 18 STEP 5A FINAL VALIDATION FAILED."
    )

# --------------------------------------------------------------------------------------------------
# 10. FREEZE FULL RAW ROOT AND STEP 5A CHECKPOINT
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(STEP5A_VALIDATION_PATH, validation)

full_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    STEP5A_VALIDATION_PATH,
]
aggregate_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in aggregate_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "AcceleratedSmokeEquivalenceRows": len(accelerated_equivalence),
    "AcceleratedSmokeEquivalenceFailures": int((~accelerated_equivalence["Pass"].astype(bool)).sum()),
    "AcceleratedEquivalenceAudit": str(ACCELERATED_EQUIVALENCE_PATH),
    "CompletedAtUTC": completed_at_utc,
    "SmokeCheckpointSHA256": smoke_checkpoint_sha256,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "RawRoot": str(FULL_RAW_RESULT_ROOT),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "RawManifest": str(RAW_MANIFEST_PATH),
    "RawManifestSHA256": sha256_file(RAW_MANIFEST_PATH),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisRun": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "FullExecutionSecondsThisInvocation": float(full_execution_seconds),
    "AggregateOutputManifest": aggregate_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To16Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "EvaluationCohortImmutable": True,
    "ResumeSafe": True,
}
atomic_json(STEP5A_REPORT_PATH, report_payload)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint_payload)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "Conditions": len(condition_inventory),
    "MLFits": len(combined_model_fits),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "FailedValidationChecks": len(failed_validation),
    "RegistryModified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5A_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_CONDITIONS,
        "ExpectedConditions": EXPECTED_CONDITIONS,
        "RawRootSHA256": raw_root_sha256,
        "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "RegistryModified": False,
        "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,

        "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,
    },
)

checkpoint_readback = load_json(STEP5A_CHECKPOINT_PATH)
status_readback = load_json(STEP5A_STATUS_PATH)
if checkpoint_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A checkpoint readback failed.")
if status_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A status readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during Step 5A finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 18 CELL 9 / STEP 5A ACCELERATED RESULT ===")
print("=" * 136)
print()
print("Project:", PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[14],
)
print(
    "Project 15 identity:",
    required_registered_identities[15],
)
print(
    "Project 16 identity:",
    required_registered_identities[16],
)
print(
    "Project 17 identity:",
    required_registered_identities[17],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Accelerated engine:")
print("Engine version:", ACCELERATED_ENGINE_VERSION)
print("Clean REC mismatches:", accelerated_clean_mismatch_values)
print("Frozen smoke-equivalence failures:", int((~accelerated_equivalence["Pass"].astype(bool)).sum()))
print()
print("Full experiment:")
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Raw result freeze:")
print("Raw files:", raw_files)
print("Raw bytes:", raw_bytes)
print("Raw root SHA-256:", raw_root_sha256)
print()
print("Checkpoint/resume:")
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Resume safe:", True)
print()
print("Baselines and metrics:")
print("Baseline invariance failures:", baseline_invariance_failures)
print("QTF global score variants:", qtf_global_score_variants)
print("QTF global rank variants:", qtf_global_rank_variants)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 18 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–17 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Step 5A checkpoint:")
print(STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print()
print("Runtime seconds this invocation:", round(full_execution_seconds, 2))
print()
print("STATUS:", STEP5A_STATUS)
print("=" * 136)


=== PROJECT 18 CELL 9 / STEP 5A: RESUME-SAFE FULL 270-CONDITION EXPERIMENT ===

Loading frozen Project 18 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 241 / 10580 / 1186

Scanning existing condition checkpoints...
Valid completed conditions: 0
Incomplete/invalid condition directories: 0
Pending conditions: 270

Loading deterministic RNG stream for seed 1.

--------------------------------------------------------------------------------------------------------------
[1/270] Running noise_00__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_00__seed_01
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 142 | condition seconds: 10.30

--------------------------------------------------------------------------------------------------------------
[9/270] Running noise_50__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_50__seed_01
  Completed: noise_50__seed_01
  Raw flips: 22497 | model-label changes: 4069 | dependent REC changes: 115478
  Training failures: 4065 | condition seconds: 10.35

--------------------------------------------------------------------------------------------------------------
[2/270] Running noise_05__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_30
  Raw flips: 18067 | model-label changes: 3230 | dependent REC changes: 112859
  Training failures: 3242 | condition seconds: 14.90

--------------------------------------------------------------------------------------------------------------
[270/270] Running noise_50__seed_30
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_30
  Raw flips: 22459 | model-label changes: 4024 | dependent REC changes: 115741
  Training failures: 4012 | condition seconds: 15.20

Validating all 270 completed conditions.

Step 5A validation:


,Check,Expected,Actual,Pass
0,Step 4B passed,PASS_PROJECT_18_TWO_CONDITION_END_TO_END_SMOKE...,PASS_PROJECT_18_TWO_CONDITION_END_TO_END_SMOKE...,True
1,Smoke checkpoint SHA-256,c558b8c3c02dfa5e9c83b56fa2143a051a9037db90b879...,c558b8c3c02dfa5e9c83b56fa2143a051a9037db90b879...,True
2,Accelerated clean REC mismatches,0,0,True
3,Accelerated smoke-equivalence rows,2,2,True
4,Accelerated smoke-equivalence keys,"[noise_00__seed_01, noise_50__seed_01]","[noise_00__seed_01, noise_50__seed_01]",True
5,Accelerated smoke-equivalence failures,0,0,True
6,Completed conditions,270,270,True
7,Noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
8,Repetition seeds,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
9,Duplicate condition keys,0,0,True



=== PROJECT 18 CELL 9 / STEP 5A ACCELERATED RESULT ===

Project: cantaloupe-project@cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Accelerated engine:
Engine version: PROJECT_18_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_ZERO_TIE_FROZEN_ORDER
Clean REC mismatches: 0
Frozen smoke-equivalence failures: 0

Full experiment:
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 4437720
Build-metric rows: 22680
Project-run rows: 1890
Condition-audit rows: 270
Training-median rows: 40770

Raw result freeze:
Raw files: 2160
Raw bytes: 78874343
Raw root SHA-256: 3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2

In [35]:
# ==================================================================================================
# PROJECT 18 — CELL 10 / STEP 5B
# CORRECTED PROJECT-SPECIFIC COUNT CONTRACT, RAW REVALIDATION, AND COMPACT AGGREGATION
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# CURRENT REGISTRY CONTRACT:
# - Projects 1–17 must be present exactly once and COMPLETE_AND_FROZEN.
# - Project 11 must be apache@shardingsphere.
# - Project 12 must be zolyfarkas@spf4j.
# - Project 13 must be jcabi@jcabi-github.
# - Project 14 must be JMRI@JMRI.
# - Project 15 must be eclipse@steady.
# - Project 16 must be apache@rocketmq.
# - Project 17 must be yamcs@Yamcs.
# - Project 18 must still be absent.
#
# THIS CELL:
# - independently hashes all 2,160 Project 18 raw files;
# - validates every condition checkpoint and compact output;
# - recounts all 4,437,720 compressed ranking rows;
# - independently validates noise hashes, REC invariance, metrics, and baselines;
# - creates analysis-ready aggregates across all 30 seeds;
# - writes the Project 18 Step 5B checkpoint;
# - does not rerun conditions or fit models;
# - does not access or modify prior-project condition outputs;
# - does not register Project 18.
#
# BASELINE-INVARIANCE CONTRACT:
# - Random and QTF-Avg are compared independently within each metric;
# - APFDc and APFD are never compared against one another.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gzip, hashlib, json, os, time
import numpy as np
import pandas as pd

print('=' * 136)
print('=== PROJECT 18 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===')
print('=' * 136)

PROJECT_NUMBER = 18
PROJECT_NAME = 'cantaloupe-project@cantaloupe'
PROJECT_SLUG = 'cantaloupe-project__cantaloupe'
PROJECT_SHORT = 'CANTALOUPE'
STEP5A_STATUS = 'PASS_PROJECT_18_FULL_270_CONDITION_EXPERIMENT_COMPLETE'
CONDITION_STATUS = 'PASS_FULL_CONDITION'
STEP5B_STATUS = 'PASS_PROJECT_18_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN'

EXPECTED_STEP5A_SHA = '3c2757656b27e8c125cec92124f1656f823cbc70a475ac92cecf075dd93f7f0b'
EXPECTED_RAW_ROOT_SHA = '3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702'
EXPECTED_REGISTRY_SHA = 'f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd'
EXPECTED_SOURCE_ROOT_SHA = 'd3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8'

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
ML_TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
INVARIANT_BASELINES = ['Random', 'QTF-Avg']
PROJECT_METRICS = ['MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']
BUILD_METRICS = ['APFDc', 'APFD']
EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    'ModelTrainingRows ascending',
    'ModelEvaluationRows ascending',
    'RawExecutionRows ascending',
    'Project ascending',
]

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 78_874_343
EXPECTED_RANKING_ROWS_PER_CONDITION = 2_348 * 7
EXPECTED_BUILD_ROWS_PER_CONDITION = 12 * 7
EXPECTED_PROJECT_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_TOTAL_RANKING_ROWS = 4_437_720
EXPECTED_TOTAL_BUILD_ROWS = 22_680
EXPECTED_TOTAL_PROJECT_ROWS = 1_890
EXPECTED_TOTAL_FIT_ROWS = 1_080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40_770

# Project 18 fixed evaluation counts.
# These are validated in Step 1A/1B, Step 2A, Step 4A, Step 4B, and Step 5A.
EXPECTED_SCORED_FAILING_BUILDS = 12
EXPECTED_EVALUATION_BUILDS = 113
EXPECTED_EVALUATION_FAILURES = 31

EXPECTED_CONDITION_FILES = {
    'rankings.csv.gz', 'build_metrics.csv', 'project_runs.csv', 'model_fits.csv',
    'training_medians.csv', 'condition_audit.csv', 'condition_summary.json', 'COMPLETE.json'
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {'condition_summary.json', 'COMPLETE.json'}

# Mount only Drive. No source re-extraction is needed for Step 5B.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Thesis_Experiment')
NOTES = ROOT / 'Notes'
RESULTS = ROOT / 'Results'
REGISTRY = NOTES / 'completed_project_registry.csv'
PROJECT_ROOT = RESULTS / 'Aggregated' / PROJECT_SLUG
RAW_ROOT = RESULTS / 'Raw' / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f'{PROJECT_SHORT}_full_experiment'
PLAN = PROJECT_ROOT / f'{PROJECT_SHORT}_noise_plan' / f'{PROJECT_SHORT}_condition_plan.csv'
STEP5A_CHECKPOINT = NOTES / 'project_18_step5a_checkpoint.json'
STEP5A_STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5a_status.json'
STEP5A_REPORT = FULL_ROOT / f'{PROJECT_SHORT}_step5a_report.json'
STEP5A_RAW_MANIFEST = FULL_ROOT / f'{PROJECT_SHORT}_raw_manifest.csv'
STEP5A_BASELINE = FULL_ROOT / f'{PROJECT_SHORT}_baseline_invariance.csv'

OUT = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b'
CURRENT_MANIFEST = OUT / f'{PROJECT_SHORT}_independent_raw_manifest.csv'
CONDITION_INVENTORY = OUT / f'{PROJECT_SHORT}_independent_condition_inventory.csv'
REVALIDATED_PROJECT_RUNS = OUT / f'{PROJECT_SHORT}_revalidated_project_runs.csv'
REVALIDATED_BUILD_METRICS = OUT / f'{PROJECT_SHORT}_revalidated_build_metrics.csv'
REVALIDATED_MODEL_FITS = OUT / f'{PROJECT_SHORT}_revalidated_model_fits.csv'
REVALIDATED_CONDITION_AUDIT = OUT / f'{PROJECT_SHORT}_revalidated_condition_audit.csv'
REVALIDATED_MEDIANS = OUT / f'{PROJECT_SHORT}_revalidated_training_medians.csv'
NOISE_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_technique_summary.csv'
SEED_DELTAS = OUT / f'{PROJECT_SHORT}_seed_level_noise_deltas.csv'
DELTA_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_delta_summary.csv'
VALIDATION_PATH = OUT / f'{PROJECT_SHORT}_step5b_validation.csv'
REPORT_PATH = OUT / f'{PROJECT_SHORT}_step5b_report.json'
STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b_status.json'
CHECKPOINT_PATH = NOTES / 'project_18_step5b_checkpoint.json'


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write('\n')
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    df.to_csv(tmp, index=False, lineterminator='\n')
    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise RuntimeError(f'Could not resolve {label}; columns={list(columns)}')


def normalize_manifest(df, label):
    p = resolve_col(df.columns, ['RelativePath'], f'{label} path')
    b = resolve_col(df.columns, ['Bytes', 'SizeBytes'], f'{label} bytes')
    s = resolve_col(df.columns, ['SHA256'], f'{label} sha')
    out = df[[p, b, s]].copy(); out.columns = ['RelativePath', 'Bytes', 'SHA256']
    out['RelativePath'] = out['RelativePath'].astype(str).str.replace('\\', '/', regex=False)
    out['Bytes'] = pd.to_numeric(out['Bytes'], errors='raise').astype('int64')
    out['SHA256'] = out['SHA256'].astype(str).str.lower()
    return out.sort_values('RelativePath', kind='mergesort').reset_index(drop=True)


def root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values('RelativePath', kind='mergesort').itertuples(index=False):
        h.update(f'{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n'.encode('utf-8'))
    return h.hexdigest()


def gzip_rows(path):
    n = 0
    with gzip.open(path, 'rb') as f:
        for _ in f:
            n += 1
    return max(0, n - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Pass': bool(passed)})


def metric_nonfinite(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int((~np.isfinite(arr)).sum())


def metric_outside(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int(((arr < 0) | (arr > 1)).sum())


# Required Drive inputs.
required = [REGISTRY, PLAN, STEP5A_CHECKPOINT, STEP5A_STATUS_PATH, STEP5A_REPORT,
            STEP5A_RAW_MANIFEST, STEP5A_BASELINE, RAW_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Project 18 Step 5B inputs:\n' + '\n'.join(missing))

# Frozen Step 5A and registry.
step5a_sha = sha256_file(STEP5A_CHECKPOINT)
step5a = load_json(STEP5A_CHECKPOINT)
step5a_status = load_json(STEP5A_STATUS_PATH)
step5a_report = load_json(STEP5A_REPORT)
if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(f'Step 5A checkpoint SHA differs. Expected={EXPECTED_STEP5A_SHA}; actual={step5a_sha}')
for label, payload in [('checkpoint', step5a), ('status', step5a_status), ('report', step5a_report)]:
    if payload.get('Status') != STEP5A_STATUS:
        raise RuntimeError(f'Step 5A {label} is not in PASS state.')
if step5a.get('SourceRootSHA256') != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError('Step 5A source-root SHA differs.')
if step5a.get('RawRootSHA256') != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError('Step 5A frozen raw-root SHA differs.')
if step5a.get('ActiveReservations') != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError('Step 5A active-reservation state differs.')
if step5a.get('RuntimePriorityRule') != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError('Step 5A runtime-priority rule differs.')

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f'Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA}; actual={registry_sha_before}')
registry = pd.read_csv(REGISTRY, dtype=str).fillna('')
pn_col = resolve_col(registry.columns, ['ProjectNumber'], 'registry ProjectNumber')
project_col = resolve_col(registry.columns, ['Project'], 'registry Project')
st_col = resolve_col(registry.columns, ['Status'], 'registry Status')
pnums = pd.to_numeric(registry[pn_col], errors='raise').astype(int)

if len(registry) != 17 or sorted(pnums.tolist()) != list(range(1, 18)):
    raise RuntimeError(
        'Registry must contain exactly Projects 1–17 before Project 18 Step 5B.'
    )

if not registry[st_col].eq('COMPLETE_AND_FROZEN').all():
    raise RuntimeError(
        'Projects 1–17 are not all COMPLETE_AND_FROZEN.'
    )

required_registered_identities = {
    11: 'apache@shardingsphere',
    12: 'zolyfarkas@spf4j',
    13: 'jcabi@jcabi-github',
    14: 'JMRI@JMRI',
    15: 'eclipse@steady',
    16: 'apache@rocketmq',
    17: 'yamcs@Yamcs',
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(required_number)
    ]

    if (
        len(matching_rows) != 1
        or matching_rows.iloc[0][project_col] != required_project
    ):
        raise RuntimeError(
            'A required frozen predecessor has a different registry identity.\n'
            f'Project number: {required_number}\n'
            f'Expected project: {required_project}'
        )

if pnums.eq(18).any() or registry[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError(
        'Project 18 is unexpectedly already present in the completion registry.'
    )

# Validate Step 5A aggregate-output manifest.
agg_manifest = step5a.get('AggregateOutputManifest', [])
if not isinstance(agg_manifest, list) or not agg_manifest:
    raise RuntimeError('Step 5A checkpoint has no AggregateOutputManifest.')
agg_failures = 0
for item in agg_manifest:
    p = Path(item['Path'])
    ok = p.is_file() and p.stat().st_size == int(item['Bytes']) and sha256_file(p) == str(item['SHA256'])
    agg_failures += int(not ok)
if agg_failures:
    raise RuntimeError(f'{agg_failures} Step 5A aggregate outputs changed.')

# Independently hash all raw files.
print('\nIndependently hashing all 2,160 raw files.')
hash_start = time.perf_counter()
paths = sorted([p for p in RAW_ROOT.rglob('*') if p.is_file()], key=lambda p: p.relative_to(RAW_ROOT).as_posix())
manifest_rows = []
for i, p in enumerate(paths, 1):
    manifest_rows.append({'RelativePath': p.relative_to(RAW_ROOT).as_posix(),
                          'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)})
    if i % 200 == 0 or i == len(paths):
        print(f'  Raw hashing progress: {i} / {len(paths)} files')
current_manifest = normalize_manifest(pd.DataFrame(manifest_rows), 'current manifest')
hash_seconds = time.perf_counter() - hash_start
frozen_manifest = normalize_manifest(pd.read_csv(STEP5A_RAW_MANIFEST), 'frozen manifest')
current_raw_sha = root_hash(current_manifest)
current_raw_bytes = int(current_manifest['Bytes'].sum())
merged_manifest = frozen_manifest.merge(current_manifest, on='RelativePath', how='outer',
                                        suffixes=('_frozen', '_current'), indicator=True)
missing_raw = int(merged_manifest['_merge'].eq('left_only').sum())
unexpected_raw = int(merged_manifest['_merge'].eq('right_only').sum())
size_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['Bytes_frozen'].ne(merged_manifest['Bytes_current'])).sum())
hash_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['SHA256_frozen'].ne(merged_manifest['SHA256_current'])).sum())

# Condition-by-condition independent validation and compact reload.
plan = pd.read_csv(PLAN, low_memory=False)
id_col = resolve_col(plan.columns, ['ConditionID', 'ConditionKey'], 'condition identifier')
order_col = resolve_col(plan.columns, ['ConditionOrder'], 'condition order')
noise_col = resolve_col(plan.columns, ['NoisePercent'], 'noise percent')
seed_col = resolve_col(plan.columns, ['RepetitionSeed'], 'repetition seed')
for col in [order_col, noise_col, seed_col]:
    plan[col] = pd.to_numeric(plan[col], errors='raise').astype(int)
plan = plan.sort_values(order_col, kind='mergesort').reset_index(drop=True)

inventory_rows, project_frames, build_frames, fit_frames, audit_frames, median_frames = [], [], [], [], [], []
marker_fail = summary_fail = file_set_fail = embedded_fail = ranking_count_fail = 0
print('\nRevalidating all 270 condition directories.')
condition_start = time.perf_counter()
for i, row in enumerate(plan.itertuples(index=False), 1):
    key = str(getattr(row, id_col)); order = int(getattr(row, order_col))
    noise = int(getattr(row, noise_col)); seed = int(getattr(row, seed_col))
    d = RAW_ROOT / key
    if not d.is_dir():
        raise FileNotFoundError(f'Missing condition directory: {d}')
    actual_files = {p.name for p in d.iterdir() if p.is_file()}
    file_ok = actual_files == EXPECTED_CONDITION_FILES
    file_set_fail += int(not file_ok)
    complete_path, summary_path = d / 'COMPLETE.json', d / 'condition_summary.json'
    complete, summary = load_json(complete_path), load_json(summary_path)
    complete_ok = (complete.get('Status') == CONDITION_STATUS and complete.get('ConditionKey') == key and
                   str(complete.get('ConditionSummaryPath')) == str(summary_path) and
                   str(complete.get('ConditionSummarySHA256')).lower() == sha256_file(summary_path))
    summary_ok = (summary.get('Status') == CONDITION_STATUS and summary.get('ConditionKey') == key and
                  int(summary.get('NoisePercent', -1)) == noise and int(summary.get('RepetitionSeed', -1)) == seed)
    marker_fail += int(not complete_ok); summary_fail += int(not summary_ok)
    output_manifest = summary.get('OutputManifest', [])
    local_embedded_fail = 0
    names = set()
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        local_embedded_fail += 1
    else:
        for item in output_manifest:
            p = Path(item.get('Path', '')); names.add(p.name)
            ok = (p.parent == d and p.is_file() and p.stat().st_size == int(item.get('Bytes', -1)) and
                  sha256_file(p) == str(item.get('SHA256', '')).lower())
            local_embedded_fail += int(not ok)
        local_embedded_fail += int(names != CONDITION_OUTPUT_FILES)
    embedded_fail += local_embedded_fail
    ranking_rows = gzip_rows(d / 'rankings.csv.gz')
    ranking_count_fail += int(ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION)
    build = pd.read_csv(d / 'build_metrics.csv', low_memory=False)
    project = pd.read_csv(d / 'project_runs.csv', low_memory=False)
    fits = pd.read_csv(d / 'model_fits.csv', low_memory=False)
    audit = pd.read_csv(d / 'condition_audit.csv', low_memory=False)
    medians = pd.read_csv(d / 'training_medians.csv', low_memory=False)
    expected_counts = [EXPECTED_BUILD_ROWS_PER_CONDITION, EXPECTED_PROJECT_ROWS_PER_CONDITION,
                       EXPECTED_FIT_ROWS_PER_CONDITION, 1, EXPECTED_MEDIAN_ROWS_PER_CONDITION]
    actual_counts = [len(build), len(project), len(fits), len(audit), len(medians)]
    if actual_counts != expected_counts:
        raise RuntimeError(f'{key}: compact output counts differ. expected={expected_counts}; actual={actual_counts}')
    for field, count in [('RankingRows', ranking_rows), ('BuildMetricRows', len(build)),
                         ('ProjectRunRows', len(project)), ('MLFits', len(fits)),
                         ('TrainingMedianRows', len(medians))]:
        if int(summary.get(field, -1)) != count:
            raise RuntimeError(f'{key}: condition_summary {field} differs.')
    project_frames.append(project); build_frames.append(build); fit_frames.append(fits)
    audit_frames.append(audit); median_frames.append(medians)
    inventory_rows.append({
        'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
        'ConditionOrder': order, 'ConditionKey': key, 'NoisePercent': noise, 'RepetitionSeed': seed,
        'ConditionDirectory': str(d), 'CompletionStatus': complete.get('Status'),
        'SummaryStatus': summary.get('Status'), 'Files': len(actual_files),
        'ConditionBytes': int(sum(p.stat().st_size for p in d.iterdir() if p.is_file())),
        'RankingRows': ranking_rows, 'BuildMetricRows': len(build), 'ProjectRunRows': len(project),
        'ModelFits': len(fits), 'ConditionAuditRows': len(audit), 'TrainingMedianRows': len(medians),
        'FileSetPass': file_ok, 'CompletionMarkerPass': complete_ok, 'ConditionSummaryPass': summary_ok,
        'EmbeddedManifestFailures': local_embedded_fail,
        'CompletionMarkerSHA256': sha256_file(complete_path), 'ConditionSummarySHA256': sha256_file(summary_path),
    })
    if i % 30 == 0 or i == len(plan):
        print(f'  Condition revalidation progress: {i} / {len(plan)}')
condition_seconds = time.perf_counter() - condition_start

inventory = pd.DataFrame(inventory_rows).sort_values('ConditionOrder', kind='mergesort').reset_index(drop=True)
project_runs = pd.concat(project_frames, ignore_index=True)
build_metrics = pd.concat(build_frames, ignore_index=True)
model_fits = pd.concat(fit_frames, ignore_index=True)
condition_audit = pd.concat(audit_frames, ignore_index=True)
training_medians = pd.concat(median_frames, ignore_index=True)

# Contract audits.
coordinate_count = len(inventory[['NoisePercent', 'RepetitionSeed']].drop_duplicates())
dup_keys = int(inventory.duplicated(['ConditionKey'], keep=False).sum())
dup_coords = int(inventory.duplicated(['NoisePercent', 'RepetitionSeed'], keep=False).sum())
order_viol = int((inventory['ConditionOrder'].to_numpy(int) != np.arange(1, 271)).sum())
files_viol = int(inventory['Files'].ne(8).sum())
ranking_viol = int(inventory['RankingRows'].ne(EXPECTED_RANKING_ROWS_PER_CONDITION).sum())
small_per_condition_viol = int(inventory['BuildMetricRows'].ne(EXPECTED_BUILD_ROWS_PER_CONDITION).sum() +
                               inventory['ProjectRunRows'].ne(7).sum() + inventory['ModelFits'].ne(4).sum() +
                               inventory['ConditionAuditRows'].ne(1).sum() + inventory['TrainingMedianRows'].ne(151).sum())
project_techniques = sorted(project_runs['Technique'].astype(str).unique().tolist())
fit_techniques = sorted(model_fits['Technique'].astype(str).unique().tolist())
dup_project = int(project_runs.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_build = int(build_metrics.duplicated(['ConditionKey', 'Technique', 'Build'], keep=False).sum())
dup_fit = int(model_fits.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_audit = int(condition_audit.duplicated(['ConditionKey'], keep=False).sum())
dup_median = int(training_medians.duplicated(['ConditionKey', 'PredictorOrder'], keep=False).sum())
fit_fail = int((~model_fits['Status'].astype(str).eq('PASS_MODEL_FIT')).sum())
fit_errors = int(model_fits['Error'].fillna('').astype(str).str.len().gt(0).sum())
project_nonfinite = metric_nonfinite(project_runs, PROJECT_METRICS)
project_outside = metric_outside(project_runs, PROJECT_METRICS)
build_nonfinite = metric_nonfinite(build_metrics, BUILD_METRICS)
build_outside = metric_outside(build_metrics, BUILD_METRICS)
median_nonfinite = int((~np.isfinite(pd.to_numeric(training_medians['TrainingMedian'], errors='coerce').to_numpy(float))).sum())
median_predictor_viol = int(training_medians.groupby('ConditionKey')['Predictor'].nunique().ne(151).sum())
scored_build_viol = int(
    project_runs[
        'ScoredFailingBuilds'
    ].ne(
        EXPECTED_SCORED_FAILING_BUILDS
    ).sum()
)

eval_build_viol = int(
    project_runs[
        'EvaluationBuilds'
    ].ne(
        EXPECTED_EVALUATION_BUILDS
    ).sum()
)

eval_failure_viol = int(
    project_runs[
        'EvaluationFailures'
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)
zero = condition_audit[condition_audit['NoisePercent'].eq(0)]
positive = condition_audit[condition_audit['NoisePercent'].gt(0)]
zero_flip_viol = int(zero['NumberFlipped'].ne(0).sum())
zero_model_viol = int(zero['ModelLabelChanges'].ne(0).sum())
zero_rec_viol = int(zero['DependentRECChanges'].ne(0).sum())
pos_raw_viol = int(positive['NumberFlipped'].le(0).sum())
pos_model_viol = int(positive['ModelLabelChanges'].le(0).sum())
pos_rec_viol = int(positive['DependentRECChanges'].le(0).sum())
independent_viol = int(condition_audit['IndependentRECChanges'].ne(0).sum())
independent_recon_viol = int(condition_audit['IndependentReconstructionMismatches'].ne(0).sum())
noise_hash_mismatch = 0
for e, a in [('ExpectedFlipMaskSHA256', 'ActualFlipMaskSHA256'),
             ('ExpectedNoisyRawVerdictSHA256', 'ActualNoisyRawVerdictSHA256'),
             ('ExpectedNoisyModelVerdictSHA256', 'ActualNoisyModelVerdictSHA256')]:
    noise_hash_mismatch += int((condition_audit[e].astype(str) != condition_audit[a].astype(str)).sum())

baseline = pd.read_csv(STEP5A_BASELINE, low_memory=False)
if 'Pass' in baseline.columns:
    bpass = baseline['Pass'].astype(str).str.strip().str.lower().isin({'true', '1'})
    ranking_baseline_fail = int((~bpass).sum())
else:
    mismatch_cols = [c for c in baseline.columns if 'mismatch' in c.lower()]
    ranking_baseline_fail = int(baseline[mismatch_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(float).sum())
# Project-metric invariance must be evaluated independently for each metric.
# The previous V1 expression compared the maximum of one metric with the
# minimum of another metric. Because APFDc and APFD naturally have different
# values, that incorrectly marked all 60 seed/baseline groups as failures even
# though each individual metric was invariant across noise.
metric_baseline_fail = 0
metric_baseline_max_range = 0.0

for _, g in project_runs[
    project_runs['Technique'].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        'RepetitionSeed',
        'Technique',
    ],
    sort=False,
):
    arr = g[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            arr,
            axis=0,
        )
        - np.min(
            arr,
            axis=0,
        )
    )

    metric_baseline_max_range = max(
        metric_baseline_max_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    metric_baseline_fail += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )

# Compact aggregates.
agg_start = time.perf_counter()
noise_summary = project_runs.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Runs=('ConditionKey', 'count'), Seeds=('RepetitionSeed', 'nunique'),
    Mean_MeanAPFDc=('MeanAPFDc', 'mean'), SD_MeanAPFDc=('MeanAPFDc', 'std'), Median_MeanAPFDc=('MeanAPFDc', 'median'),
    Mean_MedianAPFDc=('MedianAPFDc', 'mean'), SD_MedianAPFDc=('MedianAPFDc', 'std'), Median_MedianAPFDc=('MedianAPFDc', 'median'),
    Mean_MeanAPFD=('MeanAPFD', 'mean'), SD_MeanAPFD=('MeanAPFD', 'std'), Median_MeanAPFD=('MeanAPFD', 'median'),
    Mean_MedianAPFD=('MedianAPFD', 'mean'), SD_MedianAPFD=('MedianAPFD', 'std'), Median_MedianAPFD=('MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
clean = project_runs[project_runs['NoisePercent'].eq(0)][['RepetitionSeed', 'Technique'] + PROJECT_METRICS].rename(
    columns={c: f'Clean_{c}' for c in PROJECT_METRICS})
seed_deltas = project_runs.merge(clean, on=['RepetitionSeed', 'Technique'], how='left', validate='many_to_one')
for c in PROJECT_METRICS:
    seed_deltas[f'Delta_{c}'] = seed_deltas[c] - seed_deltas[f'Clean_{c}']
delta_cols = [f'Delta_{c}' for c in PROJECT_METRICS]
seed_deltas = seed_deltas[['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent',
                           'RepetitionSeed', 'Technique'] + PROJECT_METRICS +
                          [f'Clean_{c}' for c in PROJECT_METRICS] + delta_cols].sort_values(
                              ['NoisePercent', 'Technique', 'RepetitionSeed'], kind='mergesort').reset_index(drop=True)
delta_summary = seed_deltas.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Seeds=('RepetitionSeed', 'nunique'),
    Mean_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'mean'), SD_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'std'), Median_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'median'),
    Mean_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'mean'), SD_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'std'), Median_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'median'),
    Mean_Delta_MeanAPFD=('Delta_MeanAPFD', 'mean'), SD_Delta_MeanAPFD=('Delta_MeanAPFD', 'std'), Median_Delta_MeanAPFD=('Delta_MeanAPFD', 'median'),
    Mean_Delta_MedianAPFD=('Delta_MedianAPFD', 'mean'), SD_Delta_MedianAPFD=('Delta_MedianAPFD', 'std'), Median_Delta_MedianAPFD=('Delta_MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
agg_seconds = time.perf_counter() - agg_start
summary_nonfinite = metric_nonfinite(noise_summary, [c for c in noise_summary.columns if c not in {'NoisePercent', 'Technique'}])
delta_nonfinite = metric_nonfinite(delta_summary, [c for c in delta_summary.columns if c not in {'NoisePercent', 'Technique'}])
clean_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['NoisePercent'].eq(0)][delta_cols].to_numpy(float)) > 1e-15).sum())
baseline_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['Technique'].isin(INVARIANT_BASELINES)][delta_cols].to_numpy(float)) > 1e-15).sum())

# Validation table.
checks = []
add_check(checks, 'Step 5A status', STEP5A_STATUS, step5a.get('Status'), step5a.get('Status') == STEP5A_STATUS)
add_check(checks, 'Step 5A checkpoint SHA-256', EXPECTED_STEP5A_SHA, step5a_sha, step5a_sha == EXPECTED_STEP5A_SHA)
add_check(checks, 'Frozen raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, step5a.get('RawRootSHA256'), step5a.get('RawRootSHA256') == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Independent current raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, current_raw_sha, current_raw_sha == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Step 5A aggregate-manifest failures', 0, agg_failures, agg_failures == 0)
add_check(checks, 'Condition marker failures', 0, marker_fail, marker_fail == 0)
add_check(checks, 'Condition summary failures', 0, summary_fail, summary_fail == 0)
add_check(checks, 'Condition file-set failures', 0, file_set_fail, file_set_fail == 0)
add_check(checks, 'Embedded output-manifest failures', 0, embedded_fail, embedded_fail == 0)
add_check(checks, 'Conditions', 270, len(inventory), len(inventory) == 270)
add_check(checks, 'Condition coordinates', 270, coordinate_count, coordinate_count == 270)
add_check(checks, 'Duplicate condition keys', 0, dup_keys, dup_keys == 0)
add_check(checks, 'Duplicate condition coordinates', 0, dup_coords, dup_coords == 0)
add_check(checks, 'Condition-order violations', 0, order_viol, order_viol == 0)
add_check(checks, 'Files-per-condition violations', 0, files_viol, files_viol == 0)
add_check(checks, 'Raw files', EXPECTED_RAW_FILES, len(current_manifest), len(current_manifest) == EXPECTED_RAW_FILES)
add_check(checks, 'Raw bytes', EXPECTED_RAW_BYTES, current_raw_bytes, current_raw_bytes == EXPECTED_RAW_BYTES)
add_check(checks, 'Missing raw files', 0, missing_raw, missing_raw == 0)
add_check(checks, 'Unexpected raw files', 0, unexpected_raw, unexpected_raw == 0)
add_check(checks, 'Raw size mismatches', 0, size_mismatch, size_mismatch == 0)
add_check(checks, 'Raw SHA-256 mismatches', 0, hash_mismatch, hash_mismatch == 0)
add_check(checks, 'Ranking rows', EXPECTED_TOTAL_RANKING_ROWS, int(inventory['RankingRows'].sum()), int(inventory['RankingRows'].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(checks, 'Ranking row-count failures', 0, ranking_count_fail + ranking_viol, ranking_count_fail + ranking_viol == 0)
add_check(checks, 'Project-run rows', EXPECTED_TOTAL_PROJECT_ROWS, len(project_runs), len(project_runs) == EXPECTED_TOTAL_PROJECT_ROWS)
add_check(checks, 'Build-metric rows', EXPECTED_TOTAL_BUILD_ROWS, len(build_metrics), len(build_metrics) == EXPECTED_TOTAL_BUILD_ROWS)
add_check(checks, 'Model-fit rows', EXPECTED_TOTAL_FIT_ROWS, len(model_fits), len(model_fits) == EXPECTED_TOTAL_FIT_ROWS)
add_check(checks, 'Condition-audit rows', EXPECTED_TOTAL_AUDIT_ROWS, len(condition_audit), len(condition_audit) == EXPECTED_TOTAL_AUDIT_ROWS)
add_check(checks, 'Training-median rows', EXPECTED_TOTAL_MEDIAN_ROWS, len(training_medians), len(training_medians) == EXPECTED_TOTAL_MEDIAN_ROWS)
add_check(checks, 'Small rows-per-condition violations', 0, small_per_condition_viol, small_per_condition_viol == 0)
add_check(checks, 'Project-run technique set', sorted(TECHNIQUES), project_techniques, project_techniques == sorted(TECHNIQUES))
add_check(checks, 'Model-fit technique set', sorted(ML_TECHNIQUES), fit_techniques, fit_techniques == sorted(ML_TECHNIQUES))
add_check(checks, 'Duplicate project/build/fit/audit/median rows', 0, dup_project + dup_build + dup_fit + dup_audit + dup_median, dup_project + dup_build + dup_fit + dup_audit + dup_median == 0)
add_check(checks, 'Model-fit failures', 0, fit_fail + fit_errors, fit_fail + fit_errors == 0)
add_check(
    checks,
    'Scored-failing-build count violations',
    0,
    scored_build_viol,
    scored_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-build count violations',
    0,
    eval_build_viol,
    eval_build_viol == 0,
)

add_check(
    checks,
    'Evaluation-failure count violations',
    0,
    eval_failure_viol,
    eval_failure_viol == 0,
)

add_check(
    checks,
    'Combined scored/evaluated/failure count violations',
    0,
    scored_build_viol + eval_build_viol + eval_failure_viol,
    scored_build_viol + eval_build_viol + eval_failure_viol == 0,
)
add_check(checks, 'Project metric invalid values', 0, project_nonfinite + project_outside, project_nonfinite + project_outside == 0)
add_check(checks, 'Build metric invalid values', 0, build_nonfinite + build_outside, build_nonfinite + build_outside == 0)
add_check(checks, 'Training-median invalid values', 0, median_nonfinite + median_predictor_viol, median_nonfinite + median_predictor_viol == 0)
add_check(checks, 'Zero-noise conditions', 30, len(zero), len(zero) == 30)
add_check(checks, 'Zero-noise violations', 0, zero_flip_viol + zero_model_viol + zero_rec_viol, zero_flip_viol + zero_model_viol + zero_rec_viol == 0)
add_check(checks, 'Positive-noise violations', 0, pos_raw_viol + pos_model_viol + pos_rec_viol, pos_raw_viol + pos_model_viol + pos_rec_viol == 0)
add_check(checks, 'Independent REC violations', 0, independent_viol + independent_recon_viol, independent_viol + independent_recon_viol == 0)
add_check(checks, 'Noise-plan hash mismatches', 0, noise_hash_mismatch, noise_hash_mismatch == 0)
add_check(checks, 'Ranking-level baseline-invariance failures', 0, ranking_baseline_fail, ranking_baseline_fail == 0)
add_check(checks, 'Project-metric baseline-invariance failures', 0, metric_baseline_fail, metric_baseline_fail == 0)
add_check(checks, 'Noise-technique summary rows', 63, len(noise_summary), len(noise_summary) == 63)
add_check(checks, 'Noise-technique summary count/nonfinite violations', 0, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite == 0)
add_check(checks, 'Seed-level delta rows', 1890, len(seed_deltas), len(seed_deltas) == 1890)
add_check(checks, 'Clean delta non-zero values', 0, clean_delta_nonzero, clean_delta_nonzero == 0)
add_check(checks, 'Invariant-baseline delta non-zero values', 0, baseline_delta_nonzero, baseline_delta_nonzero == 0)
add_check(checks, 'Noise-delta summary rows', 63, len(delta_summary), len(delta_summary) == 63)
add_check(checks, 'Noise-delta summary count/nonfinite violations', 0, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite == 0)
add_check(checks, 'Registry rows', 17, len(registry), len(registry) == 17)
add_check(checks, 'Active reservations', EXPECTED_ACTIVE_RESERVATIONS, step5a.get('ActiveReservations'), step5a.get('ActiveReservations') == EXPECTED_ACTIVE_RESERVATIONS)
add_check(checks, 'Runtime-priority ranking rule', EXPECTED_RUNTIME_PRIORITY_RULE, step5a.get('RuntimePriorityRule'), step5a.get('RuntimePriorityRule') == EXPECTED_RUNTIME_PRIORITY_RULE)

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        pnums.eq(
            required_number
        )
    ]

    add_check(
        checks,
        f'Registry Project {required_number} rows',
        1,
        len(
            matching_rows
        ),
        len(
            matching_rows
        )
        == 1,
    )

    add_check(
        checks,
        f'Project {required_number} frozen identity',
        required_project,
        (
            str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            if len(
                matching_rows
            )
            == 1
            else None
        ),
        (
            len(
                matching_rows
            )
            == 1
            and str(
                matching_rows.iloc[
                    0
                ][
                    project_col
                ]
            )
            == required_project
        ),
    )

add_check(
    checks,
    'Registry Project 18 rows',
    0,
    int(
        pnums.eq(
            18
        ).sum()
    ),
    int(
        pnums.eq(
            18
        ).sum()
    )
    == 0,
)

validation = pd.DataFrame(checks)
failed = validation[~validation['Pass']]
print('\nProject 18 Step 5B validation:')
display(validation)
if not failed.empty:
    print('\nFailed checks:'); display(failed)
    raise RuntimeError('PROJECT 18 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.')

# Freeze outputs.
OUT.mkdir(parents=True, exist_ok=True)
for path, frame in [
    (CURRENT_MANIFEST, current_manifest), (CONDITION_INVENTORY, inventory),
    (REVALIDATED_PROJECT_RUNS, project_runs), (REVALIDATED_BUILD_METRICS, build_metrics),
    (REVALIDATED_MODEL_FITS, model_fits), (REVALIDATED_CONDITION_AUDIT, condition_audit),
    (REVALIDATED_MEDIANS, training_medians), (NOISE_SUMMARY, noise_summary),
    (SEED_DELTAS, seed_deltas), (DELTA_SUMMARY, delta_summary), (VALIDATION_PATH, validation),
]:
    atomic_csv(path, frame)
output_paths = [CURRENT_MANIFEST, CONDITION_INVENTORY, REVALIDATED_PROJECT_RUNS,
                REVALIDATED_BUILD_METRICS, REVALIDATED_MODEL_FITS, REVALIDATED_CONDITION_AUDIT,
                REVALIDATED_MEDIANS, NOISE_SUMMARY, SEED_DELTAS, DELTA_SUMMARY, VALIDATION_PATH]
output_manifest = [{'Path': str(p), 'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)} for p in output_paths]
completed = datetime.now(timezone.utc).isoformat()
report = {
    'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
    'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
    'FrozenRawRootSHA256': EXPECTED_RAW_ROOT_SHA, 'IndependentRawRootSHA256': current_raw_sha,
    'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes, 'Conditions': len(inventory),
    'ExpectedScoredFailingBuilds': EXPECTED_SCORED_FAILING_BUILDS,
    'ExpectedEvaluationBuilds': EXPECTED_EVALUATION_BUILDS,
    'ExpectedEvaluationFailures': EXPECTED_EVALUATION_FAILURES,
    'MLFits': len(model_fits), 'RankingRows': int(inventory['RankingRows'].sum()),
    'BuildMetricRows': len(build_metrics), 'ProjectRunRows': len(project_runs),
    'ConditionAuditRows': len(condition_audit), 'TrainingMedianRows': len(training_medians),
    'NoiseTechniqueSummaryRows': len(noise_summary), 'SeedLevelNoiseDeltaRows': len(seed_deltas),
    'NoiseDeltaSummaryRows': len(delta_summary),
    'StandardDeviationDefinition': 'Sample SD across 30 seeds; pandas std, ddof=1',
    'RankingLevelBaselineInvarianceFailures': int(ranking_baseline_fail),
    'ProjectMetricBaselineInvarianceFailures': int(metric_baseline_fail),
    'ProjectMetricBaselineMaximumWithinMetricRange': float(metric_baseline_max_range),
    'RawHashingSeconds': float(hash_seconds), 'ConditionRevalidationSeconds': float(condition_seconds),
    'AggregationSeconds': float(agg_seconds), 'OutputManifest': output_manifest,
    'ValidationChecks': len(validation), 'FailedValidationChecks': len(failed),
    'RegistrySHA256': registry_sha_before,
    'RegistryModified': False,
    'Projects1To17Modified': False,
    'Project17RegistryIdentity': required_registered_identities[17],
    'Project17ConditionOutputsAccessed': False,
    'Project17ConditionOutputsModified': False,
    'ActiveReservations': EXPECTED_ACTIVE_RESERVATIONS,
    'RuntimePriorityRule': EXPECTED_RUNTIME_PRIORITY_RULE,
    'PriorProjectConditionOutputsAccessed': False,
    'PriorProjectWriteAttempted': False,
    'ModelsFitted': False,
    'ConditionsRerun': False,
}
atomic_json(REPORT_PATH, report)
checkpoint = {**report, 'CheckpointVersion': 1,
              'CheckpointType': 'PROJECT_18_RAW_REVALIDATION_AND_COMPACT_AGGREGATION',
              'RawResultsRevalidated': True, 'CompactAggregatesFrozen': True,
              'ReadyForFinalPackageAndRegistration': True}
atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
status = {'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
          'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
          'RawRootSHA256': current_raw_sha, 'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes,
          'Conditions': len(inventory), 'MLFits': len(model_fits), 'Checkpoint': str(CHECKPOINT_PATH),
          'CheckpointSHA256': checkpoint_sha,
          'ReadyForFinalPackageAndRegistration': True,
          'RegistryModified': False,
          'Projects1To17Modified': False,
          'Project17ConditionOutputsAccessed': False,
          'Project17ConditionOutputsModified': False,
          'ActiveReservations': EXPECTED_ACTIVE_RESERVATIONS,
          'RuntimePriorityRule': EXPECTED_RUNTIME_PRIORITY_RULE,
          'PriorProjectConditionOutputsAccessed': False}
atomic_json(STATUS_PATH, status)

# Readback and immutability.
if load_json(CHECKPOINT_PATH).get('Status') != STEP5B_STATUS or load_json(STATUS_PATH).get('Status') != STEP5B_STATUS:
    raise RuntimeError('Step 5B checkpoint/status readback failed.')
for item in output_manifest:
    p = Path(item['Path'])
    if not p.is_file() or p.stat().st_size != item['Bytes'] or sha256_file(p) != item['SHA256']:
        raise RuntimeError(f'Step 5B output readback failed: {p}')
registry_sha_after = sha256_file(REGISTRY)
if registry_sha_after != registry_sha_before:
    raise RuntimeError('Registry changed during Project 18 Step 5B.')
if sha256_file(STEP5A_CHECKPOINT) != EXPECTED_STEP5A_SHA:
    raise RuntimeError('Step 5A checkpoint changed during Step 5B.')

print('\nNoise-technique summary:')
display(noise_summary)
print('\nNoise-delta summary:')
display(delta_summary)
print('\n' + '=' * 136)
print('=== PROJECT 18 CELL 10 / STEP 5B RESULT ===')
print('=' * 136)

print('Project:', PROJECT_NAME)
print('Project slug:', PROJECT_SLUG)
print('Step 5A checkpoint SHA-256:', step5a_sha)
print('Frozen raw-root SHA-256:', EXPECTED_RAW_ROOT_SHA)
print('Independent current raw-root SHA-256:', current_raw_sha)

print('\nRaw-output revalidation:')
print('Conditions:', len(inventory), '/', EXPECTED_CONDITIONS)
print('Raw files:', len(current_manifest), '/', EXPECTED_RAW_FILES)
print('Raw bytes:', current_raw_bytes, '/', EXPECTED_RAW_BYTES)
print(
    'Missing / unexpected / size / SHA mismatches:',
    missing_raw,
    '/',
    unexpected_raw,
    '/',
    size_mismatch,
    '/',
    hash_mismatch,
)
print('Embedded output-manifest failures:', embedded_fail)

print('\nExperiment totals:')
print('ML fits:', len(model_fits), '/', EXPECTED_TOTAL_FIT_ROWS)
print(
    'Ranking rows:',
    int(
        inventory[
            'RankingRows'
        ].sum()
    ),
    '/',
    EXPECTED_TOTAL_RANKING_ROWS,
)
print('Build-metric rows:', len(build_metrics), '/', EXPECTED_TOTAL_BUILD_ROWS)
print('Project-run rows:', len(project_runs), '/', EXPECTED_TOTAL_PROJECT_ROWS)
print('Condition-audit rows:', len(condition_audit), '/', EXPECTED_TOTAL_AUDIT_ROWS)
print('Training-median rows:', len(training_medians), '/', EXPECTED_TOTAL_MEDIAN_ROWS)

print('\nAnalysis-ready aggregates:')
print('Noise-technique summary rows:', len(noise_summary))
print('Seed-level noise-delta rows:', len(seed_deltas))
print('Noise-delta summary rows:', len(delta_summary))
print('Sample SD calculated with ddof=1:', True)
print('Ranking-level baseline-invariance failures:', ranking_baseline_fail)
print('Project-metric baseline-invariance failures:', metric_baseline_fail)
print('Maximum within-metric baseline range:', metric_baseline_max_range)

print('\nImmutability and isolation:')
print('Completion registry unchanged:', registry_sha_after == registry_sha_before)
print('Registry Project 11 rows:', int(pnums.eq(11).sum()))
print('Registry Project 12 rows:', int(pnums.eq(12).sum()))
print('Registry Project 13 rows:', int(pnums.eq(13).sum()))
print('Registry Project 14 rows:', int(pnums.eq(14).sum()))
print('Registry Project 15 rows:', int(pnums.eq(15).sum()))
print('Registry Project 16 rows:', int(pnums.eq(16).sum()))
print('Registry Project 17 rows:', int(pnums.eq(17).sum()))
print('Registry Project 18 rows:', int(pnums.eq(18).sum()))
print('Project 11 identity:', required_registered_identities[11])
print('Project 12 identity:', required_registered_identities[12])
print('Project 13 identity:', required_registered_identities[13])
print('Project 14 identity:', required_registered_identities[14])
print('Project 15 identity:', required_registered_identities[15])
print('Project 16 identity:', required_registered_identities[16])
print('Project 17 identity:', required_registered_identities[17])
print('Active reservations:', EXPECTED_ACTIVE_RESERVATIONS)
print('Runtime-priority rule:', EXPECTED_RUNTIME_PRIORITY_RULE)
print('Projects 1–17 modified:', 0)
print('Project 17 condition outputs accessed:', False)
print('Project 17 condition outputs modified:', False)
print('Prior project condition outputs accessed:', False)
print('Prior project write attempted:', False)
print('Conditions rerun:', False)
print('Models fitted:', False)

print('\nRuntime:')
print('Raw hashing seconds:', round(hash_seconds, 2))
print('Condition revalidation seconds:', round(condition_seconds, 2))
print('Compact aggregation seconds:', round(agg_seconds, 2))

print('\nValidation:')
print('Checks:', len(validation))
print('Failed checks:', len(failed))

print('\nProject 18 Step 5B checkpoint:')
print(CHECKPOINT_PATH)
print('Checkpoint SHA-256:', checkpoint_sha)

print('\nSTATUS:', STEP5B_STATUS)
print('=' * 136)


=== PROJECT 18 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Condition revalidatio

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_18_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_18_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,3c2757656b27e8c125cec92124f1656f823cbc70a475ac...,3c2757656b27e8c125cec92124f1656f823cbc70a475ac...,True
2,Frozen raw-root SHA-256,3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb...,3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb...,True
3,Independent current raw-root SHA-256,3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb...,3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb...,True
4,Step 5A aggregate-manifest failures,0,0,True
...,...,...,...,...
67,Registry Project 16 rows,1,1,True
68,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
69,Registry Project 17 rows,1,1,True
70,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.569503,0.000000,0.569503,0.642331,0.000000,0.642331,0.325883,0.000000,0.325883,0.322289,0.000000,0.322289
1,0,LightGBM,30,30,0.670107,0.000000,0.670107,0.802702,0.000000,0.802702,0.808092,0.000000,0.808092,0.897815,0.000000,0.897815
2,0,NaiveBayes,30,30,0.566164,0.000000,0.566164,0.583029,0.000000,0.583029,0.793760,0.000000,0.793760,0.833858,0.000000,0.833858
3,0,QTF-Avg,30,30,0.603369,0.000000,0.603369,0.607914,0.000000,0.607914,0.394931,0.000000,0.394931,0.229846,0.000000,0.229846
4,0,Random,30,30,0.505163,0.076668,0.503097,0.518073,0.091891,0.514719,0.503930,0.071909,0.499734,0.508517,0.085611,0.501928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.400147,0.108343,0.360459,0.405277,0.141051,0.384439,0.420255,0.150875,0.405780,0.421218,0.243532,0.339584
59,50,QTF-Avg,30,30,0.603369,0.000000,0.603369,0.607914,0.000000,0.607914,0.394931,0.000000,0.394931,0.229846,0.000000,0.229846
60,50,Random,30,30,0.505163,0.076668,0.503097,0.518073,0.091891,0.514719,0.503930,0.071909,0.499734,0.508517,0.085611,0.501928
61,50,RandomForest,30,30,0.469783,0.090931,0.464683,0.465899,0.104947,0.499017,0.455389,0.097544,0.459056,0.450161,0.114150,0.484316



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,-0.166017,0.108343,-0.205705,-0.177751,0.141051,-0.198589,-0.373505,0.150875,-0.387980,-0.412639,0.243532,-0.494274
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.257183,0.102472,-0.251870,-0.260272,0.125164,-0.247141,-0.442326,0.099847,-0.439643,-0.471999,0.119975,-0.434192



=== PROJECT 18 CELL 10 / STEP 5B RESULT ===
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Step 5A checkpoint SHA-256: 3c2757656b27e8c125cec92124f1656f823cbc70a475ac92cecf075dd93f7f0b
Frozen raw-root SHA-256: 3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702
Independent current raw-root SHA-256: 3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 78874343 / 78874343
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0
Embedded output-manifest failures: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 4437720 / 4437720
Build-metric rows: 22680 / 22680
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Sample SD calculated with ddof=1: True
Ranking-level basel

In [36]:
# ==================================================================================================
# PROJECT 18 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   cantaloupe-project@cantaloupe
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_16.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 18 Step 5A checkpoint SHA-256:
#   3c2757656b27e8c125cec92124f1656f823cbc70a475ac92cecf075dd93f7f0b
# - Project 18 Step 5B checkpoint SHA-256:
#   da6faa3eff4f9ce9879f8f66d5a8bb13f9b2931754026e1d7b07dcd9f96a1b2d
# - Project 18 raw-root SHA-256:
#   3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702
# - Registry before registration:
#   exactly Projects 1–17, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 18 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 18
PROJECT_NAME = "cantaloupe-project@cantaloupe"
PROJECT_SLUG = "cantaloupe-project__cantaloupe"
PROJECT_SHORT = "CANTALOUPE"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_18_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_18_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_18_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "f804412b0c2ac005418762a7892d7aeb74b4c543877b29100c6e3ca5b96c3dcd"
STEP5A_SHA_EXPECTED = "3c2757656b27e8c125cec92124f1656f823cbc70a475ac92cecf075dd93f7f0b"
STEP5B_SHA_EXPECTED = "da6faa3eff4f9ce9879f8f66d5a8bb13f9b2931754026e1d7b07dcd9f96a1b2d"
SOURCE_ROOT_SHA = "d3de40f4a52f7967d82e3111d5a8ee94b4be1941c175d4f760fc53d4e3cdacb8"
RAW_ROOT_SHA = "3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 78874343, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 4437720, "BuildMetricRows": 22680, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 450,
    "TrainingBuilds": 337, "EvaluationBuilds": 113, "RawRows": 67108,
    "RawTrainingRows": 45140, "RawEvaluationRows": 21968,
    "RawTrainingFailures": 144, "RawEvaluationFailures": 31, "ModelRows": 10580,
    "ModelTrainingRows": 8232, "ModelEvaluationRows": 2348,
    "ModelTrainingFailures": 142, "ModelEvaluationFailures": 31,
    "ModelFailingEvaluationBuilds": 12, "Predictors": 151, "RECFeatures": 19,
}
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_18_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_18.csv"

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)
STEP5A_CP = NOTES / "project_18_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_18_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_18_selection_checkpoint.json",
    NOTES / "project_18_rec_reconstruction_checkpoint.json",
    NOTES / "project_18_noise_plan_checkpoint.json",
    NOTES / "project_18_runtime_contract_checkpoint.json",
    NOTES / "project_18_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 18 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "Project17ConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to Project 17 condition outputs."
    )

if bool(
    step5b.get(
        "Project17ConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of Project 17 condition outputs."
    )

step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must contain exactly completed Projects 1–17, with Project 18 absent.
registry_sha_before = sha(REGISTRY)

if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(
        "Registry SHA differs before Project 18 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna("")

pn_col = resolve(reg_before.columns, "ProjectNumber")
project_col = resolve(reg_before.columns, "Project")
status_col = resolve(reg_before.columns, "Status")

pnums = pd.to_numeric(
    reg_before[pn_col],
    errors="raise",
).astype(int)

if len(reg_before) != 17 or sorted(pnums.tolist()) != list(range(1, 18)):
    raise RuntimeError("Registry must contain exactly Projects 1–17.")

if not reg_before[status_col].eq(COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–17 are not all COMPLETE_AND_FROZEN.")

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = reg_before.loc[pnums.eq(required_number)]
    if len(matching_rows) != 1 or matching_rows.iloc[0][project_col] != required_project:
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if pnums.eq(PROJECT_NUMBER).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Project 18 is already present in the completion registry.")

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-18 registry backup does not match the live registry."
    )

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

# Use the frozen Step 5B completion timestamp so the package is deterministic
# across safe reruns.
created_at = str(
    step5b.get(
        "CompletedAtUTC",
        ""
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project18_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 18 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
        "Project17ConditionOutputsAccessed": False,
        "Project17ConditionOutputsModified": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 18 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 18 final package already exists and "
                "was not modified."
            )

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems. A direct
        # os.replace(tmp_root, FINAL_ROOT) therefore raises EXDEV. Publish in
        # two stages:
        #   1. copy the completed local package to a sibling staging directory
        #      on Google Drive;
        #   2. verify every staged payload file;
        #   3. rename the staging directory to FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root,
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + comparison.loc[
                    (
                        comparison["_merge"].ne(
                            "both"
                        )
                        | comparison[
                            "Bytes_candidate"
                        ].ne(
                            comparison[
                                "Bytes_staged"
                            ]
                        )
                        | comparison[
                            "SHA256_candidate"
                        ].ne(
                            comparison[
                                "SHA256_staged"
                            ]
                        )
                    )
                ].head(
                    20
                ).to_string(
                    index=False
                )
            )

        # This rename is within the Google Drive filesystem, so it does not
        # cross a device boundary.
        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 18 package failed readback validation.")

# Build a complete Project 18 registry row.
#
# The registry has evolved across Projects 1–17. Some columns are protocol
# descriptors, some are project-specific counts, and some are paths to frozen
# audit artefacts. V2 deliberately stopped because it did not map every
# variable column. V3 handles the complete observed schema explicitly.
#
# For protocol fields whose textual formatting has varied historically
# (Seeds, NoiseLevels, Techniques, DoNotRerun), use the exact frozen
# Project 17 representation. Project 18 uses the same protocol.
project_17_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        17
    )
]

if len(
    project_17_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 17 registry template row."
    )

project_17_template = project_17_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_17_template[
                registry_column
            ]
        ).strip()


protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}


for protocol_key, fallback_value in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        ""
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds": protocol_template_values["seeds"],
    "noiselevels": protocol_template_values["noiselevels"],
    "techniques": protocol_template_values["techniques"],
    "evaluationrows": COUNTS["ModelEvaluationRows"],
    "evaluationfailures": COUNTS["ModelEvaluationFailures"],
    "finaldirectory": str(FINAL_ROOT),
    "finalauditreport": str(REPORT_PATH),
    "donotrerun": protocol_template_values["donotrerun"],
    "freezerecord": str(CHECKPOINT_PATH),
    "rawresultsmanifest": str(STEP5B_RAW_MANIFEST_PATH),
    "finalpackagemanifest": str(MANIFEST_PATH),
    "rawresultsrootsha256": RAW_ROOT_SHA,
    "finalauditstatus": STEP5C_STATUS,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )


new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; no registry write "
        "was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [reg_before, pd.DataFrame([new_row])],
    ignore_index=True,
)
reg_candidate[pn_col] = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int).astype(str)
candidate_nums = pd.to_numeric(
    reg_candidate[pn_col],
    errors="raise",
).astype(int)

project18_candidate = reg_candidate.loc[candidate_nums.eq(18)]

if len(reg_candidate) != 18 or sorted(candidate_nums.tolist()) != list(range(1, 19)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–18.")

if (
    not reg_candidate[status_col].eq(COMPLETE_STATUS).all()
    or len(project18_candidate) != 1
    or project18_candidate.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError("Candidate Project 18 registry row failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 17, len(reg_before), len(reg_before) == 17)
check(rows, "Registry rows candidate", 18, len(reg_candidate), len(reg_candidate) == 18)
check(rows, "Candidate Project 18 rows", 1, len(project18_candidate), len(project18_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}


registry_field_validation_failures = 0

for expected_column_name, expected_value in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project18_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )


check(
    rows,
    "Explicit Project 18 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(rows)
print("\nProject 18 Step 5C pre-write validation:"); display(pre)
print("\nProject 18 registry row candidate:"); display(project18_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 18 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project18_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if (
    len(tmp_read) != 18
    or sorted(tmp_nums.tolist()) != list(range(1, 19))
    or not tmp_read[status_col].eq(COMPLETE_STATUS).all()
    or int(tmp_nums.eq(18).sum()) != 1
):
    tmp_reg.unlink(missing_ok=True)
    raise RuntimeError("Temporary Project 18 registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
project12_after = reg_after.loc[after_nums.eq(12)]
project13_after = reg_after.loc[after_nums.eq(13)]
project14_after = reg_after.loc[after_nums.eq(14)]
project15_after = reg_after.loc[after_nums.eq(15)]
project16_after = reg_after.loc[after_nums.eq(16)]
project17_after = reg_after.loc[after_nums.eq(17)]
project18_after = reg_after.loc[after_nums.eq(18)]
registry_sha_after = sha(REGISTRY)

if (
    len(reg_after) != 18
    or sorted(after_nums.tolist()) != list(range(1, 19))
    or not reg_after[status_col].eq(COMPLETE_STATUS).all()
    or len(project11_after) != 1
    or project11_after.iloc[0][project_col] != "apache@shardingsphere"
    or len(project12_after) != 1
    or project12_after.iloc[0][project_col] != "zolyfarkas@spf4j"
    or len(project13_after) != 1
    or project13_after.iloc[0][project_col] != "jcabi@jcabi-github"
    or len(project14_after) != 1
    or project14_after.iloc[0][project_col] != "JMRI@JMRI"
    or len(project15_after) != 1
    or project15_after.iloc[0][project_col] != "eclipse@steady"
    or len(project16_after) != 1
    or project16_after.iloc[0][project_col] != "apache@rocketmq"
    or len(project17_after) != 1
    or project17_after.iloc[0][project_col] != "yamcs@Yamcs"
    or len(project18_after) != 1
    or project18_after.iloc[0][project_col] != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed Project 18 post-write validation. "
        f"Backup: {BACKUP_PATH}"
    )

check(rows, "Registry rows after", 18, len(reg_after), len(reg_after) == 18)
check(rows, "COMPLETE_AND_FROZEN projects after", 18, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), int(reg_after[status_col].eq(COMPLETE_STATUS).sum()) == 18)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry Project 12 rows after", 1, len(project12_after), len(project12_after) == 1)
check(rows, "Registry Project 13 rows after", 1, len(project13_after), len(project13_after) == 1)
check(rows, "Registry Project 14 rows after", 1, len(project14_after), len(project14_after) == 1)
check(rows, "Registry Project 15 rows after", 1, len(project15_after), len(project15_after) == 1)
check(rows, "Registry Project 16 rows after", 1, len(project16_after), len(project16_after) == 1)
check(rows, "Registry Project 17 rows after", 1, len(project17_after), len(project17_after) == 1)
check(rows, "Registry Project 18 rows after", 1, len(project18_after), len(project18_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed)
    raise RuntimeError("PROJECT 18 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after),
    "Project12RegistryRowsAfter": len(project12_after),
    "Project13RegistryRowsAfter": len(project13_after),
    "Project14RegistryRowsAfter": len(project14_after),
    "Project15RegistryRowsAfter": len(project15_after),
    "Project16RegistryRowsAfter": len(project16_after),
    "Project17RegistryRowsAfter": len(project17_after),
    "Project18RegistryRowsAfter": len(project18_after),
    "Project17ConditionOutputsAccessed": False,
    "Project17ConditionOutputsModified": False,
    "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectWriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "CheckpointType": "PROJECT_18_FINAL_PACKAGE_AND_REGISTRY", "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha,
    "ProjectCompleteAndFrozen": True,
    "Project17ConditionOutputsAccessed": False,
    "Project17ConditionOutputsModified": False,
    "PriorProjectConditionOutputsAccessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 18 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Project 11 identity:", required_registered_identities[11])
print("Project 12 identity:", required_registered_identities[12])
print("Project 13 identity:", required_registered_identities[13])
print("Project 14 identity:", required_registered_identities[14])
print("Project 15 identity:", required_registered_identities[15])
print("Project 16 identity:", required_registered_identities[16])
print("Project 17 identity:", required_registered_identities[17])
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Project 12 registry rows:", len(project12_after))
print("Project 13 registry rows:", len(project13_after))
print("Project 14 registry rows:", len(project14_after))
print("Project 15 registry rows:", len(project15_after))
print("Project 16 registry rows:", len(project16_after))
print("Project 17 registry rows:", len(project17_after))
print("Project 18 registry rows:", len(project18_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 17 condition outputs accessed:", False)
print("Project 17 condition outputs modified:", False)
print("Prior project condition outputs accessed:", False)
print("Prior project write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 18 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print("Explicit registry-schema fields validated:", len(required_registry_field_expectations))
print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)


=== PROJECT 18 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project 18 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,3c2757656b27e8c125cec92124f1656f823cbc70a475ac...,3c2757656b27e8c125cec92124f1656f823cbc70a475ac...,True
1,Step 5B checkpoint SHA-256,da6faa3eff4f9ce9879f8f66d5a8bb13f9b2931754026e...,da6faa3eff4f9ce9879f8f66d5a8bb13f9b2931754026e...,True
2,Step 5A manifest failures,0,0,True
3,Step 5B manifest failures,0,0,True
4,Package missing files,0,0,True
5,Package unexpected files,0,0,True
6,Package size mismatches,0,0,True
7,Package SHA-256 mismatches,0,0,True
8,Registry rows before,17,17,True
9,Registry rows candidate,18,18,True



Project 18 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
17,18,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,COMPLETE_AND_FROZEN,270,30,9,7,113,2348,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb...,173be5fd2c47cd714eb56d1bee7a7e0d84507e6c9d229a...,1080,5358150.0,PASS_PROJECT_18_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 18 CELL 11 / STEP 5C RESULT ===
Project number: 18
Project: cantaloupe-project@cantaloupe
Project slug: cantaloupe-project__cantaloupe
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 78874343
Raw root SHA-256: 3bfa16b92a374bfd42f6a2d65859e0762b86bfb491c2bb2034023eb715265702

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/cantaloupe-project__cantaloupe
Package files: 32
Package bytes: 15142811
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: 173be5fd2c47cd714eb56d1bee7a7e0d84507e6c9d229a1dff5c3574c25e9d69

Completion registry:
Registry rows: 18
COMPLETE_AND_FROZ